# 2027학년도 가천대학교 입학 안내 Agentic RAG 챗봇

**중간평가 · 생성형 AI 활용**

- **대상 문서**: `dataset.zip` 내 5개 문서 (HWP 1개, PDF 4개)
- **파이프라인**: HWP/PDF 파싱 → 청킹 → 임베딩/벡터DB → 하이브리드 검색 → Agentic RAG 답변
- **제약**: 답변 근거는 제공 문서로 한정 (웹 검색 금지), 문항당 5초 이내 응답 목표

---
## 0. 환경 설정

In [2]:
# 필요 패키지 설치 (최초 1회, 약 2~3분)
!pip install -qU langchain langchain-openai langchain-community "langchain-chroma>=0.1.2" \
    langchain-hwp-hwpx-loader rank_bm25 kiwipiepy pypdf pdfplumber
print("설치 완료")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
설치 완료


In [8]:
import os

# API 키는 코드에 직접 쓰지 않고, Colab Secrets(왼쪽 🔑) 또는 환경변수에서 읽습니다.
def _load_key(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

os.environ["OPENAI_API_KEY"] = _load_key("OPENAI_API_KEY") or ""
print("OPENAI_API_KEY:", "설정됨 ✅" if os.environ["OPENAI_API_KEY"] else "없음 ❌  → 왼쪽 🔑(Secrets)에 OPENAI_API_KEY 추가 후 '노트북 액세스' 허용")

OPENAI_API_KEY: 설정됨 ✅


In [3]:
# [개발용] 런타임에 문서가 있는지 확인
!ls -la /content | head -30

total 16
drwxr-xr-x 1 root root 4096 Oct 21 13:23 .
drwxr-xr-x 1 root root 4096 Oct 25 05:31 ..
drwxr-xr-x 4 root root 4096 Oct 21 13:23 .config
drwxr-xr-x 1 root root 4096 Oct 21 13:23 sample_data


In [7]:
# 데이터 준비: dataset.zip 압축 해제 후 5개 문서 존재 확인
import glob, zipfile

REQUIRED = [
    "01_2027학년도_재외국민_모집요강.hwp",
    "02_2027학년도_지역의사선발전형_안내.pdf",
    "03_2027학년도_수시_모집요강.pdf",
    "04_2027학년도_정시_모집요강.pdf",
    "05_2027학년도_수시_면접고사_일정.pdf",
]

for z in glob.glob("/content/**/dataset.zip", recursive=True):
    with zipfile.ZipFile(z) as f:
        f.extractall("/content/dataset")
    print("압축 해제:", z)

found = {}
for n in REQUIRED:
    hits = glob.glob(f"/content/**/{n}", recursive=True)
    found[n] = hits[0] if hits else None
    print(("✅" if hits else "❌"), n)

print("\n누락 문서 수:", sum(1 for v in found.values() if not v))

압축 해제: /content/dataset.zip
✅ 01_2027학년도_재외국민_모집요강.hwp
✅ 02_2027학년도_지역의사선발전형_안내.pdf
✅ 03_2027학년도_수시_모집요강.pdf
✅ 04_2027학년도_정시_모집요강.pdf
✅ 05_2027학년도_수시_면접고사_일정.pdf

누락 문서 수: 0


---
## 1. 문서 파싱 (HWP / PDF)

In [9]:
import inspect
from langchain_hwp_hwpx import HwpHwpxLoader

print(inspect.signature(HwpHwpxLoader.__init__))
print("---")
print((HwpHwpxLoader.__doc__ or "")[:1200])

(self, file_path: 'str | Path', mode: 'Mode' = 'single', extract_options: 'Any | None' = None, include_tables: 'bool' = True, include_notes: 'bool' = True, include_memos: 'bool' = True, include_hyperlinks: 'bool' = True, include_images: 'bool' = False, images_dir: 'str | Path | None' = None, image_document_mode: 'ImageDocumentMode' = 'metadata_only', on_encrypted: 'StatusPolicy' = 'raise', on_invalid: 'StatusPolicy' = 'raise', on_error: 'ErrorPolicy' = 'raise', extra_metadata: 'dict[str, Any] | None' = None, include_extracted_at: 'bool' = True, on_unsupported: 'StatusPolicy' = 'raise', distribution_backend: "Literal['pyhwp'] | None" = None) -> 'None'
---
Load a single HWP/HWPX file into LangChain Document objects.


In [10]:
from pathlib import Path

DATA = Path("/content/dataset")
hwp_path = DATA / "01_2027학년도_재외국민_모집요강.hwp"

d_single = HwpHwpxLoader(str(hwp_path), mode="single").load()
print("문서 수:", len(d_single), "| 문자 수:", len(d_single[0].page_content))
print("메타데이터:", d_single[0].metadata)
print("=" * 80)
print(d_single[0].page_content[:2000])

HwpHwpxLoaderError: Document skipped because it is unsupported: /content/dataset/01_2027학년도_재외국민_모집요강.hwp. Distribution HWP: install [distribution] and set distribution_backend='pyhwp'

In [11]:
# 배포용(distribution) HWP는 pyhwp 백엔드가 필요합니다
!pip install -qU "langchain-hwp-hwpx-loader[distribution]"
import pyhwp, importlib
print("pyhwp", getattr(pyhwp, "__version__", "?"))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


ModuleNotFoundError: No module named 'pyhwp'

In [12]:
try:
    import hwp5
    print("hwp5 OK", hwp5.__file__)
except Exception as e:
    print("hwp5 실패:", e)

from langchain_hwp_hwpx import HwpHwpxLoader
try:
    d = HwpHwpxLoader(str(hwp_path), mode="single", distribution_backend="pyhwp").load()
    print("문서 수:", len(d), "| 문자 수:", len(d[0].page_content))
    print(d[0].page_content[:1500])
except Exception as e:
    print("로드 실패:", type(e).__name__, e)

hwp5 OK /usr/local/lib/python3.13/dist-packages/hwp5/__init__.py
문서 수: 1 | 문자 수: 78954
Gachon University
2027학년도 재외국민 모집요강

## Tables
| [rowspan=1, colspan=3] | [merged:R1C1] | [merged:R1C1] |
| --- | --- | --- |
| 2027학년도 가천대학교 재외국민 모집요강 [rowspan=1, colspan=3] | [merged:R2C1] | [merged:R2C1] |
| [rowspan=4, colspan=1] |  | [rowspan=4, colspan=1] |
| [merged:R3C1] | [IMAGE] | [merged:R3C3] |
| [merged:R3C1] |  | [merged:R3C3] |
| [merged:R3C1] | [IMAGE] | [merged:R3C3] |

| 모집인원 [rowspan=1, colspan=6] | [merged:R1C1] | [merged:R1C1] | [merged:R1C1] | [merged:R1C1] | [merged:R1C1] |
| --- | --- | --- | --- | --- | --- |
| [rowspan=1, colspan=6] | [merged:R2C1] | [merged:R2C1] | [merged:R2C1] | [merged:R2C1] | [merged:R2C1] |
| ※ 본 내용은 변동될 수 있으며 반드시 최종 확정되는 모집요강(본 대학 입학처 홈페이지 모집요강)을 확인하시기 바랍니다. [rowspan=1, colspan=6] | [merged:R3C1] | [merged:R3C1] | [merged:R3C1] | [merged:R3C1] | [merged:R3C1] |
| [rowspan=1, colspan=6] | [merged:R4C1] | [merged:R4C1] | [merged:R4C1] | [merged:R4C1] | 

In [13]:
from collections import Counter
d_el = HwpHwpxLoader(str(hwp_path), mode="elements", distribution_backend="pyhwp").load()
print("elements:", len(d_el))
print(Counter(x.metadata.get("element_type") for x in d_el))

# 표 하나의 셀 구조 확인
for x in d_el:
    if x.metadata.get("element_type") == "table":
        print("--- page_content ---")
        print(x.page_content[:800])
        print("--- cells ---")
        print(x.metadata.get("cells", "")[:1500])
        break

elements: 19
Counter({'table': 18, 'body': 1})
--- page_content ---
| [rowspan=1, colspan=3] | [merged:R1C1] | [merged:R1C1] |
| --- | --- | --- |
| 2027학년도 가천대학교 재외국민 모집요강 [rowspan=1, colspan=3] | [merged:R2C1] | [merged:R2C1] |
| [rowspan=4, colspan=1] |  | [rowspan=4, colspan=1] |
| [merged:R3C1] | [IMAGE] | [merged:R3C3] |
| [merged:R3C1] |  | [merged:R3C3] |
| [merged:R3C1] | [IMAGE] | [merged:R3C3] |
--- cells ---
[{"row": 0, "col": 0, "rowspan": 1, "colspan": 3, "text": ""}, {"row": 1, "col": 0, "rowspan": 1, "colspan": 3, "text": "2027학년도 가천대학교 재외국민 모집요강"}, {"row": 2, "col": 0, "rowspan": 4, "colspan": 1, "text": ""}, {"row": 2, "col": 1, "rowspan": 1, "colspan": 1, "text": ""}, {"row": 2, "col": 2, "rowspan": 4, "colspan": 1, "text": ""}, {"row": 3, "col": 1, "rowspan": 1, "colspan": 1, "text": "[IMAGE]"}, {"row": 4, "col": 1, "rowspan": 1, "colspan": 1, "text": ""}, {"row": 5, "col": 1, "rowspan": 1, "colspan": 1, "text": "[IMAGE]"}]


In [14]:
body = [x for x in d_el if x.metadata.get("element_type") == "body"][0]
print("BODY 길이:", len(body.page_content))
print(body.page_content[:1500])
print("\n" + "=" * 80 + "\n")
tbl = [x for x in d_el if x.metadata.get("element_type") == "table"][5]
print("TABLE", tbl.metadata.get("element_index"), "rows/cols:", tbl.metadata.get("row_count"), tbl.metadata.get("col_count"))
print(tbl.page_content[:1200])

BODY 길이: 35
Gachon University
2027학년도 재외국민 모집요강


TABLE 6 rows/cols: 14 6
| 합격자 발표 및 등록 [rowspan=1, colspan=6] | [merged:R1C1] | [merged:R1C1] | [merged:R1C1] | [merged:R1C1] | [merged:R1C1] |
| --- | --- | --- | --- | --- | --- |
| [rowspan=1, colspan=6] | [merged:R2C1] | [merged:R2C1] | [merged:R2C1] | [merged:R2C1] | [merged:R2C1] |
| 합격자 발표 |  | 합격자 발표 | 충원합격자 발표 | 발표방법 |  |
| [rowspan=4, colspan=1] |  | 2026.08.13.(목) [rowspan=2, colspan=1] | 2026.12.23(수) ~ 29(화) [rowspan=2, colspan=1] | 본 대학 입학처 홈페이지 [rowspan=2, colspan=1] |  |
| [merged:R4C1] |  | [merged:R4C3] | [merged:R4C4] | [merged:R4C5] |  |
| [merged:R4C1] |  |  |  |  |  |
| [merged:R4C1] | 합격자 및 충원합격자는 반드시 지정된 기간 내에 등록하여야 합니다(미등록 시 불합격 처리함). 합격자 및 충원합격자 중 본교에 등록 후 입학을 포기하고자 하는 사람은 반드시 정해진 기간 내에 본 대학 입학처 홈페이지를 통한 등록포기 및 환불전차에 의거하여 등록 취소를 하여야 합니다. 합격자 명단 및 유의사항 미확인으로 인한 모든 불이익은 지원자에게 있으므로 반드시 지원자 본인이 직접 확인하여야 합니다. 충원합격자 통지기간 [2026.12.23(수) ~ 29(화) 18:00] 중 연락 두절로 합격 통지가 불가능할 경우 충원 대상에서 제외됩니다. 또한 어떠한 사유로도 이의를 제기할 수 없으며, 본 

In [15]:
!hwp5txt --loglevel=ERROR --output=/content/hwp5txt.txt "/content/dataset/01_2027학년도_재외국민_모집요강.hwp"
txt = open("/content/hwp5txt.txt", encoding="utf-8").read()
print("길이:", len(txt))
print(txt[:1200])

/usr/local/lib/python3.13/dist-packages/hwp5/importhelper.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
길이: 269
















































Gachon University
















































2027학년도 재외국민 모집요강

<표>


<표>














<표>




<표>


<표>


<표>


<표>


<표>


<표>


<표>


<표>











<표>


<표>




<표>


<표>





<표>



<표>


<표>




In [16]:
import json

def render_table(cells_json):
    """HWP 표의 cells JSON을 rowspan/colspan 전개 후 markdown으로 변환"""
    cells = json.loads(cells_json)
    nrows = max(c["row"] + c["rowspan"] for c in cells)
    ncols = max(c["col"] + c["colspan"] for c in cells)
    grid = [["" for _ in range(ncols)] for _ in range(nrows)]

    for c in cells:
        text = " ".join((c.get("text") or "").split()).replace("|", "/")
        if text == "[IMAGE]":
            text = ""
        for r in range(c["row"], min(c["row"] + c["rowspan"], nrows)):
            for col in range(c["col"], min(c["col"] + c["colspan"], ncols)):
                grid[r][col] = text

    lines, prev = [], None
    for row in grid:
        if row == prev:      # 완전히 동일한 반복행 제거
            continue
        prev = row
        if not any(row):
            continue
        lines.append("| " + " | ".join(row) + " |")
    if not lines:
        return ""
    return lines[0] + "\n|" + "---|" * ncols + "\n" + "\n".join(lines[1:])


tables = [x for x in d_el if x.metadata.get("element_type") == "table"]
for idx in (1, 6):
    md = render_table(tables[idx].metadata["cells"])
    print(f"=== TABLE {idx} ({tables[idx].metadata['row_count']}x{tables[idx].metadata['col_count']}) ===")
    print("\n".join(md.splitlines()[:14]))
    print()

=== TABLE 1 (71x6) ===
| 모집인원 | 모집인원 | 모집인원 | 모집인원 | 모집인원 | 모집인원 |
|---|---|---|---|---|---|
| ※ 본 내용은 변동될 수 있으며 반드시 최종 확정되는 모집요강(본 대학 입학처 홈페이지 모집요강)을 확인하시기 바랍니다. | ※ 본 내용은 변동될 수 있으며 반드시 최종 확정되는 모집요강(본 대학 입학처 홈페이지 모집요강)을 확인하시기 바랍니다. | ※ 본 내용은 변동될 수 있으며 반드시 최종 확정되는 모집요강(본 대학 입학처 홈페이지 모집요강)을 확인하시기 바랍니다. | ※ 본 내용은 변동될 수 있으며 반드시 최종 확정되는 모집요강(본 대학 입학처 홈페이지 모집요강)을 확인하시기 바랍니다. | ※ 본 내용은 변동될 수 있으며 반드시 최종 확정되는 모집요강(본 대학 입학처 홈페이지 모집요강)을 확인하시기 바랍니다. | ※ 본 내용은 변동될 수 있으며 반드시 최종 확정되는 모집요강(본 대학 입학처 홈페이지 모집요강)을 확인하시기 바랍니다. |
| 계열 | 계열 | 모집단위 | 모집단위 | 최대모집 가능인원 | 모집인원 |
| 인문 | 인문 | 경영학과 | 경영학과 | 15 | 16 |
| 인문 | 인문 | 회계세무학과 | 회계세무학과 | 5 | 16 |
| 인문 | 인문 | 관광경영학과 | 관광경영학과 | 4 | 16 |
| 인문 | 인문 | 의료산업경영학과 | 의료산업경영학과 | 4 | 16 |
| 자연 | 자연 | 금융·빅데이터학부 | 금융수학전공 | 10 | 16 |
| 자연 | 자연 | 금융·빅데이터학부 | 빅데이터경영전공 | 10 | 16 |
| 인문 | 인문 | 미디어커뮤니케이션학과 | 미디어커뮤니케이션학과 | 3 | 16 |
| 인문 | 인문 | 경제학과 | 경제학과 | 5 | 16 |
| 인문 | 인문 | 응용통계학과 | 응용통계학과 | 4 | 16 |
| 인문 | 인문 | 사회복지학과 | 사회복지학과 | 4 | 16 |

=== TABLE 6 (24x3) ===
| 전형방법 | 

In [19]:
import json, re
from pathlib import Path
from langchain_core.documents import Document
from langchain_hwp_hwpx import HwpHwpxLoader
from pypdf import PdfReader

DATA = Path("/content/dataset")


def render_hwp_table(cells_json):
    """HWP 표를 rowspan/colspan 전개 후 markdown으로 변환 (긴 병합셀 중복 방지)"""
    cells = json.loads(cells_json)
    nrows = max(c["row"] + c["rowspan"] for c in cells)
    ncols = max(c["col"] + c["colspan"] for c in cells)
    grid = [["" for _ in range(ncols)] for _ in range(nrows)]

    for c in cells:
        text = " ".join((c.get("text") or "").split()).replace("|", "/")
        if text == "[IMAGE]":
            text = ""
        span = c["rowspan"] * c["colspan"]
        for r in range(c["row"], min(c["row"] + c["rowspan"], nrows)):
            for col in range(c["col"], min(c["col"] + c["colspan"], ncols)):
                # 짧은 값(계열/구분)만 병합 전개, 긴 서술형은 원점에만
                if span == 1 or len(text) <= 40 or (r, col) == (c["row"], c["col"]):
                    grid[r][col] = text

    lines, prev = [], None
    for row in grid:
        if row == prev or not any(row):
            continue
        prev = row
        lines.append("| " + " | ".join(row) + " |")
    if not lines:
        return ""
    return lines[0] + "\n|" + "---|" * ncols + "\n" + "\n".join(lines[1:])


def parse_hwp(path: Path):
    els = HwpHwpxLoader(str(path), mode="elements", distribution_backend="pyhwp").load()
    out = []
    for e in els:
        if e.metadata.get("element_type") == "body" and e.page_content.strip():
            out.append(Document(page_content=e.page_content.strip(),
                                metadata={"source": path.name, "page": 1, "kind": "text"}))
        elif e.metadata.get("element_type") == "table":
            md = render_hwp_table(e.metadata["cells"])
            if md:
                out.append(Document(page_content=md,
                                    metadata={"source": path.name, "page": int(e.metadata.get("element_index", 0)) + 1, "kind": "table"}))
    return out


def parse_pdf(path: Path):
    reader = PdfReader(str(path))
    out = []
    for i, p in enumerate(reader.pages, start=1):
        t = (p.extract_text() or "").strip()
        if t:
            out.append(Document(page_content=t, metadata={"source": path.name, "page": i, "kind": "text"}))
    return out


raw_docs = []
for p in sorted(DATA.iterdir()):
    if p.suffix.lower() == ".hwp":
        raw_docs += parse_hwp(p)
    elif p.suffix.lower() == ".pdf":
        raw_docs += parse_pdf(p)

print(f"파싱 단위: {len(raw_docs)}개")
for name in sorted({d.metadata['source'] for d in raw_docs}):
    part = [d for d in raw_docs if d.metadata['source'] == name]
    print(f"  {name}: {len(part)}개, {sum(len(d.page_content) for d in part):,}자")

파싱 단위: 191개
  01_2027학년도_재외국민_모집요강.hwp: 19개, 39,889자
  02_2027학년도_지역의사선발전형_안내.pdf: 20개, 22,649자
  03_2027학년도_수시_모집요강.pdf: 104개, 95,095자
  04_2027학년도_정시_모집요강.pdf: 44개, 31,062자
  05_2027학년도_수시_면접고사_일정.pdf: 4개, 1,582자


In [23]:
import re


def normalize(text: str) -> str:
    """PDF 추출 텍스트의 탭/중복 공백과 '2027 . 01 . 04 .' 같은 날짜 붕괴를 정리"""
    out = []
    for ln in text.split("\n"):
        ln = re.sub(r"[ \t]+", " ", ln).strip()
        ln = re.sub(r"(?<=\d)\s*\.\s*(?=\d)", ".", ln)
        if ln:
            out.append(ln)
    return "\n".join(out)


raw_docs = [Document(page_content=normalize(d.page_content), metadata=d.metadata) for d in raw_docs]
print("정규화 완료:", len(raw_docs))
print(raw_docs[40].page_content[:300])

정규화 완료: 191
가천의 바람은 당신의 꿈을 이루게 합니다.
미래는 스스로 움직이지 않습니다.
그것을 움직이는 힘은 언제나 지성(知性)과 의지, 그리고 도전의 에너지입니다.
가천대학교는 변화의 흐름 속에서 새로운 지식을 발견하고,
그것을 사회로 확장하며, 사람과 기술, 배움과 혁신을 하나로 잇는 대학입니다.
인공지능과 의학, 반도체와 창업, 가천의 지성은 시대의 경계를 넘나들며
미래를 설계하고 있습니다.


In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", "다. ", " ", ""],
)


def _hard_split(text: str, limit: int):
    """한 줄이 너무 길 때 공백 기준으로 강제 분할"""
    out, cur = [], ""
    for w in text.split(" "):
        if len(cur) + len(w) + 1 > limit and cur:
            out.append(cur)
            cur = ""
        cur += (" " if cur else "") + w
    if cur:
        out.append(cur)
    return out


def split_table_md(md: str, max_chars: int = 1400):
    lines = md.splitlines()
    head, body = lines[:2], lines[2:]
    chunks, cur, size = [], [], 0
    for ln in body:
        for piece in (_hard_split(ln, max_chars) if len(ln) > max_chars else [ln]):
            if size + len(piece) > max_chars and cur:
                chunks.append("\n".join(head + cur))
                cur, size = [], 0
            cur.append(piece)
            size += len(piece)
    if cur:
        chunks.append("\n".join(head + cur))
    return chunks


chunks = []
for d in raw_docs:
    prefix = f"[{d.metadata['source']} p.{d.metadata['page']}]\n"
    pieces = split_table_md(d.page_content) if d.metadata["kind"] == "table" \
        else [c.page_content for c in splitter.split_documents([d])]
    for i, piece in enumerate(pieces):
        chunks.append(Document(page_content=prefix + piece, metadata={**d.metadata, "part": i}))

lengths = [len(c.page_content) for c in chunks]
print(f"청크 수: {len(chunks):,} | 평균 {sum(lengths)/len(lengths):.0f}자 | 최대 {max(lengths):,}자")
print(f"800자 초과: {sum(1 for l in lengths if l > 1200)}개")

청크 수: 513 | 평균 573자 | 최대 2,580자
800자 초과: 41개


---
## 2. 임베딩 · 벡터DB · 하이브리드 검색

In [47]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

EMBED_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# 재실행 시 중복 적재를 막기 위해 컬렉션을 새로 만듭니다
import time
COLLECTION = f"gachon_{int(time.time())}"

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION,
    collection_metadata={"hnsw:space": "cosine"},
)
vector_retriever = vectordb.as_retriever(search_kwargs={"k": 8})
print("벡터DB 구축 완료:", vectordb._collection.count(), "개")

벡터DB 구축 완료: 513 개


In [28]:
from openai import OpenAI

client = OpenAI()
ids = sorted(m.id for m in client.models.list().data)
for k in ["gpt-5", "gpt-4.1", "gpt-4o", "o4", "o3"]:
    hits = [i for i in ids if i.startswith(k)]
    if hits:
        print(k, "→", ", ".join(hits[:12]))

gpt-5 → gpt-5, gpt-5-2025-08-07, gpt-5-chat-latest, gpt-5-codex, gpt-5-mini, gpt-5-mini-2025-08-07, gpt-5-nano, gpt-5-nano-2025-08-07, gpt-5-pro, gpt-5-pro-2025-10-06, gpt-5-search-api, gpt-5-search-api-2025-10-14
gpt-4.1 → gpt-4.1, gpt-4.1-2025-04-14, gpt-4.1-mini, gpt-4.1-mini-2025-04-14, gpt-4.1-nano, gpt-4.1-nano-2025-04-14
gpt-4o → gpt-4o, gpt-4o-2024-05-13, gpt-4o-2024-08-06, gpt-4o-2024-11-20, gpt-4o-mini, gpt-4o-mini-2024-07-18, gpt-4o-mini-search-preview, gpt-4o-mini-search-preview-2025-03-11, gpt-4o-mini-transcribe, gpt-4o-mini-transcribe-2025-03-20, gpt-4o-mini-transcribe-2025-12-15, gpt-4o-mini-tts
o4 → o4-mini, o4-mini-2025-04-16
o3 → o3, o3-2025-04-16, o3-mini, o3-mini-2025-01-31


---
## 3. Agentic RAG 답변

In [29]:
ids = sorted(m.id for m in client.models.list().data)
print(len(ids))
print([i for i in ids if "luna" in i.lower() or "gpt-6" in i.lower()])
print([i for i in ids if i.startswith("gpt")][-25:])

133
['gpt-5.6-luna', 'gpt-6-astra', 'gpt-6-luna', 'gpt-6-sol', 'gpt-6.1-sol']
['gpt-audio-mini', 'gpt-audio-mini-2025-10-06', 'gpt-audio-mini-2025-12-15', 'gpt-image-1', 'gpt-image-1-mini', 'gpt-image-1.5', 'gpt-image-2', 'gpt-image-2-2026-04-21', 'gpt-image-2.5-flare', 'gpt-image-2.5-flare-2026-09-08', 'gpt-image-2.5-sunburst', 'gpt-image-2.5-sunburst-2026-09-08', 'gpt-live-1', 'gpt-live-transcribe', 'gpt-realtime', 'gpt-realtime-1.5', 'gpt-realtime-2', 'gpt-realtime-2.1', 'gpt-realtime-2.1-mini', 'gpt-realtime-2025-08-28', 'gpt-realtime-mini', 'gpt-realtime-mini-2025-12-15', 'gpt-realtime-translate', 'gpt-realtime-whisper', 'gpt-transcribe']


In [31]:
import time
from langchain_openai import ChatOpenAI

LLM_MODEL = "gpt-6-luna"
llm = ChatOpenAI(model=LLM_MODEL)  # gpt-6-luna는 temperature 미지원(기본값 사용)

SYSTEM = """당신은 가천대학교 입학처 상담원입니다. 아래 [근거 문서]에 있는 내용만 사용해 답하세요.

규칙:
1. 질문이 요구한 핵심 정보(수치·날짜·대상·조건)를 빠짐없이 포함해 1~3문장으로 답합니다.
2. 날짜 표기는 문서에 나온 그대로 씁니다. (예: 2027.01.04.(월) ~ 07.(목) 18:00)
3. 표에 있는 값을 답할 때는 행/열 머리글을 함께 읽어 정확한 값을 고릅니다.
4. 근거에 없으면 "제공된 문서에서 확인할 수 없습니다."라고만 답합니다.
5. 답변 끝에 (출처: 파일명 p.페이지)를 붙입니다."""


def format_context(pairs):
    return "\n\n".join(
        f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.page_content}"
        for i, (d, _) in enumerate(pairs, 1)
    )


def answer(question, k=6):
    t0 = time.perf_counter()
    pairs = rrf_retrieve(question, k=k)
    ctx = format_context(pairs)
    t_ret = time.perf_counter() - t0

    t1 = time.perf_counter()
    resp = llm.invoke([("system", SYSTEM), ("human", f"[근거 문서]\n{ctx}\n\n[질문]\n{question}")])
    t_gen = time.perf_counter() - t1
    return resp.content, t_ret, t_gen


SAMPLES = [
    ("가천대학교 대표 전화번호는 무엇인가요?", "1577-0067"),
    ("정시 원서접수는 몇 월 며칠부터 시작하고, 마감일과 마감시각은 언제인가요?", "2027년 1월 4일 ~ 1월 7일 18:00"),
    ("2027학년도 재외국민 전형의 인터넷 원서접수에 사용할 증명사진은 최근 몇 개월 이내 사진이어야 하며, 어떤 파일 형식으로 준비해야 하나요?", "최근 3개월 이내, JPG/GIF"),
]

for q, gold in SAMPLES:
    out, t_ret, t_gen = answer(q)
    print(f"Q: {q}\nA: {out}\n⏱ 검색 {t_ret:.2f}s / 생성 {t_gen:.2f}s / 합계 {t_ret+t_gen:.2f}s")
    print(f"   (정답: {gold})\n")

Q: 가천대학교 대표 전화번호는 무엇인가요?
A: 가천대학교 대표 전화번호는 1577-0067입니다. (출처: 03_2027학년도_수시_모집요강.pdf p.104)
⏱ 검색 0.60s / 생성 1.65s / 합계 2.25s
   (정답: 1577-0067)

Q: 정시 원서접수는 몇 월 며칠부터 시작하고, 마감일과 마감시각은 언제인가요?
A: 정시 원서접수는 2027.01.04.(월)에 시작하며, 2027.01.07.(목) 18:00에 마감합니다. (출처: 04_2027학년도_정시_모집요강.pdf p.4)
⏱ 검색 0.22s / 생성 1.90s / 합계 2.12s
   (정답: 2027년 1월 4일 ~ 1월 7일 18:00)

Q: 2027학년도 재외국민 전형의 인터넷 원서접수에 사용할 증명사진은 최근 몇 개월 이내 사진이어야 하며, 어떤 파일 형식으로 준비해야 하나요?
A: 최근 3개월 내 사진이어야 하며, jpg 또는 gif 파일 형식으로 준비해야 합니다. 인물 뒤에 배경이 있는 사진이나 스냅사진은 사용할 수 없습니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.5)
⏱ 검색 0.22s / 생성 1.51s / 합계 1.73s
   (정답: 최근 3개월 이내, JPG/GIF)



---
### 3-1. Agentic 재검색 판단

In [32]:
from pydantic import BaseModel, Field


class AgentStep(BaseModel):
    """1차 검색 결과에 대한 자기 점검"""
    sufficient: bool = Field(description="근거 문서만으로 질문의 모든 조건에 답할 수 있으면 True")
    missing: str = Field(default="", description="부족한 정보 (sufficient=False일 때)")
    search_query: str = Field(default="", description="부족할 때 다시 검색할 핵심 키워드형 질의")
    answer: str = Field(default="", description="충분할 때의 최종 답변")


step_llm = llm.with_structured_output(AgentStep)

JUDGE_PROMPT = """당신은 가천대학교 입학처 상담원입니다. [근거 문서]만 보고 [질문]에 답하세요.

먼저 스스로 점검하세요.
- 질문이 요구한 핵심 정보(수치·날짜·대상·조건)가 근거에 모두 있는가?
- 서로 다른 문서/표를 함께 봐야 하는 질문인데 한쪽만 있는가?

모두 있으면 sufficient=True 로 두고 answer 에 1~3문장 답변을 작성하세요.
부족하면 sufficient=False 로 두고, missing 에 무엇이 없는지, search_query 에 다시 검색할
핵심 키워드(문서에 나올 법한 명사 위주, 15자 내외)를 적으세요.

답변 규칙: 날짜는 문서 표기 그대로, 표 값은 행/열 머리글을 함께 읽어 정확히 고르고,
답변 끝에 (출처: 파일명 p.페이지)를 붙입니다."""


def agent_answer(question, k=6):
    t0 = time.perf_counter()
    pairs = rrf_retrieve(question, k=k)
    ctx = format_context(pairs)

    step = step_llm.invoke([
        ("system", JUDGE_PROMPT),
        ("human", f"[근거 문서]\n{ctx}\n\n[질문]\n{question}"),
    ])
    if step.sufficient and step.answer:
        return step.answer, 1, time.perf_counter() - t0

    # 추가 검색 (2차)
    q2 = step.search_query or question
    pairs2 = rrf_retrieve(q2, k=k)
    seen, merged = set(), []
    for d, s in pairs + pairs2:
        if d.page_content not in seen:
            seen.add(d.page_content)
            merged.append((d, s))

    resp = llm.invoke([
        ("system", SYSTEM),
        ("human", f"[근거 문서]\n{format_context(merged)}\n\n[질문]\n{question}"),
    ])
    return resp.content, 2, time.perf_counter() - t0


for q, gold in SAMPLES:
    out, rounds, sec = agent_answer(q)
    print(f"A: {out}\n⏱ {sec:.2f}s / 검색 {rounds}회  (정답: {gold})\n")

A: 가천대학교 대표 전화번호는 1577-0067입니다. (출처: 03_2027학년도_수시_모집요강.pdf p.104)
⏱ 2.42s / 검색 1회  (정답: 1577-0067)

A: 정시 원서접수는 2027.01.04.(월)에 시작하며, 마감은 2027.01.07.(목) 18:00입니다. (출처: 04_2027학년도_정시_모집요강.pdf p.4)
⏱ 2.04s / 검색 1회  (정답: 2027년 1월 4일 ~ 1월 7일 18:00)

A: 최근 3개월 내 촬영한 지원자 본인의 증명사진을 JPG 또는 GIF 파일 형식으로 준비하시면 됩니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.5)
⏱ 2.28s / 검색 1회  (정답: 최근 3개월 이내, JPG/GIF)



In [33]:
import random, json

random.seed(42)

by_src = {}
for c in chunks:
    by_src.setdefault(c.metadata["source"], []).append(c)

pool = []
for src, lst in by_src.items():
    pool += random.sample(lst, min(4, len(lst)))
print("테스트 생성에 쓸 청크:", len(pool))


class QA(BaseModel):
    question: str = Field(description="이 청크만으로 답할 수 있는 구체적 질문 (수치/날짜/조건 포함)")
    answer: str = Field(description="청크에 근거한 정답")


qa_llm = llm.with_structured_output(QA)

testset = []
for c in pool:
    try:
        qa = qa_llm.invoke([
            ("system", "주어진 문서 발췌만 근거로, 입학 상담에서 나올 법한 구체적 질문 1개와 정답을 만드세요. 질문은 수치·날짜·대상·조건 중 최소 하나를 묻도록 하세요."),
            ("human", c.page_content[:1500]),
        ])
        testset.append({"q": qa.question, "gold": qa.answer, "src": f"{c.metadata['source']} p.{c.metadata['page']}"})
    except Exception as e:
        print("생성 실패:", e)

print("생성된 문항:", len(testset))
for t in testset[:3]:
    print("-", t["q"], "\n  →", t["gold"])

테스트 생성에 쓸 청크: 20
생성된 문항: 20
- 장애인 등록을 한 지원자가 입시전형 진행 중 지원이 필요하면 어떻게 해야 하나요? 
  → 장애인복지법 제32조에 따라 장애인등록을 마쳤고 장애 또는 지체로 전형 진행 과정에서 지원이 필요한 경우, 사전에 요청해야 합니다.
- 2027학년도 재외국민 모집요강에서 경영학과의 최대모집 가능인원과 모집인원은 각각 몇 명인가요? 
  → 경영학과의 최대모집 가능인원은 15명이고, 모집인원은 16명입니다.
- 재외국민과 외국인(2% 이내) 전형에서 국외 재학기간 요건은 무엇이며, 1개 학년은 어떤 기준으로 인정되나요? 
  → 고교과정 1개 학년을 포함해 중·고교과정에서 통산 3년(3개 학년 이상) 재학해야 하며, 성적이 나온 기간만 인정됩니다. 일반적으로 학기 개시일부터 다음 학년도 같은 학기 개시일 전일까지를 1개 학년으로 보지만, 해당 기간의 모든 학기를 이수한 경우에는 다음 학년도 같은 학기의 직전 학기 종료일까지로도 1개 학년을 충족한 것으로 간주합니다. 중간에 편입학했다면 편입학일부터 역년으로 1년(365일)이 되는 날까지를 1개 학년으로 봅니다. 또한 국외 재학기간뿐 아니라 체류기간과 국외근무자 재직기간도 모두 충족해야 합니다.


In [34]:
class Verdict(BaseModel):
    correct: bool = Field(description="예측 답변이 정답의 핵심 정보(수치·날짜·대상·조건)를 모두 포함하면 True")
    reason: str = Field(default="", description="판정 이유")


judge = llm.with_structured_output(Verdict)
JUDGE = """정답과 예측 답변을 비교하세요. 표현이 달라도 핵심 정보(수치·날짜·대상·조건)가 모두 일치하면 정답입니다.
부분점수는 없습니다. 하나라도 빠지거나 틀리면 오답입니다."""

results = []
for i, t in enumerate(testset, 1):
    out, rounds, sec = agent_answer(t["q"])
    v = judge.invoke([
        ("system", JUDGE),
        ("human", f"[질문]\n{t['q']}\n\n[정답]\n{t['gold']}\n\n[예측]\n{out}"),
    ])
    results.append({"q": t["q"], "gold": t["gold"], "pred": out, "ok": v.correct, "why": v.reason,
                    "sec": sec, "rounds": rounds})
    print(f"{i:2d}. {'O' if v.correct else 'X'} {sec:4.1f}s r{rounds} | {t['q'][:60]}")

ok = sum(r["ok"] for r in results)
print(f"\n정확도: {ok}/{len(results)} = {ok/len(results)*100:.0f}%")
print(f"평균 응답: {sum(r['sec'] for r in results)/len(results):.2f}s | 최대 {max(r['sec'] for r in results):.2f}s")

 1. X  1.9s r1 | 장애인 등록을 한 지원자가 입시전형 진행 중 지원이 필요하면 어떻게 해야 하나요?
 2. O  1.7s r1 | 2027학년도 재외국민 모집요강에서 경영학과의 최대모집 가능인원과 모집인원은 각각 몇 명인가요?
 3. X  4.1s r1 | 재외국민과 외국인(2% 이내) 전형에서 국외 재학기간 요건은 무엇이며, 1개 학년은 어떤 기준으로 인정되나요
 4. O  2.1s r1 | 재외국민 지원자격에서 국외 체류는 누구에게 요구되며, 전 교육과정 이수자의 경우에는 어떻게 적용되나요?
 5. O  2.2s r1 | 대학은 학비 지원 신청을 1학기와 2학기 각각 언제까지 시·도지사에게 해야 하나요?
 6. O  2.0s r1 | 강원 지역의 속초권에는 어떤 시·군이 포함되나요?
 7. X  2.1s r1 | 지역의사선발전형에서 진료권·광역권 동시 지원이 가능한지는 어디서 확인해야 하며, 복수지원 금지 및 이중등록 
 8. X  2.2s r1 | 지역의사 지원센터가 제공하는 지원 종류에는 무엇이 있으며, 의무복무를 완료한 지역의사는 어떤 우대 지원을 받
 9. O  2.7s r1 | 가천대 길병원과 협력한 임상실무실습은 몇 개 내과 사이트에서 진행되며, 예시는 무엇인가요?
10. X  4.5s r1 | 본 대학에 합격해 문서등록까지 했지만 등록을 포기하려는 경우, 어떤 절차로 취소해야 하며 신청 후 취소를 번
11. X  2.5s r1 | 제6항 해당자가 신분증 사본을 제출해야 하는 출생 기준일은 언제이며, 제출 서류는 언제 이후 발급된 것이어야
12. O  1.7s r1 | 교내 순환 Eco 버스 ‘무당이’는 언제 도입됐으며, 현재 몇 대가 운행되고 있나요?
13. O  2.5s r1 | 2027학년도 정시모집 안내에서 원서접수 내용은 몇 페이지에 있나요?
14. X  5.2s r1 | 2027학년도 정시모집에서 메디컬캠퍼스(인천)에 개설된 바이오로직스학과의 모집인원은 몇 명인가요?
15. X  1.6s r1 | 2027학년도 정시 모집요

In [36]:
# 사용 모델은 gpt-6-luna 단일 모델입니다.
# step_llm / qa_llm / judge 는 모두 같은 llm 인스턴스에서 파생된 구조화 출력 래퍼입니다.
print("LLM_MODEL:", llm.model_name)
print("step_llm / qa_llm / judge 모두 동일 모델에서 파생 (단일 모델)")

print("\n=== 오답 상세 ===")
for r in results:
    if not r["ok"]:
        print(f"\nQ: {r['q']}")
        print(f"  정답: {r['gold'][:200]}")
        print(f"  예측: {r['pred'][:200]}")
        print(f"  판정: {r['why'][:120]}")

LLM_MODEL: gpt-6-luna
step_llm / qa_llm / judge 모두 동일 모델에서 파생 (단일 모델)

=== 오답 상세 ===

Q: 장애인 등록을 한 지원자가 입시전형 진행 중 지원이 필요하면 어떻게 해야 하나요?
  정답: 장애인복지법 제32조에 따라 장애인등록을 마쳤고 장애 또는 지체로 전형 진행 과정에서 지원이 필요한 경우, 사전에 요청해야 합니다.
  예측: 장애인복지법 제32조에 따라 장애인 등록을 마친 지원자가 입학전형 진행 과정에서 지원이 필요한 경우, 사전에 요청해야 합니다. (출처: 04_2027학년도_정시_모집요강.pdf p.8)
  판정: 정답은 장애 또는 지체로 전형 진행 과정에서 지원이 필요한 경우라고 조건을 명시하지만, 예측 답변에는 ‘지체로’라는 조건이 빠져 있습니다.

Q: 재외국민과 외국인(2% 이내) 전형에서 국외 재학기간 요건은 무엇이며, 1개 학년은 어떤 기준으로 인정되나요?
  정답: 고교과정 1개 학년을 포함해 중·고교과정에서 통산 3년(3개 학년 이상) 재학해야 하며, 성적이 나온 기간만 인정됩니다. 일반적으로 학기 개시일부터 다음 학년도 같은 학기 개시일 전일까지를 1개 학년으로 보지만, 해당 기간의 모든 학기를 이수한 경우에는 다음 학년도 같은 학기의 직전 학기 종료일까지로도 1개 학년을 충족한 것으로 간주합니다. 중간에 편입학했
  예측: 국외 재학기간은 고교과정 1개 학년을 포함해 중·고교과정에서 통산 3년(3개 학년 이상) 재학해야 하며, 성적이 나온 기간만 인정됩니다. 1개 학년은 학기 개시일부터 다음 학년도 동일 학기 개시일 전일까지(약 365일)로 보며, 그 기간 내 모든 학기를 이수한 경우에는 다음 학년도 동일 학기 개시일 직전 학기 종료일까지를 인정합니다. 중간에 편입학했다면 편
  판정: 재학기간 요건과 1개 학년 인정 기준은 포함했지만, 국외 재학기간뿐 아니라 체류기간과 국외근무자 재직기간도 모두 충족해야 한다는 조건이 빠졌습니다.

Q: 지역의사선발전형에서 진료권·광역권

In [37]:
# 청크 위치 인덱스 (재임베딩 없이 이웃 청크 확장에 사용)
CHUNK_INDEX = {c.page_content: i for i, c in enumerate(chunks)}


def expand_neighbors(pairs, span=1, top_n=4):
    """상위 검색 결과의 앞/뒤 청크를 붙여 문맥 절단을 완화"""
    out, seen = [], set()
    for d, s in pairs:
        i = CHUNK_INDEX.get(d.page_content)
        cand = [d]
        if i is not None:
            for j in range(max(0, i - span), min(len(chunks), i + span + 1)):
                if chunks[j].metadata["source"] == d.metadata["source"]:
                    cand.append(chunks[j])
        for c in cand:
            if c.page_content not in seen:
                seen.add(c.page_content)
                out.append((c, s))
    return out


def rrf_retrieve_v2(query, k=8, rrf_k=60):
    """BM25 + 벡터 + 이웃 확장"""
    ranked = [bm25_retriever.invoke(query), vector_retriever.invoke(query)]
    scores, docs = {}, {}
    for result in ranked:
        for rank, doc in enumerate(result, start=1):
            key = doc.page_content
            scores[key] = scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
            docs[key] = doc
    ordered = sorted(scores, key=scores.get, reverse=True)[:k]
    return expand_neighbors([(docs[key], scores[key]) for key in ordered])


SYSTEM_V2 = """당신은 가천대학교 입학처 상담원입니다. [근거 문서]에 있는 내용만 사용해 답하세요.

규칙:
1. 질문이 요구한 핵심 정보(수치·날짜·대상·조건)를 빠짐없이 포함해 2~4문장으로 답합니다.
2. 근거에 여러 조건이 있으면 모두 나열합니다.
3. 근거에 없는 조건·수식어를 덧붙이지 않습니다. (예: '~기준', '~에 한해', '~만')
4. 날짜는 문서 표기 그대로 씁니다. (예: 2027.01.04.(월) ~ 07.(목) 18:00)
5. 표에서 값을 읽을 때 같은 행의 다른 열 값을 섞지 말고, 행/열 머리글을 함께 확인합니다.
6. 근거에 없으면 "제공된 문서에서 확인할 수 없습니다."라고만 답합니다.
7. 답변 끝에 (출처: 파일명 p.페이지)를 붙입니다."""


def agent_answer_v2(question, k=8):
    t0 = time.perf_counter()
    pairs = rrf_retrieve_v2(question, k=k)
    ctx = format_context(pairs)

    step = step_llm.invoke([
        ("system", JUDGE_PROMPT),
        ("human", f"[근거 문서]\n{ctx}\n\n[질문]\n{question}"),
    ])
    if step.sufficient and step.answer:
        return step.answer, 1, time.perf_counter() - t0

    pairs2 = rrf_retrieve_v2(step.search_query or question, k=k)
    seen, merged = set(), []
    for d, s in pairs + pairs2:
        if d.page_content not in seen:
            seen.add(d.page_content)
            merged.append((d, s))

    resp = llm.invoke([
        ("system", SYSTEM_V2),
        ("human", f"[근거 문서]\n{format_context(merged)}\n\n[질문]\n{question}"),
    ])
    return resp.content, 2, time.perf_counter() - t0


SAMPLE_CHECK = agent_answer_v2("가천대학교 대표 전화번호는 무엇인가요?")
print(SAMPLE_CHECK[0], f"\n{SAMPLE_CHECK[2]:.2f}s")

가천대학교 대표 전화번호는 1577-0067입니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.16) 
2.08s


In [39]:
# --- 프롬프트 v3: 지시문은 영어, 답변은 한국어 ---
JUDGE_PROMPT = """You are an admissions counselor at Gachon University. Answer the question using ONLY the evidence documents.

Self-check first:
- Does the evidence contain every key fact the question asks for (numbers, dates, targets, conditions)?
- Does the question require combining facts from different documents or tables while only one side is present?

If everything needed is present: set sufficient=True and write the final answer in `answer`.
If something is missing: set sufficient=False, describe what is missing in `missing`, and put a short
keyword-style search query in `search_query` (Korean, using wording that appears in the documents).

Answer rules:
- Write the answer in Korean.
- Copy dates exactly as written in the documents (e.g., 2027.01.04.(월) ~ 07.(목) 18:00).
- When reading a table value, verify both the row header and the column header before choosing a number.
- Do not add conditions or qualifiers that are not in the evidence.
- End the answer with (출처: 파일명 p.페이지)."""

SYSTEM_V2 = """You are an admissions counselor at Gachon University. Answer using ONLY the evidence documents.

Rules:
1. Include every key fact the question asks for (numbers, dates, targets, conditions) in 2-4 sentences.
2. If the evidence lists several conditions, list all of them.
3. Never add conditions or qualifiers that are not in the evidence (e.g., "based on X", "only for Y").
4. Copy dates exactly as written in the documents (e.g., 2027.01.04.(월) ~ 07.(목) 18:00).
5. When reading a table, align the row and column headers correctly; never mix values from different columns.
6. If the evidence does not contain the answer, reply only: "제공된 문서에서 확인할 수 없습니다."
7. Write the answer in Korean and end with (출처: 파일명 p.페이지)."""

JUDGE = """Compare the gold answer with the predicted answer.
If the prediction contains all key facts of the gold answer (numbers, dates, targets, conditions), mark it correct
even when the wording differs. There is no partial credit: if any key fact is missing or wrong, mark it incorrect."""

print("프롬프트 v3(영어 지시문) 적용 완료")

프롬프트 v3(영어 지시문) 적용 완료


In [41]:
hits = [c for c in chunks if "바이오로직스" in c.page_content]
print("바이오로직스 등장 청크:", len(hits))
for c in hits:
    print("\n###", c.metadata["source"], "p.", c.metadata["page"])
    for ln in c.page_content.splitlines():
        if "바이오로직스" in ln or "모집인원" in ln:
            print("   ", ln[:200])

바이오로직스 등장 청크: 29

### 01_2027학년도_재외국민_모집요강.hwp p. 3
    | 모집인원 | 모집인원 | 모집인원 | 모집인원 | 모집인원 | 모집인원 |
    | 자연 | 자연 | 바이오로직스학과* | 바이오로직스학과* | 10 | 12 |
    | ※ | 전 교육과정 이수자는 모집인원 제한 없이 지원 가능하며, 북한이탈주민은 인공지능학과, 의예과, 약학과, 바이오로직스학과, 간호학과를 제외한 모집단위에 지원 가능합니다. | | | | |

### 03_2027학년도_수시_모집요강.pdf p. 8
    바이오로직스학과(2024)

### 03_2027학년도_수시_모집요강.pdf p. 9
    바이오로직스학과, 빅데이터경영전공, 배터리공학전공(2023)

### 03_2027학년도_수시_모집요강.pdf p. 16
    포함해 4개의 신설학과를 개설하였습니다. 이어 2023년에는 배터리공학전공을, 2024년에는 바이오로직스학과, 빅데이터경영전

### 03_2027학년도_수시_모집요강.pdf p. 17
    바이오로직스학과
    바이오로직스학과
    바이오로직스(혁신 신약) 분야 융합형 인재를

### 03_2027학년도_수시_모집요강.pdf p. 31
    바이오로직스학과 Ⓜ ★ 100

### 03_2027학년도_수시_모집요강.pdf p. 33
    바이오로직스학과 29 12 3 14 2

### 03_2027학년도_수시_모집요강.pdf p. 34
    인문계열, 간호학과, 바이오로직스학과,

### 03_2027학년도_수시_모집요강.pdf p. 40
    계열 모집단위 모집인원
    바이오로직스학과 국어, 수학, 영어, 사회/과학탐구(1과목) 2개 영역 등급 합 5 이내
    계열 모집단위 모집인원

### 03_2027학년도_수시_모집요강.pdf p. 40
    계열 모집단위 모집인원
    바이오로직스학과 29

### 03_2027학년도_수시_모집요강.pdf p. 41
    인문계열, 간호학과, 바이

In [42]:
c = [x for x in chunks if x.metadata["source"].startswith("04") and x.metadata["page"] == 5][0]
print(c.page_content[:1800])

[04_2027학년도_정시_모집요강.pdf p.5]
31
대학 모집단위 전공선택기준정원 입학정원
경영
대학
경영학과 130
회계세무학과 50
관광경영학과 45
의료산업경영학과 45
금융·빅데이터학부 ★
금융수학전공
100
빅데이터경영전공
사회
과학
대학
미디어커뮤니케이션학과 30
경제학과 50
응용통계학과 45
사회복지학과 40
유아교육학과 50
심리학과 40
패션산업학과 40
AI
인문
대학
한국어문학과 50
210
영미어문학과 40
중국어문학과 40
일본어문학과 40
유럽어문학과 40
법과
대학
법학과 60
150경찰행정학과 40
행정학과 50
공과
대학
도시계획·조경학부
도시계획학전공
65
조경학전공
건축학부
실내건축학전공
60
건축학전공(5년)
건축공학과 40
화공생명배터리
공학부 ★
화공생명공학전공 ◉
200
배터리공학전공
기계공학부
기계공학전공 ◉
160로봇공학전공
설비 · 소방공학전공
스마트팩토리학과 ★ 50
건설환경공학과 ◉ 45
신소재공학과 45
미래자동차학과 ❖ 30
바이오
나노
대학
바이오나노학과 45
식품생명공학과 45
식품영양학과 40
생명과학과 45
반도체물리학과 45
화학과 45
대학 모집단위 전공선택기준정원 입학정원
반도체
대학
전자공학과 ◉ 110
200
반도체공학과 90
시스템반도체학과 ★ 50
반 도 체·디 스 플 레 이 학 과 ❖ 130
반도체설계학과 ❖ 35
A I·컴퓨팅
대학
컴퓨터공학과 100
인공지능학과 ★ 150
인공지능시스템학과 50
IT융합
대학
정보보호학과 ★ 50
전기공학과 ◉ 60
스마트시티학과 ★ 50
의공학과 45
바이오의료기기학과 ❖ 35
게 임·영 상 학 과 ❖ 30
한의과대학 한의예과 30


In [43]:
import pdfplumber

path = "/content/dataset/04_2027학년도_정시_모집요강.pdf"
with pdfplumber.open(path) as pdf:
    page = pdf.pages[4]
    text_layout = page.extract_text(layout=True) or ""
print(text_layout[:2000])

                                                                                     
                                                                                     
                                                                                     
                                                                                     
                                                                      2027학년도 정시모집요강 
                    모집단위별          입학정원                                              
                                                                                     
                                                                                     
                                                                                     
                                                                                     
                                   전공선택 입학                              전공선택 입학      
          대학          모집단위                      대학    

In [44]:
with pdfplumber.open(path) as pdf:
    tables = pdf.pages[4].extract_tables()

print("표 개수:", len(tables))
for ti, rows in enumerate(tables[:2]):
    print(f"\n=== table {ti}: {len(rows)}행 ===")
    for r in rows[:12]:
        print([ (c or "").replace("\n", " ")[:22] for c in r ])

표 개수: 2

=== table 0: 42행 ===
['대학', '모집단위', '', '전공선택 기준정원', '입학 정원']
['', '경영학과', '', '', '130']
['', '회계세무학과', '', '', '50']
['', '관광경영학과', '', '', '45']
['', '의료산업경영학과', '', '', '45']
['', '금융·빅데이터학부 ★', '금융수학전공', '', '100']
['', '', '빅데이터경영전공', '', '']
['', '미디어커뮤니케이션학과', '', '', '30']
['', '경제학과', '', '', '50']
['', '응용통계학과', '', '', '45']
['', '사회복지학과', '', '', '40']
['', '유아교육학과', '', '', '50']

=== table 1: 37행 ===
['대학', '모집단위', '', '전공선택 기준정원', '입학 정원']
['', '전자공학과 ◉', '', '110', '200']
['', '반도체공학과', '', '90', '']
['', '시스템반도체학과 ★', '', '', '50']
['', '반도체·디스플레이학과 ❖', '', '', '130']
['', '반도체설계학과 ❖', '', '', '35']
['', '컴퓨터공학과', '', '', '100']
['', '인공지능학과 ★', '', '', '150']
['', '인공지능시스템학과', '', '', '50']
['', '정보보호학과 ★', '', '', '50']
['', '전기공학과 ◉', '', '', '60']
['', '스마트시티학과 ★', '', '', '50']


In [45]:
import pdfplumber


def rows_to_md(rows):
    if not rows:
        return ""
    ncols = max(len(r) for r in rows)
    rows = [[(c or "").replace("\n", " ").strip() for c in r] + [""] * (ncols - len(r)) for r in rows]
    keep = [j for j in range(ncols) if any(r[j] for r in rows)]
    rows = [[r[j] for j in keep] for r in rows]
    rows = [r for r in rows if any(r)]
    if len(rows) < 2:
        return ""
    body = "\n".join("| " + " | ".join(c.replace("|", "/") for c in r) + " |" for r in rows)
    head = "| " + " | ".join(rows[0]) + " |\n|" + "---|" * len(keep)
    return head + "\n" + "\n".join(body.splitlines()[1:])


pdf_table_docs = []
for p in sorted(DATA.glob("*.pdf")):
    with pdfplumber.open(str(p)) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            for t in page.extract_tables():
                md = rows_to_md(t)
                if len(md) > 60:
                    pdf_table_docs.append(Document(page_content=md,
                                                   metadata={"source": p.name, "page": i, "kind": "table"}))

print("PDF 표 청크:", len(pdf_table_docs))
raw_docs = raw_docs + [Document(page_content=normalize(d.page_content), metadata=d.metadata) for d in pdf_table_docs]
print("전체 파싱 단위:", len(raw_docs))

PDF 표 청크: 192
전체 파싱 단위: 383


In [49]:
!pip install -q pymupdf
import fitz, time

t0 = time.perf_counter()
doc = fitz.open("/content/dataset/04_2027학년도_정시_모집요강.pdf")
page = doc[4]
print("=== text ===")
print(page.get_text("text")[:900])

print("\n=== find_tables ===")
tabs = page.find_tables()
print("표 개수:", len(tabs.tables))
for t in tabs.tables:
    rows = t.extract()
    print(f"- {len(rows)}행")
    for r in rows[:6]:
        print("   ", [(c or "").replace("\n", " ")[:20] for c in r])
print(f"\n소요: {time.perf_counter()-t0:.2f}s")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


=== text ===
31
대학
모집단위
전공선택
기준정원
입학
정원
경영
대학
경영학과
130
회계세무학과
50
관광경영학과
45
의료산업경영학과
45
금융·빅데이터학부 ★
금융수학전공
100
빅데이터경영전공
사회
과학
대학
미디어커뮤니케이션학과
30
경제학과
50
응용통계학과
45
사회복지학과
40
유아교육학과
50
심리학과
40
패션산업학과
40
AI
인문
대학
한국어문학과
50
210
영미어문학과
40
중국어문학과
40
일본어문학과
40
유럽어문학과
40
법과
대학
법학과
60
150
경찰행정학과
40
행정학과
50
공과 
대학
도시계획·조경학부
도시계획학전공
65
조경학전공
건축학부
실내건축학전공
60
건축학전공(5년)
건축공학과
40
화공생명배터리
공학부 ★
화공생명공학전공 ◉
200
배터리공학전공
기계공학부
기계공학전공 ◉
160
로봇공학전공
설비·소방공학전공
스마트팩토리학과 ★
50
건설환경공학과 ◉
45
신소재공학과
45
미래자동차학과 ❖
30
바이오 
나노 
대학
바이오나노학과
45
식품생명공학과
45
식품영양학과
40
생명과학과
45
반도체물리학과
45
화학과
45
대학
모집단위
전공선택
기준정원
입학
정원
반도체 
대학
전자공학과 ◉
110
200
반도체공학과
90
시스템반도체학과 ★
50
반도체·디스플레이학과 ❖
130
반도체설계학과 ❖
35
AI·컴퓨팅
대학
컴퓨터공학과
100
인공지능학과 ★
150
인공지능시스템학과
50
IT융합 
대학
정보보호학과 ★
50
전기공학과 ◉
60
스마트시티학과 ★
50
의공학과
45
바이오의료기기학과 ❖
35
게임·영상학과 ❖
30
한의과대학
한의예과
30
예술·체육
대학
미술·디자인학부
회화전공
30
조소전공
25
시각디자인전공
30
산업디자인전공
30
음악학부
성악전공
12
기악전공
31
작곡전공
7
체육학부
체육전공
40
태권도전공
30
연기

=== find_tables ===
표 개수: 1
- 21행
    ['대학', '모집단위', '', '전공선택 기준정원', '입학 정원', '', '대

In [50]:
import time, pymupdf
from pypdf import PdfReader

p = "/content/dataset/04_2027학년도_정시_모집요강.pdf"

# 1) 날짜 표 페이지 텍스트 비교 (p.4)
d = pymupdf.open(p)
print("=== PyMuPDF (p.4) ===")
print(d[3].get_text("text")[:500])

print("\n=== pypdf (p.4) ===")
print((PdfReader(p).pages[3].extract_text() or "")[:500])

# 2) 속도 비교 (전체 페이지 텍스트)
t0 = time.perf_counter()
n = sum(len(pg.get_text("text")) for pg in pymupdf.open(p))
t_mu = time.perf_counter() - t0

t0 = time.perf_counter()
m = sum(len(pg.extract_text() or "") for pg in PdfReader(p).pages)
t_pypdf = time.perf_counter() - t0
print(f"\nPyMuPDF {n:,}자 {t_mu:.2f}s | pypdf {m:,}자 {t_pypdf:.2f}s")

=== PyMuPDF (p.4) ===
30
2027학년도 정시모집요강
전형일정
구분
일시
비고
원서접수
2027. 01. 04.(월) ~ 07.(목) 18:00
서류제출 마감
2027. 01. 08.(금) 13:00까지
·원서접수 사이트에서 제출
실기고사
가군
연기예술학과(연출)
고사장 확인 
2027. 01. 11.(월)
·세부 일정은 개별통지를 하지 않으므로 지원자가 
반드시 확인해야 합니다.
·본 대학 입학처 홈페이지에서 고사(실기, 면접) 
일정을 반드시 확인해야 합니다.
·전형일은 원서접수 마감 후 지원자 수에 의해 
변경될 수 있습니다. 
·고사(실기, 면접) 시 본인임을 확인할 수 있는 
신분증(주민등록증, 운전면허증, 여권 등)을 
지참해야 합니다.
전형일  
2027. 01. 12.(화)
면접고사
나군
조기취업형 계약학과 전형
1단계 발표 
2027. 01. 18.(월)
전형일 
2027. 01. 21.(목)
합격자 발표
수능위주 
2027. 01. 22.(금)
·본 대학 입학처 홈페이지
실기, 

=== pypdf (p.4) ===
30
2027학년도	정시모집요강
전형일정
구분 일시 비고
원서접수 2027 .	01.	04.(월)	~	07 .(목)	18:00
서류제출 마감 2027 .	01.	08.(금)	13:00까지 ·원서접수	사이트에서	제출
실기고사 가군 연기예술학과(연출)
고사장	확인	 2027 .	01.	1 1.(월) ·	세부	일정은	개별통지를	하지	않으므로	지원자가	
반드시	확인해야	합니다.
·	본	대학	입학처	홈페이지에서	고사(실기,	면접)	
일정을	반드시	확인해야	합니다.
·	 	전형일은	원서접수	마감	후	지원자	수에	의해	
변경될	수	있습니다.	
·	고사(실기,	면접)	시	본인임을	확인할	수	있는	
신분증(주민등록증,	운전면허증,	여권	등)을	
지참해야	합니다.
전형일		 2027 .	01.	12.(화)
면접고사 나군 조기취업형	계약학과	전형
1단계	발표	 2027 .	01.	18.(월)
전형일	 2027 .	01.	21

In [51]:
# ===== v4 STEP 1. 파싱 (텍스트: PyMuPDF / 표: pdfplumber / HWP: langchain-hwp-hwpx) =====
import json, re
from pathlib import Path
import pymupdf, pdfplumber
from langchain_core.documents import Document
from langchain_hwp_hwpx import HwpHwpxLoader

DATA = Path("/content/dataset")


def normalize(text):
    out = []
    for ln in text.split("\n"):
        ln = re.sub(r"[ \t]+", " ", ln).strip()
        ln = re.sub(r"(?<=\d)\s*\.\s*(?=\d)", ".", ln)
        if ln:
            out.append(ln)
    return "\n".join(out)


def render_hwp_table(cells_json):
    cells = json.loads(cells_json)
    nrows = max(c["row"] + c["rowspan"] for c in cells)
    ncols = max(c["col"] + c["colspan"] for c in cells)
    grid = [["" for _ in range(ncols)] for _ in range(nrows)]
    for c in cells:
        text = " ".join((c.get("text") or "").split()).replace("|", "/")
        if text == "[IMAGE]":
            text = ""
        span = c["rowspan"] * c["colspan"]
        for r in range(c["row"], min(c["row"] + c["rowspan"], nrows)):
            for col in range(c["col"], min(c["col"] + c["colspan"], ncols)):
                if span == 1 or len(text) <= 40 or (r, col) == (c["row"], c["col"]):
                    grid[r][col] = text
    lines, prev = [], None
    for row in grid:
        if row == prev or not any(row):
            continue
        prev = row
        lines.append("| " + " | ".join(row) + " |")
    if not lines:
        return ""
    return lines[0] + "\n|" + "---|" * ncols + "\n" + "\n".join(lines[1:])


def rows_to_md(rows):
    if not rows:
        return ""
    ncols = max(len(r) for r in rows)
    rows = [[(c or "").replace("\n", " ").strip() for c in r] + [""] * (ncols - len(r)) for r in rows]
    keep = [j for j in range(ncols) if any(r[j] for r in rows)]
    rows = [r for r in ([[r[j] for j in keep] for r in rows]) if any(r)]
    if len(rows) < 2:
        return ""
    body = "\n".join("| " + " | ".join(c.replace("|", "/") for c in r) + " |" for r in rows[1:])
    return "| " + " | ".join(rows[0]) + " |\n|" + "---|" * len(keep) + "\n" + body


def parse_hwp(path):
    els = HwpHwpxLoader(str(path), mode="elements", distribution_backend="pyhwp").load()
    out = []
    for e in els:
        if e.metadata.get("element_type") == "body" and e.page_content.strip():
            out.append(Document(page_content=e.page_content.strip(),
                                metadata={"source": path.name, "page": 1, "kind": "text"}))
        elif e.metadata.get("element_type") == "table":
            md = render_hwp_table(e.metadata["cells"])
            if md:
                out.append(Document(page_content=md,
                                    metadata={"source": path.name, "page": int(e.metadata.get("element_index", 0)) + 1, "kind": "table"}))
    return out


def parse_pdf(path):
    out = []
    doc = pymupdf.open(str(path))
    for i, page in enumerate(doc, start=1):
        t = normalize(page.get_text("text"))
        if t:
            out.append(Document(page_content=t, metadata={"source": path.name, "page": i, "kind": "text"}))
    doc.close()
    with pdfplumber.open(str(path)) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            for t in page.extract_tables():
                md = normalize(rows_to_md(t))
                if len(md) > 60:
                    out.append(Document(page_content=md, metadata={"source": path.name, "page": i, "kind": "table"}))
    return out


raw_docs = []
for p in sorted(DATA.iterdir()):
    if p.suffix.lower() == ".hwp":
        raw_docs += parse_hwp(p)
    elif p.suffix.lower() == ".pdf":
        raw_docs += parse_pdf(p)

print("파싱 단위:", len(raw_docs))
for name in sorted({d.metadata['source'] for d in raw_docs}):
    part = [d for d in raw_docs if d.metadata['source'] == name]
    print(f"  {name}: {len(part)}개, {sum(len(d.page_content) for d in part):,}자")

파싱 단위: 382
  01_2027학년도_재외국민_모집요강.hwp: 19개, 39,889자
  02_2027학년도_지역의사선발전형_안내.pdf: 45개, 33,619자
  03_2027학년도_수시_모집요강.pdf: 203개, 133,609자
  04_2027학년도_정시_모집요강.pdf: 108개, 50,842자
  05_2027학년도_수시_면접고사_일정.pdf: 7개, 5,174자


In [52]:
# ===== v4 STEP 2. 청킹 =====
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", "다. ", " ", ""],
)


def _hard_split(text, limit):
    out, cur = [], ""
    for w in text.split(" "):
        if len(cur) + len(w) + 1 > limit and cur:
            out.append(cur)
            cur = ""
        cur += (" " if cur else "") + w
    if cur:
        out.append(cur)
    return out


def split_table_md(md, max_chars=1400):
    lines = md.splitlines()
    head, body = lines[:2], lines[2:]
    chunks_, cur, size = [], [], 0
    for ln in body:
        for piece in (_hard_split(ln, max_chars) if len(ln) > max_chars else [ln]):
            if size + len(piece) > max_chars and cur:
                chunks_.append("\n".join(head + cur))
                cur, size = [], 0
            cur.append(piece)
            size += len(piece)
    if cur:
        chunks_.append("\n".join(head + cur))
    return chunks_


chunks = []
for d in raw_docs:
    prefix = f"[{d.metadata['source']} p.{d.metadata['page']}]\n"
    pieces = split_table_md(d.page_content) if d.metadata["kind"] == "table" \
        else [c.page_content for c in splitter.split_documents([d])]
    for i, piece in enumerate(pieces):
        chunks.append(Document(page_content=prefix + piece,
                               metadata={**d.metadata, "part": i, "order": len(chunks)}))

lengths = [len(c.page_content) for c in chunks]
print(f"청크 {len(chunks):,}개 | 평균 {sum(lengths)/len(lengths):.0f}자 | 최대 {max(lengths):,}자 | 1200자 초과 {sum(l>1200 for l in lengths)}개")

청크 514개 | 평균 574자 | 최대 2,580자 | 1200자 초과 41개


In [53]:
# ===== v4 STEP 3. 인덱스 + 하이브리드 검색 + Agentic 답변 =====
import time, time as _t
from pydantic import BaseModel, Field
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi

LLM_MODEL = "gpt-6-luna"

# --- 임베딩 & 벡터DB ---
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=f"gachon_{int(_t.time())}",
    collection_metadata={"hnsw:space": "cosine"},
)
vector_retriever = vectordb.as_retriever(search_kwargs={"k": 8})

# --- BM25 (Kiwi 형태소) ---
kiwi = Kiwi()
ALLOWED = {"NNG", "NNP", "NNB", "VV", "VA", "XR", "SL", "SN"}
bm25_retriever = BM25Retriever.from_documents(
    chunks, preprocess_func=lambda t: [x.form for x in kiwi.tokenize(t) if x.tag in ALLOWED]
)
bm25_retriever.k = 8

CHUNK_INDEX = {c.page_content: i for i, c in enumerate(chunks)}


def rrf_retrieve(query, k=8, rrf_k=60, span=1):
    """BM25 + Dense(RRF 융합) → 상위 결과의 앞뒤 청크 확장"""
    ranked = [bm25_retriever.invoke(query), vector_retriever.invoke(query)]
    scores, docs = {}, {}
    for result in ranked:
        for rank, doc in enumerate(result, start=1):
            key = doc.page_content
            scores[key] = scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
            docs[key] = doc
    ordered = sorted(scores, key=scores.get, reverse=True)[:k]

    out, seen = [], set()
    for key in ordered:
        d = docs[key]
        i = CHUNK_INDEX.get(key)
        cand = [d]
        if i is not None:
            for j in range(max(0, i - span), min(len(chunks), i + span + 1)):
                if chunks[j].metadata["source"] == d.metadata["source"]:
                    cand.append(chunks[j])
        for c in cand:
            if c.page_content not in seen:
                seen.add(c.page_content)
                out.append((c, scores[key]))
    return out


def format_context(pairs):
    return "\n\n".join(
        f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.page_content}"
        for i, (d, _) in enumerate(pairs, 1)
    )


# --- LLM & 프롬프트 (영어 지시문, 한국어 답변) ---
llm = ChatOpenAI(model=LLM_MODEL)

SYSTEM_PROMPT = """You are an admissions counselor at Gachon University. Answer using ONLY the evidence documents.

Rules:
1. Include every key fact the question asks for (numbers, dates, targets, conditions) in 2-4 sentences.
2. If the evidence lists several conditions, list all of them.
3. Never add conditions or qualifiers that are not in the evidence (e.g., "based on X", "only for Y").
4. Copy dates exactly as written (e.g., 2027.01.04.(월) ~ 07.(목) 18:00).
5. When reading a table, align the row and column headers correctly; never mix values from different columns.
6. If the evidence does not contain the answer, reply only: "제공된 문서에서 확인할 수 없습니다."
7. Write the answer in Korean and end with (출처: 파일명 p.페이지)."""

JUDGE_PROMPT = """You are an admissions counselor at Gachon University. Answer the question using ONLY the evidence documents.

Self-check first:
- Does the evidence contain every key fact the question asks for (numbers, dates, targets, conditions)?
- Does the question require combining facts from different documents or tables while only one side is present?

If everything needed is present: set sufficient=True and write the final answer in `answer`.
If something is missing: set sufficient=False, describe what is missing in `missing`, and put a short
keyword-style search query in `search_query` (Korean, using wording that appears in the documents).

Answer rules:
- Write the answer in Korean.
- Copy dates exactly as written in the documents.
- When reading a table value, verify both the row and column headers before choosing a number.
- Do not add conditions or qualifiers that are not in the evidence.
- End the answer with (출처: 파일명 p.페이지)."""


class AgentStep(BaseModel):
    sufficient: bool = Field(description="True if the evidence alone fully answers the question")
    missing: str = Field(default="", description="What is missing (when sufficient=False)")
    search_query: str = Field(default="", description="Short Korean keyword query for a second retrieval")
    answer: str = Field(default="", description="Final Korean answer (when sufficient=True)")


step_llm = llm.with_structured_output(AgentStep)


def agent_answer(question, k=8):
    """1차 검색 → 자기점검 → 부족하면 재검색 후 재답변"""
    t0 = time.perf_counter()
    pairs = rrf_retrieve(question, k=k)

    step = step_llm.invoke([
        ("system", JUDGE_PROMPT),
        ("human", f"[Evidence]\n{format_context(pairs)}\n\n[Question]\n{question}"),
    ])
    if step.sufficient and step.answer:
        return step.answer, 1, time.perf_counter() - t0

    pairs2 = rrf_retrieve(step.search_query or question, k=k)
    seen, merged = set(), []
    for d, s in pairs + pairs2:
        if d.page_content not in seen:
            seen.add(d.page_content)
            merged.append((d, s))

    resp = llm.invoke([
        ("system", SYSTEM_PROMPT),
        ("human", f"[Evidence]\n{format_context(merged)}\n\n[Question]\n{question}"),
    ])
    return resp.content, 2, time.perf_counter() - t0


print("인덱스:", vectordb._collection.count(), "청크 | 모델:", llm.model_name)
print(agent_answer("가천대학교 대표 전화번호는 무엇인가요?")[0])

인덱스: 514 청크 | 모델: gpt-6-luna
가천대학교 대표 전화번호는 1577-0067입니다. (출처: 03_2027학년도_수시_모집요강.pdf p.104)


In [54]:
results_v4 = evaluate(agent_answer, testset, "v4-PyMuPDF+표")

 1. O  2.4s | 장애인 등록을 한 지원자가 입시전형 진행 중 지원이 필요하면 어떻게 해야 하나요?
 2. O  1.6s | 2027학년도 재외국민 모집요강에서 경영학과의 최대모집 가능인원과 모집인원은 각각 몇 명인가요?
 3. X  3.9s | 재외국민과 외국인(2% 이내) 전형에서 국외 재학기간 요건은 무엇이며, 1개 학년은 어떤 기준으로 
 4. O  2.1s | 재외국민 지원자격에서 국외 체류는 누구에게 요구되며, 전 교육과정 이수자의 경우에는 어떻게 적용되나
 5. O  1.9s | 대학은 학비 지원 신청을 1학기와 2학기 각각 언제까지 시·도지사에게 해야 하나요?
 6. O  1.7s | 강원 지역의 속초권에는 어떤 시·군이 포함되나요?
 7. X  2.7s | 지역의사선발전형에서 진료권·광역권 동시 지원이 가능한지는 어디서 확인해야 하며, 복수지원 금지 및 
 8. X  3.8s | 지역의사 지원센터가 제공하는 지원 종류에는 무엇이 있으며, 의무복무를 완료한 지역의사는 어떤 우대 
 9. O  2.1s | 가천대 길병원과 협력한 임상실무실습은 몇 개 내과 사이트에서 진행되며, 예시는 무엇인가요?
10. O  3.9s | 본 대학에 합격해 문서등록까지 했지만 등록을 포기하려는 경우, 어떤 절차로 취소해야 하며 신청 후 
11. O  2.6s | 제6항 해당자가 신분증 사본을 제출해야 하는 출생 기준일은 언제이며, 제출 서류는 언제 이후 발급된
12. O  1.7s | 교내 순환 Eco 버스 ‘무당이’는 언제 도입됐으며, 현재 몇 대가 운행되고 있나요?
13. O  2.6s | 2027학년도 정시모집 안내에서 원서접수 내용은 몇 페이지에 있나요?
14. X 18.8s | 2027학년도 정시모집에서 메디컬캠퍼스(인천)에 개설된 바이오로직스학과의 모집인원은 몇 명인가요?
15. O  3.3s | 2027학년도 정시 모집요강에서 농어촌 전형은 몇 페이지에 안내되어 있나요?
16. O  2.2s | [가군] 일반전형 1·2에 지원하려는 2027학년도 

In [55]:
print("2회 검색 문항:", [(i+1, r["rounds"]) for i, r in enumerate(results_v4) if r["rounds"] == 2])
r = results_v4[13]
print(f"\n#14 {r['sec']:.1f}s rounds={r['rounds']}")
print("정답:", r["gold"][:150])
print("예측:", r["pred"][:300])

2회 검색 문항: []

#14 18.8s rounds=1
정답: 바이오로직스학과는 메디컬캠퍼스(인천) 모집단위이며, 모집인원은 100명입니다.
예측: 2027학년도 정시모집에서 바이오로직스학과의 모집인원은 30명입니다. (출처: 04_2027학년도_정시_모집요강.pdf p.16)


In [56]:
for c in chunks:
    if c.metadata["source"].startswith("04") and c.metadata["page"] == 16 and c.metadata["kind"] == "table":
        print(c.page_content[:1200])
        print("-" * 60)

[04_2027학년도_정시_모집요강.pdf p.16]
| 계열 | 모집단위 | 모집인원 | |
|---|---|---|---|
| | | 일반전형 1 | 일반전형 2 |
| | 경영학과 | 38 | |
| | 의료산업경영학과 | 17 | |
| | 미디어커뮤니케이션학과 | 10 | |
| | 패션산업학과 | 13 | |
| | 건축학부 | 22 | |
| | 기계공학부 | 45 | |
| | 신소재공학과 | 16 | |
| | 바이오나노학과 | 16 | |
| | 식품생명공학과 | 16 | |
| | 식품영양학과 | 14 | |
| | 화학과 | 15 | |
| | 인공지능학과 | 46 | |
| | 의공학과 | 17 | |
| | 치위생학과 | 13 | |
| | 응급구조학과 | 10 | |
| | 물리치료학과 | 13 | |
| | 방사선학과 | 13 | |
| | 운동재활학과 | 14 | |
| | 바이오로직스학과 | 30 | |
------------------------------------------------------------


In [57]:
import statistics
q = "2027학년도 정시모집에서 메디컬캠퍼스(인천)에 개설된 바이오로직스학과의 모집인원은 몇 명인가요?"
times = []
for _ in range(3):
    out, r, s = agent_answer(q)
    times.append(s)
    print(f"{s:.2f}s r{r} | {out[:120]}")
print("중앙값", f"{statistics.median(times):.2f}s")

5.53s r1 | 바이오로직스학과의 정시 모집인원은 일반전형 1 30명, 일반전형 2 15명으로 총 45명입니다. (출처: 04_2027학년도_정시_모집요강.pdf p.7)
11.96s r1 | 정시모집에서 바이오로직스학과는 일반전형 1 30명, 일반전형 2 15명으로 총 45명입니다. (출처: 04_2027학년도_정시_모집요강.pdf p.7)
4.65s r1 | 2027학년도 정시모집에서 메디컬캠퍼스(인천)에 개설된 바이오로직스학과의 모집인원은 총 45명입니다(일반전형 1 30명, 일반전형 2 15명). (출처: 04_2027학년도_정시_모집요강.pdf p.7)
중앙값 5.53s


In [58]:
from langchain_openai import ChatOpenAI

for effort in ["minimal", "low"]:
    try:
        test_llm = ChatOpenAI(model=LLM_MODEL, reasoning_effort=effort, max_completion_tokens=500)
        ts = []
        for _ in range(3):
            t0 = time.perf_counter()
            r = test_llm.invoke([("system", SYSTEM_PROMPT),
                                 ("human", f"[Evidence]\n{format_context(rrf_retrieve(q))}\n\n[Question]\n{q}")])
            ts.append(time.perf_counter() - t0)
        print(f"{effort}: {[round(t,2) for t in ts]} | {r.content[:80]}")
    except Exception as e:
        print(f"{effort} 실패: {type(e).__name__} {str(e)[:160]}")

minimal 실패: OpenAIInvalidRequestError Error code: 400 - {'error': {'message': "Unsupported value: 'reasoning_effort' does not support 'minimal' with this model. Supported values are: 'none', 'low', 
low: [1.69, 1.63, 1.63] | 2027학년도 정시모집에서 메디컬캠퍼스(인천)에 개설된 바이오로직스학과의 모집인원은 일반전형 1 30명, 일반전형 2 15명으로 총 45명입니다


In [59]:
for effort in ["none"]:
    try:
        test_llm = ChatOpenAI(model=LLM_MODEL, reasoning_effort=effort, max_completion_tokens=500)
        ts = []
        for _ in range(3):
            t0 = time.perf_counter()
            r = test_llm.invoke([("system", SYSTEM_PROMPT),
                                 ("human", f"[Evidence]\n{format_context(rrf_retrieve(q))}\n\n[Question]\n{q}")])
            ts.append(time.perf_counter() - t0)
        print(f"{effort}: {[round(t,2) for t in ts]} | {r.content[:100]}")
    except Exception as e:
        print(f"{effort} 실패: {type(e).__name__} {str(e)[:200]}")

none: [1.76, 2.79, 2.03] | 바이오로직스학과의 정시모집 인원은 일반전형 1 가군 30명, 일반전형 2 가군 15명입니다. 합계 45명입니다. (출처: 04_2027학년도_정시_모집요강.pdf p.7)


In [60]:
llm = ChatOpenAI(model=LLM_MODEL, reasoning_effort="low", max_completion_tokens=500)
step_llm = llm.with_structured_output(AgentStep)
print("설정:", llm.model_name, "| reasoning_effort=low | max_completion_tokens=500")

설정: gpt-6-luna | reasoning_effort=low | max_completion_tokens=500


In [61]:
results_v5 = evaluate(agent_answer, testset, "v5-reasoning low")

 1. O  1.8s | 장애인 등록을 한 지원자가 입시전형 진행 중 지원이 필요하면 어떻게 해야 하나요?
 2. O  1.5s | 2027학년도 재외국민 모집요강에서 경영학과의 최대모집 가능인원과 모집인원은 각각 몇 명인가요?
 3. X  2.9s | 재외국민과 외국인(2% 이내) 전형에서 국외 재학기간 요건은 무엇이며, 1개 학년은 어떤 기준으로 
 4. O  1.7s | 재외국민 지원자격에서 국외 체류는 누구에게 요구되며, 전 교육과정 이수자의 경우에는 어떻게 적용되나
 5. O  1.6s | 대학은 학비 지원 신청을 1학기와 2학기 각각 언제까지 시·도지사에게 해야 하나요?
 6. O  1.4s | 강원 지역의 속초권에는 어떤 시·군이 포함되나요?
 7. X  2.5s | 지역의사선발전형에서 진료권·광역권 동시 지원이 가능한지는 어디서 확인해야 하며, 복수지원 금지 및 
 8. X  2.3s | 지역의사 지원센터가 제공하는 지원 종류에는 무엇이 있으며, 의무복무를 완료한 지역의사는 어떤 우대 
 9. O  1.8s | 가천대 길병원과 협력한 임상실무실습은 몇 개 내과 사이트에서 진행되며, 예시는 무엇인가요?
10. O  2.1s | 본 대학에 합격해 문서등록까지 했지만 등록을 포기하려는 경우, 어떤 절차로 취소해야 하며 신청 후 
11. O  1.8s | 제6항 해당자가 신분증 사본을 제출해야 하는 출생 기준일은 언제이며, 제출 서류는 언제 이후 발급된
12. O  1.6s | 교내 순환 Eco 버스 ‘무당이’는 언제 도입됐으며, 현재 몇 대가 운행되고 있나요?
13. O  1.7s | 2027학년도 정시모집 안내에서 원서접수 내용은 몇 페이지에 있나요?
14. X  2.1s | 2027학년도 정시모집에서 메디컬캠퍼스(인천)에 개설된 바이오로직스학과의 모집인원은 몇 명인가요?
15. X  1.6s | 2027학년도 정시 모집요강에서 농어촌 전형은 몇 페이지에 안내되어 있나요?
16. O  2.3s | [가군] 일반전형 1·2에 지원하려는 2027학년도 

In [62]:
# ===== v6. 정확도 우선: 초안 → 근거 대조 검증 → (필요시) 재검색 =====
llm = ChatOpenAI(model=LLM_MODEL)          # reasoning_effort 기본값 (정확도 우선)
step_llm = llm.with_structured_output(AgentStep)

DRAFT_PROMPT = """You are an admissions counselor at Gachon University. Answer using ONLY the evidence documents.

Rules:
1. Include every key fact the question asks for (numbers, dates, targets, conditions).
2. If the evidence lists several conditions or exceptions, list ALL of them.
3. Never add conditions or qualifiers that are not in the evidence.
4. Copy dates exactly as written (e.g., 2027.01.04.(월) ~ 07.(목) 18:00).
5. When reading a table, align the row and column headers correctly.
6. Write the answer in Korean, 2-5 sentences, and end with (출처: 파일명 p.페이지)."""


class Verify(BaseModel):
    missing_facts: list[str] = Field(default_factory=list,
                                     description="Key facts/conditions in the evidence that the draft omitted")
    needs_more_search: bool = Field(default=False,
                                    description="True if the evidence itself lacks information the question asks for")
    search_query: str = Field(default="", description="Korean keyword query for a second retrieval")
    final_answer: str = Field(description="Corrected final answer in Korean, including all required facts")


verify_llm = llm.with_structured_output(Verify)

VERIFY_PROMPT = """You are a strict fact-checker for Gachon University admissions answers.

Given the question, the evidence, and a draft answer:
1. Check every key fact the question asks for (numbers, dates, targets, conditions) against the evidence.
2. If the draft omitted any key fact or condition that IS in the evidence, list it in `missing_facts`
   and produce a corrected `final_answer` that includes it.
3. Never introduce facts that are not in the evidence. Do not add invented qualifiers.
4. If the evidence itself does not contain the needed information, set needs_more_search=True and give a
   short Korean `search_query`; otherwise set it False.
5. `final_answer` must be in Korean and end with (출처: 파일명 p.페이지)."""


def agent_answer_v6(question, k=8):
    t0 = time.perf_counter()
    pairs = rrf_retrieve(question, k=k)
    ctx = format_context(pairs)

    draft = llm.invoke([("system", DRAFT_PROMPT),
                        ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content

    v = verify_llm.invoke([("system", VERIFY_PROMPT),
                           ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}\n\n[Draft]\n{draft}")])

    if not v.needs_more_search:
        return v.final_answer, 1, time.perf_counter() - t0

    pairs2 = rrf_retrieve(v.search_query or question, k=k)
    seen, merged = set(), []
    for d, s in pairs + pairs2:
        if d.page_content not in seen:
            seen.add(d.page_content)
            merged.append((d, s))
    ctx2 = format_context(merged)

    draft2 = llm.invoke([("system", DRAFT_PROMPT),
                         ("human", f"[Evidence]\n{ctx2}\n\n[Question]\n{question}")]).content
    v2 = verify_llm.invoke([("system", VERIFY_PROMPT),
                            ("human", f"[Evidence]\n{ctx2}\n\n[Question]\n{question}\n\n[Draft]\n{draft2}")])
    return v2.final_answer, 2, time.perf_counter() - t0


for q_, gold_ in SAMPLES:
    out, r, s = agent_answer_v6(q_)
    print(f"{s:4.1f}s r{r} | {out[:150]}")
    print(f"        (정답: {gold_})")

 4.0s r1 | 가천대학교 대표 전화번호는 1577-0067입니다. (출처: 03_2027학년도_수시_모집요강.pdf p.104)
        (정답: 1577-0067)
 4.3s r1 | 정시 원서접수는 2027년 1월 4일(월)에 시작하며, 1월 7일(목) 오후 6시에 마감합니다. (출처: 04_2027학년도_정시_모집요강.pdf p.4)
        (정답: 2027년 1월 4일 ~ 1월 7일 18:00)
 3.8s r1 | 인터넷 원서접수용 증명사진은 최근 3개월 이내 사진이어야 하며, JPG 또는 GIF 파일 형식으로 준비해야 합니다. 인물 뒤에 배경이 있는 사진이나 스냅사진은 사용할 수 없습니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.5)
        (정답: 최근 3개월 이내, JPG/GIF)


In [63]:
results_v6 = evaluate(agent_answer_v6, testset, "v6-draft+verify")

 1. O  5.6s | 장애인 등록을 한 지원자가 입시전형 진행 중 지원이 필요하면 어떻게 해야 하나요?
 2. O  3.7s | 2027학년도 재외국민 모집요강에서 경영학과의 최대모집 가능인원과 모집인원은 각각 몇 명인가요?
 3. X  8.8s | 재외국민과 외국인(2% 이내) 전형에서 국외 재학기간 요건은 무엇이며, 1개 학년은 어떤 기준으로 
 4. O  5.4s | 재외국민 지원자격에서 국외 체류는 누구에게 요구되며, 전 교육과정 이수자의 경우에는 어떻게 적용되나
 5. O  4.5s | 대학은 학비 지원 신청을 1학기와 2학기 각각 언제까지 시·도지사에게 해야 하나요?
 6. O  4.0s | 강원 지역의 속초권에는 어떤 시·군이 포함되나요?
 7. O  6.9s | 지역의사선발전형에서 진료권·광역권 동시 지원이 가능한지는 어디서 확인해야 하며, 복수지원 금지 및 
 8. O  8.8s | 지역의사 지원센터가 제공하는 지원 종류에는 무엇이 있으며, 의무복무를 완료한 지역의사는 어떤 우대 
 9. O  4.6s | 가천대 길병원과 협력한 임상실무실습은 몇 개 내과 사이트에서 진행되며, 예시는 무엇인가요?
10. X 11.0s | 본 대학에 합격해 문서등록까지 했지만 등록을 포기하려는 경우, 어떤 절차로 취소해야 하며 신청 후 
11. O  5.2s | 제6항 해당자가 신분증 사본을 제출해야 하는 출생 기준일은 언제이며, 제출 서류는 언제 이후 발급된
12. O  3.9s | 교내 순환 Eco 버스 ‘무당이’는 언제 도입됐으며, 현재 몇 대가 운행되고 있나요?
13. O  4.3s | 2027학년도 정시모집 안내에서 원서접수 내용은 몇 페이지에 있나요?
14. X 22.6s | 2027학년도 정시모집에서 메디컬캠퍼스(인천)에 개설된 바이오로직스학과의 모집인원은 몇 명인가요?
15. X  4.3s | 2027학년도 정시 모집요강에서 농어촌 전형은 몇 페이지에 안내되어 있나요?
16. O  5.5s | [가군] 일반전형 1·2에 지원하려는 2027학년도 

In [64]:
# 실험 중에는 추론 예산을 낮게 유지 (파싱/색인 쪽 정확도에 집중)
llm = ChatOpenAI(model=LLM_MODEL, reasoning_effort="low", max_completion_tokens=600)
step_llm = llm.with_structured_output(AgentStep)
verify_llm = llm.with_structured_output(Verify)


def evaluate_batch(fn, dataset, start, end, tag=""):
    rows = []
    for i in range(start, end):
        t = dataset[i]
        out, rounds, sec = fn(t["q"])
        v = judge.invoke([
            ("system", JUDGE),
            ("human", f"[질문]\n{t['q']}\n\n[정답]\n{t['gold']}\n\n[예측]\n{out}"),
        ])
        rows.append({"q": t["q"], "gold": t["gold"], "pred": out, "ok": v.correct,
                     "why": v.reason, "sec": sec, "rounds": rounds})
        print(f"{i+1:2d}. {'O' if v.correct else 'X'} {sec:4.1f}s | {t['q'][:50]}")
    ok = sum(r["ok"] for r in rows)
    print(f"[{tag} {start+1}-{end}] {ok}/{len(rows)} | 평균 {sum(r['sec'] for r in rows)/len(rows):.2f}s | 최대 {max(r['sec'] for r in rows):.2f}s")
    return rows


print("reasoning_effort=low 로 실험 모드 설정")

reasoning_effort=low 로 실험 모드 설정


In [65]:
r6a = evaluate_batch(agent_answer_v6, testset, 0, 10, "v6")

 1. O  3.7s | 장애인 등록을 한 지원자가 입시전형 진행 중 지원이 필요하면 어떻게 해야 하나요?
 2. O  3.4s | 2027학년도 재외국민 모집요강에서 경영학과의 최대모집 가능인원과 모집인원은 각각 몇 명인
 3. O  7.1s | 재외국민과 외국인(2% 이내) 전형에서 국외 재학기간 요건은 무엇이며, 1개 학년은 어떤 
 4. O  3.6s | 재외국민 지원자격에서 국외 체류는 누구에게 요구되며, 전 교육과정 이수자의 경우에는 어떻게
 5. O  3.1s | 대학은 학비 지원 신청을 1학기와 2학기 각각 언제까지 시·도지사에게 해야 하나요?
 6. O  3.1s | 강원 지역의 속초권에는 어떤 시·군이 포함되나요?
 7. O  4.5s | 지역의사선발전형에서 진료권·광역권 동시 지원이 가능한지는 어디서 확인해야 하며, 복수지원 
 8. O  6.8s | 지역의사 지원센터가 제공하는 지원 종류에는 무엇이 있으며, 의무복무를 완료한 지역의사는 어
 9. O  3.6s | 가천대 길병원과 협력한 임상실무실습은 몇 개 내과 사이트에서 진행되며, 예시는 무엇인가요?
10. O  3.6s | 본 대학에 합격해 문서등록까지 했지만 등록을 포기하려는 경우, 어떤 절차로 취소해야 하며 
[v6 1-10] 10/10 | 평균 4.25s | 최대 7.12s


In [66]:
r6b = evaluate_batch(agent_answer_v6, testset, 10, 20, "v6")

11. O  3.4s | 제6항 해당자가 신분증 사본을 제출해야 하는 출생 기준일은 언제이며, 제출 서류는 언제 이
12. O  3.3s | 교내 순환 Eco 버스 ‘무당이’는 언제 도입됐으며, 현재 몇 대가 운행되고 있나요?
13. O  2.9s | 2027학년도 정시모집 안내에서 원서접수 내용은 몇 페이지에 있나요?
14. X  3.3s | 2027학년도 정시모집에서 메디컬캠퍼스(인천)에 개설된 바이오로직스학과의 모집인원은 몇 명
15. O  2.8s | 2027학년도 정시 모집요강에서 농어촌 전형은 몇 페이지에 안내되어 있나요?
16. O  3.5s | [가군] 일반전형 1·2에 지원하려는 2027학년도 수능 응시자가 고교 졸업(예정)자라면 
17. O  2.6s | 이 면접고사 일정 안내는 어느 대학의 몇 학년도 수시모집 중 어떤 전형을 대상으로 하나요?
18. O  3.2s | 기회균형·농어촌(종합)·특성화고교 전형의 면접고사는 며칠에 진행되나요?
19. O  5.2s | 지역균형 전형 면접 일정은 원서접수순으로 배정되나요? 학생 개인 사정으로 면접일을 변경할 
20. O  4.1s | 가천바람개비 전형에서 간호학과 면접고사는 총 몇 개의 날짜·시간대에 편성되어 있나요?
[v6 11-20] 9/10 | 평균 3.43s | 최대 5.15s


In [67]:
!pip install -qU voyageai langchain-voyageai
import os


def _load_key(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


VOYAGE_KEY = _load_key("VOYAGEAI_API_KEY") or _load_key("VOYAGE_API_KEY")
print("VOYAGEAI_API_KEY:", "설정됨 ✅" if VOYAGE_KEY else "없음 ❌ → Colab 🔑(Secrets)에 VOYAGEAI_API_KEY 추가")

VOYAGEAI_API_KEY: 설정됨 ✅


In [68]:
import voyageai

vo = voyageai.Client(api_key=VOYAGE_KEY)

for m in ["voyage-3.5", "voyage-3-large", "voyage-3.5-lite"]:
    try:
        r = vo.embed(["가천대학교 정시 원서접수 기간"], model=m, input_type="query")
        print(f"{m}: OK dim={len(r.embeddings[0])}")
    except Exception as e:
        print(f"{m}: 실패 {type(e).__name__} {str(e)[:100]}")

for m in ["rerank-2.5", "rerank-2.5-lite"]:
    try:
        r = vo.rerank("정시 원서접수 기간", ["2027.01.04.(월) ~ 07.(목) 18:00", "수시 원서접수는 9월"], model=m, top_k=1)
        print(f"{m}: OK score={r.results[0].relevance_score:.3f}")
    except Exception as e:
        print(f"{m}: 실패 {type(e).__name__} {str(e)[:100]}")

voyage-3.5: OK dim=1024
voyage-3-large: OK dim=1024
voyage-3.5-lite: OK dim=1024
rerank-2.5: OK score=0.574
rerank-2.5-lite: OK score=0.561


In [69]:
# ===== v7. VoyageAI 임베딩 + 리랭커 =====
from langchain_voyageai import VoyageAIEmbeddings, VoyageAIRerank

import inspect
print(inspect.signature(VoyageAIEmbeddings.__init__).parameters.keys())
print(inspect.signature(VoyageAIRerank.__init__).parameters.keys())

odict_keys(['self', 'data'])
odict_keys(['self', 'data'])


In [70]:
import time as _t

vemb = VoyageAIEmbeddings(model="voyage-3.5", voyage_api_key=VOYAGE_KEY)
vdb = Chroma.from_documents(
    documents=chunks,
    embedding=vemb,
    collection_name=f"gachon_v_{int(_t.time())}",
    collection_metadata={"hnsw:space": "cosine"},
)
vret = vdb.as_retriever(search_kwargs={"k": 20})

reranker = VoyageAIRerank(model="rerank-2.5", top_k=6, voyage_api_key=VOYAGE_KEY)
bm25_retriever.k = 20

print("Voyage 인덱스:", vdb._collection.count())


def retrieve_v7(query, k_rrf=20, top_n=6, expand=2):
    ranked = [bm25_retriever.invoke(query), vret.invoke(query)]
    scores, docs = {}, {}
    for result in ranked:
        for rank, doc in enumerate(result, start=1):
            key = doc.page_content
            scores[key] = scores.get(key, 0.0) + 1.0 / (60 + rank)
            docs[key] = doc
    cand = [docs[k] for k in sorted(scores, key=scores.get, reverse=True)[:k_rrf]]

    top = reranker.compress_documents(cand, query)[:top_n]

    out, seen = [], set()
    for d in top[:expand]:
        i = CHUNK_INDEX.get(d.page_content)
        cands = [d]
        if i is not None:
            for j in range(max(0, i - 1), min(len(chunks), i + 2)):
                if chunks[j].metadata["source"] == d.metadata["source"]:
                    cands.append(chunks[j])
        for c in cands:
            if c.page_content not in seen:
                seen.add(c.page_content)
                out.append(c)
    for d in top[expand:]:
        if d.page_content not in seen:
            seen.add(d.page_content)
            out.append(d)
    return out


for q_ in [s[0] for s in SAMPLES]:
    t0 = time.perf_counter()
    docs = retrieve_v7(q_)
    print(f"\nQ: {q_[:50]}  ({time.perf_counter()-t0:.2f}s, {len(docs)}개)")
    for d in docs[:3]:
        print("   ", d.metadata["source"], "p.", d.metadata["page"])

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Voyage 인덱스: 514

Q: 가천대학교 대표 전화번호는 무엇인가요?  (0.22s, 9개)
    04_2027학년도_정시_모집요강.pdf p. 44
    04_2027학년도_정시_모집요강.pdf p. 43
    04_2027학년도_정시_모집요강.pdf p. 3

Q: 정시 원서접수는 몇 월 며칠부터 시작하고, 마감일과 마감시각은 언제인가요?  (0.33s, 10개)
    04_2027학년도_정시_모집요강.pdf p. 4
    04_2027학년도_정시_모집요강.pdf p. 3
    04_2027학년도_정시_모집요강.pdf p. 5

Q: 2027학년도 재외국민 전형의 인터넷 원서접수에 사용할 증명사진은 최근 몇 개월 이내 사진  (0.36s, 10개)
    01_2027학년도_재외국민_모집요강.hwp p. 5
    01_2027학년도_재외국민_모집요강.hwp p. 4
    01_2027학년도_재외국민_모집요강.hwp p. 5


In [71]:
tests = ["voyage-4-large", "voyage-4", "voyage-context-4", "voyage-context-4-large"]
for m in tests:
    try:
        r = vo.embed(["테스트 문장"], model=m, input_type="query")
        print(f"embed {m}: OK dim={len(r.embeddings[0])}")
    except Exception as e:
        print(f"embed {m}: X {str(e)[:90]}")

for m in ["voyage-context-4", "voyage-context-4-large"]:
    try:
        r = vo.contextualized_embed(inputs=[["첫 문단", "둘째 문단"]], model=m, input_type="document")
        print(f"context {m}: OK dim={len(r.results[0].embeddings[0])}")
    except Exception as e:
        print(f"context {m}: X {str(e)[:90]}")

for m in ["rerank-3", "rerank-3.5", "rerank-4"]:
    try:
        r = vo.rerank("질문", ["문서1", "문서2"], model=m)
        print(f"rerank {m}: OK")
    except Exception as e:
        print(f"rerank {m}: X {str(e)[:90]}")

embed voyage-4-large: OK dim=1024
embed voyage-4: OK dim=1024
embed voyage-context-4: X Model voyage-context-4 is not supported. Supported models are ['voyage-4-large', 'voyage-4
embed voyage-context-4-large: X Model voyage-context-4-large is not supported. Supported models are ['voyage-4-large', 'vo
context voyage-context-4: OK dim=1024
context voyage-context-4-large: X Model voyage-context-4-large is not supported. Supported models are ['voyage-context-3', '
rerank rerank-3: OK
rerank rerank-3.5: X Model rerank-3.5 is not supported. Supported models are ['rerank-lite-1', 'rerank-2-lite',
rerank rerank-4: X Model rerank-4 is not supported. Supported models are ['rerank-3', 'rerank-3-lite', 'reran


In [72]:
# ===== A) voyage-4-large 인덱스 =====
import time as _t

v4emb = VoyageAIEmbeddings(model="voyage-4-large", voyage_api_key=VOYAGE_KEY)
v4db = Chroma.from_documents(
    documents=chunks,
    embedding=v4emb,
    collection_name=f"gachon_v4_{int(_t.time())}",
    collection_metadata={"hnsw:space": "cosine"},
)
v4ret = v4db.as_retriever(search_kwargs={"k": 20})
print("voyage-4-large 인덱스:", v4db._collection.count())
print("쿼리 임베딩 테스트:", len(v4emb.embed_query("정시 원서접수 기간")))

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

voyage-4-large 인덱스: 514
쿼리 임베딩 테스트: 1024


In [73]:
# ===== B) voyage-context-4 인덱스 (문서 문맥을 반영한 청크 임베딩) =====
import chromadb, math

cc = chromadb.Client()
ctx_col = cc.create_collection("gachon_ctx4", metadata={"hnsw:space": "cosine"})

by_src_idx = {}
for i, c in enumerate(chunks):
    by_src_idx.setdefault(c.metadata["source"], []).append(i)

BATCH = 12
added = 0
for src, idxs in by_src_idx.items():
    for s in range(0, len(idxs), BATCH):
        group = idxs[s:s + BATCH]
        texts = [chunks[i].page_content for i in group]
        res = vo.contextualized_embed(inputs=[texts], model="voyage-context-4", input_type="document")
        ctx_col.add(
            ids=[str(i) for i in group],
            embeddings=res.results[0].embeddings,
            documents=texts,
            metadatas=[chunks[i].metadata for i in group],
        )
        added += len(group)

print("voyage-context-4 인덱스:", added)


def ctx_query_embed(query):
    r = vo.contextualized_embed(inputs=[[query]], model="voyage-context-4", input_type="query")
    return r.results[0].embeddings[0]


res = ctx_col.query(query_embeddings=[ctx_query_embed("정시 원서접수 기간")], n_results=3)
for d, m in zip(res["documents"][0], res["metadatas"][0]):
    print(" -", m["source"], "p.", m["page"], "|", d.splitlines()[1][:60] if len(d.splitlines()) > 1 else "")

voyage-context-4 인덱스: 514
 - 04_2027학년도_정시_모집요강.pdf p. 8 | 34
 - 04_2027학년도_정시_모집요강.pdf p. 8 | | 모집시기 | 접수기간 | 인터넷 접수 |
 - 04_2027학년도_정시_모집요강.pdf p. 4 | 30


In [74]:
# ===== C) 두 인덱스용 검색·답변 파이프라인 =====
def _rrf(lists, k=20, rrf_k=60):
    scores, docs = {}, {}
    for result in lists:
        for rank, d in enumerate(result, start=1):
            key = d.page_content
            scores[key] = scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
            docs[key] = d
    return [docs[key] for key in sorted(scores, key=scores.get, reverse=True)[:k]]


def _rerank(query, docs, top_n=6):
    r = vo.rerank(query, [d.page_content for d in docs], model="rerank-3", top_k=top_n)
    return [docs[x.index] for x in r.results]


def _expand(docs, span=1, top=2):
    out, seen = [], set()
    for n, d in enumerate(docs):
        cand = [d]
        if n < top:
            i = CHUNK_INDEX.get(d.page_content)
            if i is not None:
                for j in range(max(0, i - span), min(len(chunks), i + span + 1)):
                    if chunks[j].metadata["source"] == d.metadata["source"]:
                        cand.append(chunks[j])
        for c in cand:
            if c.page_content not in seen:
                seen.add(c.page_content)
                out.append(c)
    return out


def retrieve_large4(query, top_n=6, k_rrf=20):
    return _expand(_rerank(query, _rrf([bm25_retriever.invoke(query), v4ret.invoke(query)], k_rrf), top_n))


def _ctx_dense(query, k=20):
    res = ctx_col.query(query_embeddings=[ctx_query_embed(query)], n_results=k)
    return [Document(page_content=t, metadata=m) for t, m in zip(res["documents"][0], res["metadatas"][0])]


def retrieve_ctx4(query, top_n=6, k_rrf=20):
    return _expand(_rerank(query, _rrf([bm25_retriever.invoke(query), _ctx_dense(query)], k_rrf), top_n))


def ctx_of(docs):
    return "\n\n".join(f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.page_content}"
                       for i, d in enumerate(docs, 1))


def build_answer_fn(retrieve_fn):
    def fn(question, top_n=6):
        t0 = time.perf_counter()
        docs = retrieve_fn(question, top_n)
        ctx = ctx_of(docs)
        draft = llm.invoke([("system", DRAFT_PROMPT),
                            ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content
        v = verify_llm.invoke([("system", VERIFY_PROMPT),
                               ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}\n\n[Draft]\n{draft}")])
        if not v.needs_more_search:
            return v.final_answer, 1, time.perf_counter() - t0

        docs2 = retrieve_fn(v.search_query or question, top_n)
        seen, merged = set(), []
        for d in docs + docs2:
            if d.page_content not in seen:
                seen.add(d.page_content)
                merged.append(d)
        ctx2 = ctx_of(merged)
        draft2 = llm.invoke([("system", DRAFT_PROMPT),
                             ("human", f"[Evidence]\n{ctx2}\n\n[Question]\n{question}")]).content
        v2 = verify_llm.invoke([("system", VERIFY_PROMPT),
                                ("human", f"[Evidence]\n{ctx2}\n\n[Question]\n{question}\n\n[Draft]\n{draft2}")])
        return v2.final_answer, 2, time.perf_counter() - t0
    return fn


answer_large4 = build_answer_fn(retrieve_large4)
answer_ctx4 = build_answer_fn(retrieve_ctx4)

for name, fn in [("large4", retrieve_large4), ("ctx4", retrieve_ctx4)]:
    t0 = time.perf_counter()
    ds = fn(SAMPLES[0][0])
    print(name, f"{time.perf_counter()-t0:.2f}s", [f"{d.metadata['source'][:2]}p{d.metadata['page']}" for d in ds[:4]])

large4 0.28s ['03p104', '03p103', '03p29', '04p44']
ctx4 0.27s ['03p104', '03p103', '03p29', '04p44']


In [75]:
r_l4_a = evaluate_batch(answer_large4, testset, 0, 10, "voyage-4-large")

 1. O  4.6s | 장애인 등록을 한 지원자가 입시전형 진행 중 지원이 필요하면 어떻게 해야 하나요?
 2. O  2.7s | 2027학년도 재외국민 모집요강에서 경영학과의 최대모집 가능인원과 모집인원은 각각 몇 명인
 3. O  7.0s | 재외국민과 외국인(2% 이내) 전형에서 국외 재학기간 요건은 무엇이며, 1개 학년은 어떤 
 4. O  3.4s | 재외국민 지원자격에서 국외 체류는 누구에게 요구되며, 전 교육과정 이수자의 경우에는 어떻게
 5. O  3.5s | 대학은 학비 지원 신청을 1학기와 2학기 각각 언제까지 시·도지사에게 해야 하나요?
 6. O  3.0s | 강원 지역의 속초권에는 어떤 시·군이 포함되나요?
 7. O  5.6s | 지역의사선발전형에서 진료권·광역권 동시 지원이 가능한지는 어디서 확인해야 하며, 복수지원 
 8. O  7.5s | 지역의사 지원센터가 제공하는 지원 종류에는 무엇이 있으며, 의무복무를 완료한 지역의사는 어
 9. O  3.4s | 가천대 길병원과 협력한 임상실무실습은 몇 개 내과 사이트에서 진행되며, 예시는 무엇인가요?
10. O  3.5s | 본 대학에 합격해 문서등록까지 했지만 등록을 포기하려는 경우, 어떤 절차로 취소해야 하며 
[voyage-4-large 1-10] 10/10 | 평균 4.43s | 최대 7.51s


In [76]:
r_ctx_a = evaluate_batch(answer_ctx4, testset, 0, 10, "voyage-context-4")

 1. O  3.4s | 장애인 등록을 한 지원자가 입시전형 진행 중 지원이 필요하면 어떻게 해야 하나요?
 2. O  2.6s | 2027학년도 재외국민 모집요강에서 경영학과의 최대모집 가능인원과 모집인원은 각각 몇 명인


LengthFinishReasonError: Could not parse response content as the length limit was reached - CompletionUsage(completion_tokens=600, prompt_tokens=8657, total_tokens=9257, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=600, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=8654, cached_tokens=0))

In [77]:
llm = ChatOpenAI(model=LLM_MODEL, reasoning_effort="low", max_completion_tokens=2000)
step_llm = llm.with_structured_output(AgentStep)
verify_llm = llm.with_structured_output(Verify)


def _safe_verify(ctx, question, draft):
    try:
        return verify_llm.invoke([("system", VERIFY_PROMPT),
                                  ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}\n\n[Draft]\n{draft}")])
    except Exception as e:
        print("   (verify 폴백:", type(e).__name__, ")")
        return Verify(missing_facts=[], needs_more_search=False, search_query="", final_answer=draft)


def build_answer_fn(retrieve_fn):
    def fn(question, top_n=6):
        t0 = time.perf_counter()
        docs = retrieve_fn(question, top_n)
        ctx = ctx_of(docs)
        draft = llm.invoke([("system", DRAFT_PROMPT),
                            ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content
        v = _safe_verify(ctx, question, draft)
        if not v.needs_more_search:
            return v.final_answer, 1, time.perf_counter() - t0

        docs2 = retrieve_fn(v.search_query or question, top_n)
        seen, merged = set(), []
        for d in docs + docs2:
            if d.page_content not in seen:
                seen.add(d.page_content)
                merged.append(d)
        ctx2 = ctx_of(merged)
        draft2 = llm.invoke([("system", DRAFT_PROMPT),
                             ("human", f"[Evidence]\n{ctx2}\n\n[Question]\n{question}")]).content
        v2 = _safe_verify(ctx2, question, draft2)
        return v2.final_answer, 2, time.perf_counter() - t0
    return fn


answer_large4 = build_answer_fn(retrieve_large4)
answer_ctx4 = build_answer_fn(retrieve_ctx4)
print("토큰 상한 2000 + 폴백 적용")

토큰 상한 2000 + 폴백 적용


In [78]:
r_ctx_a = evaluate_batch(answer_ctx4, testset, 0, 10, "voyage-context-4")

 1. O  5.5s | 장애인 등록을 한 지원자가 입시전형 진행 중 지원이 필요하면 어떻게 해야 하나요?
 2. O  2.7s | 2027학년도 재외국민 모집요강에서 경영학과의 최대모집 가능인원과 모집인원은 각각 몇 명인
 3. O  5.9s | 재외국민과 외국인(2% 이내) 전형에서 국외 재학기간 요건은 무엇이며, 1개 학년은 어떤 
 4. O  3.2s | 재외국민 지원자격에서 국외 체류는 누구에게 요구되며, 전 교육과정 이수자의 경우에는 어떻게
 5. O  3.3s | 대학은 학비 지원 신청을 1학기와 2학기 각각 언제까지 시·도지사에게 해야 하나요?
 6. O  2.7s | 강원 지역의 속초권에는 어떤 시·군이 포함되나요?
 7. O  3.5s | 지역의사선발전형에서 진료권·광역권 동시 지원이 가능한지는 어디서 확인해야 하며, 복수지원 
 8. O  6.2s | 지역의사 지원센터가 제공하는 지원 종류에는 무엇이 있으며, 의무복무를 완료한 지역의사는 어
 9. O  3.2s | 가천대 길병원과 협력한 임상실무실습은 몇 개 내과 사이트에서 진행되며, 예시는 무엇인가요?
10. O  3.3s | 본 대학에 합격해 문서등록까지 했지만 등록을 포기하려는 경우, 어떤 절차로 취소해야 하며 
[voyage-context-4 1-10] 10/10 | 평균 3.94s | 최대 6.19s


In [79]:
r_l4_b = evaluate_batch(answer_large4, testset, 10, 20, "voyage-4-large")

11. O  3.2s | 제6항 해당자가 신분증 사본을 제출해야 하는 출생 기준일은 언제이며, 제출 서류는 언제 이
12. O  2.8s | 교내 순환 Eco 버스 ‘무당이’는 언제 도입됐으며, 현재 몇 대가 운행되고 있나요?
13. O  2.9s | 2027학년도 정시모집 안내에서 원서접수 내용은 몇 페이지에 있나요?
14. X  8.9s | 2027학년도 정시모집에서 메디컬캠퍼스(인천)에 개설된 바이오로직스학과의 모집인원은 몇 명
15. O  2.9s | 2027학년도 정시 모집요강에서 농어촌 전형은 몇 페이지에 안내되어 있나요?
16. O  4.1s | [가군] 일반전형 1·2에 지원하려는 2027학년도 수능 응시자가 고교 졸업(예정)자라면 
17. O  3.1s | 이 면접고사 일정 안내는 어느 대학의 몇 학년도 수시모집 중 어떤 전형을 대상으로 하나요?
18. O  3.8s | 기회균형·농어촌(종합)·특성화고교 전형의 면접고사는 며칠에 진행되나요?
19. O  2.9s | 지역균형 전형 면접 일정은 원서접수순으로 배정되나요? 학생 개인 사정으로 면접일을 변경할 
20. O  3.7s | 가천바람개비 전형에서 간호학과 면접고사는 총 몇 개의 날짜·시간대에 편성되어 있나요?
[voyage-4-large 11-20] 9/10 | 평균 3.84s | 최대 8.88s


In [80]:
r_ctx_b = evaluate_batch(answer_ctx4, testset, 10, 20, "voyage-context-4")

11. O  4.1s | 제6항 해당자가 신분증 사본을 제출해야 하는 출생 기준일은 언제이며, 제출 서류는 언제 이
12. O  2.8s | 교내 순환 Eco 버스 ‘무당이’는 언제 도입됐으며, 현재 몇 대가 운행되고 있나요?
13. O  3.1s | 2027학년도 정시모집 안내에서 원서접수 내용은 몇 페이지에 있나요?
14. X  4.8s | 2027학년도 정시모집에서 메디컬캠퍼스(인천)에 개설된 바이오로직스학과의 모집인원은 몇 명
15. X  3.3s | 2027학년도 정시 모집요강에서 농어촌 전형은 몇 페이지에 안내되어 있나요?
16. O  3.9s | [가군] 일반전형 1·2에 지원하려는 2027학년도 수능 응시자가 고교 졸업(예정)자라면 
17. O  3.0s | 이 면접고사 일정 안내는 어느 대학의 몇 학년도 수시모집 중 어떤 전형을 대상으로 하나요?
18. O  3.0s | 기회균형·농어촌(종합)·특성화고교 전형의 면접고사는 며칠에 진행되나요?
19. O  3.2s | 지역균형 전형 면접 일정은 원서접수순으로 배정되나요? 학생 개인 사정으로 면접일을 변경할 
20. X  3.9s | 가천바람개비 전형에서 간호학과 면접고사는 총 몇 개의 날짜·시간대에 편성되어 있나요?
[voyage-context-4 11-20] 7/10 | 평균 3.49s | 최대 4.79s


In [40]:
results_v3 = evaluate(agent_answer_v2, testset, "v3-English prompts")

 1. O  2.3s | 장애인 등록을 한 지원자가 입시전형 진행 중 지원이 필요하면 어떻게 해야 하나요?
 2. O  1.6s | 2027학년도 재외국민 모집요강에서 경영학과의 최대모집 가능인원과 모집인원은 각각 몇 명인가요?
 3. X  4.5s | 재외국민과 외국인(2% 이내) 전형에서 국외 재학기간 요건은 무엇이며, 1개 학년은 어떤 기준으로 
 4. O  2.3s | 재외국민 지원자격에서 국외 체류는 누구에게 요구되며, 전 교육과정 이수자의 경우에는 어떻게 적용되나
 5. O  2.1s | 대학은 학비 지원 신청을 1학기와 2학기 각각 언제까지 시·도지사에게 해야 하나요?
 6. O  1.8s | 강원 지역의 속초권에는 어떤 시·군이 포함되나요?
 7. X  2.6s | 지역의사선발전형에서 진료권·광역권 동시 지원이 가능한지는 어디서 확인해야 하며, 복수지원 금지 및 
 8. X  2.5s | 지역의사 지원센터가 제공하는 지원 종류에는 무엇이 있으며, 의무복무를 완료한 지역의사는 어떤 우대 
 9. O  2.2s | 가천대 길병원과 협력한 임상실무실습은 몇 개 내과 사이트에서 진행되며, 예시는 무엇인가요?
10. O  4.2s | 본 대학에 합격해 문서등록까지 했지만 등록을 포기하려는 경우, 어떤 절차로 취소해야 하며 신청 후 
11. O  2.3s | 제6항 해당자가 신분증 사본을 제출해야 하는 출생 기준일은 언제이며, 제출 서류는 언제 이후 발급된
12. O  2.0s | 교내 순환 Eco 버스 ‘무당이’는 언제 도입됐으며, 현재 몇 대가 운행되고 있나요?
13. O  2.9s | 2027학년도 정시모집 안내에서 원서접수 내용은 몇 페이지에 있나요?
14. X  5.2s | 2027학년도 정시모집에서 메디컬캠퍼스(인천)에 개설된 바이오로직스학과의 모집인원은 몇 명인가요?
15. X  2.2s | 2027학년도 정시 모집요강에서 농어촌 전형은 몇 페이지에 안내되어 있나요?
16. O  3.3s | [가군] 일반전형 1·2에 지원하려는 2027학년도 

In [48]:
from kiwipiepy import Kiwi
from langchain_community.retrievers import BM25Retriever

kiwi = Kiwi()
ALLOWED = {"NNG", "NNP", "NNB", "VV", "VA", "XR", "SL", "SN"}


def korean_tokenizer(text):
    return [t.form for t in kiwi.tokenize(text) if t.tag in ALLOWED]


bm25_retriever = BM25Retriever.from_documents(chunks, preprocess_func=korean_tokenizer)
bm25_retriever.k = 8


def rrf_retrieve(query: str, k: int = 6, rrf_k: int = 60):
    """BM25 + 벡터 검색 결과를 RRF로 융합"""
    ranked = [bm25_retriever.invoke(query), vector_retriever.invoke(query)]
    scores, docs = {}, {}
    for result in ranked:
        for rank, doc in enumerate(result, start=1):
            key = doc.page_content
            scores[key] = scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
            docs[key] = doc
    ordered = sorted(scores, key=scores.get, reverse=True)[:k]
    return [(docs[key], scores[key]) for key in ordered]


for q in [
    "가천대학교 대표 전화번호는 무엇인가요?",
    "정시 원서접수는 몇 월 며칠부터 시작하고, 마감일과 마감시각은 언제인가요?",
    "2027학년도 재외국민 전형의 인터넷 원서접수에 사용할 증명사진은 최근 몇 개월 이내 사진이어야 하며, 어떤 파일 형식으로 준비해야 하나요?",
]:
    print("Q:", q)
    for doc, s in rrf_retrieve(q, k=3):
        print(f"  {s:.4f} | {doc.metadata['source']} p.{doc.metadata['page']}")
        print("    " + doc.page_content.replace("\n", " ")[:180])
    print()

Q: 가천대학교 대표 전화번호는 무엇인가요?
  0.0164 | 01_2027학년도_재외국민_모집요강.hwp p.16
    [01_2027학년도_재외국민_모집요강.hwp p.16] | 등록포기 및 환불 | 등록포기 및 환불 | 등록포기 및 환불 | |---|---|---| | 등록포기 및 환불 | 재외국민 전형 합격자가 충원합격자 발표기간 중 다른 대학의 충원합격 통보를 받은 경우 차순위 수험생의 합격기회 박탈 등 선의의 피해자 발생을 사전에
  0.0164 | 03_2027학년도_수시_모집요강.pdf p.104
    [03_2027학년도_수시_모집요강.pdf p.104] 가천대학교 13120 경기도 성남시 수정구 성남대로 1342 Tel. 1577-0067 메디컬캠퍼스 21936 인천광역시 연수구 함박뫼로 191 강화캠퍼스 23051 인천광역시 강화군 길상면 해안남로 646-1 하와이가천글로벌센터 315 Saratoga Road, H
  0.0161 | 03_2027학년도_수시_모집요강.pdf p.7
    [03_2027학년도_수시_모집요강.pdf p.7] 명의 학생이 3주에서 최장 15주까지 머물며 영어 공부와 현지 문화체험을 하고 있다. 하와이가천글로벌센터 기숙형 글로벌 어학센터 경험중심 교육혁신 구현을 위해 국내 최초로 프 로젝트 수업 운영이 가능한 ‘P(프로젝트)–유연학 기제’를 시행했다. 3학년 2학기를 P–학기로

Q: 정시 원서접수는 몇 월 며칠부터 시작하고, 마감일과 마감시각은 언제인가요?
  0.0313 | 04_2027학년도_정시_모집요강.pdf p.4
    [04_2027학년도_정시_모집요강.pdf p.4] 30 2027학년도 정시모집요강 전형일정 구분 일시 비고 원서접수 2027.01.04.(월) ~ 07 .(목) 18:00 서류제출 마감 2027.01.08.(금) 13:00까지 ·원서접수 사이트에서 제출 실기고사 가군 연기예술학과(연출) 고사장 확인 2027.01.1 1
  0.0294 | 03_2027학년도_수시_모집요강.pd

In [81]:
import re
hwp_chunks = [c for c in chunks if c.metadata["source"].startswith("01")]
ph = [c for c in hwp_chunks if "[merged:" in c.page_content]
print("placeholder 포함 청크:", len(ph), "/", len(hwp_chunks))
for c in ph[:2]:
    print("-" * 70)
    print(c.metadata["page"], c.page_content[:700])

placeholder 포함 청크: 3 / 38
----------------------------------------------------------------------
8 [01_2027학년도_재외국민_모집요강.hwp p.8]
| 전형방법 | 전형방법 | 전형방법 |
|---|---|---|
| 전형방법 | 1. 전체 모집단위 | 1. 전체 모집단위 |
|  | 1. 전체 모집단위 | 1. 전체 모집단위 |
|  | 모집단위 전형방법 서류평가 면접평가 비고 전체 모집단위 일괄합산 50% 50% 대면 면접 |  |
|  | ※ 학교폭력예방 및 대책에 관한 법률 제17조에 따른 처분을 받은 사람은 서류평가 및 면접평가에서 정성평가로 반영합니다. |  |
|  | 2. 의예과, 약학과 | 2. 의예과, 약학과 |
|  | 모집단위 구분 서류평가 면접평가 비고 의예과, 약학과 [rowspan=2, colspan=1] 1단계 100% 모집인원의 7배수 선발 [merged:R2C1] 2단계 1단계 평가 50% 50% 대면 면접 |  |
|  | ※ 학교폭력예방 및 대책에 관한 법률 제17조에 따른 처분을 받은 사람은 1단계 서류평가 및 2단계 면접평가에서 정성평가로 반영합니다. |  |
| 서류평가 | 구분 평가내용 비고 평가 자료 학교생활기록부 및 학교생활기록부 대체서식 평가 영역 [rowspan=3, colspan=1] ◉ 인성 (공동체역량, 나눔과 배려, 성실성) [rowspan=3, colspan=1] [merged:R3C1] ◉ 진학의지 (교과역량(계열적합성), 학습경험(계열적합성), 능동적 학업태도, 
----------------------------------------------------------------------
8 [01_2027학년도_재외국민_모집요강.hwp p.8]
| 전형방법 | 전형방법 | 전형방법 |
|---|---|---|
| 면접평가 | 구분 평가내용 [rowspan=1, colspan=2] [merged:R1C2] 비고 평가

In [82]:
els = HwpHwpxLoader(str(DATA / "01_2027학년도_재외국민_모집요강.hwp"),
                    mode="elements", distribution_backend="pyhwp").load()
tabs = [e for e in els if e.metadata.get("element_type") == "table"]
print("표 요소:", len(tabs))
cells = json.loads(tabs[3].metadata["cells"])
print("cell keys:", cells[0].keys())
print(json.dumps(cells[0], ensure_ascii=False)[:400])
print("\n표 page_content 예시:")
print(tabs[3].page_content[:600])

표 요소: 18
cell keys: dict_keys(['row', 'col', 'rowspan', 'colspan', 'text'])
{"row": 0, "col": 0, "rowspan": 1, "colspan": 6, "text": "원서접수"}

표 page_content 예시:
| 원서접수 [rowspan=1, colspan=6] | [merged:R1C1] | [merged:R1C1] | [merged:R1C1] | [merged:R1C1] | [merged:R1C1] |
| --- | --- | --- | --- | --- | --- |
| [rowspan=1, colspan=6] | [merged:R2C1] | [merged:R2C1] | [merged:R2C1] | [merged:R2C1] | [merged:R2C1] |
| 원서접수 기간 |  | 구분 | 접수기간 | 장소 |  |
| [rowspan=7, colspan=1] |  | 원서접수 | 2026. 07. 06(월) 10:00 ~ 10(금) 16:00 [rowspan=2, colspan=1] | 본 대학 입학처 홈페이지 |  |
| [merged:R4C1] |  | 서류제출 | [merged:R4C4] | 온라인, 우편 가능 |  |
| [merged:R4C1] |  |  |  |  |  |
| [merged:R4C1] | [rowspan=2, colspan=1] | 온라인 : 원서접수 사이트를 통해 서류 업로드 [rowspan=2, colspan=3] | [mer


In [83]:
for i, t in enumerate(testset, 1):
    print(f"{i}. Q: {t['q']}")
    print(f"   G: {t['gold']}")
    print(f"   S: {t['src']}")

1. Q: 장애인 등록을 한 지원자가 입시전형 진행 중 지원이 필요하면 어떻게 해야 하나요?
   G: 장애인복지법 제32조에 따라 장애인등록을 마쳤고 장애 또는 지체로 전형 진행 과정에서 지원이 필요한 경우, 사전에 요청해야 합니다.
   S: 01_2027학년도_재외국민_모집요강.hwp p.5
2. Q: 2027학년도 재외국민 모집요강에서 경영학과의 최대모집 가능인원과 모집인원은 각각 몇 명인가요?
   G: 경영학과의 최대모집 가능인원은 15명이고, 모집인원은 16명입니다.
   S: 01_2027학년도_재외국민_모집요강.hwp p.3
3. Q: 재외국민과 외국인(2% 이내) 전형에서 국외 재학기간 요건은 무엇이며, 1개 학년은 어떤 기준으로 인정되나요?
   G: 고교과정 1개 학년을 포함해 중·고교과정에서 통산 3년(3개 학년 이상) 재학해야 하며, 성적이 나온 기간만 인정됩니다. 일반적으로 학기 개시일부터 다음 학년도 같은 학기 개시일 전일까지를 1개 학년으로 보지만, 해당 기간의 모든 학기를 이수한 경우에는 다음 학년도 같은 학기의 직전 학기 종료일까지로도 1개 학년을 충족한 것으로 간주합니다. 중간에 편입학했다면 편입학일부터 역년으로 1년(365일)이 되는 날까지를 1개 학년으로 봅니다. 또한 국외 재학기간뿐 아니라 체류기간과 국외근무자 재직기간도 모두 충족해야 합니다.
   S: 01_2027학년도_재외국민_모집요강.hwp p.10
4. Q: 재외국민 지원자격에서 국외 체류는 누구에게 요구되며, 전 교육과정 이수자의 경우에는 어떻게 적용되나요?
   G: 일반적으로 부모와 학생 모두 국외에 체류해야 합니다. 다만 전 교육과정 이수자는 학생만 국외 체류하면 됩니다.
   S: 01_2027학년도_재외국민_모집요강.hwp p.10
5. Q: 대학은 학비 지원 신청을 1학기와 2학기 각각 언제까지 시·도지사에게 해야 하나요?
   G: 1학기는 2월 14일까지, 2학기는 8월 14일까지 신청해야 합니다.
   S: 02_2027학년도_지역의사선발전형

In [85]:
# ===== 평가셋 확장: 구어체/애매한 표현/용어 오용 20문항 =====
testset_hard = [
    {"q": "몸이 좀 불편한 학생인데요, 시험 볼 때 도와주는 사람 받을 수 있나요? 있으면 어떻게 해야 해요?",
     "gold": "장애인복지법 제32조에 따라 장애인등록을 마쳤고 장애 또는 지체로 전형 진행 과정에서 지원이 필요한 경우, 사전에 요청해야 합니다.",
     "src": "01 p.5"},
    {"q": "외국에서 살다 온 사람들 뽑는 거 있잖아요, 경영학과는 최대 몇명까지 뽑을 수 있고 실제로는 몇명 뽑아요?",
     "gold": "경영학과의 최대모집 가능인원은 15명이고, 모집인원은 16명입니다.",
     "src": "01 p.3"},
    {"q": "외국 학교를 3년 다녀야 된다고 들었는데 한 학년은 어떻게 세는 거예요? 중간에 전학 가면 어떻게 되고요?",
     "gold": "고교과정 1개 학년을 포함해 중·고교과정에서 통산 3년(3개 학년 이상) 재학해야 하며, 성적이 나온 기간만 인정됩니다. 일반적으로 학기 개시일부터 다음 학년도 같은 학기 개시일 전일까지를 1개 학년으로 보지만, 해당 기간의 모든 학기를 이수한 경우에는 다음 학년도 같은 학기의 직전 학기 종료일까지로도 1개 학년을 충족한 것으로 간주합니다. 중간에 편입학했다면 편입학일부터 역년으로 1년(365일)이 되는 날까지를 1개 학년으로 봅니다. 또한 국외 재학기간뿐 아니라 체류기간과 국외근무자 재직기간도 모두 충족해야 합니다.",
     "src": "01 p.10"},
    {"q": "외국에 있는 부모님이랑 저 둘 다 외국에 있어야 되나요? 저만 있으면 안 되나요?",
     "gold": "일반적으로 부모와 학생 모두 국외에 체류해야 합니다. 다만 전 교육과정 이수자는 학생만 국외 체류하면 됩니다.",
     "src": "01 p.10"},
    {"q": "지역에서 의사 하면 학비 지원해준다던데 신청은 언제까지 해요? 1학기랑 2학기 따로 알려주세요.",
     "gold": "1학기는 2월 14일까지, 2학기는 8월 14일까지 신청해야 합니다.",
     "src": "02 p.8"},
    {"q": "강원도 속초 쪽이라고 하면 어디어디가 들어가요?",
     "gold": "속초시, 인제군, 고성군, 양양군이 포함됩니다.",
     "src": "02 p.5"},
    {"q": "지역의사 전형에서 두 군데 지역에 같이 넣을 수 있나요? 어디에 물어봐야 되고, 다른 전형처럼 여러 개 지원하면 안 되는 건가요?",
     "gold": "진료권·광역권 동시 지원 가능 여부는 개별 대학 모집 요강에서 확인해야 합니다. 한국대학교육협의회의 「대학입학전형 기본사항」에 따른 복수지원 금지 및 이중등록 금지 규정은 지역의사선발전형에도 동일하게 적용됩니다.",
     "src": "02 p.4"},
    {"q": "지역의사로 뽑히면 뭐 지원해주는 게 있어요? 돈 말고요. 그리고 의무 다 채우고 나면 취직할 때 좋은 거 있나요?",
     "gold": "지역의사 지원센터는 주거 안정, 직무교육, 경력개발, 국내외 연수, 권익보호, 진로상담·정보 제공을 지원합니다. 의무복무 완료 후에는 보건복지부장관, 지방자치단체의 장 또는 의무복무기관의 장이 해당 기관이나 지역 내 공공보건의료기관에 우선 채용하는 우대조치를 할 수 있습니다.",
     "src": "02 p.18"},
    {"q": "가천대 길병원에서 실습한다고 들었는데 몇 군데서 해요? 예를 들면 어디어디요?",
     "gold": "8개 내과 사이트에서 진행되며, 순환기내과와 내분비내과 등이 예시입니다.",
     "src": "03 p.15"},
    {"q": "합격하고 등록금까지 냈는데 마음이 바뀌었어요. 어떻게 취소해요? 취소하고 나서 다시 번복할 수는 있나요?",
     "gold": "정해진 기간 내에 본 대학 입학처 홈페이지에서 등록포기 및 환불 절차에 따라 등록 취소를 신청해야 합니다. 등록포기 및 환불신청을 완료하면 다시 취소할 수 없으므로 신중하게 신청해야 합니다.",
     "src": "03 p.90"},
    {"q": "예전에 태어난 사람은 신분증 사본도 내야 한다던데 저도 해당되나요? 서류는 언제 발급받은 걸로 내야 해요?",
     "gold": "1997년 3월 1일 이전 출생자에 해당하면 신분증 사본을 제출해야 합니다. 모든 제출 서류는 2026년 8월 1일 이후 발급된 것이어야 합니다.",
     "src": "03 p.63"},
    {"q": "학교 안에 도는 버스 있다던데 언제부터 있었고 지금은 몇대 돌아요?",
     "gold": "2012년에 도입됐으며, 현재 6대가 운행되고 있습니다.",
     "src": "03 p.7"},
    {"q": "정시 요강에서 원서접수 얘기는 몇 쪽에 나와요?",
     "gold": "원서접수 내용은 34페이지에 있습니다.",
     "src": "04 p.2"},
    {"q": "인천 캠퍼스에 바이오로직스학과 있잖아요, 거기 정시로 몇명 뽑아요?",
     "gold": "일반전형 1에서 30명, 일반전형 2에서 15명으로 총 45명입니다.",
     "src": "04 p.7"},
    {"q": "농어촌 전형은 정시 요강 몇 쪽에 있어요?",
     "gold": "44페이지에 안내되어 있습니다.",
     "src": "04 p.11"},
    {"q": "수능으로 가군 일반전형에 넣으려는데 서류 뭐 내야 해요? 고등학교 졸업했으면 아무것도 안 내도 되나요?",
     "gold": "기본적으로 제출서류는 없습니다. 다만 학생부 온라인 제공에 동의하지 않은 지원자, 또는 검정고시 합격자 중 국내 고교 재학 이력이 있는 지원자는 학교생활기록부를 제출해야 합니다.",
     "src": "04 p.12"},
    {"q": "이 면접 일정표는 어디 대학 거예요? 수시 중에 어떤 전형 대상이에요?",
     "gold": "가천대학교 2027학년도 수시모집의 학생부종합 및 지역균형 전형을 대상으로 합니다.",
     "src": "05 p.1"},
    {"q": "기회균형이나 농어촌, 특성화고 애들은 면접 언제 봐요?",
     "gold": "10월 3일(토)과 10월 4일(일)에 진행됩니다.",
     "src": "05 p.3"},
    {"q": "지역균형은 면접 날짜를 접수 순서로 정해줘요? 아프면 바꿀 수 있어요?",
     "gold": "면접은 원서접수순으로 배정될 예정이며, 학생 개인 사정에 따른 면접일 변경은 불가합니다.",
     "src": "05 p.4"},
    {"q": "바람개비 전형 간호학과는 면접을 몇 번 봐요? 날짜랑 시간이 몇 개 있어요?",
     "gold": "총 6개의 날짜·시간대에 편성되어 있습니다.",
     "src": "05 p.2"},
]

print("구어체 세트:", len(testset_hard), "문항")

구어체 세트: 20 문항


In [86]:
# ===== 최종 제출본과 동일한 검색·답변 파이프라인 (기준선) =====
DENSE20 = vectordb.as_retriever(search_kwargs={"k": 20})
bm25_retriever.k = 20
CHUNK_INDEX = {c.page_content: i for i, c in enumerate(chunks)}


def retrieve_final(query, top_n=6, rrf_k=60, span=1, expand_top=2):
    """RRF(BM25 + Dense) → 상위 top_n → 상위 expand_top개만 이웃 청크 확장."""
    scores, docs = {}, {}
    for result in (bm25_retriever.invoke(query), DENSE20.invoke(query)):
        for rank, d in enumerate(result, 1):
            key = d.page_content
            scores[key] = scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
            docs[key] = d
    top = [docs[k] for k in sorted(scores, key=scores.get, reverse=True)[:top_n]]

    out, seen = [], set()
    for n, d in enumerate(top):
        if d.page_content not in seen:
            out.append(d)
            seen.add(d.page_content)
        if n >= expand_top:
            continue
        i = CHUNK_INDEX[d.page_content]
        for j in range(max(0, i - span), min(len(chunks), i + span + 1)):
            c = chunks[j]
            if c.metadata["source"] == d.metadata["source"] and c.page_content not in seen:
                out.append(c)
                seen.add(c.page_content)
    return out


def ask_final(question, top_n=6):
    t0 = time.perf_counter()
    docs = retrieve_final(question, top_n)
    ctx = ctx_of(docs)

    draft = llm.invoke([("system", DRAFT_PROMPT),
                        ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content
    v = _safe_verify(ctx, question, draft)
    rounds, answer = 1, (v.final_answer or draft)

    if v.needs_more_search:
        rounds = 2
        docs2 = retrieve_final(v.search_query or question, top_n)
        merged, seen = [], set()
        for d in docs + docs2:
            if d.page_content not in seen:
                seen.add(d.page_content)
                merged.append(d)
        ctx2 = ctx_of(merged)
        draft2 = llm.invoke([("system", DRAFT_PROMPT),
                             ("human", f"[Evidence]\n{ctx2}\n\n[Question]\n{question}")]).content
        v2 = _safe_verify(ctx2, question, draft2)
        answer = v2.final_answer or draft2

    return answer, rounds, time.perf_counter() - t0


for t in testset_hard[:2]:
    a, r, s = ask_final(t["q"])
    print(f"{s:.2f}s r{r} | {a[:140]}")
    print(f"   gold: {t['gold'][:100]}")

13.17s r2 | 장애인복지법 제32조에 따라 장애인 등록을 마쳤고, 장애나 지체로 입학전형 진행 과정에서 지원이 필요한 경우 사전에 요청하시면 됩니다. 다만, 제공된 모집요강 내용에는 요청처나 구체적인 신청 방법은 나와 있지 않습니다. 장애학생의 지원 및 선발에는 차
   gold: 장애인복지법 제32조에 따라 장애인등록을 마쳤고 장애 또는 지체로 전형 진행 과정에서 지원이 필요한 경우, 사전에 요청해야 합니다.
10.11s r2 | 재외국민 전형 경영학과는 최대모집 가능인원이 15명이며, 모집인원은 16명입니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.3)
   gold: 경영학과의 최대모집 가능인원은 15명이고, 모집인원은 16명입니다.


In [87]:
r_h1 = evaluate_batch(ask_final, testset_hard, 0, 5, "hard")

 1. O 12.5s | 몸이 좀 불편한 학생인데요, 시험 볼 때 도와주는 사람 받을 수 있나요? 있으면 어떻게 해
 2. O 12.0s | 외국에서 살다 온 사람들 뽑는 거 있잖아요, 경영학과는 최대 몇명까지 뽑을 수 있고 실제로
 3. X  9.3s | 외국 학교를 3년 다녀야 된다고 들었는데 한 학년은 어떻게 세는 거예요? 중간에 전학 가면
 4. O 17.3s | 외국에 있는 부모님이랑 저 둘 다 외국에 있어야 되나요? 저만 있으면 안 되나요?
 5. O  4.3s | 지역에서 의사 하면 학비 지원해준다던데 신청은 언제까지 해요? 1학기랑 2학기 따로 알려주
[hard 1-5] 4/5 | 평균 11.08s | 최대 17.34s


In [88]:
r_h2 = evaluate_batch(ask_final, testset_hard, 5, 10, "hard")

 6. O  4.5s | 강원도 속초 쪽이라고 하면 어디어디가 들어가요?
 7. O  8.9s | 지역의사 전형에서 두 군데 지역에 같이 넣을 수 있나요? 어디에 물어봐야 되고, 다른 전형
 8. O  7.4s | 지역의사로 뽑히면 뭐 지원해주는 게 있어요? 돈 말고요. 그리고 의무 다 채우고 나면 취직
 9. O  5.9s | 가천대 길병원에서 실습한다고 들었는데 몇 군데서 해요? 예를 들면 어디어디요?
10. O  6.9s | 합격하고 등록금까지 냈는데 마음이 바뀌었어요. 어떻게 취소해요? 취소하고 나서 다시 번복할
[hard 6-10] 5/5 | 평균 6.72s | 최대 8.86s


In [89]:
r_h3 = evaluate_batch(ask_final, testset_hard, 10, 15, "hard")

11. O  8.8s | 예전에 태어난 사람은 신분증 사본도 내야 한다던데 저도 해당되나요? 서류는 언제 발급받은 
12. O  3.9s | 학교 안에 도는 버스 있다던데 언제부터 있었고 지금은 몇대 돌아요?
13. X  4.3s | 정시 요강에서 원서접수 얘기는 몇 쪽에 나와요?
14. O 17.6s | 인천 캠퍼스에 바이오로직스학과 있잖아요, 거기 정시로 몇명 뽑아요?
15. O  3.9s | 농어촌 전형은 정시 요강 몇 쪽에 있어요?
[hard 11-15] 4/5 | 평균 7.71s | 최대 17.64s


In [90]:
r_h4 = evaluate_batch(ask_final, testset_hard, 15, 20, "hard")

16. O  6.0s | 수능으로 가군 일반전형에 넣으려는데 서류 뭐 내야 해요? 고등학교 졸업했으면 아무것도 안 
17. O  4.0s | 이 면접 일정표는 어디 대학 거예요? 수시 중에 어떤 전형 대상이에요?
18. O  7.7s | 기회균형이나 농어촌, 특성화고 애들은 면접 언제 봐요?
19. O  3.5s | 지역균형은 면접 날짜를 접수 순서로 정해줘요? 아프면 바꿀 수 있어요?
20. O 21.1s | 바람개비 전형 간호학과는 면접을 몇 번 봐요? 날짜랑 시간이 몇 개 있어요?
[hard 16-20] 5/5 | 평균 8.44s | 최대 21.08s


In [91]:
from collections import Counter

rows = r_h1 + r_h2 + r_h3 + r_h4
print("rounds 분포:", Counter(r["rounds"] for r in rows))
print(f"평균 {sum(r['sec'] for r in rows)/len(rows):.2f}s | 5초 초과 {sum(r['sec']>5 for r in rows)}개 | 10초 초과 {sum(r['sec']>10 for r in rows)}개")
print("llm 설정:", getattr(llm, "reasoning_effort", None), getattr(llm, "max_completion_tokens", None))

for t in testset_hard[:3]:
    t0 = time.perf_counter()
    docs = retrieve_final(t["q"])
    t1 = time.perf_counter()
    ctx = ctx_of(docs)
    d = llm.invoke([("system", DRAFT_PROMPT),
                    ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{t['q']}")]).content
    t2 = time.perf_counter()
    v = _safe_verify(ctx, t["q"], d)
    t3 = time.perf_counter()
    print(f"검색 {t1-t0:.2f}s | 초안 {t2-t1:.2f}s | 검증 {t3-t2:.2f}s | 재검색={v.needs_more_search} | 근거 {len(ctx):,}자 | {t['q'][:26]}")

rounds 분포: Counter({1: 15, 2: 5})
평균 8.49s | 5초 초과 13개 | 10초 초과 5개
llm 설정: low None
검색 0.36s | 초안 2.11s | 검증 2.77s | 재검색=True | 근거 5,302자 | 몸이 좀 불편한 학생인데요, 시험 볼 때 도와주
검색 0.19s | 초안 4.27s | 검증 4.72s | 재검색=True | 근거 10,184자 | 외국에서 살다 온 사람들 뽑는 거 있잖아요, 경
검색 0.23s | 초안 2.79s | 검증 5.04s | 재검색=False | 근거 12,875자 | 외국 학교를 3년 다녀야 된다고 들었는데 한 학


In [92]:
q = testset_hard[1]["q"]
docs = retrieve_final(q)
print("문서 수:", len(docs))
for i, d in enumerate(docs):
    print(f"  {i+1}. {d.metadata['kind']:5s} {len(d.page_content):5,}자 | {d.metadata['source'][:2]} p.{d.metadata['page']}")

for n in (3, 5, 8):
    ctx = ctx_of(docs[:n])
    t0 = time.perf_counter()
    d1 = llm.invoke([("system", DRAFT_PROMPT), ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{q}")]).content
    t1 = time.perf_counter()
    v = _safe_verify(ctx, q, d1)
    t2 = time.perf_counter()
    print(f"근거 {n}개({len(ctx):,}자): 초안 {t1-t0:.2f}s + 검증 {t2-t1:.2f}s = {t2-t0:.2f}s | 재검색={v.needs_more_search}")

문서 수: 10
  1. table   847자 | 01 p.9
  2. table   258자 | 01 p.8
  3. table   940자 | 01 p.10
  4. table 1,713자 | 01 p.12
  5. table 1,333자 | 01 p.11
  6. table 1,475자 | 01 p.12
  7. text    789자 | 03 p.80
  8. text    822자 | 02 p.2
  9. text    818자 | 03 p.64
  10. text    807자 | 03 p.19
근거 3개(2,158자): 초안 1.78s + 검증 3.27s = 5.06s | 재검색=True
근거 5개(5,282자): 초안 2.32s + 검증 3.58s = 5.90s | 재검색=True
근거 8개(8,484자): 초안 2.71s + 검증 3.46s = 6.17s | 재검색=True


In [93]:
q = testset_hard[1]["q"]
ctx = ctx_of(docs[:5])

llm_none = ChatOpenAI(model=LLM_MODEL, reasoning_effort="none", max_completion_tokens=1500, timeout=30)
verify_none = llm_none.with_structured_output(Verify)

# 1) 현재 구성 (low / low)
t0 = time.perf_counter()
d1 = llm.invoke([("system", DRAFT_PROMPT), ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{q}")]).content
t1 = time.perf_counter()
v1_ = _safe_verify(ctx, q, d1)
t2 = time.perf_counter()
print(f"[low/low]   초안 {t1-t0:.2f}s + 검증 {t2-t1:.2f}s = {t2-t0:.2f}s")

# 2) none / none
t0 = time.perf_counter()
d2 = llm_none.invoke([("system", DRAFT_PROMPT), ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{q}")]).content
t1 = time.perf_counter()
v2_ = verify_none.invoke([("system", VERIFY_PROMPT),
                          ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{q}\n\n[Draft]\n{d2}")])
t2 = time.perf_counter()
print(f"[none/none] 초안 {t1-t0:.2f}s + 검증 {t2-t1:.2f}s = {t2-t0:.2f}s")

# 3) none 초안 + low 검증 (짧은 근거 3개)
ctx3 = ctx_of(docs[:3])
t0 = time.perf_counter()
d3 = llm_none.invoke([("system", DRAFT_PROMPT), ("human", f"[Evidence]\n{ctx3}\n\n[Question]\n{q}")]).content
t1 = time.perf_counter()
v3_ = _safe_verify(ctx3, q, d3)
t2 = time.perf_counter()
print(f"[none/low,근거3] 초안 {t1-t0:.2f}s + 검증 {t2-t1:.2f}s = {t2-t0:.2f}s")

print("\n--- none/none 답변 ---")
print(v2_.final_answer[:200])

[low/low]   초안 1.90s + 검증 3.69s = 5.59s
[none/none] 초안 1.63s + 검증 2.49s = 4.13s
[none/low,근거3] 초안 1.88s + 검증 4.92s = 6.80s

--- none/none 답변 ---
재외국민과 외국인 전형은 전체 모집인원의 2% 이내로 선발하지만, 제시된 자료에는 경영학과의 최대 모집 인원과 실제 모집 인원이 나와 있지 않아 확인할 수 없습니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.9)


In [94]:
# ===== v8. 지연시간 개선: 검증은 none, 재검색은 초안 1회, 시간 예산 가드 =====
verify_fast = ChatOpenAI(
    model=LLM_MODEL, reasoning_effort="none", max_completion_tokens=1200, timeout=25
).with_structured_output(Verify)

TIME_BUDGET = 7.0   # 1차 답변이 이 시간을 넘기면 재검색 생략 (20초 초과 방지)


def _verify_fast(question, ctx, draft):
    try:
        return verify_fast.invoke([("system", VERIFY_PROMPT),
                                   ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}\n\n[Draft]\n{draft}")])
    except Exception as e:
        print("   (검증 생략:", type(e).__name__, ")")
        return Verify(needs_more_search=False, final_answer=draft)


def ask_v8(question, top_n=5, expand_top=1):
    t0 = time.perf_counter()
    docs = retrieve_final(question, top_n=top_n, expand_top=expand_top)
    ctx = ctx_of(docs)

    draft = llm.invoke([("system", DRAFT_PROMPT),
                        ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content
    v = _verify_fast(question, ctx, draft)
    answer, rounds = (v.final_answer or draft), 1

    elapsed = time.perf_counter() - t0
    if v.needs_more_search and elapsed < TIME_BUDGET:
        rounds = 2
        docs2 = retrieve_final(v.search_query or question, top_n=top_n, expand_top=0)
        merged, seen = [], set()
        for d in docs + docs2:
            if d.page_content not in seen:
                seen.add(d.page_content)
                merged.append(d)
        ctx2 = ctx_of(merged)
        answer = llm.invoke([("system", DRAFT_PROMPT),
                             ("human", f"[Evidence]\n{ctx2}\n\n[Question]\n{question}")]).content

    return answer, rounds, time.perf_counter() - t0


for t in testset_hard[:2]:
    a, r, s = ask_v8(t["q"])
    print(f"{s:.2f}s r{r} | {a[:120]}")

6.57s r2 | 제공된 모집요강 근거에는 장애 또는 신체 불편이 있는 수험생을 위한 시험 중 지원 인력 제공 여부나 신청 방법이 안내되어 있지 않습니다. 따라서 이 자료만으로는 도움을 받을 수 있는지, 어떻게 신청해야 하는지 확인할
5.73s r2 | 재외국민 모집에서 경영학과는 최대 15명까지 모집 가능하며, 모집인원은 16명으로 안내되어 있습니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.3)


In [95]:
# ===== v9. 검색은 기준선 유지, 검증만 none + 2차는 초안 1회 + 시간 가드 =====
def ask_v9(question, top_n=6, expand_top=2, budget=6.5):
    t0 = time.perf_counter()
    docs = retrieve_final(question, top_n=top_n, expand_top=expand_top)
    ctx = ctx_of(docs)

    draft = llm.invoke([("system", DRAFT_PROMPT),
                        ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content
    v = _verify_fast(question, ctx, draft)
    answer, rounds = (v.final_answer or draft), 1

    if v.needs_more_search and time.perf_counter() - t0 < budget:
        rounds = 2
        docs2 = retrieve_final(v.search_query or question, top_n=top_n, expand_top=1)
        merged, seen = [], set()
        for d in docs + docs2:
            if d.page_content not in seen:
                seen.add(d.page_content)
                merged.append(d)
        ctx2 = ctx_of(merged)
        draft2 = llm.invoke([("system", DRAFT_PROMPT),
                             ("human", f"[Evidence]\n{ctx2}\n\n[Question]\n{question}")]).content
        v2 = _verify_fast(question, ctx2, draft2)
        answer = v2.final_answer or draft2

    return answer, rounds, time.perf_counter() - t0


for t in testset_hard[:5]:
    a, r, s = ask_v9(t["q"])
    print(f"{s:5.2f}s r{r} | {a[:110]}")

 9.36s r2 | 입학전형 진행 과정에서 장애 또는 지체로 지원이 필요하면, 장애인복지법 제32조에 따라 장애인 등록을 마친 뒤 사전에 요청하시면 됩니다. 다만 ‘도와주는 사람’을 별도로 제공하는지는 안내되어 있지 
 8.57s r2 | 재외국민 전형에서 경영학과의 최대모집 가능인원은 15명이고, 모집인원은 16명입니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.3)
 5.26s r1 | 한 학년은 학기 개시일부터 다음 학년도 같은 학기 개시일 전일까지(약 365일)로 계산합니다. 다만 그 1개 학년 기간 안의 모든 학기를 이수했다면 학기 개시일부터 다음 학년도 같은 학기 직전 학기
 4.97s r1 | 모집요강상 지원 유형에 따라 기준이 다릅니다. 영주교포의 자녀는 부모와 학생 모두 외국에서 영주해야 합니다. 국외근무자의 자녀는 외국에서 근무한 사실이 있는 공무원·상사직원·외국정부 또는 국제기구 
 3.45s r1 | 대학이 시·도지사에게 학비 지원을 신청하는 기한은 1학기는 2월 14일까지, 2학기는 8월 14일까지입니다. (출처: 02_2027학년도_지역의사선발전형_안내.pdf p.8)


In [96]:
# ===== v10. 2차 검색은 '검증형 재작성' 1회로 축약 + 시간 가드 =====
def ask_v10(question, top_n=6, expand_top=2, t_budget=5.5):
    t0 = time.perf_counter()
    docs = retrieve_final(question, top_n=top_n, expand_top=expand_top)
    ctx = ctx_of(docs)

    draft = llm.invoke([("system", DRAFT_PROMPT),
                        ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content
    v = _verify_fast(question, ctx, draft)
    answer, rounds = (v.final_answer or draft), 1

    if v.needs_more_search and time.perf_counter() - t0 < t_budget:
        rounds = 2
        docs2 = retrieve_final(v.search_query or question, top_n=top_n, expand_top=1)
        merged, seen = [], set()
        for d in docs + docs2:
            if d.page_content not in seen:
                seen.add(d.page_content)
                merged.append(d)
        ctx2 = ctx_of(merged)
        v2 = _safe_verify(ctx2, question, draft)      # low 모델로 1회 재작성
        answer = v2.final_answer or draft

    return answer, rounds, time.perf_counter() - t0


for t in testset_hard[:5]:
    a, r, s = ask_v10(t["q"])
    print(f"{s:5.2f}s r{r} | {a[:110]}")

 9.01s r2 | 제공된 자료에는 몸이 불편한 수험생의 시험 편의 제공 여부나 신청 방법이 안내되어 있지 않아 확인이 필요합니다. (출처: 03_2027학년도_수시_모집요강.pdf p.56)
 5.65s r1 | 재외국민과 외국인 전형은 정원의 2% 이내로 모집하지만, 제공된 자료에는 경영학과의 최대 선발 인원이나 실제 모집 인원이 나와 있지 않습니다. 따라서 두 인원은 이 자료만으로 확인할 수 없습니다. 
 5.83s r1 | 국외 재학은 고교과정 1개 학년을 포함해 중·고교 과정 통산 3년(3개 학년 이상)이어야 하며, 성적이 나온 경우만 인정됩니다. 1개 학년은 학기 개시일부터 다음 학년도 동일 학기 개시일 전일까지(
12.28s r2 | 지원 유형에 따라 다릅니다. 「재외국민과 외국인(2% 이내)」으로 지원한다면 부모와 학생 모두 국외에 체류해야 하므로, 학생만 외국에 있는 경우에는 해당 요건을 충족하지 못합니다. 이 유형은 학생이
 4.55s r1 | 대학이 시·도지사에게 학비 지원을 신청하는 기한은 1학기 2월 14일까지, 2학기 8월 14일까지입니다. (출처: 02_2027학년도_지역의사선발전형_안내.pdf p.8)


In [97]:
print("model_kwargs:", llm.model_kwargs)
print("max_tokens:", getattr(llm, "max_tokens", None))
print("max_completion_tokens:", getattr(llm, "max_completion_tokens", None))
print("reasoning_effort:", getattr(llm, "reasoning_effort", None))

model_kwargs: {}
max_tokens: 2000
max_completion_tokens: None
reasoning_effort: low


In [98]:
# ===== v11. 기준선 품질 유지 + 2차 재검색을 초안 1회로 축약 + 시간 가드 =====
def ask_v11(question, top_n=6, expand_top=2, t_budget=6.0):
    t0 = time.perf_counter()
    docs = retrieve_final(question, top_n=top_n, expand_top=expand_top)
    ctx = ctx_of(docs)

    try:
        draft = llm.invoke([("system", DRAFT_PROMPT),
                            ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content
    except Exception as e:
        print("   (초안 실패:", type(e).__name__, ")")
        return "제공된 문서에서 확인할 수 없습니다.", 1, time.perf_counter() - t0

    v = _safe_verify(ctx, question, draft)          # low 모델 검증
    answer, rounds = (v.final_answer or draft), 1

    if v.needs_more_search and time.perf_counter() - t0 < t_budget:
        rounds = 2
        docs2 = retrieve_final(v.search_query or question, top_n=top_n, expand_top=1)
        merged, seen = [], set()
        for d in docs + docs2:
            if d.page_content not in seen:
                seen.add(d.page_content)
                merged.append(d)
        ctx2 = ctx_of(merged)
        try:
            answer = llm.invoke([("system", DRAFT_PROMPT),
                                 ("human", f"[Evidence]\n{ctx2}\n\n[Question]\n{question}")]).content
        except Exception:
            pass

    return answer, rounds, time.perf_counter() - t0


for t in testset_hard[:5]:
    a, r, s = ask_v11(t["q"])
    print(f"{s:5.2f}s r{r} | {a[:110]}")

 7.40s r2 | 장애인복지법 제32조에 따라 장애인 등록을 마친 학생이 각종 장애 또는 지체로 인해 입시전형 진행 과정에서 지원이 필요한 경우 사전 요청하면 됩니다. 장애학생의 지원 및 선발에 차별은 없으며, 입학
 6.36s r1 | 재외국민과 외국인 전형은 모집인원의 2% 이내로 선발하지만, 제시된 자료에는 경영학과의 최대 선발 인원이나 실제 선발 인원이 나와 있지 않습니다. (출처: 01_2027학년도_재외국민_모집요강.hw
 7.78s r1 | 고교과정 1개 학년을 포함해 중·고교과정에서 통산 3년(3개 학년 이상) 재학해야 하며, 성적이 나온 기간만 인정됩니다. 한 학년은 학기 개시일부터 다음 학년도 같은 학기 개시일 전날까지 약 365
 9.31s r1 | 부모님과 지원자 모두 외국에 있어야 하는지는 지원 유형에 따라 다릅니다. ‘영주교포의 자녀’ 유형은 부모와 학생 모두 외국에서 영주해야 합니다. 반면 다른 유형은 부모 등의 해외 근무·활동·유학 사
 4.01s r1 | 학비 지원 신청은 대학이 시·도지사에게 합니다. 신청 기한은 1학기는 2월 14일까지, 2학기는 8월 14일까지입니다. (출처: 02_2027학년도_지역의사선발전형_안내.pdf p.8)


In [99]:
q = testset_hard[1]["q"]
print("Q:", q)

print("\n[BM25 top10]")
for d in bm25_retriever.invoke(q)[:10]:
    print(f"  {d.metadata['source'][:2]} p.{d.metadata['page']:>3} {len(d.page_content):5,}자 | {d.page_content.splitlines()[1][:70] if len(d.page_content.splitlines())>1 else ''}")

print("\n[Dense top10]")
for d in DENSE20.invoke(q)[:10]:
    print(f"  {d.metadata['source'][:2]} p.{d.metadata['page']:>3} {len(d.page_content):5,}자 | {d.page_content.splitlines()[1][:70] if len(d.page_content.splitlines())>1 else ''}")

print("\n[경영학과 포함 청크]")
for i, c in enumerate(chunks):
    if c.metadata["source"].startswith("01") and "경영학과" in c.page_content:
        print(f"  idx={i} p.{c.metadata['page']} {len(c.page_content)}자")
        print("   ", c.page_content[:300].replace("\n", " "))

Q: 외국에서 살다 온 사람들 뽑는 거 있잖아요, 경영학과는 최대 몇명까지 뽑을 수 있고 실제로는 몇명 뽑아요?

[BM25 top10]
  02 p.  2   822자 | 당신이 나고 자란 그곳을 책임집니다
  03 p. 19   807자 | 2027학년도 수시모집요강
  03 p. 57   788자 | Q. 활동의 종류나 개수가 많을수록 유리한가요?
  03 p.  4   801자 | 4
  01 p.  9   847자 | | 지원유형 | 지원유형 |
  03 p.  7   805자 | 가천대역은 2010년 가천대학교 시그니처
  01 p. 13 1,359자 | | 지원자 유의사항 | 지원자 유의사항 |
  03 p. 74   825자 | 74
  01 p. 18 2,554자 | | <초 · 중 · 고 재학기간 기재사항> | <초 · 중 · 고 재학기간 기재사항> | <초 · 중 · 고 재학기간 기재사항
  03 p. 18   436자 | 가천대학교가 반도체 산업에 필요한 대규모 실무형 인재 양성을 선도합니다. 세계 최초·최대 입학정원의 단독 단과대학으로

[Dense top10]
  03 p. 64   818자 | 64
  03 p. 68   787자 | 68
  04 p. 22   775자 | 48
  03 p. 48   829자 | 48
  01 p. 10   940자 | | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 
  01 p. 10 1,461자 | | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 
  01 p. 12 1,475자 | | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기
  01 p. 17 1,706자 | | [IMAGE] 가천대학교 | 2027학년도 재외국민 제출서류 목록 | 2027학년도 재외국민 제출

In [100]:
def keywords(text, k=8):
    out = []
    for t in kiwi.tokenize(text):
        if t.tag in {"NNG", "NNP", "SL", "SN"} and len(t.form) > 1 and t.form not in out:
            out.append(t.form)
    return out[:k]


q = testset_hard[1]["q"]
kw = " ".join(keywords(q))
print("키워드 질의:", kw)

print("\n[BM25(키워드) top8]")
for d in bm25_retriever.invoke(kw)[:8]:
    print(f"  {d.metadata['source'][:2]} p.{d.metadata['page']:>3} {len(d.page_content):5,}자 | {d.page_content.splitlines()[1][:60] if len(d.page_content.splitlines())>1 else ''}")

print("\n[Dense(키워드) top8]")
for d in DENSE20.invoke(kw)[:8]:
    print(f"  {d.metadata['source'][:2]} p.{d.metadata['page']:>3} {len(d.page_content):5,}자 | {d.page_content.splitlines()[1][:60] if len(d.page_content.splitlines())>1 else ''}")

키워드 질의: 외국 사람 경영학과 최대

[BM25(키워드) top8]
  01 p.  9   847자 | | 지원유형 | 지원유형 |
  01 p. 12 1,713자 | | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 
  01 p. 13 1,359자 | | 지원자 유의사항 | 지원자 유의사항 |
  01 p. 12 1,475자 | | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 
  03 p. 24   157자 | 창업학기제 이수학생
  04 p. 22   775자 | 48
  03 p. 72   680자 | 72
  03 p. 76   732자 | 76

[Dense(키워드) top8]
  01 p. 12 1,475자 | | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 
  01 p. 12 1,713자 | | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 | 외국학교 인정기준 
  03 p. 76   732자 | 76
  03 p. 72   680자 | 72
  01 p. 10 1,764자 | | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원
  03 p. 80   789자 | 80
  01 p. 10 1,461자 | | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원
  01 p.  1    66자 | Gachon University


In [101]:
print("BM25 문서 수:", len(bm25_retriever.docs), "| 전체 청크:", len(chunks))
target = chunks[1].page_content
print("p.3 청크가 BM25 코퍼스에 있나:", any(d.page_content == target for d in bm25_retriever.docs))
print("idx1 토큰 앞부분:", korean_tokenizer(target)[:20] if 'korean_tokenizer' in globals() else "?")

for test_q in ["경영학과", "경영학과 최대모집 가능인원", "최대모집 가능인원 모집인원"]:
    hits = bm25_retriever.invoke(test_q)[:5]
    print(f"\nBM25('{test_q}')")
    for d in hits:
        print(f"   {d.metadata['source'][:2]} p.{d.metadata['page']} | {d.page_content.splitlines()[1][:50] if len(d.page_content.splitlines())>1 else ''}")

BM25 문서 수: 514 | 전체 청크: 514
p.3 청크가 BM25 코퍼스에 있나: True
idx1 토큰 앞부분: ['01', '2027', '학년도', '재외국민', '모집', '요강', 'hwp', 'p.', '3', '모집', '인원', '모집', '인원', '모집', '인원', '모집', '인원', '모집', '인원', '모집']

BM25('경영학과')
   03 p.50 | | 계열 | 모집단위 | | 모집인원 |
   03 p.62 | | 계열 | 모집단위 | | 모집인원 |
   03 p.40 | | 계열 | 모집단위 | | 모집인원 |
   03 p.42 | | 계열 | 모집단위 | | 모집인원 |
   03 p.58 | | 계열 | 모집단위 | | 모집인원 |

BM25('경영학과 최대모집 가능인원')
   01 p.3 | | 모집인원 | 모집인원 | 모집인원 | 모집인원 | 모집인원 | 모집인원 |
   04 p.22 | | 계열 | 모집단위 | 모집인원 |
   03 p.68 | | 계열 | 모집단위 | 모집인원 |
   03 p.88 | 88
   03 p.89 | 89

BM25('최대모집 가능인원 모집인원')
   01 p.3 | | 모집인원 | 모집인원 | 모집인원 | 모집인원 | 모집인원 | 모집인원 |
   03 p.89 | 89
   01 p.3 | | 모집인원 | 모집인원 | 모집인원 | 모집인원 | 모집인원 | 모집인원 |
   01 p.3 | | 모집인원 | 모집인원 | 모집인원 | 모집인원 | 모집인원 | 모집인원 |
   01 p.13 | | 지원자 유의사항 | 지원자 유의사항 |


In [102]:
SYNONYMS = {
    "뽑": "모집 선발", "선발": "모집", "몇명": "모집인원", "몇 명": "모집인원",
    "인원": "모집인원", "해외": "재외국민", "외국": "재외국민", "유학": "재외국민",
    "등록금": "등록 포기 환불", "환불": "등록포기 환불",
    "시험": "전형", "서류": "제출서류", "조건": "지원자격",
}

q = testset_hard[1]["q"]
extra = " ".join(v for k, v in SYNONYMS.items() if k in q)
q_exp = f"{q} {extra}"
print("확장 질의:", q_exp)

for name, d in [("BM25", bm25_retriever.invoke(q_exp)), ("Dense", DENSE20.invoke(q_exp))]:
    rank = next((i for i, x in enumerate(d[:20], 1) if x.metadata["page"] == 3 and x.metadata["source"].startswith("01")), None)
    print(f"{name}: p.3 순위 = {rank}")
    for x in d[:5]:
        print(f"   {x.metadata['source'][:2]} p.{x.metadata['page']}")

확장 질의: 외국에서 살다 온 사람들 뽑는 거 있잖아요, 경영학과는 최대 몇명까지 뽑을 수 있고 실제로는 몇명 뽑아요? 모집 선발 모집인원 재외국민
BM25: p.3 순위 = 17
   03 p.19
   01 p.9
   03 p.74
   02 p.2
   01 p.13
Dense: p.3 순위 = 9
   01 p.10
   03 p.68
   04 p.22
   01 p.9
   01 p.1


In [103]:
def keywords(text, k=8):
    out = []
    for t in kiwi.tokenize(text):
        if t.tag in {"NNG", "NNP", "SL", "SN"} and len(t.form) > 1 and t.form not in out:
            out.append(t.form)
    return out[:k]


def expand_query(q):
    syn = [v for k, v in SYNONYMS.items() if k in q]
    return " ".join(keywords(q) + syn)


def retrieve_v12(query, top_n=6, expand_top=2, rrf_k=60):
    """질문 + 키워드·동의어 확장 질의를 각각 BM25/Dense로 검색해 RRF 융합."""
    q_exp = expand_query(query)
    lists = [
        bm25_retriever.invoke(query), DENSE20.invoke(query),
        bm25_retriever.invoke(q_exp), DENSE20.invoke(q_exp),
    ]
    scores, docs = {}, {}
    for result in lists:
        for rank, d in enumerate(result, 1):
            key = d.page_content
            scores[key] = scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
            docs[key] = d
    top = [docs[k] for k in sorted(scores, key=scores.get, reverse=True)[:top_n]]

    out, seen = [], set()
    for n, d in enumerate(top):
        if d.page_content not in seen:
            out.append(d)
            seen.add(d.page_content)
        if n >= expand_top:
            continue
        i = CHUNK_INDEX[d.page_content]
        for j in range(max(0, i - 1), min(len(chunks), i + 2)):
            c = chunks[j]
            if c.metadata["source"] == d.metadata["source"] and c.page_content not in seen:
                out.append(c)
                seen.add(c.page_content)
    return out


q = testset_hard[1]["q"]
print("확장 질의:", expand_query(q))
t0 = time.perf_counter()
ds = retrieve_v12(q)
print(f"검색 {time.perf_counter()-t0:.2f}s, {len(ds)}개")
for d in ds:
    print(f"  {d.metadata['source'][:2]} p.{d.metadata['page']:>3} {len(d.page_content):5,}자")

확장 질의: 외국 사람 경영학과 최대 모집 선발 모집인원 재외국민
검색 0.39s, 10개
  01 p.  9   847자
  01 p.  8   258자
  01 p. 10   940자
  04 p. 22   775자
  04 p. 21   186자
  04 p. 22   491자
  03 p. 80   789자
  01 p.  1    66자
  01 p. 17 1,706자
  01 p. 17 1,392자


In [104]:
q = testset_hard[1]["q"]
tests = {
    "도메인 prefix": "2027학년도 가천대학교 입학 모집요강 " + q,
    "깨끗한 키워드": "경영학과 최대 모집인원",
    "키워드+동의어": "경영학과 최대 모집인원 재외국민",
}
for name, query in tests.items():
    bm = bm25_retriever.invoke(query)
    de = DENSE20.invoke(query)
    rb = next((i for i, x in enumerate(bm[:20], 1) if x.metadata["page"] == 3 and x.metadata["source"].startswith("01")), None)
    rd = next((i for i, x in enumerate(de[:20], 1) if x.metadata["page"] == 3 and x.metadata["source"].startswith("01")), None)
    print(f"{name:14s} | BM25 p.3 순위={rb} | Dense p.3 순위={rd}")

도메인 prefix     | BM25 p.3 순위=None | Dense p.3 순위=5
깨끗한 키워드        | BM25 p.3 순위=1 | Dense p.3 순위=None
키워드+동의어        | BM25 p.3 순위=4 | Dense p.3 순위=2


In [105]:
GENERIC = {
    "사람", "사람들", "학생", "학생들", "경우", "때", "거", "것", "수", "등", "분",
    "년", "월", "일", "좀", "저", "제", "우리", "나", "너", "그", "이", "뭐", "무엇",
    "어디", "언제", "얼마", "때문", "정도", "그거", "이거", "저거", "애들", "친구",
}


def search_keywords(q, k=10):
    """질문 → 검색용 키워드(일반명사 제거 + 동의어 치환)."""
    out = []
    for t in kiwi.tokenize(q):
        if t.tag not in {"NNG", "NNP", "SL", "SN"} or len(t.form) < 2:
            continue
        w = t.form
        if w in GENERIC or any(key in w for key in SYNONYMS):
            continue
        if w not in out:
            out.append(w)
    for key, val in SYNONYMS.items():
        if key in q:
            for v in val.split():
                if v not in out:
                    out.append(v)
    return " ".join(out[:k])


def retrieve_v13(query, top_n=6, expand_top=2, rrf_k=60):
    """원 질문 + 키워드/동의어 질의를 각각 BM25·Dense로 검색해 4-리스트 RRF."""
    kw = search_keywords(query)
    lists = [
        bm25_retriever.invoke(query), DENSE20.invoke(query),
        bm25_retriever.invoke(kw), DENSE20.invoke(kw),
    ]
    scores, docs = {}, {}
    for result in lists:
        for rank, d in enumerate(result, 1):
            key = d.page_content
            scores[key] = scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
            docs[key] = d
    top = [docs[k] for k in sorted(scores, key=scores.get, reverse=True)[:top_n]]

    out, seen = [], set()
    for n, d in enumerate(top):
        if d.page_content not in seen:
            out.append(d)
            seen.add(d.page_content)
        if n >= expand_top:
            continue
        i = CHUNK_INDEX[d.page_content]
        for j in range(max(0, i - 1), min(len(chunks), i + 2)):
            c = chunks[j]
            if c.metadata["source"] == d.metadata["source"] and c.page_content not in seen:
                out.append(c)
                seen.add(c.page_content)
    return out


for qi in (1, 0):
    print("Q:", testset_hard[qi]["q"])
    print("키워드:", search_keywords(testset_hard[qi]["q"]))
    t0 = time.perf_counter()
    ds = retrieve_v13(testset_hard[qi]["q"])
    print(f"  검색 {time.perf_counter()-t0:.2f}s")
    for d in ds:
        print(f"   {d.metadata['source'][:2]} p.{d.metadata['page']:>3} {len(d.page_content):5,}자")
    print()

Q: 외국에서 살다 온 사람들 뽑는 거 있잖아요, 경영학과는 최대 몇명까지 뽑을 수 있고 실제로는 몇명 뽑아요?
키워드: 경영학과 최대 모집 선발 모집인원 재외국민
  검색 0.51s
   01 p.  9   847자
   01 p.  8   258자
   01 p. 10   940자
   01 p. 10 1,766자
   01 p.  1    66자
   01 p. 17 1,706자
   01 p. 10 1,764자
   04 p. 22   118자

Q: 몸이 좀 불편한 학생인데요, 시험 볼 때 도와주는 사람 받을 수 있나요? 있으면 어떻게 해야 해요?
키워드: 불편 전형
  검색 0.44s
   03 p.  4   312자
   03 p.  4   795자
   03 p.  5   301자
   02 p.  2   822자
   02 p.  1   678자
   02 p.  2   454자
   03 p. 56   241자
   03 p. 38   353자
   01 p.  4   328자



In [106]:
SYNONYMS.update({
    "불편": "장애 지원", "도와": "지원", "도움": "지원", "장애인": "장애",
    "사진": "증명사진", "원서": "원서접수", "수시": "수시모집", "정시": "정시모집",
    "등록": "등록 포기 환불", "합격": "합격자 발표",
})


def retrieve_kw(query, top_n=5, expand_top=1, rrf_k=60):
    """키워드·동의어 질의만으로 재검색 (구어체 질문 대응)."""
    kw = search_keywords(query)
    lists = [bm25_retriever.invoke(kw), DENSE20.invoke(kw)]
    scores, docs = {}, {}
    for result in lists:
        for rank, d in enumerate(result, 1):
            key = d.page_content
            scores[key] = scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
            docs[key] = d
    top = [docs[k] for k in sorted(scores, key=scores.get, reverse=True)[:top_n]]

    out, seen = [], set()
    for n, d in enumerate(top):
        if d.page_content not in seen:
            out.append(d)
            seen.add(d.page_content)
        if n >= expand_top:
            continue
        i = CHUNK_INDEX[d.page_content]
        for j in range(max(0, i - 1), min(len(chunks), i + 2)):
            c = chunks[j]
            if c.metadata["source"] == d.metadata["source"] and c.page_content not in seen:
                out.append(c)
                seen.add(c.page_content)
    return out


for qi in (0, 1):
    qq = testset_hard[qi]["q"]
    print("Q:", qq)
    print("키워드:", search_keywords(qq))
    for d in retrieve_kw(qq):
        print(f"   {d.metadata['source'][:2]} p.{d.metadata['page']:>3} {len(d.page_content):5,}자")
    print()

Q: 몸이 좀 불편한 학생인데요, 시험 볼 때 도와주는 사람 받을 수 있나요? 있으면 어떻게 해야 해요?
키워드: 전형 장애 지원
   04 p.  8   394자
   04 p.  8   804자
   04 p.  9   594자
   03 p. 35   390자
   01 p.  5   527자
   01 p. 13 1,359자
   02 p. 10   824자

Q: 외국에서 살다 온 사람들 뽑는 거 있잖아요, 경영학과는 최대 몇명까지 뽑을 수 있고 실제로는 몇명 뽑아요?
키워드: 경영학과 최대 모집 선발 모집인원 재외국민
   01 p.  1    66자
   01 p.  3 1,520자
   04 p. 22   118자
   03 p. 50   316자
   03 p. 64   448자



In [107]:
# ===== v14. '못 찾음'이면 키워드 재검색으로 근거를 보강해 1회 더 작성 =====
NOT_FOUND = "확인할 수 없습니다"


def _need_retry(answer, v):
    return v.needs_more_search or NOT_FOUND in answer


def ask_v14(question, top_n=6, expand_top=2, t_budget=6.5):
    t0 = time.perf_counter()
    docs = retrieve_final(question, top_n=top_n, expand_top=expand_top)
    ctx = ctx_of(docs)

    try:
        draft = llm.invoke([("system", DRAFT_PROMPT),
                            ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content
    except Exception as e:
        print("   (초안 실패:", type(e).__name__, ")")
        return NOT_FOUND + ".", 1, time.perf_counter() - t0

    v = _safe_verify(ctx, question, draft)
    answer, rounds = (v.final_answer or draft), 1

    if _need_retry(answer, v) and time.perf_counter() - t0 < t_budget:
        rounds = 2
        docs2 = retrieve_kw(question, top_n=5, expand_top=1)
        merged, seen = [], set()
        for d in docs + docs2:
            if d.page_content not in seen:
                seen.add(d.page_content)
                merged.append(d)
        ctx2 = ctx_of(merged)
        try:
            answer = llm.invoke([("system", DRAFT_PROMPT),
                                 ("human", f"[Evidence]\n{ctx2}\n\n[Question]\n{question}")]).content
        except Exception:
            pass

    return answer, rounds, time.perf_counter() - t0


for t in testset_hard[:2]:
    a, r, s = ask_v14(t["q"])
    print(f"{s:5.2f}s r{r} | {a[:160]}")
    print(f"        gold: {t['gold'][:90]}")

 6.75s r1 | 제공된 모집요강 발췌에는 몸이 불편한 수험생이 시험 중 도우미를 받을 수 있는지와 신청 방법이 안내되어 있지 않아 확인할 수 없습니다. 장애 수험생 편의 제공 및 신청 절차가 안내된 내용을 추가로 확인해야 합니다. (출처: 03_2027학년도_수시_모집요강.pdf p.77)
        gold: 장애인복지법 제32조에 따라 장애인등록을 마쳤고 장애 또는 지체로 전형 진행 과정에서 지원이 필요한 경우, 사전에 요청해야 합니다.
 7.81s r1 | 재외국민과 외국인 전형은 모집인원의 2% 이내로 선발합니다. 다만 제시된 자료에는 경영학과의 최대 선발 인원이나 실제 선발 인원이 기재되어 있지 않아 확인할 수 없습니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.9)
        gold: 경영학과의 최대모집 가능인원은 15명이고, 모집인원은 16명입니다.


In [108]:
for t in testset_hard[:2]:
    a, r, s = ask_v14(t["q"], t_budget=9.0)
    print(f"{s:5.2f}s r{r} | {a[:170]}")
    print(f"        gold: {t['gold'][:90]}")

 8.56s r2 | 장애인복지법 제32조에 따라 장애인 등록을 마친 분이 장애 또는 지체로 입학전형 진행 과정에서 지원이 필요하면 사전에 요청하시면 됩니다. 안내된 자료에는 지원 요청 방법이나 구체적인 지원 내용은 나와 있지 않습니다. (출처: 03_2027학년도_수시_모집요강.pdf p.35)
        gold: 장애인복지법 제32조에 따라 장애인등록을 마쳤고 장애 또는 지체로 전형 진행 과정에서 지원이 필요한 경우, 사전에 요청해야 합니다.
 8.29s r2 | 재외국민 전형의 경영학과는 최대모집 가능인원이 15명이고, 모집인원은 16명으로 안내되어 있습니다. 다만 모집인원은 변동될 수 있으므로 최종 확정되는 모집요강을 확인해야 합니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.3)
        gold: 경영학과의 최대모집 가능인원은 15명이고, 모집인원은 16명입니다.


In [109]:
import sys
print(sys.version.split()[0])
print("chunks:", len(globals().get("chunks", [])))
print("testset:", len(globals().get("testset", [])), "| hard:", len(globals().get("testset_hard", [])))
print("fns:", [k for k in ("retrieve_final", "retrieve_kw", "ask_v14", "evaluate_batch", "search_keywords") if k in globals()])
print("llm:", llm.model_name, llm.reasoning_effort, llm.max_tokens)
print("bm25 k:", bm25_retriever.k)


3.13.16
chunks: 514
testset: 20 | hard: 20
fns: ['retrieve_final', 'retrieve_kw', 'ask_v14', 'evaluate_batch', 'search_keywords']
llm: gpt-6-luna low 2000
bm25 k: 20


In [110]:
print("metadata keys:", sorted(chunks[0].metadata.keys()))
print("kind 분포:", __import__("collections").Counter(c.metadata.get("kind") for c in chunks))
print("문서별:", __import__("collections").Counter(c.metadata["source"][:2] for c in chunks))

print("\n=== 표 청크 예시 (01 p.3) ===")
for c in chunks:
    if c.metadata["source"].startswith("01") and c.metadata["page"] == 3:
        print(f"[kind={c.metadata['kind']} len={len(c.page_content)}]")
        print(c.page_content[:1200])
        print("-" * 60)

print("\n=== 목차 청크 예시 (04 p.2) ===")
for c in chunks:
    if c.metadata["source"].startswith("04") and c.metadata["page"] == 2:
        print(f"[kind={c.metadata['kind']} len={len(c.page_content)}]")
        print(c.page_content[:900])


metadata keys: ['kind', 'order', 'page', 'part', 'source']
kind 분포: Counter({'text': 278, 'table': 236})
문서별: Counter({'03': 276, '04': 124, '02': 68, '01': 38, '05': 8})

=== 표 청크 예시 (01 p.3) ===
[kind=table len=1520]
[01_2027학년도_재외국민_모집요강.hwp p.3]
| 모집인원 | 모집인원 | 모집인원 | 모집인원 | 모집인원 | 모집인원 |
|---|---|---|---|---|---|
| ※ 본 내용은 변동될 수 있으며 반드시 최종 확정되는 모집요강(본 대학 입학처 홈페이지 모집요강)을 확인하시기 바랍니다. |  |  |  |  |  |
| 계열 | 계열 | 모집단위 | 모집단위 | 최대모집 가능인원 | 모집인원 |
| 인문 | 인문 | 경영학과 | 경영학과 | 15 | 16 |
| 인문 | 인문 | 회계세무학과 | 회계세무학과 | 5 | 16 |
| 인문 | 인문 | 관광경영학과 | 관광경영학과 | 4 | 16 |
| 인문 | 인문 | 의료산업경영학과 | 의료산업경영학과 | 4 | 16 |
| 자연 | 자연 | 금융·빅데이터학부 | 금융수학전공 | 10 | 16 |
| 자연 | 자연 | 금융·빅데이터학부 | 빅데이터경영전공 | 10 | 16 |
| 인문 | 인문 | 미디어커뮤니케이션학과 | 미디어커뮤니케이션학과 | 3 | 16 |
| 인문 | 인문 | 경제학과 | 경제학과 | 5 | 16 |
| 인문 | 인문 | 응용통계학과 | 응용통계학과 | 4 | 16 |
| 인문 | 인문 | 사회복지학과 | 사회복지학과 | 4 | 16 |
| 인문 | 인문 | 심리학과 | 심리학과 | 4 | 16 |
| 인문 | 인문 | 패션산업학과 | 패션산업학과 | 4 | 16 |
| 인문 | 인문 | 한국어문학과 | AI인문대학 | 21 | 5 |
| 인문 | 인문 | 영미어문학과 | AI인문대학 

In [111]:
import collections

print("part 값 분포(상위 20):")
for p, n in collections.Counter(c.metadata.get("part") for c in chunks).most_common(20):
    print(f"  {n:4d} | {p}")

print("\n문서별 페이지 수:")
pages = collections.defaultdict(set)
for c in chunks:
    pages[c.metadata["source"]].add(c.metadata["page"])
for s, ps in pages.items():
    print(f"  {s}: {len(ps)}p ({min(ps)}~{max(ps)})")

print("\n=== part 예시가 있는 청크 ===")
seen = set()
for c in chunks:
    p = c.metadata.get("part")
    if p and p not in seen:
        seen.add(p)
        print(f"  [{c.metadata['source'][:2]} p.{c.metadata['page']}] part={p!r}")
    if len(seen) >= 12:
        break


part 값 분포(상위 20):
   381 | 0
   105 | 1
    15 | 2
     3 | 3
     3 | 4
     3 | 5
     2 | 6
     1 | 7
     1 | 8

문서별 페이지 수:
  01_2027학년도_재외국민_모집요강.hwp: 18p (1~19)
  02_2027학년도_지역의사선발전형_안내.pdf: 20p (1~20)
  03_2027학년도_수시_모집요강.pdf: 104p (1~104)
  04_2027학년도_정시_모집요강.pdf: 44p (1~44)
  05_2027학년도_수시_면접고사_일정.pdf: 4p (1~4)

=== part 예시가 있는 청크 ===
  [01 p.3] part=1
  [01 p.3] part=2
  [01 p.10] part=3
  [01 p.10] part=4
  [01 p.10] part=5
  [01 p.10] part=6
  [01 p.10] part=7
  [01 p.10] part=8


In [112]:
print("=== DRAFT_PROMPT ===")
print(DRAFT_PROMPT)
print("\n=== VERIFY_PROMPT ===")
print(VERIFY_PROMPT[:1500])
print("\n=== Verify 스키마 ===")
print(Verify.model_json_schema()["properties"].keys())
print("\n=== ctx_of ===")
import inspect
print(inspect.getsource(ctx_of))

=== DRAFT_PROMPT ===
You are an admissions counselor at Gachon University. Answer using ONLY the evidence documents.

Rules:
1. Include every key fact the question asks for (numbers, dates, targets, conditions).
2. If the evidence lists several conditions or exceptions, list ALL of them.
3. Never add conditions or qualifiers that are not in the evidence.
4. Copy dates exactly as written (e.g., 2027.01.04.(월) ~ 07.(목) 18:00).
5. When reading a table, align the row and column headers correctly.
6. Write the answer in Korean, 2-5 sentences, and end with (출처: 파일명 p.페이지).

=== VERIFY_PROMPT ===
You are a strict fact-checker for Gachon University admissions answers.

Given the question, the evidence, and a draft answer:
1. Check every key fact the question asks for (numbers, dates, targets, conditions) against the evidence.
2. If the draft omitted any key fact or condition that IS in the evidence, list it in `missing_facts`
   and produce a corrected `final_answer` that includes it.
3. Never

In [113]:
import re
from langchain_core.documents import Document

DOC_TITLES = {
    "01": "2027학년도 재외국민 모집요강",
    "02": "2027학년도 지역의사선발전형 안내",
    "03": "2027학년도 수시 모집요강",
    "04": "2027학년도 정시 모집요강",
    "05": "2027학년도 수시 면접고사 일정",
}
HEADER_HINTS = ("모집단위", "구분", "전형", "계열", "일시", "날짜", "대상", "내용",
                "학과", "기준", "인원", "비율", "점수", "등급", "시간", "장소", "비고", "지역")
GENERIC_COLS = {"계열", "구분", "비고", "연번", "번호"}


def doc_title(source):
    return DOC_TITLES.get(source[:2], source)


def md_rows(text):
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line.startswith("|"):
            continue
        cells = [c.strip() for c in line.strip("|").split("|")]
        if all(set(c) <= set("-: ") for c in cells):
            continue
        rows.append(cells)
    return rows


def dedup_cells(cells):
    out = []
    for c in cells:
        if not out or c != out[-1]:
            out.append(c)
    return out


def parse_table(text):
    """마크다운 표 → (표제, 헤더, 데이터행 리스트). 병합셀 반복은 제거."""
    rows = [dedup_cells(r) for r in md_rows(text)]
    if not rows:
        return None, [], []

    # 헤더: 앞 6행 중 힌트를 가장 많이 포함한 행
    best_i, best_score = None, 0
    for i, r in enumerate(rows[:6]):
        score = sum(any(h in c for h in HEADER_HINTS) for c in r)
        if score > best_score:
            best_i, best_score = i, score
    if best_i is None or best_score == 0:
        best_i = 0
    header = rows[best_i]

    # 표제: 헤더 앞 행들의 가장 긴 셀
    caption = ""
    for r in rows[:best_i]:
        cand = max(r, key=len) if r else ""
        if 2 < len(cand) < 60 and not cand.startswith("※"):
            caption = cand
    if not caption:
        cand = [c for c in header if c and not c.replace(" ", "").isdigit()]
        caption = max(cand, key=len) if cand else ""

    data = [r for r in rows[best_i + 1:] if r and not r[0].startswith("※")]
    return caption, header, data


def row_summary(header, data, max_rows=30, max_chars=2600):
    """표 데이터행을 '라벨: 열=값' 형태로 직렬화 (BM25/Dense 매칭용)."""
    cols = header
    lines, used = [], 0
    for r in data:
        if not r or (r[0] in ("합계", "총계")):
            continue
        vals = [c for c in r if c]
        if len(vals) < 2:
            continue
        label = next((c for c in vals if any(k in c for k in ("학과", "학부", "전공", "전형", "지역", "권역"))), vals[0])
        pairs = []
        for i, c in enumerate(r):
            if not c or c == label or c in GENERIC_COLS:
                continue
            col = cols[i] if i < len(cols) else ""
            if col in GENERIC_COLS or col == c:
                col = ""
            if col and (c.replace(",", "").replace(".", "").isdigit() or len(c) <= 25):
                pairs.append(f"{col}={c}")
            elif not col and c.replace(",", "").replace(".", "").isdigit():
                pairs.append(c)
        if not pairs:
            continue
        line = f"- {label}: " + ", ".join(pairs[:5])
        if used + len(line) > max_chars:
            break
        lines.append(line)
        used += len(line)
        if len(lines) >= max_rows:
            break
    return "\n".join(lines)


def build_contextual_chunks(chunks):
    """각 청크에 문서/페이지/표 맥락 헤더 + 표 행 요약을 붙인다."""
    out = []
    for c in chunks:
        src, page, kind = c.metadata["source"], c.metadata["page"], c.metadata["kind"]
        title = doc_title(src)
        raw = c.page_content
        header = f"[문서] {title} | p.{page} | {'표' if kind == 'table' else '본문'}"
        summary = ""
        if kind == "table":
            caption, hdr, data = parse_table(raw)
            if caption:
                header += f" | 표제: {caption}"
            summary = row_summary(hdr, data)
        body = header
        if summary:
            body += "\n[표 행 요약]\n" + summary
        body += "\n[원문]\n" + raw
        md = dict(c.metadata)
        md["raw"] = raw
        md["title"] = title
        md["contextual"] = True
        out.append(Document(page_content=body, metadata=md))
    return out


chunks_v15 = build_contextual_chunks(chunks)
raws = [len(c.metadata["raw"]) for c in chunks_v15]
alls = [len(c.page_content) for c in chunks_v15]
print(f"청크 {len(chunks_v15)} | 원문 평균 {sum(raws)/len(raws):.0f}자 → 색인 평균 {sum(alls)/len(alls):.0f}자")

for src, pg in (("01", 3), ("04", 7), ("05", 2)):
    for c in chunks_v15:
        if c.metadata["source"].startswith(src) and c.metadata["page"] == pg and c.metadata["kind"] == "table":
            print(f"\n=== {src} p.{pg} 색인 텍스트 앞부분 ===")
            print(c.page_content[:900])
            break


청크 514 | 원문 평균 574자 → 색인 평균 699자

=== 01 p.3 색인 텍스트 앞부분 ===
[문서] 2027학년도 재외국민 모집요강 | p.3 | 표 | 표제: 모집인원
[표 행 요약]
- 경영학과: 최대모집 가능인원=15, 모집인원=16
- 회계세무학과: 최대모집 가능인원=5, 모집인원=16
- 관광경영학과: 최대모집 가능인원=4, 모집인원=16
- 의료산업경영학과: 최대모집 가능인원=4, 모집인원=16
- 금융·빅데이터학부: 최대모집 가능인원=금융수학전공, 모집인원=10, 16
- 금융·빅데이터학부: 최대모집 가능인원=빅데이터경영전공, 모집인원=10, 16
- 미디어커뮤니케이션학과: 최대모집 가능인원=3, 모집인원=16
- 경제학과: 최대모집 가능인원=5, 모집인원=16
- 응용통계학과: 최대모집 가능인원=4, 모집인원=16
- 사회복지학과: 최대모집 가능인원=4, 모집인원=16
- 심리학과: 최대모집 가능인원=4, 모집인원=16
- 패션산업학과: 최대모집 가능인원=4, 모집인원=16
- 한국어문학과: 최대모집 가능인원=AI인문대학, 모집인원=21, 5
- 영미어문학과: 최대모집 가능인원=AI인문대학, 모집인원=21, 5
- 중국어문학과: 최대모집 가능인원=AI인문대학, 모집인원=21, 5
- 일본어문학과: 최대모집 가능인원=AI인문대학, 모집인원=21, 5
- 유럽어문학과: 최대모집 가능인원=AI인문대학, 모집인원=21, 5
- 법학과: 최대모집 가능인원=법과대학, 모집인원=15, 5
- 경찰행정학과: 최대모집 가능인원=법과대학, 모집인원=15, 5
- 행정학과: 최대모집 가능인원=법과대학, 모집인원=15, 5
- 도시계획·조경학부: 최대모집 가능인원=도시계획학전공, 모집인원=6, 27
- 도시계획·조경학부: 최대모집 가능인원=조경학전공, 모집인원=6, 27
- 건축학부: 최대모집 가능인원=실내건축학전공, 모집인원=10, 27
- 건축학부: 최대모집 가능인원=건축학전공

=== 04 p.7 색인 텍스트 앞부분 ===
[문서] 2027학년도

In [114]:
NUMERIC = re.compile(r"^[\d,./~%()~\-–\s]+$")
SYMBOLS = {"★", "○", "●", "◉", "O", "X", "△", "▲", "■", "□"}


def is_value(c):
    return bool(c) and (bool(NUMERIC.match(c)) or c in SYMBOLS)


def parse_table2(text):
    """마크다운 표 → (표제, 모드, 확장헤더, 데이터행)."""
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line.startswith("|"):
            continue
        cells = [c.strip() for c in line.strip("|").split("|")]
        if all(set(c) <= set("-: ") for c in cells):
            continue
        rows.append(cells)
    if not rows:
        return "", "roster", [], []

    # 데이터 시작: 학과/학부/전공/지역명이 처음 나오는 행
    start = None
    for i, r in enumerate(rows):
        if any(any(k in c for k in ("학과", "학부", "전공", "의예과", "지역")) for c in r):
            start = i
            break
    if start is None or start == 0:
        start = 1 if len(rows) > 1 else 0

    header_rows = rows[:start]
    data = [r for r in rows[start:] if r and not r[0].startswith("※")]

    # 확장 헤더: 열별로 위→아래 값을 중복 없이 결합
    width = max([len(r) for r in header_rows] + [len(r) for r in data] + [1])
    cols = []
    for j in range(width):
        parts = []
        for r in header_rows:
            v = r[j] if j < len(r) else ""
            if v and (not parts or parts[-1] != v):
                parts.append(v)
        cols.append(" ".join(parts))

    sched = any(re.search(r"\d+월|\d+일|시간|일시|오전|오후", c) for c in cols)
    mode = "wide" if (width >= 7 or sched) else "roster"

    caption = ""
    if sched:
        caption = "면접/전형 일정"
    else:
        cand = [c for c in cols if c and not is_value(c) and not c.startswith("※")]
        caption = max(cand, key=len) if cand else ""
    return caption, mode, cols, data


def row_summary2(cols, data, mode, max_rows=30, max_chars=2600):
    lines, used = [], 0
    for r in data:
        if not r:
            continue
        if mode == "wide":
            label = next((c for c in r if any(k in c for k in ("학과", "학부", "전공", "지역"))), "")
            pairs = []
            for j, v in enumerate(r):
                if not v or v == label or v in GENERIC_COLS:
                    continue
                col = cols[j] if j < len(cols) else ""
                if col and col != v:
                    pairs.append(f"{col}={v}")
            if not label or not pairs:
                continue
        else:
            # 병합셀 반복 제거 후, 숫자 꼬리를 헤더 오른쪽에 정렬
            ded = dedup_cells(r)
            vals = [c for c in ded if c]
            tail = []
            for c in reversed(vals):
                if is_value(c):
                    tail.append(c)
                else:
                    break
            tail.reverse()
            if len(tail) < 1:
                continue
            head = vals[:len(vals) - len(tail)]
            label = " ".join(c for c in head if c not in GENERIC_COLS)
            if not label or is_value(label):
                continue
            pairs = [f"{cols[-len(tail) + i] if len(cols) >= len(tail) else ''}={v}".lstrip("=")
                     for i, v in enumerate(tail)]
        line = f"- {label}: " + ", ".join(pairs[:6])
        if used + len(line) > max_chars or len(lines) >= max_rows:
            break
        lines.append(line)
        used += len(line)
    return "\n".join(lines)


def build_contextual_chunks(chunks):
    out = []
    for c in chunks:
        src, page, kind = c.metadata["source"], c.metadata["page"], c.metadata["kind"]
        title = doc_title(src)
        raw = c.page_content
        header = f"[문서] {title} | p.{page} | {'표' if kind == 'table' else '본문'}"
        summary = ""
        if kind == "table":
            caption, mode, cols, data = parse_table2(raw)
            if caption:
                header += f" | 표제: {caption}"
            summary = row_summary2(cols, data, mode)
        body = header + (("\n[표 행 요약]\n" + summary) if summary else "") + "\n[원문]\n" + raw
        md = dict(c.metadata)
        md.update(raw=raw, title=title, table_mode=locals().get("mode", ""), contextual=True)
        out.append(Document(page_content=body, metadata=md))
    return out


chunks_v15 = build_contextual_chunks(chunks)
alls = [len(c.page_content) for c in chunks_v15]
print(f"청크 {len(chunks_v15)} | 색인 평균 {sum(alls)/len(alls):.0f}자 | 최대 {max(alls):,}자")

for src, pg in (("01", 3), ("04", 7), ("05", 2)):
    for c in chunks_v15:
        if c.metadata["source"].startswith(src) and c.metadata["page"] == pg and c.metadata["kind"] == "table":
            print(f"\n=== {src} p.{pg} ===")
            print(c.page_content[:750])
            break


청크 514 | 색인 평균 664자 | 최대 2,983자

=== 01 p.3 ===
[문서] 2027학년도 재외국민 모집요강 | p.3 | 표 | 표제: 모집인원 ※ 본 내용은 변동될 수 있으며 반드시 최종 확정되는 모집요강(본 대학 입학처 홈페이지 모집요강)을 확인하시기 바랍니다. 계열
[표 행 요약]
- 인문 경영학과: 모집인원 최대모집 가능인원=15, 모집인원=16
- 인문 회계세무학과: 모집인원 최대모집 가능인원=5, 모집인원=16
- 인문 관광경영학과: 모집인원 최대모집 가능인원=4, 모집인원=16
- 인문 의료산업경영학과: 모집인원 최대모집 가능인원=4, 모집인원=16
- 자연 금융·빅데이터학부 금융수학전공: 모집인원 최대모집 가능인원=10, 모집인원=16
- 자연 금융·빅데이터학부 빅데이터경영전공: 모집인원 최대모집 가능인원=10, 모집인원=16
- 인문 미디어커뮤니케이션학과: 모집인원 최대모집 가능인원=3, 모집인원=16
- 인문 경제학과: 모집인원 최대모집 가능인원=5, 모집인원=16
- 인문 응용통계학과: 모집인원 최대모집 가능인원=4, 모집인원=16
- 인문 사회복지학과: 모집인원 최대모집 가능인원=4, 모집인원=16
- 인문 심리학과: 모집인원 최대모집 가능인원=4, 모집인원=16
- 인문 패션산업학과: 모집인원 최대모집 가능인원=4, 모집인원=16
- 인문 한국어문학과 AI인문대학: 모집인원 최대모집 가능인원=21, 모집인원=5
- 인문 영미어문학과 AI인문대학: 모집인원 최대모집 가능인원=21, 모집인원=5
- 인문 중국어문학과 AI인문대학: 모집인원 최대모집 가능인원=21, 모

=== 04 p.7 ===
[문서] 2027학년도 정시 모집요강 | p.7 | 표 | 표제: 일반전형 1 가군
[표 행 요약]
- 시스템반도체학과: 나군=18
- 컴퓨터공학과: 일반전형 1 가군=30, 일반전형 2 가군=18
- 인공지능학과: 다군=46, 다군=21
- 인공지능시스템학과: 일반전형 1 가군=17
- 정보보호학과: 나군=17
- 전기공학과: 

In [115]:
def expand_header(header_rows, width):
    """다단 헤더: 병합(빈칸) 구간을 가로로 전개해 열 이름을 만든다."""
    rows = []
    for r in header_rows:
        r = list(r) + [""] * (width - len(r))
        nonempty = sum(bool(c) for c in r)
        if nonempty * 2 <= width:          # 그룹 헤더행만 가로 전개
            last = ""
            for j in range(width):
                if r[j]:
                    last = r[j]
                elif last:
                    r[j] = last
        rows.append(r)
    cols = []
    for j in range(width):
        parts = []
        for r in rows:
            v = r[j]
            if v and (not parts or parts[-1] != v):
                parts.append(v)
        cols.append(" ".join(parts))
    return cols


def parse_table2(text):
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line.startswith("|"):
            continue
        cells = [c.strip() for c in line.strip("|").split("|")]
        if all(set(c) <= set("-: ") for c in cells):
            continue
        rows.append(cells)
    if not rows:
        return "", "roster", [], []

    start = None
    for i, r in enumerate(rows):
        if any(any(k in c for k in ("학과", "학부", "전공", "의예과", "지역")) for c in r):
            start = i
            break
    if start is None or start == 0:
        start = 1 if len(rows) > 1 else 0

    header_rows = rows[:start]
    data = [r for r in rows[start:] if r and not r[0].startswith("※")]
    width = max([len(r) for r in rows] + [1])

    sched = any(re.search(r"\d+월|\d+일|시간|일시|오전|오후", c)
                for r in header_rows for c in r)
    mode = "wide" if (width >= 7 or sched) else "roster"

    if mode == "wide":
        cols = expand_header(header_rows, width)
    else:
        base = header_rows[-1] if header_rows else []
        cols = dedup_cells([c for c in base if c]) or [c for c in base if c]

    caption = ""
    if sched:
        caption = "면접/전형 일정"
    else:
        first = [c for c in (header_rows[0] if header_rows else []) if c and c not in GENERIC_COLS]
        caption = first[0] if first else ""
        if len(caption) > 30:
            caption = caption[:30]
    return caption, mode, cols, data


def row_summary2(cols, data, mode, max_rows=30, max_chars=2600):
    H = len(cols)
    lines, used = [], 0
    for r in data:
        if not r:
            continue
        if mode == "wide":
            label = next((c for c in r if any(k in c for k in ("학과", "학부", "전공", "지역"))), "")
            pairs = []
            for j, v in enumerate(r):
                if not v or v == label or v in GENERIC_COLS:
                    continue
                col = cols[j] if j < len(cols) else ""
                if col and col != v:
                    pairs.append(f"{col}={v}")
            if not label or not pairs:
                continue
        else:
            ded = dedup_cells(r)
            # 헤더 폭에 맞춰 정렬 (병합 라벨 1칸 초과 / 꼬리 빈칸 허용)
            if len(ded) == H + 1:
                ded = [ded[0], f"{ded[1]} {ded[2]}".strip()] + ded[3:]
            if len(ded) < H:
                ded = [""] * (H - len(ded)) + ded
            if len(ded) > H:
                ded = ded[-H:]
            vals = [(j, v) for j, v in enumerate(ded) if v]
            if len(vals) < 2:
                continue
            first_val = next((j for j, v in vals if is_value(v)), H)
            label = " ".join(v for j, v in vals if j < first_val and v not in GENERIC_COLS)
            if not label or is_value(label):
                continue
            pairs = [f"{cols[j]}={v}" for j, v in vals if j >= first_val and cols[j]]
            if not pairs:
                continue
        line = f"- {label}: " + ", ".join(pairs[:6])
        if used + len(line) > max_chars or len(lines) >= max_rows:
            break
        lines.append(line)
        used += len(line)
    return "\n".join(lines)


def build_contextual_chunks(chunks):
    out, with_summary = [], 0
    for c in chunks:
        src, page, kind = c.metadata["source"], c.metadata["page"], c.metadata["kind"]
        title = doc_title(src)
        raw = c.page_content
        header = f"[문서] {title} | p.{page} | {'표' if kind == 'table' else '본문'}"
        summary, mode = "", ""
        if kind == "table":
            caption, mode, cols, data = parse_table2(raw)
            if caption:
                header += f" | 표제: {caption}"
            summary = row_summary2(cols, data, mode)
            if summary:
                with_summary += 1
        body = header + (("\n[표 행 요약]\n" + summary) if summary else "") + "\n[원문]\n" + raw
        md = dict(c.metadata)
        md.update(raw=raw, title=title, table_mode=mode, contextual=True)
        out.append(Document(page_content=body, metadata=md))
    print(f"표 청크 {sum(c.metadata['kind']=='table' for c in chunks)}개 중 행 요약 생성 {with_summary}개")
    return out


chunks_v15 = build_contextual_chunks(chunks)
print(f"색인 평균 {sum(len(c.page_content) for c in chunks_v15)/len(chunks_v15):.0f}자")

for src, pg in (("01", 3), ("04", 7), ("05", 2)):
    for c in chunks_v15:
        if c.metadata["source"].startswith(src) and c.metadata["page"] == pg and c.metadata["kind"] == "table":
            print(f"\n=== {src} p.{pg} (mode={c.metadata['table_mode']}) ===")
            print(c.page_content[:700])
            break


IndexError: list index out of range

In [116]:
def row_summary2(cols, data, mode, max_rows=30, max_chars=2600):
    H = len(cols)
    if H == 0:
        return ""
    lines, used = [], 0
    for r in data:
        if not r:
            continue
        if mode == "wide":
            label = next((c for c in r if any(k in c for k in ("학과", "학부", "전공", "지역"))), "")
            pairs = []
            for j, v in enumerate(r):
                if not v or v == label or v in GENERIC_COLS:
                    continue
                col = cols[j] if j < len(cols) else ""
                if col and col != v:
                    pairs.append(f"{col}={v}")
            if not label or not pairs:
                continue
        else:
            ded = dedup_cells(r)
            if len(ded) == H + 1 and len(ded) >= 3:
                ded = [ded[0], f"{ded[1]} {ded[2]}".strip()] + ded[3:]
            if len(ded) < H:
                ded = [""] * (H - len(ded)) + ded
            if len(ded) > H:
                ded = ded[-H:]
            vals = [(j, v) for j, v in enumerate(ded) if v]
            if len(vals) < 2:
                continue
            first_val = next((j for j, v in vals if is_value(v)), H)
            label = " ".join(v for j, v in vals if j < first_val and v not in GENERIC_COLS)
            if not label or is_value(label):
                continue
            pairs = [f"{cols[j]}={v}" for j, v in vals if j >= first_val and cols[j]]
            if not pairs:
                continue
        line = f"- {label}: " + ", ".join(pairs[:6])
        if used + len(line) > max_chars or len(lines) >= max_rows:
            break
        lines.append(line)
        used += len(line)
    return "\n".join(lines)


chunks_v15 = build_contextual_chunks(chunks)
print(f"색인 평균 {sum(len(c.page_content) for c in chunks_v15)/len(chunks_v15):.0f}자")

for src, pg in (("01", 3), ("04", 7), ("05", 2)):
    for c in chunks_v15:
        if c.metadata["source"].startswith(src) and c.metadata["page"] == pg and c.metadata["kind"] == "table":
            print(f"\n=== {src} p.{pg} (mode={c.metadata['table_mode']}) ===")
            print(c.page_content[:700])
            break


표 청크 236개 중 행 요약 생성 57개
색인 평균 668자

=== 01 p.3 (mode=roster) ===
[문서] 2027학년도 재외국민 모집요강 | p.3 | 표 | 표제: 모집인원
[표 행 요약]
- 인문 경영학과: 최대모집 가능인원=15, 모집인원=16
- 인문 회계세무학과: 최대모집 가능인원=5, 모집인원=16
- 인문 관광경영학과: 최대모집 가능인원=4, 모집인원=16
- 인문 의료산업경영학과: 최대모집 가능인원=4, 모집인원=16
- 자연 금융·빅데이터학부 금융수학전공: 최대모집 가능인원=10, 모집인원=16
- 자연 금융·빅데이터학부 빅데이터경영전공: 최대모집 가능인원=10, 모집인원=16
- 인문 미디어커뮤니케이션학과: 최대모집 가능인원=3, 모집인원=16
- 인문 경제학과: 최대모집 가능인원=5, 모집인원=16
- 인문 응용통계학과: 최대모집 가능인원=4, 모집인원=16
- 인문 사회복지학과: 최대모집 가능인원=4, 모집인원=16
- 인문 심리학과: 최대모집 가능인원=4, 모집인원=16
- 인문 패션산업학과: 최대모집 가능인원=4, 모집인원=16
- 인문 한국어문학과 AI인문대학: 최대모집 가능인원=21, 모집인원=5
- 인문 영미어문학과 AI인문대학: 최대모집 가능인원=21, 모집인원=5
- 인문 중국어문학과 AI인문대학: 최대모집 가능인원=21, 모집인원=5
- 인문 일본어문학과 AI인문대학: 최대모집 가능인원=21, 모집인원=5
- 인문 유럽어문학과 AI인문대학: 최대모집 가능인원=21, 모집인원=5
- 인문 법학과

=== 04 p.7 (mode=wide) ===
[문서] 2027학년도 정시 모집요강 | p.7 | 표 | 표제: 모집단위
[표 행 요약]
- 시스템반도체학과: 나군=18
- 컴퓨터공학과: 일반전형 1 가군=30, 일반전형 2 가군=18
- 인공지능학과: 다군=46, 다군=21
- 인공지능시스템학과: 일반전형 1 가군=17
- 정보보호학과: 나군=17
- 전기공학과: 일반전형 1 가군=21
- 스마트시티학과: 나군

In [117]:
# 다단 헤더: 내부 빈칸만 가로 전개 (그룹 헤더 병합 복원)
def expand_header(header_rows, width):
    rows = []
    for r in header_rows:
        r = list(r) + [""] * (width - len(r))
        last_nonempty = max((j for j, v in enumerate(r) if v), default=-1)
        for j in range(last_nonempty):
            if not r[j] and j > 0 and r[j - 1]:
                r[j] = r[j - 1]
        rows.append(r)
    cols = []
    for j in range(width):
        parts = []
        for r in rows:
            v = r[j]
            if v and (not parts or parts[-1] != v):
                parts.append(v)
        cols.append(" ".join(parts))
    return cols


# parse_table2 / row_summary2 / build_contextual_chunks 재정의 (expand_header 교체 반영)
def parse_table2(text):
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line.startswith("|"):
            continue
        cells = [c.strip() for c in line.strip("|").split("|")]
        if all(set(c) <= set("-: ") for c in cells):
            continue
        rows.append(cells)
    if not rows:
        return "", "roster", [], []
    start = None
    for i, r in enumerate(rows):
        if any(any(k in c for k in ("학과", "학부", "전공", "의예과", "지역")) for c in r):
            start = i
            break
    if start is None or start == 0:
        start = 1 if len(rows) > 1 else 0
    header_rows = rows[:start]
    data = [r for r in rows[start:] if r and not r[0].startswith("※")]
    width = max([len(r) for r in rows] + [1])
    sched = any(re.search(r"\d+월|\d+일|시간|일시|오전|오후", c) for r in header_rows for c in r)
    mode = "wide" if (width >= 7 or sched) else "roster"
    if mode == "wide":
        cols = expand_header(header_rows, width)
    else:
        base = header_rows[-1] if header_rows else []
        cols = dedup_cells([c for c in base if c]) or [c for c in base if c]
    if sched:
        caption = "면접/전형 일정"
    else:
        first = [c for c in (header_rows[0] if header_rows else []) if c and c not in GENERIC_COLS]
        caption = (first[0][:30] if first else "")
    return caption, mode, cols, data


def row_summary2(cols, data, mode, max_rows=30, max_chars=2600):
    H = len(cols)
    if H == 0:
        return ""
    lines, used = [], 0
    for r in data:
        if not r:
            continue
        if mode == "wide":
            label = next((c for c in r if any(k in c for k in ("학과", "학부", "전공", "지역"))), "")
            pairs = []
            for j, v in enumerate(r):
                if not v or v == label or v in GENERIC_COLS:
                    continue
                col = cols[j] if j < len(cols) else ""
                if col and col != v and col != "모집단위":
                    pairs.append(f"{col}={v}")
            if not label or not pairs:
                continue
        else:
            ded = dedup_cells(r)
            if len(ded) == H + 1 and len(ded) >= 3:
                ded = [ded[0], f"{ded[1]} {ded[2]}".strip()] + ded[3:]
            if len(ded) < H:
                ded = [""] * (H - len(ded)) + ded
            if len(ded) > H:
                ded = ded[-H:]
            vals = [(j, v) for j, v in enumerate(ded) if v]
            if len(vals) < 2:
                continue
            first_val = next((j for j, v in vals if is_value(v)), H)
            label = " ".join(v for j, v in vals if j < first_val and v not in GENERIC_COLS)
            if not label or is_value(label):
                continue
            pairs = [f"{cols[j]}={v}" for j, v in vals if j >= first_val and cols[j]]
            if not pairs:
                continue
        line = f"- {label}: " + ", ".join(pairs[:6])
        if used + len(line) > max_chars or len(lines) >= max_rows:
            break
        lines.append(line)
        used += len(line)
    return "\n".join(lines)


def build_contextual_chunks(chunks):
    out, with_summary = [], 0
    for c in chunks:
        src, page, kind = c.metadata["source"], c.metadata["page"], c.metadata["kind"]
        title = doc_title(src)
        raw = c.page_content
        header = f"[문서] {title} | p.{page} | {'표' if kind == 'table' else '본문'}"
        summary, mode = "", ""
        if kind == "table":
            caption, mode, cols, data = parse_table2(raw)
            if caption:
                header += f" | 표제: {caption}"
            summary = row_summary2(cols, data, mode)
            if summary:
                with_summary += 1
        body = header + (("\n[표 행 요약]\n" + summary) if summary else "") + "\n[원문]\n" + raw
        md = dict(c.metadata)
        md.update(raw=raw, title=title, table_mode=mode, contextual=True)
        out.append(Document(page_content=body, metadata=md))
    print(f"표 청크 {sum(c.metadata['kind']=='table' for c in chunks)}개 중 행 요약 {with_summary}개")
    return out


chunks_v15 = build_contextual_chunks(chunks)
for src, pg in (("04", 7), ("05", 2)):
    for c in chunks_v15:
        if c.metadata["source"].startswith(src) and c.metadata["page"] == pg and c.metadata["kind"] == "table":
            print(f"\n=== {src} p.{pg} ===")
            print("\n".join(c.page_content.splitlines()[:12]))
            break


표 청크 236개 중 행 요약 57개

=== 04 p.7 ===
[문서] 2027학년도 정시 모집요강 | p.7 | 표 | 표제: 모집단위
[표 행 요약]
- 시스템반도체학과: 일반전형 1 나군=18
- 컴퓨터공학과: 일반전형 1 가군=30, 일반전형 2 가군=18
- 인공지능학과: 일반전형 1 다군=46, 일반전형 2 다군=21
- 인공지능시스템학과: 일반전형 1 가군=17
- 정보보호학과: 일반전형 1 나군=17
- 전기공학과: 일반전형 1 가군=21
- 스마트시티학과: 일반전형 1 나군=16
- 의공학과: 일반전형 1 다군=17
- 간호학과: 일반전형 1 나군=70, 일반전형 2 나군=30
- 치위생학과: 일반전형 1 다군=13

=== 05 p.2 ===
[문서] 2027학년도 수시 면접고사 일정 | p.2 | 표 | 표제: 면접/전형 일정
[표 행 요약]
- 경영학과: 10월 31일(토) 오전=★, 10월 31일(토) 오후=★, 11월 01일(일) 오전=★, 11월 01일(일) 오후=★
- 회계세무학과: 11월 07일(토) 오전=★, 11월 07일(토) 오후=★
- 관광경영학과: 10월 31일(토) 오후=★
- 의료산업경영학과: 10월 31일(토) 오전=★, 10월 31일(토) 오후=★
- 금융·빅데이터학부: 11월 07일(토) 오전=★, 11월 07일(토) 오후=★
- 미디어커뮤니케이션학과: 11월 07일(토) 오후=★
- 경제학과: 11월 08일(일) 오전=★, 오후=★
- 응용통계학과: 11월 01일(일) 오전=★, 11월 01일(일) 오후=★
- 사회복지학과: 11월 07일(토) 오전=★, 11월 07일(토) 오후=★
- 유아교육학과: 11월 07일(토) 오전=★, 11월 07일(토) 오후=★


In [118]:
import time
from collections import defaultdict
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever

# ---------- 1) 청크 인덱스 (contextual + table-aware) ----------
t0 = time.perf_counter()
vectordb_v15 = Chroma.from_documents(
    documents=chunks_v15,
    embedding=embeddings,
    collection_name=f"gachon_v15_{int(time.time())}",
    collection_metadata={"hnsw:space": "cosine"},
)
dense_v15 = vectordb_v15.as_retriever(search_kwargs={"k": 20})
bm25_v15 = BM25Retriever.from_documents(chunks_v15, preprocess_func=korean_tokenizer, k=20)
print(f"청크 인덱스: {time.perf_counter()-t0:.1f}s")

# ---------- 2) 페이지 인덱스 (hierarchical) ----------
groups = defaultdict(list)
for c in chunks_v15:
    groups[(c.metadata["source"], c.metadata["page"])].append(c)

page_docs = []
for (src, page), cs in sorted(groups.items()):
    body = []
    used = 0
    for c in cs:
        t = c.metadata["raw"].strip()
        if used + len(t) > 3200:
            t = t[: max(0, 3200 - used)]
        body.append(t)
        used += len(t)
        if used >= 3200:
            break
    page_docs.append(Document(
        page_content=f"[{doc_title(src)} p.{page}]\n" + "\n".join(body),
        metadata={"source": src, "page": page, "title": doc_title(src)},
    ))

t0 = time.perf_counter()
page_db = Chroma.from_documents(
    documents=page_docs,
    embedding=embeddings,
    collection_name=f"gachon_pages_{int(time.time())}",
    collection_metadata={"hnsw:space": "cosine"},
)
page_dense = page_db.as_retriever(search_kwargs={"k": 8})
page_bm25 = BM25Retriever.from_documents(page_docs, preprocess_func=korean_tokenizer, k=8)
print(f"페이지 인덱스: {len(page_docs)}p, {time.perf_counter()-t0:.1f}s")
print("전체 구축 완료")


청크 인덱스: 9.3s
페이지 인덱스: 190p, 5.9s
전체 구축 완료


In [119]:
import re

CHUNK_INDEX_V15 = {c.page_content: i for i, c in enumerate(chunks_v15)}

ROUTE_PATTERNS = [
    ("page", r"(몇\s*쪽|페이지|쪽에|몇\s*장)"),
    ("table", r"(몇\s*명|몇명|인원|모집|선발|정원|명까지|명씩|비율|점수|등급|컷)"),
    ("procedure", r"(어떻게|신청|제출|절차|방법|해야\s*하|어디서)"),
    ("eligibility", r"(자격|조건|대상|가능|되나요|될까|수\s*있나|받을\s*수)"),
    ("list", r"(어디어디|무엇무엇|종류|예시|포함|목록|어떤\s*게)"),
]


def route_query(q):
    for name, pat in ROUTE_PATTERNS:
        if re.search(pat, q):
            return name
    return "general"


ROUTE_SUFFIX = {
    "table": " 모집인원 표",
    "page": " 목차 페이지",
    "procedure": " 신청 방법 절차",
    "eligibility": " 지원자격 조건",
    "list": " 목록",
    "general": "",
}


def route_queries(q):
    kw = search_keywords(q)
    return route_query(q), kw, (kw + ROUTE_SUFFIX[route_query(q)]).strip()


def rrf_fuse(result_lists, weights, rrf_k=60):
    scores, docs = {}, {}
    for w, result in zip(weights, result_lists):
        for rank, d in enumerate(result, 1):
            key = d.page_content
            scores[key] = scores.get(key, 0.0) + w / (rrf_k + rank)
            docs[key] = d
    return scores, docs


def page_prior(query, top_pages=5):
    """계층 검색: 페이지 단위로 먼저 찾고, 해당 페이지 청크에 가산점."""
    s, d = rrf_fuse([page_bm25.invoke(query), page_dense.invoke(query)], [1.0, 1.0], rrf_k=40)
    ordered = sorted(s, key=s.get, reverse=True)[:top_pages]
    return {(d[k].metadata["source"], d[k].metadata["page"]): i + 1 for i, k in enumerate(ordered)}


def retrieve_v15(query, top_n=6, expand_top=2, rrf_k=60):
    route, kw, kw_route = route_queries(query)
    lists = [bm25_v15.invoke(query), dense_v15.invoke(query),
             bm25_v15.invoke(kw), dense_v15.invoke(kw_route)]
    weights = [1.0, 1.0, 1.15, 1.15]
    scores, docs = rrf_fuse(lists, weights, rrf_k)

    prior = page_prior(kw_route)
    for key, d in docs.items():
        r = prior.get((d.metadata["source"], d.metadata["page"]))
        if r:
            scores[key] += 0.45 / (40 + r)
        if route == "table" and d.metadata["kind"] == "table":
            scores[key] *= 1.15
        if route in ("procedure", "eligibility", "page") and d.metadata["kind"] == "text":
            scores[key] *= 1.12

    top = [docs[k] for k in sorted(scores, key=scores.get, reverse=True)[:top_n]]
    out, seen = [], set()
    for n, d in enumerate(top):
        if d.page_content not in seen:
            out.append(d)
            seen.add(d.page_content)
        if n >= expand_top:
            continue
        i = CHUNK_INDEX_V15[d.page_content]
        for j in range(max(0, i - 1), min(len(chunks_v15), i + 2)):
            c = chunks_v15[j]
            if c.metadata["source"] == d.metadata["source"] and c.page_content not in seen:
                out.append(c)
                seen.add(c.page_content)
    return out, route


def parse_gold(src):
    m = re.search(r"(\d{2}).*?p\.\s*(\d+)", src)
    return (m.group(1), int(m.group(2))) if m else (None, None)


def recall_eval(questions, k=6, verbose=True):
    hits, near, misses = 0, 0, []
    for t in questions:
        gs, gp = parse_gold(t["src"])
        docs, route = retrieve_v15(t["q"], top_n=k)
        pages = [(d.metadata["source"][:2], d.metadata["page"]) for d in docs]
        ok = (gs, gp) in pages
        near_ok = ok or any(s == gs and abs(p - gp) <= 1 for s, p in pages)
        hits += ok
        near += near_ok
        if not ok:
            misses.append((t["q"][:40], f"{gs} p.{gp}", route, pages[:6]))
    if verbose:
        for m in misses:
            print("  MISS", m[1], f"[{m[2]}]", "|", m[0], "|", m[3])
    return hits, near, len(questions)


print("=== 구어체 20문항 검색 평가 ===")
h1, n1, tot1 = recall_eval(testset_hard)
print(f"Recall@6: {h1}/{tot1} | ±1p: {n1}/{tot1}\n")
print("=== 기존 20문항 검색 평가 ===")
h2, n2, tot2 = recall_eval(testset)
print(f"Recall@6: {h2}/{tot2} | ±1p: {n2}/{tot2}")


=== 구어체 20문항 검색 평가 ===
  MISS 02 p.4 [eligibility] | 지역의사 전형에서 두 군데 지역에 같이 넣을 수 있나요? 어디에 물어봐야 | [('02', 18), ('02', 17), ('02', 18), ('02', 3), ('02', 2), ('02', 3)]
  MISS 04 p.2 [page] | 정시 요강에서 원서접수 얘기는 몇 쪽에 나와요? | [('04', 9), ('04', 8), ('04', 10), ('03', 36), ('03', 35), ('03', 37)]
  MISS 04 p.11 [page] | 농어촌 전형은 정시 요강 몇 쪽에 있어요? | [('04', 18), ('04', 18), ('04', 19), ('04', 17), ('04', 2), ('03', 48)]
  MISS 05 p.1 [eligibility] | 이 면접 일정표는 어디 대학 거예요? 수시 중에 어떤 전형 대상이에요? | [('03', 61), ('03', 60), ('03', 62), ('03', 89), ('03', 88), ('03', 89)]
Recall@6: 16/20 | ±1p: 17/20

=== 기존 20문항 검색 평가 ===
  MISS 05 p.1 [table] | 이 면접고사 일정 안내는 어느 대학의 몇 학년도 수시모집 중 어떤 전형을 | [('03', 76), ('03', 76), ('03', 76), ('03', 61), ('03', 60), ('03', 62)]
Recall@6: 19/20 | ±1p: 19/20


In [120]:
def recall_old(questions, k=6):
    hits = 0
    misses = []
    for t in questions:
        gs, gp = parse_gold(t["src"])
        docs = retrieve_final(t["q"], top_n=k)
        pages = [(d.metadata["source"][:2], d.metadata["page"]) for d in docs]
        if (gs, gp) in pages:
            hits += 1
        else:
            misses.append((t["q"][:35], f"{gs} p.{gp}", pages[:6]))
    return hits, misses


for name, qs in (("hard", testset_hard), ("orig", testset)):
    h, ms = recall_old(qs)
    print(f"[기존 retrieve_final] {name}: {h}/{len(qs)}")
    for m in ms:
        print("   MISS", m[1], "|", m[0], "|", m[2])

print("\n" + "=" * 70)
for src, pg in (("02", 4), ("04", 2), ("04", 11), ("05", 1)):
    for c in chunks_v15:
        if c.metadata["source"].startswith(src) and c.metadata["page"] == pg:
            print(f"\n=== {src} p.{pg} raw ({len(c.metadata['raw'])}자) ===")
            print(c.metadata["raw"][:800])
            break


[기존 retrieve_final] hard: 16/20
   MISS 01 p.5 | 몸이 좀 불편한 학생인데요, 시험 볼 때 도와주는 사람 받을 수 | [('03', 4), ('03', 4), ('03', 5), ('03', 56), ('03', 56), ('03', 57)]
   MISS 01 p.3 | 외국에서 살다 온 사람들 뽑는 거 있잖아요, 경영학과는 최대 몇 | [('01', 9), ('01', 8), ('01', 10), ('01', 12), ('01', 11), ('01', 12)]
   MISS 04 p.2 | 정시 요강에서 원서접수 얘기는 몇 쪽에 나와요? | [('04', 9), ('04', 8), ('04', 10), ('03', 36), ('03', 35), ('03', 37)]
   MISS 04 p.7 | 인천 캠퍼스에 바이오로직스학과 있잖아요, 거기 정시로 몇명 뽑아 | [('04', 5), ('04', 5), ('04', 6), ('01', 3), ('01', 3), ('01', 3)]
[기존 retrieve_final] orig: 20/20


=== 02 p.4 raw (817자) ===
[02_2027학년도_지역의사선발전형_안내.pdf p.4]
02
출신 중학교·고등학교 소재지 및 거주 요건
1
법 제4조, 시행령 제2조
요건 구분
세부 내용
학교 소재지 요건
해당 의과대학이 소재한 지역 또는 인접지역으로서 대통령령으로 정하는 지역 참고 1 의
중학교 및 고등학교를 입학부터 졸업까지 전 교육과정 이수 후 졸업한 사람(고등학교는 졸업예정자 포함)
거주 요건
중학교 및 고등학교 재학기간 내 해당 학교가 소재한 지역에 실제 거주해야 함
※ 재학기간 중 일부라도 해당 지역 외로 주소를 이전하면 자격요건 미충족 가능
※ 기숙사 거주 등 거주요건 해당 여부는 개별 대학 모집 요강 확인 필요
※ 부모 거주지는 지원요건에 해당하지 않음
선발 비율
100% 위 요건 충족자로 선발(지역별 세부 비율 - 선발고시 제4조 및 별표 참조)
※ 

In [121]:
for i in (6, 12, 14, 16):
    t = testset_hard[i]
    route, kw, kwr = route_queries(t["q"])
    print(f"\n[{i}] {t['q'][:60]}")
    print(f"  route={route} | kw={kw!r}")
    print(f"  kw_route={kwr!r}")

print("\n=== '복수지원' 포함 청크 ===")
for c in chunks_v15:
    if "복수지원" in c.metadata["raw"] or "동시 지원" in c.metadata["raw"]:
        print(f"  {c.metadata['source'][:2]} p.{c.metadata['page']} | {c.metadata['kind']}")
        idx = c.metadata["raw"].find("복수지원")
        print("   ", c.metadata["raw"][max(0, idx-120):idx+180].replace("\n", " "))

print("\n=== TOC 후보 페이지 (숫자 시작 줄 3개 이상) ===")
toc = []
for c in chunks_v15:
    if c.metadata["kind"] != "text":
        continue
    lines = [l.strip() for l in c.metadata["raw"].splitlines() if l.strip()]
    n_num = sum(bool(re.match(r"^\d{1,3}\s+\S", l)) for l in lines)
    if n_num >= 3:
        toc.append((c.metadata["source"][:2], c.metadata["page"], n_num, len(lines)))
for t in toc:
    print("  ", t)



[6] 지역의사 전형에서 두 군데 지역에 같이 넣을 수 있나요? 어디에 물어봐야 되고, 다른 전형처럼 여러 개 지원
  route=eligibility | kw='지역 의사 전형 지원'
  kw_route='지역 의사 전형 지원 지원자격 조건'

[12] 정시 요강에서 원서접수 얘기는 몇 쪽에 나와요?
  route=page | kw='요강 접수 얘기 원서접수 정시모집'
  kw_route='요강 접수 얘기 원서접수 정시모집 목차 페이지'

[14] 농어촌 전형은 정시 요강 몇 쪽에 있어요?
  route=page | kw='농어촌 전형 요강 정시모집'
  kw_route='농어촌 전형 요강 정시모집 목차 페이지'

[16] 이 면접 일정표는 어디 대학 거예요? 수시 중에 어떤 전형 대상이에요?
  route=eligibility | kw='면접 일정표 대학 전형 대상 수시모집'
  kw_route='면접 일정표 대학 전형 대상 수시모집 지원자격 조건'

=== '복수지원' 포함 청크 ===
  01 p.13 | table
    자 유의사항 | |---|---| | 지원자는 불이익이 발생하지 않도록 다음 사항에 각별히 유의하여 주시기 바랍니다. 수시모집에 해당하는 모든 전형을 대상으로 지원가능한 횟수는 최대 6회로 제한합니다. |  | | 복수지원 허용범위 | 수시모집 대학(교육대학 포함, 산업대학·전문대학 제외)에 있어서는 전형기간이 같아도 최대 6개 전형 이내 에서 복수지원이 가능합니다. | | 복수지원 금지 | 수시모집 대학(산업대학, 교육대학, 전문대학 포함)의 합격자(최초합격자 및 충원합격자)는 등록(예치)과 상관없이 ‘정시모집 및 추가모집’에 지
  02 p.4 | text
    (시행령 별표2 제2호나목) ※ ‌진료권·광역권 동시 지원 가능 여부 및 지역의사선발전형 외 일반전형과의 중복지원 지원 가능 여부는 개별 대학 모집 요강 확인 필요 ※ 한국대학교육협의회 「대학입학전형 기본사항」의 복수지원 금지 및 이중등록 금지 규정은 지역의사선발전형

In [122]:
SYNONYMS.update({
    "두 군데": "복수지원", "같이 넣": "복수지원", "중복 지원": "복수지원",
    "동시 지원": "복수지원", "여러 개 지원": "복수지원", "동시에": "복수지원",
    "일정표": "일정", "몇 쪽": "페이지", "몇쪽": "페이지",
})

# TOC 페이지 자동 탐지
TOC_PAGES = {}
for c in chunks_v15:
    if c.metadata["kind"] != "text":
        continue
    lines = [l.strip() for l in c.metadata["raw"].splitlines() if l.strip()]
    n_num = sum(bool(re.match(r"^\d{1,3}\s+\S", l)) for l in lines)
    if n_num >= 5 and n_num / max(1, len(lines)) >= 0.3:
        TOC_PAGES.setdefault(c.metadata["source"][:2], set()).add(c.metadata["page"])
print("TOC 페이지:", {k: sorted(v) for k, v in TOC_PAGES.items()})

DOC_KEYWORDS = [
    ("05", r"(면접.*(일정|일정표|날짜|시간)|(일정|일정표).*면접)"),
    ("04", r"정시"),
    ("03", r"수시"),
    ("02", r"지역\s*의사"),
    ("01", r"(재외국민|외국)"),
]


def target_doc(q):
    for prefix, pat in DOC_KEYWORDS:
        if re.search(pat, q):
            return prefix
    return None


def route_query(q):
    if re.search(r"(면접.*(일정|일정표|날짜|시간)|(일정|일정표).*면접)", q):
        return "schedule"
    for name, pat in ROUTE_PATTERNS:
        if re.search(pat, q):
            return name
    return "general"


ROUTE_SUFFIX["schedule"] = " 면접 일정"


def overlap_score(doc, kw):
    toks = [t for t in kw.split() if len(t) > 1]
    text = doc.metadata.get("raw", doc.page_content)
    return sum(text.count(t) for t in toks)


def inject(scores, docs, pool, top_score, factor=1.05, limit=2):
    """경로별 필수 근거(목차/일정표)를 후보에 강제 주입."""
    ranked = sorted(pool, key=lambda d: overlap_score(d, " ".join(docs)), reverse=True)[:limit]
    for d in ranked:
        key = d.page_content
        scores[key] = max(scores.get(key, 0.0), top_score * factor)
        docs[key] = d
    return ranked


def retrieve_v15(query, top_n=6, expand_top=2, rrf_k=60):
    route, kw, kw_route = route_queries(query)
    lists = [bm25_v15.invoke(query), dense_v15.invoke(query),
             bm25_v15.invoke(kw), dense_v15.invoke(kw_route)]
    scores, docs = rrf_fuse(lists, [1.0, 1.0, 1.15, 1.15], rrf_k)

    prior = page_prior(kw_route)
    for key, d in docs.items():
        r = prior.get((d.metadata["source"], d.metadata["page"]))
        if r:
            scores[key] += 0.45 / (40 + r)
        if route == "table" and d.metadata["kind"] == "table":
            scores[key] *= 1.15
        if route in ("procedure", "eligibility", "page") and d.metadata["kind"] == "text":
            scores[key] *= 1.12

    top_score = max(scores.values()) if scores else 0.0
    tdoc = target_doc(query)

    # 목차 질문 → 해당 문서의 TOC 페이지를 근거로 강제 주입
    if route == "page" and tdoc and tdoc in TOC_PAGES:
        pool = [c for c in chunks_v15
                if c.metadata["source"].startswith(tdoc) and c.metadata["page"] in TOC_PAGES[tdoc]]
        inject(scores, [kw], pool, top_score, factor=1.12, limit=2)

    # 면접 일정 질문 → 05 문서(일정표)로 라우팅
    if route == "schedule":
        pool = [c for c in chunks_v15 if c.metadata["source"].startswith("05")]
        inject(scores, [kw], pool, top_score, factor=1.15, limit=3)

    top = [docs[k] for k in sorted(scores, key=scores.get, reverse=True)[:top_n]]
    out, seen = [], set()
    for n, d in enumerate(top):
        if d.page_content not in seen:
            out.append(d)
            seen.add(d.page_content)
        if n >= expand_top:
            continue
        i = CHUNK_INDEX_V15[d.page_content]
        for j in range(max(0, i - 1), min(len(chunks_v15), i + 2)):
            c = chunks_v15[j]
            if c.metadata["source"] == d.metadata["source"] and c.page_content not in seen:
                out.append(c)
                seen.add(c.page_content)
    return out, route


print("\n=== 재측정 ===")
h1, n1, _ = recall_eval(testset_hard)
print(f"hard Recall@6 {h1}/20 | ±1p {n1}/20\n")
h2, n2, _ = recall_eval(testset)
print(f"orig Recall@6 {h2}/20 | ±1p {n2}/20")


TOC 페이지: {'03': [3, 28, 38, 82, 96, 102], '04': [2, 11, 31]}

=== 재측정 ===


TypeError: list indices must be integers or slices, not str

In [123]:
def inject(scores, docs, kw, pool, top_score, factor=1.05, limit=2):
    ranked = sorted(pool, key=lambda d: overlap_score(d, kw), reverse=True)[:limit]
    for d in ranked:
        key = d.page_content
        scores[key] = max(scores.get(key, 0.0), top_score * factor)
        docs[key] = d
    return ranked


def retrieve_v15(query, top_n=6, expand_top=2, rrf_k=60):
    route, kw, kw_route = route_queries(query)
    lists = [bm25_v15.invoke(query), dense_v15.invoke(query),
             bm25_v15.invoke(kw), dense_v15.invoke(kw_route)]
    scores, docs = rrf_fuse(lists, [1.0, 1.0, 1.15, 1.15], rrf_k)

    prior = page_prior(kw_route)
    for key, d in docs.items():
        r = prior.get((d.metadata["source"], d.metadata["page"]))
        if r:
            scores[key] += 0.45 / (40 + r)
        if route == "table" and d.metadata["kind"] == "table":
            scores[key] *= 1.15
        if route in ("procedure", "eligibility", "page") and d.metadata["kind"] == "text":
            scores[key] *= 1.12

    top_score = max(scores.values()) if scores else 0.0
    tdoc = target_doc(query)

    if route == "page" and tdoc and tdoc in TOC_PAGES:
        pool = [c for c in chunks_v15
                if c.metadata["source"].startswith(tdoc) and c.metadata["page"] in TOC_PAGES[tdoc]]
        inject(scores, docs, kw, pool, top_score, factor=1.12, limit=2)

    if route == "schedule":
        pool = [c for c in chunks_v15 if c.metadata["source"].startswith("05")]
        inject(scores, docs, kw, pool, top_score, factor=1.15, limit=3)

    top = [docs[k] for k in sorted(scores, key=scores.get, reverse=True)[:top_n]]
    out, seen = [], set()
    for n, d in enumerate(top):
        if d.page_content not in seen:
            out.append(d)
            seen.add(d.page_content)
        if n >= expand_top:
            continue
        i = CHUNK_INDEX_V15[d.page_content]
        for j in range(max(0, i - 1), min(len(chunks_v15), i + 2)):
            c = chunks_v15[j]
            if c.metadata["source"] == d.metadata["source"] and c.page_content not in seen:
                out.append(c)
                seen.add(c.page_content)
    return out, route


print("=== 재측정 ===")
h1, n1, _ = recall_eval(testset_hard)
print(f"hard Recall@6 {h1}/20 | ±1p {n1}/20\n")
h2, n2, _ = recall_eval(testset)
print(f"orig Recall@6 {h2}/20 | ±1p {n2}/20")


=== 재측정 ===
hard Recall@6 20/20 | ±1p 20/20

orig Recall@6 20/20 | ±1p 20/20


In [124]:
import inspect
print(inspect.getsource(evaluate_batch))

def evaluate_batch(fn, dataset, start, end, tag=""):
    rows = []
    for i in range(start, end):
        t = dataset[i]
        out, rounds, sec = fn(t["q"])
        v = judge.invoke([
            ("system", JUDGE),
            ("human", f"[질문]\n{t['q']}\n\n[정답]\n{t['gold']}\n\n[예측]\n{out}"),
        ])
        rows.append({"q": t["q"], "gold": t["gold"], "pred": out, "ok": v.correct,
                     "why": v.reason, "sec": sec, "rounds": rounds})
        print(f"{i+1:2d}. {'O' if v.correct else 'X'} {sec:4.1f}s | {t['q'][:50]}")
    ok = sum(r["ok"] for r in rows)
    print(f"[{tag} {start+1}-{end}] {ok}/{len(rows)} | 평균 {sum(r['sec'] for r in rows)/len(rows):.2f}s | 최대 {max(r['sec'] for r in rows):.2f}s")
    return rows



In [125]:
DRAFT_PROMPT_V2 = """You are an admissions counselor at Gachon University. Answer using ONLY the evidence documents.

Rules:
1. Include EVERY key fact the question asks for (numbers, dates, targets, conditions, exceptions).
2. If the evidence lists several conditions or exceptions, list ALL of them.
3. Never add facts, conditions, or qualifiers that are not in the evidence.
4. For tables, align the row label and column headers by position. Never swap columns.
5. If the evidence contains the answer, NEVER say you cannot confirm it.
6. If the evidence covers only part of the question, answer the supported part and briefly state what is missing.
7. If the question asks which page, give the page number exactly as listed in the table of contents.
8. Write in Korean, 2-5 sentences, and end with (출처: 파일명 p.페이지)."""

REPAIR_PROMPT = """You are a strict fact-checker. Rewrite the draft using ONLY the evidence.

- Remove any number, date, or name that is not present in the evidence.
- Add any key fact from the evidence that the question asks for but the draft omitted.
- Keep the answer in Korean, 2-5 sentences, ending with (출처: 파일명 p.페이지).
- If the evidence truly lacks the answer, say what is missing in one sentence."""

NUM_RE = re.compile(r"\d[\d,]*(?:\.\d+)?")
DATE_RE = re.compile(r"\d{1,2}\s*월\s*\d{1,2}\s*일")
ENT_RE = re.compile(r"[가-힣A-Za-z0-9·]+(?:학과|학부|전공|전형|대학|고등학교|중학교)")
CITE_RE = re.compile(r"\(출처:\s*([^)]*?)\s*p\.\s*(\d+)\)")
NOT_FOUND = "확인할 수 없습니다"


def _norm(s):
    return re.sub(r"\s+", "", s)


def _strip_cite(s):
    return CITE_RE.sub("", s)


def ground_answer(answer, docs, question=""):
    """답변의 숫자·날짜·개체·인용을 근거와 대조해 grounding 점수를 계산."""
    evidence = _norm("\n".join(d.metadata.get("raw", d.page_content) for d in docs))
    body = _strip_cite(answer)

    nums = [n for n in NUM_RE.findall(body) if len(n.replace(",", "")) >= 1]
    dates = DATE_RE.findall(body)
    ents = [e for e in ENT_RE.findall(body) if len(e) >= 3]

    num_ok = sum(_norm(n) in evidence for n in nums) / len(nums) if nums else 1.0
    date_ok = sum(_norm(d) in evidence for d in dates) / len(dates) if dates else 1.0
    ent_ok = sum(_norm(e) in evidence for e in ents) / len(ents) if ents else 1.0

    cites = CITE_RE.findall(answer)
    valid = 0
    for fname, page in cites:
        key = fname.strip()[:2]
        if any(d.metadata["source"].startswith(key) and str(d.metadata["page"]) == page for d in docs):
            valid += 1
    cite_ok = valid / len(cites) if cites else 0.0

    score = 0.40 * num_ok + 0.20 * date_ok + 0.25 * ent_ok + 0.15 * cite_ok
    return {
        "score": round(score, 3), "num_ok": round(num_ok, 3), "date_ok": round(date_ok, 3),
        "ent_ok": round(ent_ok, 3), "cite_ok": round(cite_ok, 3),
        "n_num": len(nums), "n_date": len(dates), "n_ent": len(ents),
        "not_found": NOT_FOUND in body,
    }


def lexical_coverage(kw, docs):
    toks = [t for t in kw.split() if len(t) > 1]
    if not toks:
        return 0.0
    ev = _norm(" ".join(d.metadata.get("raw", "") for d in docs[:5]))
    return sum(_norm(t) in ev for t in toks) / len(toks)


def make_confidence(route, kw, docs, g):
    cov = lexical_coverage(kw, docs)
    route_bonus = {"table": 0.06, "page": 0.05, "schedule": 0.05}.get(route, 0.0)
    conf = 0.50 * g["score"] + 0.35 * cov + 0.15 * (1.0 if g["cite_ok"] >= 0.99 else 0.5) + route_bonus
    if g["not_found"]:
        conf -= 0.25
    return round(max(0.0, min(1.0, conf)), 3), round(cov, 3)


def ask_v16(question, top_n=6, t_budget=9.0, return_meta=False):
    t0 = time.perf_counter()
    docs, route = retrieve_v15(question, top_n=top_n)
    kw = search_keywords(question)
    ctx = "\n\n".join(f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.metadata['raw']}"
                      for i, d in enumerate(docs, 1))

    try:
        answer = llm.invoke([("system", DRAFT_PROMPT_V2),
                             ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content
    except Exception as e:
        print("   (draft 실패:", type(e).__name__, ")")
        answer = NOT_FOUND + "."

    g = ground_answer(answer, docs, question)
    conf, cov = make_confidence(route, kw, docs, g)
    retried = False

    # 근거 미충족 → 키워드 재검색 병합 후 교정 1회
    if (g["score"] < 0.78 or g["not_found"]) and time.perf_counter() - t0 < t_budget:
        retried = True
        docs2 = [d for d in retrieve_kw(question, top_n=4, expand_top=1)
                if d.metadata["source"] not in {x.metadata["source"] for x in docs}]
        merged, seen = [], set()
        for d in docs + docs2:
            if d.metadata["raw"] not in seen:
                seen.add(d.metadata["raw"])
                merged.append(d)
        ctx2 = "\n\n".join(f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.metadata['raw']}"
                           for i, d in enumerate(merged, 1))
        try:
            answer = llm.invoke([("system", REPAIR_PROMPT),
                                 ("human", f"[Evidence]\n{ctx2}\n\n[Question]\n{question}\n\n[Draft]\n{answer}")]).content
        except Exception:
            pass
        docs = merged
        g = ground_answer(answer, docs, question)
        conf, cov = make_confidence(route, kw, docs, g)

    sec = time.perf_counter() - t0
    if return_meta:
        return {"answer": answer, "route": route, "sec": sec, "conf": conf, "cov": cov,
                "ground": g, "retried": retried, "docs": [(d.metadata["source"][:2], d.metadata["page"]) for d in docs]}
    return answer, 1 + int(retried), sec


for t in (testset_hard[0], testset_hard[1], testset_hard[12]):
    r = ask_v16(t["q"], return_meta=True)
    print(f"\n[{r['sec']:.2f}s] route={r['route']} conf={r['conf']} ground={r['ground']['score']} retry={r['retried']}")
    print("  ", r["answer"][:200])
    print("   gold:", t["gold"][:120])



[2.41s] route=procedure conf=1.0 ground=1.0 retry=False
   장애인복지법 제32조에 따라 장애인 등록을 마친 지원자가 장애나 지체로 인해 입학전형 진행 중 지원이 필요한 경우, 사전에 요청하면 됩니다. 장애학생의 지원 및 선발에 차별은 없으며, 입학 시 본교 장애학생지원 관련 규정이 적용됩니다. (출처: 04_2027학년도_정시_모집요강.pdf p.8)
   gold: 장애인복지법 제32조에 따라 장애인등록을 마쳤고 장애 또는 지체로 전형 진행 과정에서 지원이 필요한 경우, 사전에 요청해야 합니다.

[1.78s] route=table conf=1.0 ground=1.0 retry=False
   경영학과는 최대 모집 가능 인원이 15명이고, 모집인원은 16명입니다. 다만 모집요강에는 내용이 변동될 수 있으므로 최종 확정 모집요강을 확인하라고 안내되어 있습니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.3)
   gold: 경영학과의 최대모집 가능인원은 15명이고, 모집인원은 16명입니다.

[2.04s] route=page conf=0.992 ground=1.0 retry=False
   정시 모집요강 목차에는 원서접수가 34쪽으로 안내되어 있고, 원서접수 내용은 34~35쪽에 나옵니다. (출처: 04_2027학년도_정시_모집요강.pdf p.2)
   gold: 원서접수 내용은 34페이지에 있습니다.


In [126]:
def eval_meta(dataset, start, end, tag="", fn=ask_v16):
    rows = []
    for i in range(start, end):
        t = dataset[i]
        r = fn(t["q"], return_meta=True)
        v = judge.invoke([("system", JUDGE),
                          ("human", f"[질문]\n{t['q']}\n\n[정답]\n{t['gold']}\n\n[예측]\n{r['answer']}")])
        rows.append({"idx": i, "q": t["q"], "gold": t["gold"], "pred": r["answer"],
                     "ok": bool(v.correct), "why": v.reason, "sec": r["sec"], "conf": r["conf"],
                     "ground": r["ground"]["score"], "cov": r["cov"], "route": r["route"],
                     "retried": r["retried"], "docs": r["docs"]})
        print(f"{i+1:2d}. {'O' if v.correct else 'X'} {r['sec']:4.1f}s conf={r['conf']:.2f} g={r['ground']['score']:.2f} [{r['route']}] {t['q'][:42]}")
    print(f"[{tag} {start+1}-{end}] {sum(x['ok'] for x in rows)}/{len(rows)} | "
          f"평균 {sum(x['sec'] for x in rows)/len(rows):.2f}s | 최대 {max(x['sec'] for x in rows):.2f}s | "
          f"재시도 {sum(x['retried'] for x in rows)}")
    return rows


h16 = eval_meta(testset_hard, 0, 5, "hard-v16")


 1. O  2.2s conf=1.00 g=1.00 [procedure] 몸이 좀 불편한 학생인데요, 시험 볼 때 도와주는 사람 받을 수 있나요? 있
 2. O  2.0s conf=1.00 g=1.00 [table] 외국에서 살다 온 사람들 뽑는 거 있잖아요, 경영학과는 최대 몇명까지 뽑을 
 3. X  3.1s conf=1.00 g=1.00 [procedure] 외국 학교를 3년 다녀야 된다고 들었는데 한 학년은 어떻게 세는 거예요? 중
 4. O  2.0s conf=1.00 g=1.00 [eligibility] 외국에 있는 부모님이랑 저 둘 다 외국에 있어야 되나요? 저만 있으면 안 되
 5. O  2.0s conf=1.00 g=1.00 [procedure] 지역에서 의사 하면 학비 지원해준다던데 신청은 언제까지 해요? 1학기랑 2학
[hard-v16 1-5] 4/5 | 평균 2.26s | 최대 3.10s | 재시도 0


In [127]:
print("PRED:", h16[2]["pred"])
print("\nWHY:", h16[2]["why"])
print("\nGOLD:", h16[2]["gold"])
print("\nDOCS:", h16[2]["docs"])

PRED: 재외국민과 외국인(2% 이내)의 국외 재학기간은 고교과정 1개 학년을 포함해 중·고교 과정에서 통산 3년(3개 학년 이상)이며, 성적이 나온 기간만 인정됩니다. 1개 학년은 원칙적으로 학기 개시일부터 다음 학년도 같은 학기 개시일 전일까지 약 365일이며, 그 기간의 모든 학기를 이수했다면 다음 학년도 같은 학기 개시일 직전 학기 종료일까지로도 충족한 것으로 봅니다. 중간에 편입학한 경우에는 편입학일부터 역년으로 1년(365일)이 되는 날까지를 1개 학년으로 산정합니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.10)

WHY: 예측 답변은 학년 산정 방식과 3년 재학 요건은 포함했지만, 국외 재학기간뿐 아니라 체류기간과 국외근무자 재직기간도 모두 충족해야 한다는 조건을 빠뜨렸습니다.

GOLD: 고교과정 1개 학년을 포함해 중·고교과정에서 통산 3년(3개 학년 이상) 재학해야 하며, 성적이 나온 기간만 인정됩니다. 일반적으로 학기 개시일부터 다음 학년도 같은 학기 개시일 전일까지를 1개 학년으로 보지만, 해당 기간의 모든 학기를 이수한 경우에는 다음 학년도 같은 학기의 직전 학기 종료일까지로도 1개 학년을 충족한 것으로 간주합니다. 중간에 편입학했다면 편입학일부터 역년으로 1년(365일)이 되는 날까지를 1개 학년으로 봅니다. 또한 국외 재학기간뿐 아니라 체류기간과 국외근무자 재직기간도 모두 충족해야 합니다.

DOCS: [('01', 11), ('01', 10), ('01', 12), ('01', 10), ('01', 10), ('01', 10), ('01', 12), ('01', 17)]


In [128]:
h16 += eval_meta(testset_hard, 5, 10, "hard-v16")

 6. O  2.0s conf=0.82 g=1.00 [list] 강원도 속초 쪽이라고 하면 어디어디가 들어가요?
 7. X  2.7s conf=0.93 g=1.00 [eligibility] 지역의사 전형에서 두 군데 지역에 같이 넣을 수 있나요? 어디에 물어봐야 되
 8. X  2.7s conf=0.95 g=1.00 [general] 지역의사로 뽑히면 뭐 지원해주는 게 있어요? 돈 말고요. 그리고 의무 다 채
 9. O  2.5s conf=1.00 g=1.00 [list] 가천대 길병원에서 실습한다고 들었는데 몇 군데서 해요? 예를 들면 어디어디요
10. O  2.4s conf=0.91 g=1.00 [procedure] 합격하고 등록금까지 냈는데 마음이 바뀌었어요. 어떻게 취소해요? 취소하고 나
[hard-v16 6-10] 3/5 | 평균 2.45s | 최대 2.67s | 재시도 0


In [129]:
for r in h16:
    if not r["ok"]:
        print(f"\n[{r['idx']+1}] {r['q'][:70]}")
        print("  PRED:", r["pred"][:350])
        print("  WHY :", r["why"][:220])
        print("  DOCS:", r["docs"][:6])


[3] 외국 학교를 3년 다녀야 된다고 들었는데 한 학년은 어떻게 세는 거예요? 중간에 전학 가면 어떻게 되고요?
  PRED: 재외국민과 외국인(2% 이내)의 국외 재학기간은 고교과정 1개 학년을 포함해 중·고교 과정에서 통산 3년(3개 학년 이상)이며, 성적이 나온 기간만 인정됩니다. 1개 학년은 원칙적으로 학기 개시일부터 다음 학년도 같은 학기 개시일 전일까지 약 365일이며, 그 기간의 모든 학기를 이수했다면 다음 학년도 같은 학기 개시일 직전 학기 종료일까지로도 충족한 것으로 봅니다. 중간에 편입학한 경우에는 편입학일부터 역년으로 1년(365일)이 되는 날까지를 1개 학년으로 산정합니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.10)
  WHY : 예측 답변은 학년 산정 방식과 3년 재학 요건은 포함했지만, 국외 재학기간뿐 아니라 체류기간과 국외근무자 재직기간도 모두 충족해야 한다는 조건을 빠뜨렸습니다.
  DOCS: [('01', 11), ('01', 10), ('01', 12), ('01', 10), ('01', 10), ('01', 10)]

[7] 지역의사 전형에서 두 군데 지역에 같이 넣을 수 있나요? 어디에 물어봐야 되고, 다른 전형처럼 여러 개 지원하면 안 되는 건가
  PRED: 제공된 안내에는 지역의사 전형을 두 군데에 동시에 지원할 수 있는지, 또는 여러 전형에 중복 지원할 수 있는지에 대한 규정이 없습니다. 지원 가능 여부와 중복 지원 제한은 지원하려는 대학의 입학전형 공고나 모집요강에서 확인하시기 바랍니다. 안내책자는 지원 대학의 입학전형 공고에서 의무복무지역을 확인하도록 안내하고 있습니다. (출처: 02_2027학년도_지역의사선발전형_안내.pdf p.19)
  WHY : 대학별 모집요강에서 동시 지원 가능 여부를 확인해야 한다는 점은 포함했지만, 대교협 「대학입학전형 기본사항」의 복수지원 금지 및 이중등록 금지 규정이 지역의사선발전형에도 동일하게 적용된다는 핵심 내용이 빠져 있습니다.
  DOCS: [('02

In [130]:
DRAFT_PROMPT_V3 = """You are an admissions counselor at Gachon University. Answer using ONLY the evidence documents.

Procedure:
1. Scan EVERY evidence passage for sentences containing the key terms of the question, including passages near the end.
2. Collect all facts that answer any part of the question.
3. Write the answer from those facts only.

Rules:
- Include EVERY key fact the question asks for (numbers, dates, targets, conditions, exceptions).
- If the evidence lists several conditions or exceptions, list ALL of them, including additional requirements stated in the same passage.
- Copy the exact wording of list items (e.g., "국내외 연수", "주거 안정") instead of shortening them.
- For tables, align the row label and column headers by position. Never swap columns.
- Never add facts or qualifiers that are not in the evidence.
- NEVER say a rule or information is absent when any evidence passage mentions the question's key terms.
- If the question asks which page, give the page number exactly as listed in the table of contents.
- Write in Korean, 2-5 sentences, and end with (출처: 파일명 p.페이지)."""

REPAIR_PROMPT_V2 = """You are a strict fact-checker. Rewrite the draft using ONLY the evidence.

- The evidence DOES contain sentences relevant to the question. Search every passage, including the last one, before deciding something is missing.
- Remove any number, date, or name not present in the evidence.
- Add every key fact from the evidence that the question asks for but the draft omitted, including additional requirements in the same passage.
- Copy list items exactly (e.g., "국내외 연수").
- Keep it Korean, 2-5 sentences, ending with (출처: 파일명 p.페이지)."""


def ask_v17(question, top_n=8, t_budget=9.0, return_meta=False):
    t0 = time.perf_counter()
    docs, route = retrieve_v15(question, top_n=top_n)
    kw = search_keywords(question)
    ctx = "\n\n".join(f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.metadata['raw']}"
                      for i, d in enumerate(docs, 1))
    answer = llm.invoke([("system", DRAFT_PROMPT_V3),
                         ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content
    g = ground_answer(answer, docs, question)
    conf, cov = make_confidence(route, kw, docs, g)
    retried = False

    weak = g["score"] < 0.78 or g["not_found"] or re.search(
        r"(규정이 없|안내되어 있지 않|나와 있지 않|확인할 수 없|제공된 자료에는.{0,20}없)", answer)
    if weak and time.perf_counter() - t0 < t_budget:
        retried = True
        answer = llm.invoke([("system", REPAIR_PROMPT_V2),
                             ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}\n\n[Draft]\n{answer}")]).content
        g = ground_answer(answer, docs, question)
        conf, cov = make_confidence(route, kw, docs, g)

    sec = time.perf_counter() - t0
    if return_meta:
        return {"answer": answer, "route": route, "sec": sec, "conf": conf, "cov": cov,
                "ground": g, "retried": retried,
                "docs": [(d.metadata["source"][:2], d.metadata["page"]) for d in docs]}
    return answer, 1 + int(retried), sec


for idx in (2, 6, 7):
    t = testset_hard[idx]
    r = ask_v17(t["q"], return_meta=True)
    print(f"\n[{idx+1}] {r['sec']:.2f}s retry={r['retried']} conf={r['conf']}")
    print("  PRED:", r["answer"][:420])
    print("  GOLD:", t["gold"][:220])
    print("  DOCS:", r["docs"][:6])



[3] 2.84s retry=False conf=1.0
  PRED: 재외국민과 외국인(2% 이내)은 고교과정 1개 학년을 포함해 중·고교과정 통산 3년(3개 학년 이상) 재학해야 하며, 성적이 나온 경우만 인정합니다. 1개 학년은 원칙적으로 학기 개시일부터 다음 학년도 같은 학기 개시일 전일까지(약 365일)로 계산합니다. 다만 해당 기간 내 모든 학기를 이수했다면 학기 개시일부터 다음 학년도 같은 학기 직전 학기 종료일까지를 1개 학년으로 인정하고, 중간에 편입학한 경우에는 편입학일부터 역년으로 1년(365일)이 되는 날까지를 1개 학년으로 봅니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.10)
  GOLD: 고교과정 1개 학년을 포함해 중·고교과정에서 통산 3년(3개 학년 이상) 재학해야 하며, 성적이 나온 기간만 인정됩니다. 일반적으로 학기 개시일부터 다음 학년도 같은 학기 개시일 전일까지를 1개 학년으로 보지만, 해당 기간의 모든 학기를 이수한 경우에는 다음 학년도 같은 학기의 직전 학기 종료일까지로도 1개 학년을 충족한 것으로 간주합니다. 중간에 편입학했다면 편입학일부터 역년으로 1년(36
  DOCS: [('01', 11), ('01', 10), ('01', 12), ('01', 10), ('01', 10), ('01', 10)]

[7] 3.16s retry=False conf=0.93
  PRED: 진료권과 광역권에 동시에 지원할 수 있는지는 대학별로 다를 수 있으므로, 지원하려는 대학의 모집 요강에서 확인해야 합니다. 지역의사선발전형 외 일반전형과의 중복지원 가능 여부도 대학별 모집 요강을 확인하세요. 다만 한국대학교육협의회의 「대학입학전형 기본사항」에 따른 복수지원 금지 및 이중등록 금지 규정은 지역의사선발전형에도 동일하게 적용됩니다. (출처: 02_2027학년도_지역의사선발전형_안내.pdf p.4)
  GOLD: 진료권·광역권 동시 지원 가능 여부는 개별 대학 모집 요강에서 확인해야 합니다. 한국대학교육협의회의 「대학입

In [131]:
h17 = eval_meta(testset_hard, 0, 5, "hard-v17", fn=ask_v17)

 1. O  2.0s conf=1.00 g=1.00 [procedure] 몸이 좀 불편한 학생인데요, 시험 볼 때 도와주는 사람 받을 수 있나요? 있
 2. O  1.7s conf=1.00 g=1.00 [table] 외국에서 살다 온 사람들 뽑는 거 있잖아요, 경영학과는 최대 몇명까지 뽑을 
 3. X  2.0s conf=1.00 g=1.00 [procedure] 외국 학교를 3년 다녀야 된다고 들었는데 한 학년은 어떻게 세는 거예요? 중
 4. O  2.1s conf=1.00 g=1.00 [eligibility] 외국에 있는 부모님이랑 저 둘 다 외국에 있어야 되나요? 저만 있으면 안 되
 5. O  1.8s conf=1.00 g=1.00 [procedure] 지역에서 의사 하면 학비 지원해준다던데 신청은 언제까지 해요? 1학기랑 2학
[hard-v17 1-5] 4/5 | 평균 1.91s | 최대 2.06s | 재시도 0


In [132]:
h17 += eval_meta(testset_hard, 5, 10, "hard-v17", fn=ask_v17)

 6. O  2.2s conf=0.82 g=1.00 [list] 강원도 속초 쪽이라고 하면 어디어디가 들어가요?
 7. O  2.1s conf=0.93 g=1.00 [eligibility] 지역의사 전형에서 두 군데 지역에 같이 넣을 수 있나요? 어디에 물어봐야 되
 8. X  4.1s conf=0.95 g=1.00 [general] 지역의사로 뽑히면 뭐 지원해주는 게 있어요? 돈 말고요. 그리고 의무 다 채
 9. O  4.1s conf=1.00 g=1.00 [list] 가천대 길병원에서 실습한다고 들었는데 몇 군데서 해요? 예를 들면 어디어디요
10. O  2.6s conf=0.91 g=1.00 [procedure] 합격하고 등록금까지 냈는데 마음이 바뀌었어요. 어떻게 취소해요? 취소하고 나
[hard-v17 6-10] 4/5 | 평균 3.03s | 최대 4.13s | 재시도 1


In [133]:
h17 += eval_meta(testset_hard, 10, 15, "hard-v17", fn=ask_v17)

11. O  1.9s conf=0.84 g=0.80 [eligibility] 예전에 태어난 사람은 신분증 사본도 내야 한다던데 저도 해당되나요? 서류는 
12. O  1.8s conf=0.88 g=1.00 [general] 학교 안에 도는 버스 있다던데 언제부터 있었고 지금은 몇대 돌아요?
13. O  1.8s conf=0.99 g=1.00 [page] 정시 요강에서 원서접수 얘기는 몇 쪽에 나와요?
14. X  1.7s conf=1.00 g=1.00 [table] 인천 캠퍼스에 바이오로직스학과 있잖아요, 거기 정시로 몇명 뽑아요?
15. X  1.8s conf=1.00 g=1.00 [page] 농어촌 전형은 정시 요강 몇 쪽에 있어요?
[hard-v17 11-15] 3/5 | 평균 1.80s | 최대 1.85s | 재시도 0


In [134]:
h17 += eval_meta(testset_hard, 15, 20, "hard-v17", fn=ask_v17)

16. O  3.9s conf=0.96 g=1.00 [eligibility] 수능으로 가군 일반전형에 넣으려는데 서류 뭐 내야 해요? 고등학교 졸업했으면
17. O  1.8s conf=1.00 g=1.00 [schedule] 이 면접 일정표는 어디 대학 거예요? 수시 중에 어떤 전형 대상이에요?
18. O  3.9s conf=0.75 g=0.65 [general] 기회균형이나 농어촌, 특성화고 애들은 면접 언제 봐요?
19. O  1.8s conf=0.93 g=1.00 [schedule] 지역균형은 면접 날짜를 접수 순서로 정해줘요? 아프면 바꿀 수 있어요?
20. O  4.2s conf=0.87 g=0.87 [schedule] 바람개비 전형 간호학과는 면접을 몇 번 봐요? 날짜랑 시간이 몇 개 있어요?
[hard-v17 16-20] 5/5 | 평균 3.11s | 최대 4.16s | 재시도 1


In [135]:
for r in h17:
    if not r["ok"]:
        print(f"\n[{r['idx']+1}] {r['q'][:75]}")
        print("  PRED:", r["pred"][:400])
        print("  WHY :", r["why"][:250])
        print("  GOLD:", r["gold"][:180])
        print("  DOCS:", r["docs"][:6])


[3] 외국 학교를 3년 다녀야 된다고 들었는데 한 학년은 어떻게 세는 거예요? 중간에 전학 가면 어떻게 되고요?
  PRED: 재외국민과 외국인(2% 이내)은 고교과정 1개 학년을 포함해 중·고교과정에서 통산 3년, 3개 학년 이상 재학해야 하며 성적이 나온 경우만 인정됩니다. 1개 학년은 학기 개시일부터 다음 학년도 동일 학기 개시일 전일까지(약 365일)로 계산합니다. 중간에 편입학한 경우에는 편입학일부터 역년으로 1년(365일)이 되는 날까지를 1개 학년으로 봅니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.10)
  WHY : 예측 답변은 학기 개시일부터 다음 학년도 같은 학기 개시일 전일까지라는 기준과 편입학 시 365일 기준은 포함하지만, 해당 기간의 모든 학기를 이수한 경우 다음 학년도 같은 학기의 직전 학기 종료일까지도 1개 학년으로 인정된다는 조건을 빠뜨렸습니다. 국외 재학기간 외에 체류기간과 국외근무자 재직기간도 충족해야 한다는 조건도 누락했습니다.
  GOLD: 고교과정 1개 학년을 포함해 중·고교과정에서 통산 3년(3개 학년 이상) 재학해야 하며, 성적이 나온 기간만 인정됩니다. 일반적으로 학기 개시일부터 다음 학년도 같은 학기 개시일 전일까지를 1개 학년으로 보지만, 해당 기간의 모든 학기를 이수한 경우에는 다음 학년도 같은 학기의 직전 학기 종료일까지로도 1개 학년을 충족한
  DOCS: [('01', 11), ('01', 10), ('01', 12), ('01', 10), ('01', 10), ('01', 10)]

[8] 지역의사로 뽑히면 뭐 지원해주는 게 있어요? 돈 말고요. 그리고 의무 다 채우고 나면 취직할 때 좋은 거 있나요?
  PRED: 네. 금전 지원 외에도 복무 중 지역의사지원센터를 통해 주거 안정, 직무교육 및 연수 프로그램, 진로·경력개발 상담·지원, 권익보호, 진로상담·정보 제공을 지원합니다. 의무복무를 마치면 보건복지부장관·지방자치단체장·의무복무기관장이 해당 기관 또는 지역 내 공공보건의료기

In [136]:
q14 = testset_hard[13]["q"]
print("Q14:", q14)
print("route/kw:", route_queries(q14))

for name, ret in (("BM25 orig", bm25_v15), ("BM25 kw", None)):
    query = q14 if ret else search_keywords(q14)
    r = (ret or bm25_v15).invoke(query)
    print(f"\n[{name}] {query!r}")
    for i, d in enumerate(r[:10], 1):
        print(f"  {i}. {d.metadata['source'][:2]} p.{d.metadata['page']} {d.metadata['kind']}")

print("\n[04 p.7 청크들]")
for i, c in enumerate(chunks_v15):
    if c.metadata["source"].startswith("04") and c.metadata["page"] == 7:
        print(f"  idx={i} kind={c.metadata['kind']} len={len(c.metadata['raw'])}")
        print("   ", c.metadata["raw"][:120].replace("\n", " "))

print("\n[04 p.7 바이오로직스 행 포함 여부]")
for i, c in enumerate(chunks_v15):
    if "바이오로직스" in c.metadata["raw"]:
        print(f"  {c.metadata['source'][:2]} p.{c.metadata['page']} kind={c.metadata['kind']}")

print("\nQ15 TOC lines:")
for c in chunks_v15:
    if c.metadata["source"].startswith("04") and c.metadata["page"] in TOC_PAGES["04"]:
        for line in c.metadata["raw"].splitlines():
            if "농어촌" in line:
                print(f"  p.{c.metadata['page']}: {line!r}")


Q14: 인천 캠퍼스에 바이오로직스학과 있잖아요, 거기 정시로 몇명 뽑아요?
route/kw: ('table', '인천 캠퍼스 바이오로직스 학과 모집 선발 모집인원 정시모집', '인천 캠퍼스 바이오로직스 학과 모집 선발 모집인원 정시모집 모집인원 표')

[BM25 orig] '인천 캠퍼스에 바이오로직스학과 있잖아요, 거기 정시로 몇명 뽑아요?'
  1. 04 p.43 text
  2. 03 p.7 text
  3. 03 p.95 text
  4. 01 p.3 table
  5. 04 p.5 text
  6. 03 p.19 text
  7. 03 p.3 text
  8. 02 p.12 table
  9. 03 p.31 text
  10. 04 p.7 text

[BM25 kw] '인천 캠퍼스 바이오로직스 학과 모집 선발 모집인원 정시모집'
  1. 01 p.3 table
  2. 04 p.43 text
  3. 04 p.5 text
  4. 03 p.64 table
  5. 03 p.48 table
  6. 03 p.95 text
  7. 03 p.46 table
  8. 03 p.31 text
  9. 04 p.16 text
  10. 04 p.16 table

[04 p.7 청크들]
  idx=389 kind=text len=490
    [04_2027학년도_정시_모집요강.pdf p.7] 33 2027학년도 정시모집요강 계열 모집단위 일반전형 1 일반전형 2 실기우수자 가군 나군 다군 가군 나군 다군 가군 자연 시스템반도체학과 18 컴퓨터공학과 30
  idx=448 kind=table len=1146
    [04_2027학년도_정시_모집요강.pdf p.7] | 계열 | 모집단위 | 일반전형 1 | | | 일반전형 2 | | | 실기우수자 | |---|---|---|---|---|---|---|---|---| | | |

[04 p.7 바이오로직스 행 포함 여부]
  01 p.3 kind=table
  03 p.8 kind=text
  03 p.9 ki

In [137]:
def retrieve_v15(query, top_n=6, expand_top=2, rrf_k=60):
    route, kw, kw_route = route_queries(query)
    lists = [bm25_v15.invoke(query), dense_v15.invoke(query),
             bm25_v15.invoke(kw), dense_v15.invoke(kw_route)]
    scores, docs = rrf_fuse(lists, [1.0, 1.0, 1.15, 1.15], rrf_k)

    prior = page_prior(kw_route)
    tdoc = target_doc(query)
    for key, d in docs.items():
        r = prior.get((d.metadata["source"], d.metadata["page"]))
        if r:
            scores[key] += 0.45 / (40 + r)
        if tdoc and d.metadata["source"].startswith(tdoc):
            scores[key] *= 1.10
        if route == "table" and d.metadata["kind"] == "table":
            scores[key] *= 1.15
        if route in ("procedure", "eligibility", "page") and d.metadata["kind"] == "text":
            scores[key] *= 1.12

    top_score = max(scores.values()) if scores else 0.0

    if route == "page" and tdoc and tdoc in TOC_PAGES:
        pool = [c for c in chunks_v15
                if c.metadata["source"].startswith(tdoc) and c.metadata["page"] in TOC_PAGES[tdoc]]
        inject(scores, docs, kw, pool, top_score, factor=1.12, limit=2)

    if route == "schedule":
        pool = [c for c in chunks_v15 if c.metadata["source"].startswith("05")]
        inject(scores, docs, kw, pool, top_score, factor=1.15, limit=3)

    ranking = sorted(scores, key=scores.get, reverse=True)
    # 희귀 키워드 근거 보장: 상위 6개에 없으면 해당 키워드를 포함한 최고 순위 청크 주입
    top_keys = set(ranking[:6])
    rare = sorted([t for t in kw.split() if len(t) >= 3], key=lambda t: sum(t in c.metadata["raw"] for c in chunks_v15))[:2]
    for t in rare:
        if any(t in docs[k].metadata["raw"] for k in top_keys):
            continue
        cand = next((k for k in ranking if t in docs[k].metadata["raw"]), None)
        if cand:
            scores[cand] = max(scores[cand], top_score * 1.05)
            top_keys.add(cand)

    top = [docs[k] for k in sorted(scores, key=scores.get, reverse=True)[:top_n]]
    out, seen = [], set()
    for n, d in enumerate(top):
        if d.page_content not in seen:
            out.append(d)
            seen.add(d.page_content)
        if n >= expand_top:
            continue
        i = CHUNK_INDEX_V15[d.page_content]
        for j in range(max(0, i - 1), min(len(chunks_v15), i + 2)):
            c = chunks_v15[j]
            if c.metadata["source"] == d.metadata["source"] and c.page_content not in seen:
                out.append(c)
                seen.add(c.page_content)
    return out, route


# page-lookup: 목차에서 질문 주제의 페이지 번호를 결정적으로 추출
def toc_lookup(question, tdoc):
    if not tdoc or tdoc not in TOC_PAGES:
        return None
    kw = search_keywords(question)
    toks = [t for t in kw.split() if len(t) >= 2 and t not in ("페이지", "목차", "요강", "모집", "전형", "안내")]
    best = (0, None, None, None)
    for c in chunks_v15:
        if not (c.metadata["source"].startswith(tdoc) and c.metadata["page"] in TOC_PAGES[tdoc]):
            continue
        for line in c.metadata["raw"].splitlines():
            m = re.match(r"^\s*(\d{1,3})\s+(.+)$", line.strip())
            if not m:
                continue
            num, title = m.group(1), m.group(2).strip()
            score = sum(t in title for t in toks)
            if score > best[0]:
                best = (score, num, title, c)
    if best[0] < 1:
        return None
    return {"page": best[1], "title": best[2], "doc_page": best[3].metadata["page"],
            "source": best[3].metadata["source"]}


def _particle(word):
    last = word.strip()[-1]
    if not ("가" <= last <= "힣"):
        return "은"
    return "은" if (ord(last) - 0xAC00) % 28 else "는"


for idx in (12, 14):
    t = testset_hard[idx]
    r = toc_lookup(t["q"], target_doc(t["q"]))
    print(f"[{idx+1}] {t['q'][:50]} → {r}")

q14 = testset_hard[13]["q"]
docs, route = retrieve_v15(q14, top_n=8)
print("\nQ14 top8:")
for d in docs[:8]:
    print(f"  {d.metadata['source'][:2]} p.{d.metadata['page']} {d.metadata['kind']}")


[13] 정시 요강에서 원서접수 얘기는 몇 쪽에 나와요? → {'page': '34', 'title': '원서접수', 'doc_page': 2, 'source': '04_2027학년도_정시_모집요강.pdf'}
[15] 농어촌 전형은 정시 요강 몇 쪽에 있어요? → {'page': '32', 'title': '정시모집인원', 'doc_page': 2, 'source': '04_2027학년도_정시_모집요강.pdf'}

Q14 top8:
  01 p.3 table
  01 p.3 table
  01 p.3 table
  04 p.5 text
  04 p.5 text
  04 p.6 text
  03 p.31 text
  04 p.16 table


In [138]:
q14 = testset_hard[13]["q"]
kw14 = search_keywords(q14)
toks = korean_tokenizer(kw14)
bm_scores = bm25_v15.vectorizer.get_scores(toks)

print("04 문서 내 BM25 순위:")
rank04 = sorted([(bm_scores[i], i) for i, c in enumerate(chunks_v15) if c.metadata["source"].startswith("04")], reverse=True)
for s, i in rank04[:10]:
    c = chunks_v15[i]
    print(f"  {s:6.2f} p.{c.metadata['page']} {c.metadata['kind']} | {c.metadata['raw'][:50].replace(chr(10),' ')}")

print("\n전체 BM25 상위 20 중 04 p.7:")
for r, i in enumerate(sorted(range(len(chunks_v15)), key=lambda i: bm_scores[i], reverse=True)[:20], 1):
    c = chunks_v15[i]
    if c.metadata["source"].startswith("04") and c.metadata["page"] == 7:
        print(f"  rank={r} score={bm_scores[i]:.2f} kind={c.metadata['kind']}")


04 문서 내 BM25 순위:
   18.40 p.43 text | [04_2027학년도_정시_모집요강.pdf p.43] 69 2027학년도 정시모집요강 자유
   17.78 p.5 text | [04_2027학년도_정시_모집요강.pdf p.5] 대학 정보보호학과 ★ 50 전기공학과 
   15.84 p.16 text | [04_2027학년도_정시_모집요강.pdf p.16] 42 수능위주 전형방법 100% 수능
   15.75 p.16 table | [04_2027학년도_정시_모집요강.pdf p.16] | 계열 | 모집단위 | 모집인원 |
   14.76 p.24 text | [04_2027학년도_정시_모집요강.pdf p.24] 50 특별전형 전형방법 1단계 (3배
   14.68 p.7 text | [04_2027학년도_정시_모집요강.pdf p.7] 33 2027학년도 정시모집요강 계열 
   13.79 p.42 table | [04_2027학년도_정시_모집요강.pdf p.42] | 모집인원 | 지원자격 | 제한범위
   13.73 p.24 table | [04_2027학년도_정시_모집요강.pdf p.24] | 계열 | 모집단위 | 비고 | |
   13.73 p.30 text | [04_2027학년도_정시_모집요강.pdf p.30] 56 실기위주 2027학년도 정시모집
   13.15 p.20 text | [04_2027학년도_정시_모집요강.pdf p.20] 46 특별전형 전형방법 100% 수능

전체 BM25 상위 20 중 04 p.7:
  rank=13 score=14.68 kind=text


In [139]:
def dept_terms(q):
    return list(dict.fromkeys(re.findall(r"([가-힣A-Za-z0-9·]{2,})\s*(?:학과|학부|전공)", q)))


def toc_lookup(question, tdoc):
    if not tdoc or tdoc not in TOC_PAGES:
        return None
    kw = search_keywords(question)
    skip = {"페이지", "목차", "요강", "모집", "전형", "안내", "정시모집", "수시모집", "대학"}
    toks = [t for t in kw.split() if len(t) >= 2 and t not in skip]
    best = (0, None, None, None)
    for c in chunks_v15:
        if not (c.metadata["source"].startswith(tdoc) and c.metadata["page"] in TOC_PAGES[tdoc]):
            continue
        for line in c.metadata["raw"].splitlines():
            m = re.match(r"^\s*(\d{1,3})\s+(.+)$", line.strip())
            if not m:
                continue
            num, title = m.group(1), m.group(2).strip()
            score = sum(len(t) for t in toks if t in title)
            if score > best[0]:
                best = (score, num, title, c)
    if best[0] < 2:
        return None
    return {"page": best[1], "title": best[2], "doc_page": best[3].metadata["page"],
            "source": best[3].metadata["source"]}


def table_dept_boost(scores, docs, query, kw, tdoc, top_score):
    terms = dept_terms(query)
    if not terms:
        return []
    cands = []
    for c in chunks_v15:
        if c.metadata["kind"] != "table" or not c.metadata["source"].startswith(tdoc):
            continue
        if not any(t in c.metadata["raw"] for t in terms):
            continue
        summary_hit = sum(t in c.page_content.split("[원문]")[0] for t in terms)
        cands.append((overlap_score(c, kw) + 5 * summary_hit, c))
    cands.sort(key=lambda x: x[0], reverse=True)
    picked = [c for _, c in cands[:2]]
    for c in picked:
        scores[c.page_content] = max(scores.get(c.page_content, 0.0), top_score * 1.18)
        docs[c.page_content] = c
    return picked


def retrieve_v15(query, top_n=6, expand_top=2, rrf_k=60):
    route, kw, kw_route = route_queries(query)
    lists = [bm25_v15.invoke(query), dense_v15.invoke(query),
             bm25_v15.invoke(kw), dense_v15.invoke(kw_route)]
    scores, docs = rrf_fuse(lists, [1.0, 1.0, 1.15, 1.15], rrf_k)

    prior = page_prior(kw_route)
    tdoc = target_doc(query)
    for key, d in docs.items():
        r = prior.get((d.metadata["source"], d.metadata["page"]))
        if r:
            scores[key] += 0.45 / (40 + r)
        if tdoc and d.metadata["source"].startswith(tdoc):
            scores[key] *= 1.12
        if route == "table" and d.metadata["kind"] == "table":
            scores[key] *= 1.15
        if route in ("procedure", "eligibility", "page") and d.metadata["kind"] == "text":
            scores[key] *= 1.12

    top_score = max(scores.values()) if scores else 0.0

    if route == "table" and tdoc:
        table_dept_boost(scores, docs, query, kw, tdoc, top_score)
    if route == "page" and tdoc and tdoc in TOC_PAGES:
        pool = [c for c in chunks_v15
                if c.metadata["source"].startswith(tdoc) and c.metadata["page"] in TOC_PAGES[tdoc]]
        inject(scores, docs, kw, pool, top_score, factor=1.12, limit=2)
    if route == "schedule":
        pool = [c for c in chunks_v15 if c.metadata["source"].startswith("05")]
        inject(scores, docs, kw, pool, top_score, factor=1.15, limit=3)

    ranking = sorted(scores, key=scores.get, reverse=True)
    top_keys = set(ranking[:6])
    rare = sorted([t for t in kw.split() if len(t) >= 3],
                  key=lambda t: sum(t in c.metadata["raw"] for c in chunks_v15))[:2]
    for t in rare:
        if any(t in docs[k].metadata["raw"] for k in top_keys):
            continue
        cand = next((k for k in ranking if t in docs[k].metadata["raw"]), None)
        if cand:
            scores[cand] = max(scores[cand], top_score * 1.05)
            top_keys.add(cand)

    top = [docs[k] for k in sorted(scores, key=scores.get, reverse=True)[:top_n]]
    out, seen = [], set()
    for n, d in enumerate(top):
        if d.page_content not in seen:
            out.append(d)
            seen.add(d.page_content)
        if n >= expand_top:
            continue
        i = CHUNK_INDEX_V15[d.page_content]
        for j in range(max(0, i - 1), min(len(chunks_v15), i + 2)):
            c = chunks_v15[j]
            if c.metadata["source"] == d.metadata["source"] and c.page_content not in seen:
                out.append(c)
                seen.add(c.page_content)
    return out, route


print("Q15 toc:", toc_lookup(testset_hard[14]["q"], "04"))
print("Q13 toc:", toc_lookup(testset_hard[12]["q"], "04"))
docs, route = retrieve_v15(testset_hard[13]["q"], top_n=8)
print("\nQ14 top8:")
for d in docs[:8]:
    print(f"  {d.metadata['source'][:2]} p.{d.metadata['page']} {d.metadata['kind']}")


Q15 toc: {'page': '44', 'title': '농어촌 전형', 'doc_page': 11, 'source': '04_2027학년도_정시_모집요강.pdf'}
Q13 toc: {'page': '34', 'title': '원서접수', 'doc_page': 2, 'source': '04_2027학년도_정시_모집요강.pdf'}

Q14 top8:
  04 p.5 table
  04 p.5 table
  04 p.6 table
  04 p.7 table
  04 p.8 table
  01 p.3 table
  04 p.5 text
  03 p.31 text


In [140]:
def ask_v18(question, top_n=8, t_budget=9.0, return_meta=False):
    t0 = time.perf_counter()
    route = route_query(question)
    tdoc = target_doc(question)

    # ---- Adaptive routing: page-lookup은 목차에서 결정적으로 추출 ----
    if route == "page" and tdoc:
        hit = toc_lookup(question, tdoc)
        if hit:
            topic = hit["title"]
            answer = (f"{topic}{_particle(topic)} {doc_title(hit['source'])} "
                      f"{hit['page']}페이지에 안내되어 있습니다. "
                      f"(출처: {hit['source']} p.{hit['doc_page']})")
            toc_doc = next(c for c in chunks_v15
                           if c.metadata["source"] == hit["source"] and c.metadata["page"] == hit["doc_page"])
            g = ground_answer(answer, [toc_doc], question)
            sec = time.perf_counter() - t0
            if return_meta:
                return {"answer": answer, "route": "page_lookup", "sec": sec, "conf": 0.99,
                        "cov": 1.0, "ground": g, "retried": False,
                        "docs": [(hit["source"][:2], hit["doc_page"])]}
            return answer, 1, sec

    # ---- 그 외: 검색 → 초안 → 근거 검증 → (필요 시) 교정 ----
    docs, route = retrieve_v15(question, top_n=top_n)
    kw = search_keywords(question)
    ctx = "\n\n".join(f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.metadata['raw']}"
                      for i, d in enumerate(docs, 1))
    answer = llm.invoke([("system", DRAFT_PROMPT_V3),
                         ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content
    g = ground_answer(answer, docs, question)
    conf, cov = make_confidence(route, kw, docs, g)
    retried = False

    weak = g["score"] < 0.78 or g["not_found"] or re.search(
        r"(규정이 없|안내되어 있지 않|나와 있지 않|확인할 수 없|제공된 자료에는.{0,20}없)", answer)
    if weak and time.perf_counter() - t0 < t_budget:
        retried = True
        answer = llm.invoke([("system", REPAIR_PROMPT_V2),
                             ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}\n\n[Draft]\n{answer}")]).content
        g = ground_answer(answer, docs, question)
        conf, cov = make_confidence(route, kw, docs, g)

    sec = time.perf_counter() - t0
    if return_meta:
        return {"answer": answer, "route": route, "sec": sec, "conf": conf, "cov": cov,
                "ground": g, "retried": retried,
                "docs": [(d.metadata["source"][:2], d.metadata["page"]) for d in docs]}
    return answer, 1 + int(retried), sec


for idx in (12, 13, 14):
    t = testset_hard[idx]
    r = ask_v18(t["q"], return_meta=True)
    print(f"\n[{idx+1}] {r['sec']:.2f}s route={r['route']} conf={r['conf']}")
    print("  PRED:", r["answer"][:300])
    print("  GOLD:", t["gold"][:150])



[13] 0.00s route=page_lookup conf=0.99
  PRED: 원서접수는 2027학년도 정시 모집요강 34페이지에 안내되어 있습니다. (출처: 04_2027학년도_정시_모집요강.pdf p.2)
  GOLD: 원서접수 내용은 34페이지에 있습니다.

[14] 1.92s route=table conf=0.841
  PRED: 바이오로직스학과는 메디컬캠퍼스(인천)에 개설되어 있으며, 정시 일반전형 1에서 30명, 일반전형 2에서 15명을 모집합니다. (출처: 04_2027학년도_정시_모집요강.pdf p.7)
  GOLD: 일반전형 1에서 30명, 일반전형 2에서 15명으로 총 45명입니다.

[15] 0.00s route=page_lookup conf=0.99
  PRED: 농어촌 전형은 2027학년도 정시 모집요강 44페이지에 안내되어 있습니다. (출처: 04_2027학년도_정시_모집요강.pdf p.11)
  GOLD: 44페이지에 안내되어 있습니다.


In [141]:
h18 = eval_meta(testset_hard, 0, 5, "hard-v18", fn=ask_v18)

 1. O  2.1s conf=1.00 g=1.00 [procedure] 몸이 좀 불편한 학생인데요, 시험 볼 때 도와주는 사람 받을 수 있나요? 있
 2. O  1.7s conf=1.00 g=1.00 [table] 외국에서 살다 온 사람들 뽑는 거 있잖아요, 경영학과는 최대 몇명까지 뽑을 
 3. O  2.8s conf=1.00 g=1.00 [procedure] 외국 학교를 3년 다녀야 된다고 들었는데 한 학년은 어떻게 세는 거예요? 중
 4. O  2.0s conf=1.00 g=1.00 [eligibility] 외국에 있는 부모님이랑 저 둘 다 외국에 있어야 되나요? 저만 있으면 안 되
 5. O  1.8s conf=1.00 g=1.00 [procedure] 지역에서 의사 하면 학비 지원해준다던데 신청은 언제까지 해요? 1학기랑 2학
[hard-v18 1-5] 5/5 | 평균 2.08s | 최대 2.84s | 재시도 0


In [142]:
h18 += eval_meta(testset_hard, 5, 10, "hard-v18", fn=ask_v18)

 6. O  1.7s conf=0.82 g=1.00 [list] 강원도 속초 쪽이라고 하면 어디어디가 들어가요?
 7. O  2.3s conf=1.00 g=1.00 [eligibility] 지역의사 전형에서 두 군데 지역에 같이 넣을 수 있나요? 어디에 물어봐야 되
 8. X  2.1s conf=0.95 g=1.00 [general] 지역의사로 뽑히면 뭐 지원해주는 게 있어요? 돈 말고요. 그리고 의무 다 채
 9. O  2.1s conf=1.00 g=1.00 [list] 가천대 길병원에서 실습한다고 들었는데 몇 군데서 해요? 예를 들면 어디어디요
10. O  1.8s conf=0.91 g=1.00 [procedure] 합격하고 등록금까지 냈는데 마음이 바뀌었어요. 어떻게 취소해요? 취소하고 나
[hard-v18 6-10] 4/5 | 평균 2.01s | 최대 2.28s | 재시도 0


In [143]:
LIST_INTENT = re.compile(r"((뭐|무엇|어떤).{0,12}(있|지원|혜택|항목|종류)|목록|종류|어디어디|무엇무엇)")


def missing_list_items(question, answer, docs, max_items=4):
    """질문이 목록을 요구할 때, 근거의 쉼표 항목이 답변에 그대로 있는지 확인."""
    if not LIST_INTENT.search(question):
        return []
    kws = [t for t in search_keywords(question).split() if len(t) >= 2][:4]
    missing = []
    for d in docs[:3]:
        for sent in re.split(r"[.\n]", d.metadata.get("raw", "")):
            if ("," not in sent and "·" not in sent) or len(sent) < 15:
                continue
            if kws and not any(t in sent for t in kws):
                continue
            items = [x.strip(" ①②③④⑤⑥⑦⑧⑨⑩") for x in re.split(r"[,·]", sent)]
            items = [x for x in items if 2 <= len(x) <= 14]
            if len(items) < 3:
                continue
            for it in items:
                if it not in answer and it not in missing:
                    missing.append(it)
            if missing:
                return missing[:max_items]
    return []


def ask_v19(question, top_n=8, t_budget=9.0, return_meta=False):
    t0 = time.perf_counter()
    route = route_query(question)
    tdoc = target_doc(question)

    if route == "page" and tdoc:
        hit = toc_lookup(question, tdoc)
        if hit:
            topic = hit["title"]
            answer = (f"{topic}{_particle(topic)} {doc_title(hit['source'])} "
                      f"{hit['page']}페이지에 안내되어 있습니다. "
                      f"(출처: {hit['source']} p.{hit['doc_page']})")
            toc_doc = next(c for c in chunks_v15
                           if c.metadata["source"] == hit["source"] and c.metadata["page"] == hit["doc_page"])
            g = ground_answer(answer, [toc_doc], question)
            sec = time.perf_counter() - t0
            if return_meta:
                return {"answer": answer, "route": "page_lookup", "sec": sec, "conf": 0.99,
                        "cov": 1.0, "ground": g, "retried": False,
                        "docs": [(hit["source"][:2], hit["doc_page"])]}
            return answer, 1, sec

    docs, route = retrieve_v15(question, top_n=top_n)
    kw = search_keywords(question)
    ctx = "\n\n".join(f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.metadata['raw']}"
                      for i, d in enumerate(docs, 1))
    answer = llm.invoke([("system", DRAFT_PROMPT_V3),
                         ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content
    g = ground_answer(answer, docs, question)
    conf, cov = make_confidence(route, kw, docs, g)
    retried = False

    missing = missing_list_items(question, answer, docs)
    weak = (g["score"] < 0.78 or g["not_found"] or bool(missing) or re.search(
        r"(규정이 없|안내되어 있지 않|나와 있지 않|확인할 수 없|제공된 자료에는.{0,20}없)", answer))
    if weak and time.perf_counter() - t0 < t_budget:
        retried = True
        hint = f"\n\n[Exact items to include]\n{', '.join(missing)}" if missing else ""
        answer = llm.invoke([("system", REPAIR_PROMPT_V2),
                             ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}\n\n[Draft]\n{answer}{hint}")]).content
        g = ground_answer(answer, docs, question)
        conf, cov = make_confidence(route, kw, docs, g)

    sec = time.perf_counter() - t0
    if return_meta:
        return {"answer": answer, "route": route, "sec": sec, "conf": conf, "cov": cov,
                "ground": g, "retried": retried,
                "docs": [(d.metadata["source"][:2], d.metadata["page"]) for d in docs]}
    return answer, 1 + int(retried), sec


r = ask_v19(testset_hard[7]["q"], return_meta=True)
print(f"{r['sec']:.2f}s retry={r['retried']}")
print(r["answer"])
print("\nmissing check:", missing_list_items(testset_hard[7]["q"], r["answer"], [c for c in chunks_v15 if c.metadata["source"].startswith("02")]))


6.15s retry=True
네. 지역의사지원센터에서 주거 안정 지원, 직무교육 및 연수 프로그램 지원, 진로·경력개발 상담·지원, 권익보호 및 관련 의견 제시, 진로상담·정보 제공을 받을 수 있고, **국내외 연수 프로그램 지원**도 있습니다. 의무복무를 마치면 보건복지부장관·지방자치단체의 장·의무복무기관의 장이 해당 기관 또는 지역 내 공공보건의료기관에 우선 채용 우대조치를 할 수 있으며, 의료취약지에서 의료기관을 개설·운영하려는 경우 시·도지사가 행정적·재정적으로 지원할 수 있습니다. (출처: 02_2027학년도_지역의사선발전형_안내.pdf p.18; 법 제12, 14조, 시행령 제11조, 시행규칙 제7조)

missing check: ['필수', '중증', '응급의료 책임의사']


In [144]:
h19 = eval_meta(testset_hard, 0, 5, "hard-v19", fn=ask_v19)

 1. O  1.9s conf=1.00 g=1.00 [procedure] 몸이 좀 불편한 학생인데요, 시험 볼 때 도와주는 사람 받을 수 있나요? 있
 2. O  1.5s conf=1.00 g=1.00 [table] 외국에서 살다 온 사람들 뽑는 거 있잖아요, 경영학과는 최대 몇명까지 뽑을 
 3. X  2.7s conf=1.00 g=1.00 [procedure] 외국 학교를 3년 다녀야 된다고 들었는데 한 학년은 어떻게 세는 거예요? 중
 4. O  1.7s conf=1.00 g=1.00 [eligibility] 외국에 있는 부모님이랑 저 둘 다 외국에 있어야 되나요? 저만 있으면 안 되
 5. O  1.9s conf=1.00 g=1.00 [procedure] 지역에서 의사 하면 학비 지원해준다던데 신청은 언제까지 해요? 1학기랑 2학
[hard-v19 1-5] 4/5 | 평균 1.94s | 최대 2.67s | 재시도 0


In [145]:
COND_MARK = re.compile(r"(경우|때|해야|충족|인정|간주|다만|또한|반드시|가능)")


def _content_tokens(text):
    return {t.form for t in kiwi.tokenize(text) if t.tag in {"NNG", "NNP", "VV", "VA", "XR", "SL", "SN"} and len(t.form) >= 2}


def missing_conditions(answer, docs, max_sent=3):
    """답변이 인용한 근거 페이지 안의 조건문을 실제로 반영했는지 확인."""
    cited = []
    for fname, page in CITE_RE.findall(answer):
        for d in docs:
            if d.metadata["source"].startswith(fname.strip()[:2]) and str(d.metadata["page"]) == page:
                cited.append(d)
    if not cited:
        cited = docs[:1]

    ans_toks = _content_tokens(answer)
    missing = []
    for d in cited[:2]:
        for sent in re.split(r"(?<=[.다])\s+|\n", d.metadata.get("raw", "")):
            sent = sent.strip()
            if len(sent) < 25 or not COND_MARK.search(sent):
                continue
            s_toks = _content_tokens(sent)
            if len(s_toks) < 3:
                continue
            overlap = len(s_toks & ans_toks) / len(s_toks)
            if overlap < 0.45:
                missing.append((round(overlap, 2), sent))
            if len(missing) >= max_sent:
                return [s for _, s in missing]
    return [s for _, s in missing]


# 현재 Q3 답변으로 검증
t3 = testset_hard[2]
docs3, _ = retrieve_v15(t3["q"], top_n=8)
a3 = llm.invoke([("system", DRAFT_PROMPT_V3),
                 ("human", f"[Evidence]\n" + "\n\n".join(
                     f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.metadata['raw']}"
                     for i, d in enumerate(docs3, 1)) + f"\n\n[Question]\n{t3['q']}")]).content
print("ANSWER:", a3[:400])
miss = missing_conditions(a3, docs3)
print(f"\n미반영 조건문 {len(miss)}개:")
for s in miss:
    print("  -", s[:220])


ANSWER: 재외국민과 외국인(2% 이내)은 고교과정 1개 학년을 포함해 중·고교과정 통산 3년(3개 학년 이상)을 재학해야 하며, 성적이 나온 기간만 인정됩니다. 1개 학년은 원칙적으로 학기 개시일부터 다음 학년도 동일 학기 개시일 전일까지 약 365일이며, 해당 기간 안의 모든 학기를 이수했다면 다음 학년도 동일 학기 개시일 직전 학기 종료일까지를 1개 학년으로 간주합니다. 중간에 편입학했다면 편입학일부터 역년으로 1년(365일) 되는 날까지를 1개 학년으로 봅니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.10)

미반영 조건문 3개:
  - |  | 재학기간 : | 재학기간 : | 재학기간 : | 재학기간 : | 영주 및 거주기간 내에서 이수한 학기 또는 학년의 해당 기간 학년 또는 학기의 이수여부와 인정하는 재학 기간 등은 해당국의 교육관계법령 등에 의한 학제, 당해 학교의 교육과정, 그리고 대한민국의 학제와 동일급 별 학교의 교육과정 등을 고려하여 결정.
  - ※ 단, 학제차이로 인한 중복은 재학이 연속적인 경우에 한하여 인정함(복수 발생 인정) ※ 누락된 학기는 한 학기에 한하여 중복된 1개 학기로 대체할 수 있음.
  - 2개 학기 이상 연속 중복 학기의 경우 누락된 1개 학기로 대체할 수 있음.


In [146]:
COND_MARK = re.compile(r"(경우|해야|충족|인정|간주|다만|반드시)")


def missing_conditions(answer, docs, max_sent=3):
    cited = []
    for fname, page in CITE_RE.findall(answer):
        key = (fname.strip()[:2], page)
        matches = [d for d in docs if d.metadata["source"].startswith(key[0]) and str(d.metadata["page"]) == page]
        matches.sort(key=lambda d: d.metadata["kind"] != "text")     # 본문 우선
        if matches and key not in {(d.metadata["source"][:2], str(d.metadata["page"])) for d in cited}:
            cited.append(matches[0])
    if not cited:
        cited = [d for d in docs[:3] if d.metadata["kind"] == "text"][:1] or docs[:1]

    ans_toks = _content_tokens(answer)
    missing = []
    for d in cited[:2]:
        for sent in re.split(r"(?<=[.다])\s+|\n", d.metadata.get("raw", "")):
            sent = sent.strip()
            if len(sent) < 25 or not COND_MARK.search(sent) or sent.count("|") > 1:
                continue
            s_toks = _content_tokens(sent)
            if len(s_toks) < 3:
                continue
            overlap = len(s_toks & ans_toks) / len(s_toks)
            if overlap < 0.45:
                missing.append((overlap, sent))
            if len(missing) >= max_sent:
                return [s for _, s in missing]
    return [s for _, s in missing]


miss = missing_conditions(a3, docs3)
print(f"미반영 조건문 {len(miss)}개:")
for s in miss:
    print("  -", s[:260])


미반영 조건문 3개:
  - ※ 단, 학제차이로 인한 중복은 재학이 연속적인 경우에 한하여 인정함(복수 발생 인정) ※ 누락된 학기는 한 학기에 한하여 중복된 1개 학기로 대체할 수 있음.
  - 2개 학기 이상 연속 중복 학기의 경우 누락된 1개 학기로 대체할 수 있음.
  - | ※ 기간산정의 기준 : 부모 중 1인의 국외 근무기간과 학생의 재학기간(방학기간 포함)이 중첩된 기간만을 인정합니다.


In [147]:
print("[체류기간 문장 포함 청크]")
for i, c in enumerate(chunks_v15):
    if "체류기간" in c.metadata["raw"] and "국외근무자" in c.metadata["raw"]:
        print(f"  idx={i} {c.metadata['source'][:2]} p.{c.metadata['page']} {c.metadata['kind']} len={len(c.metadata['raw'])}")
        print("   ", c.metadata["raw"][:300].replace("\n", " "))

print("\n[현재 Q3 검색된 01 p.10 청크]")
for d in docs3:
    if d.metadata["source"].startswith("01") and d.metadata["page"] == 10:
        print(f"  kind={d.metadata['kind']} len={len(d.metadata['raw'])} contains={'체류기간' in d.metadata['raw']}")
        print("   ", d.metadata["raw"][:200].replace("\n", " "))

print("\n[01 p.10 전체 청크 목록]")
for i, c in enumerate(chunks_v15):
    if c.metadata["source"].startswith("01") and c.metadata["page"] == 10:
        print(f"  idx={i} {c.metadata['kind']} len={len(c.metadata['raw'])} | {c.metadata['raw'][:80].replace(chr(10),' ')}")


[체류기간 문장 포함 청크]
  idx=17 01 p.10 table len=1461
    [01_2027학년도_재외국민_모집요강.hwp p.10] | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | |---|---|---|-
  idx=18 01 p.10 table len=1764
    [01_2027학년도_재외국민_모집요강.hwp p.10] | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | |---|---|---|-

[현재 Q3 검색된 01 p.10 청크]
  kind=table len=1404 contains=True
    [01_2027학년도_재외국민_모집요강.hwp p.10] | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 | 지원자격 
  kind=table len=1461 contains=True
    [01_2

In [148]:
def iter_sentences(raw):
    text = raw.replace("|", "\n")
    for sent in re.split(r"(?<=[.다])\s+|\n", text):
        sent = sent.strip(" -•\t")
        if len(sent) >= 25:
            yield sent


def missing_conditions(answer, docs, max_sent=3):
    cited = []
    for fname, page in CITE_RE.findall(answer):
        key = (fname.strip()[:2], page)
        matches = [d for d in docs if d.metadata["source"].startswith(key[0]) and str(d.metadata["page"]) == page]
        matches.sort(key=lambda d: d.metadata["kind"] != "text")
        if matches and key not in {(d.metadata["source"][:2], str(d.metadata["page"])) for d in cited}:
            cited.append(matches[0])
    if not cited:
        cited = [d for d in docs[:3] if d.metadata["kind"] == "text"][:1] or docs[:1]

    ans_toks = _content_tokens(answer)
    missing = []
    for d in cited[:2]:
        for sent in iter_sentences(d.metadata.get("raw", "")):
            if not COND_MARK.search(sent):
                continue
            s_toks = _content_tokens(sent)
            if len(s_toks) < 3:
                continue
            overlap = len(s_toks & ans_toks) / len(s_toks)
            if overlap < 0.5:
                missing.append((overlap, sent))
            if len(missing) >= max_sent:
                return [s for _, s in missing]
    return [s for _, s in missing]


miss = missing_conditions(a3, docs3)
print(f"미반영 조건문 {len(miss)}개:")
for s in miss:
    print(f"  - {s[:240]}")


미반영 조건문 3개:
  - 영주 및 거주기간 내에서 이수한 학기 또는 학년의 해당 기간 학년 또는 학기의 이수여부와 인정하는 재학 기간 등은 해당국의 교육관계법령 등에 의한 학제, 당해 학교의 교육과정, 그리고 대한민국의 학제와 동일급 별 학교의 교육과정 등을 고려하여 결정.
  - ※ 단, 학제차이로 인한 중복은 재학이 연속적인 경우에 한하여 인정함(복수 발생 인정) ※ 누락된 학기는 한 학기에 한하여 중복된 1개 학기로 대체할 수 있음.
  - 2개 학기 이상 연속 중복 학기의 경우 누락된 1개 학기로 대체할 수 있음.


In [149]:
for d in docs3:
    if d.metadata["source"].startswith("01") and d.metadata["page"] == 10 and "체류기간" in d.metadata["raw"]:
        sents = list(iter_sentences(d.metadata["raw"]))
        print(f"\n=== chunk kind={d.metadata['kind']} len={len(sents)} ===")
        for i, s in enumerate(sents):
            if "체류" in s or "또한" in s or "편입" in s:
                print(f"  [{i}] {s[:200]}")
                if i > 0:
                    print(f"      prev: {sents[i-1][:150]}")
        break



=== chunk kind=table len=9 ===
  [1] 재외국민등록부등본상 해당국의 거주기간에서 출입국사실증명서상 국내체류일수를 제외한 기간
      prev: [01_2027학년도_재외국민_모집요강.hwp p.10]


In [150]:
for d in docs3:
    if d.metadata["source"].startswith("01") and d.metadata["page"] == 10 and "국외근무자" in d.metadata["raw"]:
        print(f"\n=== chunk kind={d.metadata['kind']} len={len(d.metadata['raw'])} ===")
        for i, s in enumerate(iter_sentences(d.metadata["raw"])):
            print(f"  [{i}] {s[:160]}")
        break



=== chunk kind=table len=1461 ===
  [0] [01_2027학년도_재외국민_모집요강.hwp p.10]
  [1] 고교과정 1개 학년을 포함하여, 중·고교과정 통산 3년 재학한 기간 (3개 학년 이상) (성적이 나온 경우만 인정) 학기 개시일 ~ 다음학년도 동일학기 개시일 전일(약 365일)을 1개 학년으로 함 단, 해당 1개 학년 기간 내 모든 학기를 이수한 자는 학기 개시일부터 다음 학년도 동일
  [2] ※ 국외 재학기간, 체류기간, 국외근무자 재직기간을 모두 충족하여야 함.


In [151]:
def missing_conditions(answer, docs, max_sent=2):
    """답변을 실질적으로 뒷받침하는 청크 안의 조건문이 답변에 반영됐는지 확인."""
    ans_toks = _content_tokens(answer)
    scored = []
    for d in docs[:6]:
        c_toks = _content_tokens(d.metadata.get("raw", ""))
        if not c_toks:
            continue
        scored.append((len(c_toks & ans_toks) / len(c_toks), d))
    scored.sort(key=lambda x: x[0], reverse=True)

    missing = []
    for cov, d in scored[:2]:
        if cov < 0.30:
            continue
        for sent in iter_sentences(d.metadata.get("raw", "")):
            if not COND_MARK.search(sent):
                continue
            s_toks = _content_tokens(sent)
            if len(s_toks) < 3:
                continue
            overlap = len(s_toks & ans_toks) / len(s_toks)
            if overlap < 0.5:
                missing.append((round(overlap, 2), sent))
        if len(missing) >= max_sent:
            break
    return [s for _, s in missing[:max_sent]]


miss = missing_conditions(a3, docs3)
print(f"미반영 조건문 {len(miss)}개:")
for s in miss:
    print("  -", s[:240])


미반영 조건문 2개:
  - ※ 국외 재학기간, 체류기간, 국외근무자 재직기간을 모두 충족하여야 함.
  - 유형별 요건 충족 기간은 학생의 재학기간과 동일하거나 그 기간을 포함해야 함


In [152]:
def ask_v20(question, top_n=8, t_budget=10.0, return_meta=False):
    t0 = time.perf_counter()
    route = route_query(question)
    tdoc = target_doc(question)

    if route == "page" and tdoc:
        hit = toc_lookup(question, tdoc)
        if hit:
            topic = hit["title"]
            answer = (f"{topic}{_particle(topic)} {doc_title(hit['source'])} "
                      f"{hit['page']}페이지에 안내되어 있습니다. "
                      f"(출처: {hit['source']} p.{hit['doc_page']})")
            toc_doc = next(c for c in chunks_v15
                           if c.metadata["source"] == hit["source"] and c.metadata["page"] == hit["doc_page"])
            g = ground_answer(answer, [toc_doc], question)
            sec = time.perf_counter() - t0
            if return_meta:
                return {"answer": answer, "route": "page_lookup", "sec": sec, "conf": 0.99,
                        "cov": 1.0, "ground": g, "retried": False,
                        "docs": [(hit["source"][:2], hit["doc_page"])]}
            return answer, 1, sec

    docs, route = retrieve_v15(question, top_n=top_n)
    kw = search_keywords(question)
    ctx = "\n\n".join(f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.metadata['raw']}"
                      for i, d in enumerate(docs, 1))
    answer = llm.invoke([("system", DRAFT_PROMPT_V3),
                         ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content
    g = ground_answer(answer, docs, question)
    conf, cov = make_confidence(route, kw, docs, g)
    retried = False

    missing_items = missing_list_items(question, answer, docs)
    missing_conds = missing_conditions(answer, docs, max_sent=1)
    weak = (g["score"] < 0.78 or g["not_found"] or bool(missing_items) or bool(missing_conds) or re.search(
        r"(규정이 없|안내되어 있지 않|나와 있지 않|확인할 수 없|제공된 자료에는.{0,20}없)", answer))
    if weak and time.perf_counter() - t0 < t_budget:
        retried = True
        hints = []
        if missing_items:
            hints.append("[Exact items to include]\n" + ", ".join(missing_items))
        if missing_conds:
            hints.append("[Conditions that must be reflected]\n" + "\n".join(missing_conds))
        answer = llm.invoke([("system", REPAIR_PROMPT_V2),
                             ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}\n\n[Draft]\n{answer}\n\n"
                                       + "\n\n".join(hints))]).content
        g = ground_answer(answer, docs, question)
        conf, cov = make_confidence(route, kw, docs, g)

    sec = time.perf_counter() - t0
    if return_meta:
        return {"answer": answer, "route": route, "sec": sec, "conf": conf, "cov": cov,
                "ground": g, "retried": retried,
                "docs": [(d.metadata["source"][:2], d.metadata["page"]) for d in docs]}
    return answer, 1 + int(retried), sec


for run in range(2):
    r = ask_v20(testset_hard[2]["q"], return_meta=True)
    print(f"\n[run {run+1}] {r['sec']:.2f}s retry={r['retried']} conf={r['conf']}")
    print(r["answer"][:520])



[run 1] 5.47s retry=True conf=1.0
재외국민과 외국인(2% 이내)은 고교과정 1개 학년을 포함해 중·고교과정 통산 3년(3개 학년 이상) 재학해야 하며, 성적이 나온 경우만 인정합니다. 한 학년은 원칙적으로 학기 개시일부터 다음 학년도 동일 학기 개시일 전일까지(약 365일)로 보며, 해당 기간 내 모든 학기를 이수한 경우에는 다음 학년도 동일 학기 개시일 직전 학기 종료일까지를 1개 학년으로 인정합니다. 중간에 편입학한 경우 편입학일부터 역년으로 1년(365일)이 되는 날까지를 1개 학년으로 보고, 국외 재학기간·체류기간·국외근무자 재직기간을 모두 충족해야 합니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.10)

[run 2] 5.71s retry=True conf=1.0
재외국민과 외국인(2% 이내)은 고교과정 1개 학년을 포함해 중·고교과정 통산 3년(3개 학년 이상) 재학해야 하며, 성적을 취득한 경우만 인정됩니다. 한 학년은 학기 개시일부터 다음 학년도 같은 학기 개시일 전날까지(약 365일)로 보며, 그 기간 안의 모든 학기를 이수했다면 다음 학년도 같은 학기 직전 학기 종료일까지를 1개 학년으로 인정합니다. 중간에 편입학한 경우에는 편입학일부터 역년으로 1년(365일)이 되는 날까지를 1개 학년으로 봅니다. 국외 재학기간, 체류기간, 국외근무자 재직기간을 모두 충족해야 합니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.10)


In [153]:
h20 = eval_meta(testset_hard, 0, 5, "hard-v20", fn=ask_v20)

 1. O  4.6s conf=1.00 g=1.00 [procedure] 몸이 좀 불편한 학생인데요, 시험 볼 때 도와주는 사람 받을 수 있나요? 있
 2. O  1.9s conf=1.00 g=1.00 [table] 외국에서 살다 온 사람들 뽑는 거 있잖아요, 경영학과는 최대 몇명까지 뽑을 
 3. O  6.4s conf=1.00 g=1.00 [procedure] 외국 학교를 3년 다녀야 된다고 들었는데 한 학년은 어떻게 세는 거예요? 중
 4. O  2.2s conf=1.00 g=1.00 [eligibility] 외국에 있는 부모님이랑 저 둘 다 외국에 있어야 되나요? 저만 있으면 안 되
 5. O  2.3s conf=1.00 g=1.00 [procedure] 지역에서 의사 하면 학비 지원해준다던데 신청은 언제까지 해요? 1학기랑 2학
[hard-v20 1-5] 5/5 | 평균 3.47s | 최대 6.41s | 재시도 2


In [154]:
h20 += eval_meta(testset_hard, 5, 10, "hard-v20", fn=ask_v20)

 6. O  2.7s conf=0.82 g=1.00 [list] 강원도 속초 쪽이라고 하면 어디어디가 들어가요?
 7. O  5.5s conf=1.00 g=1.00 [eligibility] 지역의사 전형에서 두 군데 지역에 같이 넣을 수 있나요? 어디에 물어봐야 되
 8. O  5.3s conf=0.95 g=1.00 [general] 지역의사로 뽑히면 뭐 지원해주는 게 있어요? 돈 말고요. 그리고 의무 다 채
 9. O  2.2s conf=1.00 g=1.00 [list] 가천대 길병원에서 실습한다고 들었는데 몇 군데서 해요? 예를 들면 어디어디요
10. O  5.1s conf=0.91 g=1.00 [procedure] 합격하고 등록금까지 냈는데 마음이 바뀌었어요. 어떻게 취소해요? 취소하고 나
[hard-v20 6-10] 5/5 | 평균 4.15s | 최대 5.45s | 재시도 3


In [155]:
h20 += eval_meta(testset_hard, 10, 15, "hard-v20", fn=ask_v20)

11. O  2.5s conf=0.84 g=0.80 [eligibility] 예전에 태어난 사람은 신분증 사본도 내야 한다던데 저도 해당되나요? 서류는 
12. O  2.4s conf=0.88 g=1.00 [general] 학교 안에 도는 버스 있다던데 언제부터 있었고 지금은 몇대 돌아요?
13. O  0.0s conf=0.99 g=1.00 [page_lookup] 정시 요강에서 원서접수 얘기는 몇 쪽에 나와요?
14. O  1.9s conf=0.84 g=1.00 [table] 인천 캠퍼스에 바이오로직스학과 있잖아요, 거기 정시로 몇명 뽑아요?
15. O  0.0s conf=0.99 g=1.00 [page_lookup] 농어촌 전형은 정시 요강 몇 쪽에 있어요?
[hard-v20 11-15] 5/5 | 평균 1.38s | 최대 2.54s | 재시도 0


In [156]:
h20 += eval_meta(testset_hard, 15, 20, "hard-v20", fn=ask_v20)

16. O  4.2s conf=0.96 g=1.00 [eligibility] 수능으로 가군 일반전형에 넣으려는데 서류 뭐 내야 해요? 고등학교 졸업했으면
17. O  1.9s conf=1.00 g=1.00 [schedule] 이 면접 일정표는 어디 대학 거예요? 수시 중에 어떤 전형 대상이에요?
18. O  4.8s conf=0.75 g=0.65 [general] 기회균형이나 농어촌, 특성화고 애들은 면접 언제 봐요?
19. O  4.6s conf=0.93 g=1.00 [schedule] 지역균형은 면접 날짜를 접수 순서로 정해줘요? 아프면 바꿀 수 있어요?
20. O  1.9s conf=0.87 g=0.87 [schedule] 바람개비 전형 간호학과는 면접을 몇 번 봐요? 날짜랑 시간이 몇 개 있어요?
[hard-v20 16-20] 5/5 | 평균 3.48s | 최대 4.85s | 재시도 3


In [157]:
o20 = eval_meta(testset, 0, 5, "orig-v20", fn=ask_v20)

 1. O  5.9s conf=0.93 g=1.00 [procedure] 장애인 등록을 한 지원자가 입시전형 진행 중 지원이 필요하면 어떻게 해야 하
 2. O  1.7s conf=1.00 g=1.00 [table] 2027학년도 재외국민 모집요강에서 경영학과의 최대모집 가능인원과 모집인원은
 3. O  4.2s conf=0.96 g=1.00 [eligibility] 재외국민과 외국인(2% 이내) 전형에서 국외 재학기간 요건은 무엇이며, 1개
 4. O  4.0s conf=0.96 g=1.00 [procedure] 재외국민 지원자격에서 국외 체류는 누구에게 요구되며, 전 교육과정 이수자의 
 5. O  1.8s conf=1.00 g=1.00 [procedure] 대학은 학비 지원 신청을 1학기와 2학기 각각 언제까지 시·도지사에게 해야 
[orig-v20 1-5] 5/5 | 평균 3.52s | 최대 5.86s | 재시도 2


In [158]:
o20 += eval_meta(testset, 5, 10, "orig-v20", fn=ask_v20)

 6. O  1.8s conf=0.91 g=1.00 [eligibility] 강원 지역의 속초권에는 어떤 시·군이 포함되나요?
 7. O  2.1s conf=1.00 g=1.00 [table] 지역의사선발전형에서 진료권·광역권 동시 지원이 가능한지는 어디서 확인해야 하
 8. X  4.4s conf=1.00 g=1.00 [eligibility] 지역의사 지원센터가 제공하는 지원 종류에는 무엇이 있으며, 의무복무를 완료한
 9. O  2.1s conf=0.96 g=1.00 [list] 가천대 길병원과 협력한 임상실무실습은 몇 개 내과 사이트에서 진행되며, 예시
10. O  4.9s conf=0.96 g=1.00 [procedure] 본 대학에 합격해 문서등록까지 했지만 등록을 포기하려는 경우, 어떤 절차로 
[orig-v20 6-10] 4/5 | 평균 3.05s | 최대 4.89s | 재시도 2


In [159]:
o20 += eval_meta(testset, 10, 15, "orig-v20", fn=ask_v20)

11. O  2.5s conf=1.00 g=1.00 [procedure] 제6항 해당자가 신분증 사본을 제출해야 하는 출생 기준일은 언제이며, 제출 
12. O  4.0s conf=0.95 g=1.00 [general] 교내 순환 Eco 버스 ‘무당이’는 언제 도입됐으며, 현재 몇 대가 운행되고
13. O  0.0s conf=0.99 g=1.00 [page_lookup] 2027학년도 정시모집 안내에서 원서접수 내용은 몇 페이지에 있나요?
14. X  1.9s conf=0.89 g=1.00 [table] 2027학년도 정시모집에서 메디컬캠퍼스(인천)에 개설된 바이오로직스학과의 모
15. O  0.0s conf=0.99 g=1.00 [page_lookup] 2027학년도 정시 모집요강에서 농어촌 전형은 몇 페이지에 안내되어 있나요?
[orig-v20 11-15] 4/5 | 평균 1.68s | 최대 4.05s | 재시도 0


In [160]:
o20 += eval_meta(testset, 15, 20, "orig-v20", fn=ask_v20)

16. O  2.4s conf=1.00 g=1.00 [procedure] [가군] 일반전형 1·2에 지원하려는 2027학년도 수능 응시자가 고교 졸업
17. O  1.8s conf=1.00 g=1.00 [schedule] 이 면접고사 일정 안내는 어느 대학의 몇 학년도 수시모집 중 어떤 전형을 대
18. O  1.8s conf=0.86 g=0.80 [eligibility] 기회균형·농어촌(종합)·특성화고교 전형의 면접고사는 며칠에 진행되나요?
19. O  3.9s conf=1.00 g=1.00 [schedule] 지역균형 전형 면접 일정은 원서접수순으로 배정되나요? 학생 개인 사정으로 면
20. O  2.0s conf=0.93 g=1.00 [schedule] 가천바람개비 전형에서 간호학과 면접고사는 총 몇 개의 날짜·시간대에 편성되어
[orig-v20 16-20] 5/5 | 평균 2.39s | 최대 3.94s | 재시도 1


In [161]:
for r in o20:
    if not r["ok"]:
        print(f"\n[{r['idx']+1}] {r['q'][:80]}")
        print("  PRED:", r["pred"][:450])
        print("  WHY :", r["why"][:260])
        print("  GOLD:", r["gold"][:220])
        print("  DOCS:", r["docs"][:6])


[8] 지역의사 지원센터가 제공하는 지원 종류에는 무엇이 있으며, 의무복무를 완료한 지역의사는 어떤 우대 지원을 받을 수 있나요?
  PRED: 지역의사지원센터는 **주거 안정 지원, 직무교육 지원, 경력개발 지원, 해외 연수 지원, 권익보호, 진로상담·정보 제공**을 제공합니다. 법 제12·14조, 시행령 제11조, 시행규칙 제7조에 따라 의무복무 완료 후에는 해당 기관 또는 지역 내 공공보건의료기관의 우선 채용 우대조치를 받을 수 있으며, 해당 시·도 내 의료취약지에서 의료기관을 개설·운영하려는 경우 시·도지사의 행정적·재정적 지원을 받을 수 있습니다. (출처: 02_2027학년도_지역의사선발전형_안내.pdf p.18)
  WHY : 지원 종류 중 정답에 있는 ‘국내 연수’를 예측이 누락했습니다. 또한 우선 채용 조치를 할 수 있는 주체인 보건복지부장관, 지방자치단체의 장 또는 의무복무기관의 장을 제시하지 않았습니다.
  GOLD: 지역의사 지원센터는 주거 안정, 직무교육, 경력개발, 국내외 연수, 권익보호, 진로상담·정보 제공을 지원합니다. 의무복무 완료 후에는 보건복지부장관, 지방자치단체의 장 또는 의무복무기관의 장이 해당 기관이나 지역 내 공공보건의료기관에 우선 채용하는 우대조치를 할 수 있습니다.
  DOCS: [('02', 18), ('02', 17), ('02', 18), ('02', 20), ('02', 20), ('02', 3)]

[14] 2027학년도 정시모집에서 메디컬캠퍼스(인천)에 개설된 바이오로직스학과의 모집인원은 몇 명인가요?
  PRED: 2027학년도 정시모집에서 메디컬캠퍼스(인천)에 개설된 바이오로직스학과의 모집인원은 일반전형 1 30명, 일반전형 2 15명입니다. (출처: 04_2027학년도_정시_모집요강.pdf p.7)
  WHY : 정답은 모집인원 100명인데, 예측은 일반전형 1 30명과 일반전형 2 15명이라고 하여 핵심 수치가 일치하지 않습니다.
  GOLD: 바이오로직스학과는 메디컬캠퍼스(인천) 모집단위이며, 모

In [162]:
q8o = testset[7]["q"]
docs8, _ = retrieve_v15(q8o, top_n=8)
print("[02 p.18 raw 발췌]")
for d in docs8:
    if d.metadata["source"].startswith("02") and d.metadata["page"] == 18:
        idx = d.metadata["raw"].find("연수")
        print(" ...", d.metadata["raw"][max(0, idx-200):idx+250].replace("\n", " "))
        break

print("\n[missing_list_items]", missing_list_items(q8o, o20[7]["pred"], docs8))
print("[missing_conditions]", missing_conditions(o20[7]["pred"], docs8))
print("[ground]", ground_answer(o20[7]["pred"], docs8, q8o))


[02 p.18 raw 발췌]
 ...  지역의사지원센터 1 법 제12·14조, 시행령 제11조, 시행규칙 제7조, 지원고시 제12·13조 보건복지부장관은 지역의사에게 사명감 부여 및 지속적 근무 촉진을 위한 지원을 실시하며, 이를 위해 지역의사 지원센터를 운영합니다. 지원 종류 세부 내용 주거 안정 지원 의무복무지역 내 주거 안정을 위한 지원 직무교육 지원 의무복무 수행에 필요한 직무교육 및 연수 프로그램 지원 경력개발 지원 진로 및 경력개발을 위한 상담·지원 해외 연수 지원 국내외 연수 프로그램 지원 권익보호 지역의사의 권익보호 및 관련 의견 제시 진로상담·정보 제공 지역의사선발전형 입학 학생이 지역의사가 되기 위한 진로상담 및 정보 제공 � 지역의사지원센터 구성(지원고시 제12조) 구 분 업무 내용 중앙지역의사지원센터 • 권역지원센터 업무 총괄·조정 및 협력 지원 • 지역의사 지원 관련 표준교육 프로그램 개발·보급 • 지역의사 

[missing_list_items] ['지원고시 제12', '13조']
[missing_conditions] []
[ground] {'score': 1.0, 'num_ok': 1.0, 'date_ok': 1.0, 'ent_ok': 1.0, 'cite_ok': 1.0, 'n_num': 4, 'n_date': 0, 'n_ent': 0, 'not_found': False}


In [163]:
print("[02 p.18 청크 목록]")
for i, c in enumerate(chunks_v15):
    if c.metadata["source"].startswith("02") and c.metadata["page"] == 18:
        print(f"  idx={i} kind={c.metadata['kind']} len={len(c.metadata['raw'])}")
        print("   ", c.metadata["raw"][:260].replace("\n", " "))


[02 p.18 청크 목록]
  idx=72 kind=text len=834
    [02_2027학년도_지역의사선발전형_안내.pdf p.18] 16 지역의사 지원 Ⅸ 복무 중 지원 - 지역의사지원센터 1 법 제12·14조, 시행령 제11조, 시행규칙 제7조, 지원고시 제12·13조 보건복지부장관은 지역의사에게 사명감 부여 및 지속적 근무 촉진을 위한 지원을 실시하며, 이를 위해 지역의사 지원센터를 운영합니다. 지원 종류 세부 내용 주거 안정 지원 의무복무지역 내 주거 안정을 위한 지원 직무교육 지원 의무복무 수행에 필요한 직무교육 및 연수 프
  idx=73 kind=text len=174
    [02_2027학년도_지역의사선발전형_안내.pdf p.18] 해당 기관 또는 지역 내 공공보건의료기관에 우선 채용 우대조치 가능 의료취약지 개원 지원 의무복무 완료 후 해당 시·도 내 「공공보건의료에 관한 법률」 제12조에 따른 의료취약지에서 의료기관을 개설·운영하려는 경우 시·도지사가 행정적·재정적 지원 가능
  idx=102 kind=table len=147
    [02_2027학년도_지역의사선발전형_안내.pdf p.18] | 지원 종류 | 세부 내용 | |---|---| | 주거 안정 지원 | | | 직무교육 지원 | | | 경력개발 지원 | | | 해외 연수 지원 | | | 권익보호 | | | 진로상담·정보 제공 | |
  idx=103 kind=table len=96
    [02_2027학년도_지역의사선발전형_안내.pdf p.18] | 지원 종류 | 내 용 | |---|---| | 공공의료기관 우선 채용 | | | 의료취약지 개원 지원 | |


In [164]:
cands = [("02", 17), ("04", 9), ("05", 3), ("03", 6), ("01", 6), ("02", 10)]
for src, pg in cands:
    for c in chunks_v15:
        if c.metadata["source"].startswith(src) and c.metadata["page"] == pg and c.metadata["kind"] == "text":
            print(f"\n=== {src} p.{pg} ===")
            print(c.metadata["raw"][:650].replace("\n", " "))
            break



=== 02 p.17 ===
[02_2027학년도_지역의사선발전형_안내.pdf p.17] 15 시정명령 → 면허 정지 →면허 취소 3단계 절차 1 법 제16·17조, 시행규칙 제9·10조 의무복무를 이행하지 않으면 다음과 같은 제재가 단계적으로 부과됩니다. 단계 제재 종류 발동 요건 내용 및 절차 1단계 시정명령 복무형 지역의사가 다음을 위반한 경우 • 의무복무 기간·지역 위반(법 제7조제1항) • 무단 의무복무지역 변경(법 제9조) • 겸직금지 위반(법 제11조제1항) • 복무상황 보고 의무 위반(법 제15조제3항) 지역의사가 근무하는 의료기관의 장이 다음을 위반하는 경우 • 법 제15조제1항(자료제출 무단 거부) 위반 위반 사항, 법령 근거, 이행기간을 명시한 서면으로 시정명령 이행기간: 위반 유형별 1~2개월 ※ ‌시·도지사가 시정명령 시 보건복지부장관에게 위반사항과 시정명령의 내용 통보 2단계 면허 자격 정지 시정명령 수령 후 이행기간 내 미이행 1년 이내 의사 면허 자격 정지 + 과태료 최대 1천만 원 ※ 면허 정지 기간은 의무복무 기간에서 제외됨 3단계 면허 취소 다음 중 하나에 해당하는 경우 • 면허 자격 정지를 3회 이상 받은 경우 • ‌법 제7조를 위반하여 의무복무를 이행하지 않은 경우(지속적 불이행) 청문 절차 후 의사 면허 취소 ※ 취소 후 ‘의무복무

=== 04 p.9 ===
[04_2027학년도_정시_모집요강.pdf p.9] 35 원서접수 시 유의사항 1. 원서접수 중 유의사항을 반드시 확인하고 안내문의 진행요령에 따라 작성하여야 합니다. 2. 원서접수는 전형료 결제 후 접수가 완료됩니다. 3.  접수가 완료된 후에는 학과 및 전형 변경, 지원 포기 등은 불가능하므로 「모집요강」을 정확히 숙지하고 전형 유형, 지원학과/전공, 지원자격 등을 정확히 입력한 후 확인하고 접수해야 합니다. (제출서류 및 전형료는 반환하지 않음) 4. 성명, 주민등록번호는 반드시 주민등록상에 기재된 내용과 같아야 하며, 주소나 전화번호가 전형기간 

In [165]:
for src, pg in [("03", 10), ("03", 12), ("04", 35), ("02", 14), ("01", 7), ("03", 30)]:
    found = False
    for c in chunks_v15:
        if c.metadata["source"].startswith(src) and c.metadata["page"] == pg:
            print(f"\n=== {src} p.{pg} ({c.metadata['kind']}) ===")
            print(c.metadata["raw"][:520].replace("\n", " "))
            found = True
            break
    if not found:
        print(f"\n=== {src} p.{pg} 없음 ===")



=== 03 p.10 (text) ===
[03_2027학년도_수시_모집요강.pdf p.10] 가천대 길병원과 3대 연구소(뇌과학연구원, 암·당뇨연구원, 바이오나노연구원)는 첨단 의료 기술과 융합 연구를 통해 세계적인 혁 신을 이끌고 있습니다. 최첨단 연구 시설과 글로벌 협력을 기반으로 미래 의료의 새로운 기준을 제시합니다. 가천대 길병원·3대 연구소 국내 최초 연구중심병원 가천대 길병원 가천대 길병원은 2013년 국내 1기 10대 연구중심병원에 선정되었고 3년 주기 평가에서 지속적으로 우수한 연구 성과를 인 정받아 2022년 3회 연속으로 연구중심병원에 재지정됐습니다. 특히 뇌질환 영상진단 분야 및 대사성 질환 분야에서 뇌과학 연구원, 이길여 암·당뇨연구원과 연계한 연구를 통해 세계적인 연구기관과 어깨를 나란히 하고 있습니다. 한의과대학 한의학교육 인증평가 ‘인증’ 한국한의학교육평가원 2027.01. ~ 2028.12. 간호대학, 간호학 학사학위 프로그램 ‘5년 인증’ 한국간호교육평가원 2026.06. ~ 2031.06. 10 미래 의료의 새로운 기준을 세

=== 03 p.12 (text) ===
[03_2027학년도_수시_모집요강.pdf p.12] 의과대학 12 인류의 건강과 생명을 최우선으로 여기는 의료인들의 사명감은 수많은 질병과 도전에 맞서며 의료 혁신을 이끌어왔습니다. 가천대학교 의과대학은 기초의학 연구를 바탕으로 박애, 봉사, 애국의 정신을 실천하며, 미래를 선도할 참된 의료인을 양성합니다. 의료 혁신의 중심에서 인재를 키우는 가천 의과대학 의과대학 의학교육 인증평가 ‘인증’ 한국의학교육평가원 2026.03. ~ 2031.02.

=== 04 p.35 (text) ===
[04_2027학년도_정시_모집요강.pdf p.35] 61 2027학년도 정시모집요강 학교폭력조치사항에 따른 반영방법 전형 학교폭력조치사항 비고 1호 2호 3호 4호 5호 6호 7호 8호 9호 논술위주(논술) - - - –10 –20 –100 1,000점 만점 기준 감점 학생부

In [166]:
testset_holdout = [
    {"q": "지역의사가 의무복무를 안 지키면 어떻게 되나요? 단계별로 알려주세요.",
     "gold": "시정명령 → 면허 자격 정지 → 면허 취소 3단계로 제재가 부과됩니다. 2단계는 시정명령을 이행하지 않으면 1년 이내 의사 면허 자격 정지와 과태료 최대 1천만 원이 부과되며, 면허 정지 기간은 의무복무 기간에서 제외됩니다.",
     "src": "02 p.17"},
    {"q": "정시 원서 접수하고 나서 학과를 바꿀 수 있나요? 접수는 언제 완료되나요?",
     "gold": "원서접수는 전형료 결제 후 접수가 완료되며, 접수 완료 후에는 학과 및 전형 변경, 지원 포기가 불가능합니다. 제출서류 및 전형료는 반환하지 않습니다.",
     "src": "04 p.9"},
    {"q": "지역의사 하다가 그만두면 돈을 내야 한다는데, 금액별로 납부 기한이 어떻게 되나요?",
     "gold": "반환금 5천만 원 이하는 1년 이내, 5천만 원 초과~1억 원 이하는 3년 이내, 1억 원 초과는 5년 이내로 납부기한이 설정됩니다.",
     "src": "02 p.10"},
    {"q": "지역의사가 될 사람이 수련할 수 있는 전문과목은 모두 몇 개인가요? 수련병원은 어디에 있어야 하나요?",
     "gold": "전문과목은 총 26개이며, 수련병원은 원칙적으로 본인의 의무복무지역 소재 병원이어야 합니다. 부득이한 경우 의무복무지역 외 수련이 가능하나 수련기간은 의무복무 기간에 불산입됩니다.",
     "src": "02 p.14"},
    {"q": "재외국민 전형 합격자 발표는 언제이고, 충원합격자 발표는 언제까지 하나요?",
     "gold": "합격자 발표는 2026.08.13.(목), 충원합격자 발표는 2026.12.23(수)~29(화)에 본 대학 입학처 홈페이지에서 이루어집니다.",
     "src": "01 p.7"},
    {"q": "정시에서 학교폭력 6호 처분을 받으면 몇 점 깎이나요?",
     "gold": "수능위주 등 1,000점 만점 기준 전형에서 6호 처분은 -100점 감점입니다. (4호 -10점, 5호 -20점)",
     "src": "04 p.35"},
    {"q": "2027학년도 수시에서 새로 생긴 학과나 전형이 있나요?",
     "gold": "인공지능시스템학과가 신설되었고, 논술전형 모집단위에 한의예과와 약학과가 신설되었습니다.",
     "src": "03 p.30"},
    {"q": "가천대 길병원은 연구중심병원으로 언제 선정됐고, 이후 평가는 어땠나요?",
     "gold": "2013년 국내 1기 10대 연구중심병원에 선정되었고, 3년 주기 평가에서 우수한 성과를 인정받아 2022년 3회 연속 연구중심병원으로 재지정되었습니다.",
     "src": "03 p.10"},
    {"q": "가천대 의과대학 의학교육 인증평가는 언제부터 언제까지 유효한가요?",
     "gold": "한국의학교육평가원의 의학교육 인증평가 결과 '인증'을 받았으며, 유효기간은 2026.03.부터 2031.02.까지입니다.",
     "src": "03 p.12"},
    {"q": "지역의사가 의무복무지를 벗어나서 수련하면 그 기간은 어떻게 되나요?",
     "gold": "부득이한 경우 의무복무지역 외 수련이 가능하지만, 그 수련기간은 의무복무 기간에 불산입됩니다.",
     "src": "02 p.14"},
]

print("held-out:", len(testset_holdout))
for i, t in enumerate(testset_holdout, 1):
    print(f"{i:2d}. {t['q'][:55]} | {t['src']}")


held-out: 10
 1. 지역의사가 의무복무를 안 지키면 어떻게 되나요? 단계별로 알려주세요. | 02 p.17
 2. 정시 원서 접수하고 나서 학과를 바꿀 수 있나요? 접수는 언제 완료되나요? | 04 p.9
 3. 지역의사 하다가 그만두면 돈을 내야 한다는데, 금액별로 납부 기한이 어떻게 되나요? | 02 p.10
 4. 지역의사가 될 사람이 수련할 수 있는 전문과목은 모두 몇 개인가요? 수련병원은 어디에 있어야 하나요 | 02 p.14
 5. 재외국민 전형 합격자 발표는 언제이고, 충원합격자 발표는 언제까지 하나요? | 01 p.7
 6. 정시에서 학교폭력 6호 처분을 받으면 몇 점 깎이나요? | 04 p.35
 7. 2027학년도 수시에서 새로 생긴 학과나 전형이 있나요? | 03 p.30
 8. 가천대 길병원은 연구중심병원으로 언제 선정됐고, 이후 평가는 어땠나요? | 03 p.10
 9. 가천대 의과대학 의학교육 인증평가는 언제부터 언제까지 유효한가요? | 03 p.12
10. 지역의사가 의무복무지를 벗어나서 수련하면 그 기간은 어떻게 되나요? | 02 p.14


In [167]:
h, n, _ = recall_eval(testset_holdout, k=6)
print(f"held-out Recall@6 {h}/10 | ±1p {n}/10")

  MISS 03 p.30 [general] | 2027학년도 수시에서 새로 생긴 학과나 전형이 있나요? | [('03', 28), ('03', 27), ('03', 29), ('03', 41), ('03', 40), ('03', 41)]
held-out Recall@6 9/10 | ±1p 10/10


In [168]:
ho20 = eval_meta(testset_holdout, 0, 5, "holdout-v20", fn=ask_v20)

 1. X  6.3s conf=1.00 g=1.00 [procedure] 지역의사가 의무복무를 안 지키면 어떻게 되나요? 단계별로 알려주세요.
 2. X  4.7s conf=0.72 g=0.60 [eligibility] 정시 원서 접수하고 나서 학과를 바꿀 수 있나요? 접수는 언제 완료되나요?
 3. O  2.0s conf=1.00 g=1.00 [procedure] 지역의사 하다가 그만두면 돈을 내야 한다는데, 금액별로 납부 기한이 어떻게 
 4. O  2.3s conf=1.00 g=1.00 [general] 지역의사가 될 사람이 수련할 수 있는 전문과목은 모두 몇 개인가요? 수련병원
 5. X  2.5s conf=1.00 g=1.00 [general] 재외국민 전형 합격자 발표는 언제이고, 충원합격자 발표는 언제까지 하나요?
[holdout-v20 1-5] 2/5 | 평균 3.59s | 최대 6.33s | 재시도 2


In [169]:
for r in ho20:
    if not r["ok"]:
        print(f"\n[{r['idx']+1}] {r['q'][:70]}")
        print("  PRED:", r["pred"][:450])
        print("  WHY :", r["why"][:260])
        print("  DOCS:", r["docs"][:6])


[1] 지역의사가 의무복무를 안 지키면 어떻게 되나요? 단계별로 알려주세요.
  PRED: 의무복무 위반 시 제재는 단계적으로 진행됩니다. **1단계 시정명령:** 의무복무 기간·지역 위반, 무단 지역 변경, 겸직금지 위반, 복무상황 보고 의무 위반 등이 있으면 위반 사항과 이행기간(위반 유형별 1~2개월)을 명시한 서면 시정명령을 받습니다. **2단계 면허 자격 정지:** 시정명령을 받고도 이행기간 내 이행하지 않으면 의사 면허가 1년 이내 정지되고 과태료 최대 1천만 원이 부과될 수 있으며, 정지 기간은 의무복무 기간에서 제외됩니다. **3단계 면허 취소:** 면허 자격 정지를 3회 이상 받거나 의무복무를 계속 이행하지 않으면 청문 절차 후 면허가 취소될 수 있고, 재교부는 의무복무 이행 조건으로만 가능합니다. 이와 별도로 의무복무 불이행이 확인되면 학비 전액과 법정이율 이자를 반환해야 하며, 근무 관련 상황을 보고하지 않거나 거짓으로 보고한 경우에도 과태료 5백만 원 이하가 부과
  WHY : 정답은 시정명령을 이행하지 않으면 과태료 최대 1천만 원이 부과된다고 명시하지만, 예측은 과태료가 부과될 수 있다고 표현해 제재의 확정 여부가 달라집니다.
  DOCS: [('02', 19), ('02', 18), ('02', 19), ('02', 17), ('02', 16), ('02', 17)]

[2] 정시 원서 접수하고 나서 학과를 바꿀 수 있나요? 접수는 언제 완료되나요?
  PRED: 정시 원서접수가 완료된 후에는 학과 및 전형 변경이나 지원 포기가 불가능하므로, 접수 전에 지원학과·전공과 지원자격 등을 정확히 확인해야 합니다. 원서접수는 전형료 결제 후 완료되며, 완료 여부는 원서와 수험표를 통해 직접 확인할 수 있습니다. (출처: 04_2027학년도_정시_모집요강.pdf p.8-9)
  WHY : 전형료 결제 후 접수가 완료되고, 완료 후 학과·전형 변경이나 지원 포기가 불가능하다는 핵심 내용은 포함했지만, 제출서류와 전형료가 반환되지 않는다는 내용

In [170]:
GENERIC_DETAILS = {
    "2027학년도", "모집요강", "경우", "내용", "지원", "대학", "학생", "기간", "방법", "절차",
    "안내", "확인", "가능", "해당", "관련", "본교", "때문", "정도", "이상", "이하", "경우에는",
    "지원자", "전형", "모집", "입학", "서류", "평가", "결과", "사항", "신청", "제출",
}


def _noun_tokens(text):
    return {t.form for t in kiwi.tokenize(text)
            if t.tag in {"NNG", "NNP"} and len(t.form) >= 2 and t.form not in GENERIC_DETAILS}


def missing_details(answer, docs, max_items=3):
    """답변을 뒷받침하는 근거 문장의 핵심 명사(장소·서류·항목)가 답변에 있는지 확인."""
    ans_toks = _content_tokens(answer)
    best = (0.0, "")
    for d in docs[:4]:
        for sent in iter_sentences(d.metadata.get("raw", "")):
            s_toks = _content_tokens(sent)
            if len(s_toks) < 4:
                continue
            ov = len(s_toks & ans_toks) / len(s_toks)
            if ov > best[0]:
                best = (ov, sent)
    if best[0] < 0.45:
        return []
    missing = [t for t in _noun_tokens(best[1]) if t not in answer]
    # 숫자·날짜는 별도 검증에서 다루므로 명사만 반환
    return missing[:max_items]


for idx in (0, 1, 4):
    r = ho20[idx]
    print(f"[{idx+1}] missing_details:", missing_details(r["pred"], [c for c in chunks_v15 if c.metadata['source'].startswith(r['docs'][0][0]) and c.metadata['page'] == r['docs'][0][1]]))


[1] missing_details: []
[2] missing_details: []
[5] missing_details: []


In [171]:
ho20 += eval_meta(testset_holdout, 5, 10, "holdout-v20", fn=ask_v20)

 6. X  1.9s conf=1.00 g=1.00 [general] 정시에서 학교폭력 6호 처분을 받으면 몇 점 깎이나요?
 7. X  4.0s conf=0.75 g=1.00 [general] 2027학년도 수시에서 새로 생긴 학과나 전형이 있나요?
 8. O  2.2s conf=0.95 g=1.00 [general] 가천대 길병원은 연구중심병원으로 언제 선정됐고, 이후 평가는 어땠나요?
 9. O  1.5s conf=0.96 g=1.00 [general] 가천대 의과대학 의학교육 인증평가는 언제부터 언제까지 유효한가요?
10. O  2.5s conf=1.00 g=1.00 [procedure] 지역의사가 의무복무지를 벗어나서 수련하면 그 기간은 어떻게 되나요?
[holdout-v20 6-10] 3/5 | 평균 2.44s | 최대 4.04s | 재시도 1


In [172]:
for r in ho20:
    if not r["ok"] and r["idx"] >= 5:
        print(f"\n[{r['idx']+1}] {r['q'][:70]}")
        print("  PRED:", r["pred"][:420])
        print("  WHY :", r["why"][:260])
        print("  DOCS:", r["docs"][:6])


[6] 정시에서 학교폭력 6호 처분을 받으면 몇 점 깎이나요?
  PRED: 정시 학교폭력 6호 처분(출석정지)은 1,000점 만점 기준으로 20점 감점됩니다. 적용 시점은 전 학년 전 학기이며, 학교생활기록부 기재사항은 2026년 12월 1일 기준입니다. (출처: 04_2027학년도_정시_모집요강.pdf p.35)
  WHY : 정답은 1,000점 만점 기준 6호 처분 시 100점 감점입니다. 예측은 20점 감점이라고 하여 핵심 수치가 틀렸습니다.
  DOCS: [('04', 35), ('04', 34), ('04', 35), ('04', 36), ('03', 101), ('04', 31)]

[7] 2027학년도 수시에서 새로 생긴 학과나 전형이 있나요?
  PRED: 모집요강에는 2027학년도 수시에서 신설된 학과나 전형이라고 명시된 내용은 없습니다. 다만 조기취업형계약학과 전형의 모집단위로 반도체·디스플레이학과, 반도체설계학과, 바이오의료기기학과, 게임·영상학과, 미래자동차학과가 안내되어 있으나, 신설 여부는 확인할 수 없습니다. (출처: 03_2027학년도_수시_모집요강.pdf p.29, p.70)
  WHY : 정답의 핵심 내용인 인공지능시스템학과 신설과 논술전형 모집단위에 한의예과·약학과가 추가된 사실이 예측 답변에 포함되어 있지 않습니다.
  DOCS: [('03', 28), ('03', 27), ('03', 29), ('03', 41), ('03', 40), ('03', 41)]


In [173]:
print("=== 04 p.35 청크 ===")
for c in chunks_v15:
    if c.metadata["source"].startswith("04") and c.metadata["page"] == 35:
        print(f"\nkind={c.metadata['kind']} len={len(c.metadata['raw'])}")
        print(c.metadata["raw"][:900].replace("\n", " "))

print("\n\n=== Q7 검색 디버그 ===")
q7 = testset_holdout[6]["q"]
print("kw:", search_keywords(q7))
for name, r in (("BM25", bm25_v15.invoke(search_keywords(q7))), ("Dense", dense_v15.invoke(q7))):
    print(f"[{name}]")
    for i, d in enumerate(r[:8], 1):
        print(f"  {i}. {d.metadata['source'][:2]} p.{d.metadata['page']} {d.metadata['kind']}")
print("\n[03 p.30 raw]")
for c in chunks_v15:
    if c.metadata["source"].startswith("03") and c.metadata["page"] == 30:
        print(c.metadata["raw"][:400].replace("\n", " "))


=== 04 p.35 청크 ===

kind=text len=788
[04_2027학년도_정시_모집요강.pdf p.35] 61 2027학년도 정시모집요강 학교폭력조치사항에 따른 반영방법 전형 학교폭력조치사항 비고 1호 2호 3호 4호 5호 6호 7호 8호 9호 논술위주(논술) - - - –10 –20 –100 1,000점 만점 기준 감점 학생부교과(학생부우수자, 지역의사선발, 농어촌(교과)) - - - –10 –20 –100 1,000점 만점 기준 감점 학생부교과(지역균형, 특성화고교) 1단계 - - - –10 –20 –100 1,000점 만점 기준 감점 2단계 정성평가 학생부종합(가천의약학, 가천바람개비, 기회균형, 농어촌(종합), 교육기회균형, 조기취업형계약학과) 정성평가 1, 2단계 모두 정성평가 (평가요소 중 인성(40%)에서 각 호에 따라 평가 반영) 학생부종합(특성화고졸재직자) 정성평가 실기위주 일괄합산전형 - - - –10 –20 –100 1,000점 만점 기준 감점 단계전형 1단계 - - - –10 –20 –100 1,000점 만점 기준 감점 2단계 정성평가 수능위주 - - - –10 –20 –100 1,000점 만점 기준 감점 학교폭력 가해학생에 대한 조치사항 제1호 (피해학생에 대한 서면사과) 제2호 (피해학생 및 신고·고발 학생에 대한 접촉, 협박 및 보복행위의 금지) 제3호 (학교에서의 봉사) 제4호 (사회봉사) 제5호 (학내외 전문가에 의한 특별 교육이수 또는 심리치료) 제6호 (출석정지) 제7호 (학급교체) 제8호 (전학) 제9호 (퇴학처분)–고등학생만 해당됨 기타사항 – 학교폭력조치사항의 최종 적용시점은 전 학년 전 학기로 함

kind=text len=173
[04_2027학년도_정시_모집요강.pdf p.35] 제6호 (출석정지) 제7호 (학급교체) 제8호 (전학) 제9호 (퇴학처분)–고등학생만 해당됨 기타사항 – 학교폭력조치사항의 최종 적용시점은 전 학년 전 학기로 함 – 학교생활기록부 학교폭력조치사항의 반영은 2026년 12월

In [174]:
SYNONYMS.update({"새로 생": "신설", "새로 만": "신설", "신설된": "신설"})
ROUTE_PATTERNS = [
    ("page", r"(몇\s*쪽|페이지|쪽에|몇\s*장)"),
    ("table", r"(몇\s*명|몇명|인원|모집|선발|정원|명까지|명씩|비율|점수|등급|컷|몇\s*점|감점)"),
    ("procedure", r"(어떻게|신청|제출|절차|방법|해야\s*하|어디서)"),
    ("eligibility", r"(자격|조건|대상|가능|되나요|될까|수\s*있나|받을\s*수)"),
    ("list", r"(어디어디|무엇무엇|종류|예시|포함|목록|어떤\s*게)"),
]


def row_summary3(cols, data, mode, max_rows=30, max_chars=2600):
    H = len(cols)
    if H == 0:
        return ""
    num_cols = [c for c in cols if re.search(r"\d+\s*호", c)]
    lines, used = [], 0
    for r in data:
        if not r:
            continue
        vals = [(j, v) for j, v in enumerate(r) if v]
        if len(vals) < 3:
            continue

        # 숫자/대시로만 이루어진 행: 빈칸을 압축해 숫자 열에 순서대로 정렬
        body = [v for j, v in vals[1:]]
        if len(num_cols) >= 3 and body and all(v == "-" or NUMERIC.match(v) for v in body):
            label = vals[0][1]
            pairs = [f"{num_cols[i]}={v}" for i, v in enumerate(body) if i < len(num_cols)]
        elif mode == "wide":
            label = next((c for c in r if any(k in c for k in ("학과", "학부", "전공", "지역"))), "")
            pairs = []
            for j, v in enumerate(r):
                if not v or v == label or v in GENERIC_COLS:
                    continue
                col = cols[j] if j < len(cols) else ""
                if col and col != v and col != "모집단위":
                    pairs.append(f"{col}={v}")
            if not label:
                continue
        else:
            ded = dedup_cells(r)
            if len(ded) == H + 1 and len(ded) >= 3:
                ded = [ded[0], f"{ded[1]} {ded[2]}".strip()] + ded[3:]
            if len(ded) < H:
                ded = [""] * (H - len(ded)) + ded
            if len(ded) > H:
                ded = ded[-H:]
            vv = [(j, v) for j, v in enumerate(ded) if v]
            if len(vv) < 2:
                continue
            first_val = next((j for j, v in vv if is_value(v)), H)
            label = " ".join(v for j, v in vv if j < first_val and v not in GENERIC_COLS)
            if not label or is_value(label):
                continue
            pairs = [f"{cols[j]}={v}" for j, v in vv if j >= first_val and cols[j]]
        if not label or not pairs:
            continue
        line = f"- {label}: " + ", ".join(pairs[:6])
        if used + len(line) > max_chars or len(lines) >= max_rows:
            break
        lines.append(line)
        used += len(line)
    return "\n".join(lines)


def build_contextual_chunks(chunks):
    out, with_summary = [], 0
    for c in chunks:
        src, page, kind = c.metadata["source"], c.metadata["page"], c.metadata["kind"]
        title = doc_title(src)
        raw = c.page_content
        header = f"[문서] {title} | p.{page} | {'표' if kind == 'table' else '본문'}"
        summary, mode = "", ""
        if kind == "table":
            caption, mode, cols, data = parse_table2(raw)
            if caption:
                header += f" | 표제: {caption}"
            summary = row_summary3(cols, data, mode)
            if summary:
                with_summary += 1
        body = header + (("\n[표 행 요약]\n" + summary) if summary else "") + "\n[원문]\n" + raw
        md = dict(c.metadata)
        md.update(raw=raw, title=title, table_mode=mode, contextual=True)
        out.append(Document(page_content=body, metadata=md))
    print(f"표 청크 {sum(c.metadata['kind']=='table' for c in chunks)}개 중 행 요약 {with_summary}개")
    return out


chunks_v15 = build_contextual_chunks(chunks)
for c in chunks_v15:
    if c.metadata["source"].startswith("04") and c.metadata["page"] == 35 and c.metadata["kind"] == "table":
        print("\n[04 p.35 요약]")
        print(c.page_content.split("[원문]")[0][:600])
        break


표 청크 236개 중 행 요약 49개

[04 p.35 요약]
[문서] 2027학년도 정시 모집요강 | p.35 | 표 | 표제: 전형



In [175]:
for src, pg in (("01", 3), ("04", 7), ("05", 2), ("04", 35)):
    for c in chunks_v15:
        if c.metadata["source"].startswith(src) and c.metadata["page"] == pg and c.metadata["kind"] == "table":
            pre = c.page_content.split("[원문]")[0]
            print(f"\n=== {src} p.{pg} ===\n{pre[:420]}")
            break



=== 01 p.3 ===
[문서] 2027학년도 재외국민 모집요강 | p.3 | 표 | 표제: 모집인원
[표 행 요약]
- 인문 경영학과: 최대모집 가능인원=15, 모집인원=16
- 인문 회계세무학과: 최대모집 가능인원=5, 모집인원=16
- 인문 관광경영학과: 최대모집 가능인원=4, 모집인원=16
- 인문 의료산업경영학과: 최대모집 가능인원=4, 모집인원=16
- 자연 금융·빅데이터학부 금융수학전공: 최대모집 가능인원=10, 모집인원=16
- 자연 금융·빅데이터학부 빅데이터경영전공: 최대모집 가능인원=10, 모집인원=16
- 인문 미디어커뮤니케이션학과: 최대모집 가능인원=3, 모집인원=16
- 인문 경제학과: 최대모집 가능인원=5, 모집인원=16
- 인문 응용통계학과: 최대모집 가능인원=4, 모집인원=16
- 인문 사회복지학과: 최대모집 가능인원=4, 모집인원=16

=== 04 p.7 ===
[문서] 2027학년도 정시 모집요강 | p.7 | 표 | 표제: 모집단위
[표 행 요약]
- 컴퓨터공학과: 일반전형 1 가군=30, 일반전형 2 가군=18
- 인공지능학과: 일반전형 1 다군=46, 일반전형 2 다군=21
- 간호학과: 일반전형 1 나군=70, 일반전형 2 나군=30
- 바이오로직스학과: 일반전형 1 다군=30, 일반전형 2 다군=15


=== 05 p.2 ===
[문서] 2027학년도 수시 면접고사 일정 | p.2 | 표 | 표제: 면접/전형 일정
[표 행 요약]
- 경영학과: 10월 31일(토) 오전=★, 10월 31일(토) 오후=★, 11월 01일(일) 오전=★, 11월 01일(일) 오후=★
- 회계세무학과: 11월 07일(토) 오전=★, 11월 07일(토) 오후=★
- 의료산업경영학과: 10월 31일(토) 오전=★, 10월 31일(토) 오후=★
- 금융·빅데이터학부: 11월 07일(토) 오전=★, 11월 07일(토) 오후=★
- 경제학과: 11월 08일(일) 오전=★, 오후=★
- 응용통계학과: 11월 01일(일) 오전=★, 11월 01일

In [176]:
def build_contextual_chunks(chunks):
    out, with_summary = [], 0
    for c in chunks:
        src, page, kind = c.metadata["source"], c.metadata["page"], c.metadata["kind"]
        title = doc_title(src)
        raw = c.page_content
        header = f"[문서] {title} | p.{page} | {'표' if kind == 'table' else '본문'}"
        summary, mode = "", ""
        if kind == "table":
            caption, mode, cols, data = parse_table2(raw)
            if caption:
                header += f" | 표제: {caption}"
            summary = row_summary2(cols, data, mode)
            if summary:
                with_summary += 1
        body = header + (("\n[표 행 요약]\n" + summary) if summary else "") + "\n[원문]\n" + raw
        md = dict(c.metadata)
        md.update(raw=raw, title=title, table_mode=mode, contextual=True)
        out.append(Document(page_content=body, metadata=md))
    print(f"표 청크 {sum(c.metadata['kind']=='table' for c in chunks)}개 중 행 요약 {with_summary}개")
    return out


chunks_v15 = build_contextual_chunks(chunks)
DRAFT_PROMPT_V3 = DRAFT_PROMPT_V3.replace(
    "- For tables, align the row label and column headers by position. Never swap columns.",
    "- For tables, align the row label and column headers by position. Never swap columns.\n"
    "- For numbered columns (1호, 2호, ...), count every cell from the left, including dashes; match each value to its column."
)

t0 = time.perf_counter()
vectordb_v15 = Chroma.from_documents(
    documents=chunks_v15, embedding=embeddings,
    collection_name=f"gachon_v15b_{int(time.time())}",
    collection_metadata={"hnsw:space": "cosine"},
)
dense_v15 = vectordb_v15.as_retriever(search_kwargs={"k": 20})
bm25_v15 = BM25Retriever.from_documents(chunks_v15, preprocess_func=korean_tokenizer, k=20)
CHUNK_INDEX_V15 = {c.page_content: i for i, c in enumerate(chunks_v15)}
print(f"재색인 {time.perf_counter()-t0:.1f}s")


표 청크 236개 중 행 요약 57개
재색인 12.7s


In [177]:
for idx in (5, 6):
    t = testset_holdout[idx]
    print(f"\n[{idx+1}] {t['q']}")
    print("  kw:", search_keywords(t["q"]))
    docs, route = retrieve_v15(t["q"], top_n=8)
    print("  route:", route, "| docs:", [(d.metadata["source"][:2], d.metadata["page"]) for d in docs[:8]])
    r = ask_v20(t["q"], return_meta=True)
    print(f"  {r['sec']:.1f}s retry={r['retried']} conf={r['conf']}")
    print("  PRED:", r["answer"][:320])
    print("  GOLD:", t["gold"][:180])



[6] 정시에서 학교폭력 6호 처분을 받으면 몇 점 깎이나요?
  kw: 학교 폭력 처분 정시모집
  route: table | docs: [('04', 35), ('04', 34), ('04', 35), ('04', 35), ('04', 34), ('04', 38), ('04', 31), ('03', 101)]
  2.7s retry=False conf=1.0
  PRED: 정시에서 학교폭력 6호(출석정지) 처분은 1,000점 만점 기준으로 20점 감점됩니다. (출처: 04_2027학년도_정시_모집요강.pdf p.35)
  GOLD: 수능위주 등 1,000점 만점 기준 전형에서 6호 처분은 -100점 감점입니다. (4호 -10점, 5호 -20점)

[7] 2027학년도 수시에서 새로 생긴 학과나 전형이 있나요?
  kw: 2027 학년도 학과 전형 수시모집 신설
  route: general | docs: [('03', 28), ('03', 27), ('03', 29), ('03', 33), ('03', 33), ('03', 34), ('03', 44), ('03', 16)]
  5.0s retry=True conf=0.911
  PRED: 제공된 자료에는 2027학년도 수시에서 새로 생긴 학과나 전형이라고 명시된 내용은 없습니다. 다만 첨단학과 소개에는 인공지능학과(2020학년도), 배터리공학전공(2023학년도), 바이오로직스학과·빅데이터경영전공·반도체공학과(2024학년도) 신설이 안내되어 있습니다. 본 내용은 변동될 수 있으므로 반드시 최종 확정되는 모집요강(본 대학 입학처 홈페이지)을 확인하시기 바랍니다. (출처: 03_2027학년도_수시_모집요강.pdf p.16, p.33)
  GOLD: 인공지능시스템학과가 신설되었고, 논술전형 모집단위에 한의예과와 약학과가 신설되었습니다.


In [178]:
q7 = testset_holdout[6]["q"]
kw7 = search_keywords(q7)
print("kw7:", kw7)
for name, query in (("orig", q7), ("kw", kw7)):
    for ret_name, ret in (("BM25", bm25_v15), ("Dense", dense_v15)):
        r = ret.invoke(query)
        ranks = [(i, d.metadata["page"]) for i, d in enumerate(r, 1) if d.metadata["source"].startswith("03") and d.metadata["page"] == 30]
        print(f"  {ret_name}({name}) p.30 ranks: {ranks}")

# BM25 전체 점수에서 p.30 순위
toks = korean_tokenizer(kw7)
bm = bm25_v15.vectorizer.get_scores(toks)
order = sorted(range(len(chunks_v15)), key=lambda i: bm[i], reverse=True)
for rank, i in enumerate(order[:20], 1):
    c = chunks_v15[i]
    if c.metadata["source"].startswith("03") and c.metadata["page"] in (28, 29, 30, 33):
        print(f"  BM25 rank {rank}: p.{c.metadata['page']} score={bm[i]:.1f} kind={c.metadata['kind']}")


kw7: 2027 학년도 학과 전형 수시모집 신설
  BM25(orig) p.30 ranks: []
  Dense(orig) p.30 ranks: []
  BM25(kw) p.30 ranks: [(4, 30), (6, 30)]
  Dense(kw) p.30 ranks: []
  BM25 rank 4: p.30 score=13.3 kind=text
  BM25 rank 6: p.30 score=12.4 kind=table
  BM25 rank 7: p.29 score=11.2 kind=text
  BM25 rank 16: p.28 score=10.4 kind=text


In [179]:
for q in ("신설", "2027 수시 신설", "수시 신설 학과"):
    print(f"\n[{q}]")
    for name, r in (("BM25", bm25_v15.invoke(q)), ("Dense", dense_v15.invoke(q))):
        pages = [(d.metadata["source"][:2], d.metadata["page"]) for d in r[:8]]
        print(f"  {name}: {pages}")

# Q6 few-shot 프롬프트
DRAFT_PROMPT_V4 = DRAFT_PROMPT_V3.replace(
    "- For numbered columns (1호, 2호, ...), count every cell from the left, including dashes; match each value to its column.",
    "- For numbered columns (1호, 2호, ...), count every cell from the left, including dashes; match each value to its column.\n"
    "  Example: header \"1호 2호 3호 4호 5호 6호\" + row \"- - - -10 -20 -100\" means 4호=-10, 5호=-20, 6호=-100."
)

t6 = testset_holdout[5]
docs6, _ = retrieve_v15(t6["q"], top_n=8)
ctx6 = "\n\n".join(f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.metadata['raw']}" for i, d in enumerate(docs6, 1))
for name, prompt in (("V3", DRAFT_PROMPT_V3), ("V4", DRAFT_PROMPT_V4)):
    ans = llm.invoke([("system", prompt), ("human", f"[Evidence]\n{ctx6}\n\n[Question]\n{t6['q']}")]).content
    print(f"\n[{name}] {ans[:220]}")



[신설]
  BM25: [('03', 16), ('03', 9), ('03', 8), ('03', 17), ('02', 20), ('03', 30), ('03', 30), ('03', 6)]
  Dense: [('04', 27), ('02', 16), ('02', 15), ('03', 81), ('02', 10), ('02', 16), ('03', 73), ('03', 81)]

[2027 수시 신설]
  BM25: [('03', 16), ('03', 9), ('03', 8), ('03', 17), ('03', 30), ('03', 30), ('03', 15), ('03', 6)]
  Dense: [('03', 80), ('03', 5), ('03', 74), ('03', 1), ('03', 76), ('03', 33), ('03', 65), ('03', 9)]

[수시 신설 학과]
  BM25: [('03', 16), ('03', 9), ('03', 8), ('03', 17), ('03', 30), ('03', 30), ('03', 6), ('03', 15)]
  Dense: [('03', 62), ('03', 50), ('03', 33), ('03', 48), ('03', 92), ('03', 64), ('03', 81), ('03', 64)]

[V3] 정시모집에서 학교폭력 6호 처분은 1,000점 만점 기준 100점 감점(–100)입니다. (출처: 04_2027학년도_정시_모집요강.pdf p.35)

[V4] 정시에서 학교폭력 6호 처분(출석정지)을 받으면 1,000점 만점 기준으로 100점 감점됩니다. (출처: 04_2027학년도_정시_모집요강.pdf p.35)


In [180]:
DRAFT_PROMPT_V3 = DRAFT_PROMPT_V4

DF = {}
for c in chunks_v15:
    for t in set(c.metadata["raw"].split()):
        DF[t] = DF.get(t, 0) + 1


def rare_terms(kw, n=1):
    toks = [t for t in kw.split() if len(t) >= 2 and not t.isdigit()]
    toks.sort(key=lambda t: sum(t in c.metadata["raw"] for c in chunks_v15))
    return toks[:n]


def retrieve_v15(query, top_n=6, expand_top=2, rrf_k=60):
    route, kw, kw_route = route_queries(query)
    lists = [bm25_v15.invoke(query), dense_v15.invoke(query),
             bm25_v15.invoke(kw), dense_v15.invoke(kw_route)]
    weights = [1.0, 1.0, 1.15, 1.15]

    # 희귀 키워드 단독 검색: RRF 희석 방지
    rare = rare_terms(kw, 1)
    if rare and sum(t in c.metadata["raw"] for c in chunks_v15) < 80:
        lists.append(bm25_v15.invoke(rare[0]))
        weights.append(1.0)

    scores, docs = rrf_fuse(lists, weights, rrf_k)
    prior = page_prior(kw_route)
    tdoc = target_doc(query)
    for key, d in docs.items():
        r = prior.get((d.metadata["source"], d.metadata["page"]))
        if r:
            scores[key] += 0.45 / (40 + r)
        if tdoc and d.metadata["source"].startswith(tdoc):
            scores[key] *= 1.12
        if route == "table" and d.metadata["kind"] == "table":
            scores[key] *= 1.15
        if route in ("procedure", "eligibility", "page") and d.metadata["kind"] == "text":
            scores[key] *= 1.12

    top_score = max(scores.values()) if scores else 0.0
    if route == "table" and tdoc:
        table_dept_boost(scores, docs, query, kw, tdoc, top_score)
    if route == "page" and tdoc and tdoc in TOC_PAGES:
        pool = [c for c in chunks_v15
                if c.metadata["source"].startswith(tdoc) and c.metadata["page"] in TOC_PAGES[tdoc]]
        inject(scores, docs, kw, pool, top_score, factor=1.12, limit=2)
    if route == "schedule":
        pool = [c for c in chunks_v15 if c.metadata["source"].startswith("05")]
        inject(scores, docs, kw, pool, top_score, factor=1.15, limit=3)

    top = [docs[k] for k in sorted(scores, key=scores.get, reverse=True)[:top_n]]
    out, seen = [], set()
    for n, d in enumerate(top):
        if d.page_content not in seen:
            out.append(d)
            seen.add(d.page_content)
        if n >= expand_top:
            continue
        i = CHUNK_INDEX_V15[d.page_content]
        for j in range(max(0, i - 1), min(len(chunks_v15), i + 2)):
            c = chunks_v15[j]
            if c.metadata["source"] == d.metadata["source"] and c.page_content not in seen:
                out.append(c)
                seen.add(c.page_content)
    return out, route


t7 = testset_holdout[6]
docs7, _ = retrieve_v15(t7["q"], top_n=8)
print("Q7 docs:", [(d.metadata["source"][:2], d.metadata["page"]) for d in docs7[:8]])
r7 = ask_v20(t7["q"], return_meta=True)
print(f"{r7['sec']:.1f}s retry={r7['retried']} conf={r7['conf']}")
print("PRED:", r7["answer"][:400])


Q7 docs: [('03', 28), ('03', 27), ('03', 29), ('03', 16), ('03', 15), ('03', 17), ('03', 33), ('03', 9)]
2.3s retry=False conf=1.0
PRED: 네. 2027학년도 수시에는 모집단위로 인공지능시스템학과가 신설됐고, 논술전형 모집단위에 한의예과와 약학과가 새로 추가됐습니다. (출처: 03_2027학년도_수시_모집요강.pdf p.30)


In [181]:
ho2 = eval_meta(testset_holdout, 0, 5, "holdout-v21", fn=ask_v20)

 1. O  3.9s conf=0.93 g=1.00 [procedure] 지역의사가 의무복무를 안 지키면 어떻게 되나요? 단계별로 알려주세요.
 2. X  2.6s conf=1.00 g=1.00 [eligibility] 정시 원서 접수하고 나서 학과를 바꿀 수 있나요? 접수는 언제 완료되나요?
 3. O  2.8s conf=1.00 g=1.00 [procedure] 지역의사 하다가 그만두면 돈을 내야 한다는데, 금액별로 납부 기한이 어떻게 
 4. O  2.4s conf=1.00 g=1.00 [general] 지역의사가 될 사람이 수련할 수 있는 전문과목은 모두 몇 개인가요? 수련병원
 5. X  2.6s conf=0.90 g=0.80 [general] 재외국민 전형 합격자 발표는 언제이고, 충원합격자 발표는 언제까지 하나요?
[holdout-v21 1-5] 3/5 | 평균 2.85s | 최대 3.91s | 재시도 0


In [182]:
DRAFT_PROMPT_V5 = DRAFT_PROMPT_V3.replace(
    "- Include EVERY key fact the question asks for (numbers, dates, targets, conditions, exceptions).",
    "- Include EVERY key fact the question asks for (numbers, dates, targets, conditions, exceptions), "
    "and also closely related details stated in the same evidence sentence (e.g., venue, refund policy, issuing authority)."
)

for idx in (1, 4):
    t = testset_holdout[idx]
    docs, _ = retrieve_v15(t["q"], top_n=8)
    ctx = "\n\n".join(f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.metadata['raw']}" for i, d in enumerate(docs, 1))
    ans = llm.invoke([("system", DRAFT_PROMPT_V5), ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{t['q']}")]).content
    print(f"\n[{idx+1}] {ans[:360]}")


TypeError: 'in <string>' requires string as left operand, not dict

In [184]:
# rare term 조건 변수 오타 수정 후 Q2/Q5 프롬프트 시험
import re as _re

_src = None
for idx in (1, 4):
    t = testset_holdout[idx]
    docs, _ = retrieve_v15(t["q"], top_n=8)
    ctx = "\n\n".join(f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.metadata['raw']}" for i, d in enumerate(docs, 1))
    ans = llm.invoke([("system", DRAFT_PROMPT_V5), ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{t['q']}")]).content
    print(f"\n[{idx+1}] {ans[:380]}")



[2] 정시 원서접수가 완료된 후에는 학과 및 전형 변경, 지원 포기가 불가능하므로 접수 전에 지원학과·전공을 정확히 확인해야 합니다. 원서접수는 전형료 결제 후 완료되며, 접수 완료 여부는 원서와 수험표를 통해 직접 확인하면 됩니다. (출처: 04_2027학년도_정시_모집요강.pdf p.9, p.8)

[5] 재외국민 전형 합격자 발표는 2026년 8월 13일(목)이며, 충원합격자 발표는 2026년 12월 23일(수)부터 29일(화)까지입니다. 충원합격자 통지는 12월 29일(화) 18:00까지이며, 이 기간에 연락이 두절되어 합격 통지가 불가능하면 충원 대상에서 제외됩니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.7)


In [183]:
def retrieve_v15(query, top_n=6, expand_top=2, rrf_k=60):
    route, kw, kw_route = route_queries(query)
    lists = [bm25_v15.invoke(query), dense_v15.invoke(query),
             bm25_v15.invoke(kw), dense_v15.invoke(kw_route)]
    weights = [1.0, 1.0, 1.15, 1.15]

    rare = rare_terms(kw, 1)
    if rare:
        tok = rare[0]
        if sum(tok in c.metadata["raw"] for c in chunks_v15) < 80:
            lists.append(bm25_v15.invoke(tok))
            weights.append(1.0)

    scores, docs = rrf_fuse(lists, weights, rrf_k)
    prior = page_prior(kw_route)
    tdoc = target_doc(query)
    for key, d in docs.items():
        r = prior.get((d.metadata["source"], d.metadata["page"]))
        if r:
            scores[key] += 0.45 / (40 + r)
        if tdoc and d.metadata["source"].startswith(tdoc):
            scores[key] *= 1.12
        if route == "table" and d.metadata["kind"] == "table":
            scores[key] *= 1.15
        if route in ("procedure", "eligibility", "page") and d.metadata["kind"] == "text":
            scores[key] *= 1.12

    top_score = max(scores.values()) if scores else 0.0
    if route == "table" and tdoc:
        table_dept_boost(scores, docs, query, kw, tdoc, top_score)
    if route == "page" and tdoc and tdoc in TOC_PAGES:
        pool = [c for c in chunks_v15
                if c.metadata["source"].startswith(tdoc) and c.metadata["page"] in TOC_PAGES[tdoc]]
        inject(scores, docs, kw, pool, top_score, factor=1.12, limit=2)
    if route == "schedule":
        pool = [c for c in chunks_v15 if c.metadata["source"].startswith("05")]
        inject(scores, docs, kw, pool, top_score, factor=1.15, limit=3)

    top = [docs[k] for k in sorted(scores, key=scores.get, reverse=True)[:top_n]]
    out, seen = [], set()
    for n, d in enumerate(top):
        if d.page_content not in seen:
            out.append(d)
            seen.add(d.page_content)
        if n >= expand_top:
            continue
        i = CHUNK_INDEX_V15[d.page_content]
        for j in range(max(0, i - 1), min(len(chunks_v15), i + 2)):
            c = chunks_v15[j]
            if c.metadata["source"] == d.metadata["source"] and c.page_content not in seen:
                out.append(c)
                seen.add(c.page_content)
    return out, route


print("fixed")


fixed


In [185]:
for idx in (1, 4):
    t = testset_holdout[idx]
    docs, _ = retrieve_v15(t["q"], top_n=8)
    ans = ho2[idx]["pred"]
    print(f"\n=== Q{idx+1} ===")
    ans_toks = _content_tokens(ans)
    for d in docs[:4]:
        if d.metadata["kind"] != "table":
            continue
        for line in d.metadata["raw"].splitlines():
            if line.count("|") < 2:
                continue
            cells = [c.strip() for c in line.strip("|").split("|") if c.strip()]
            if len(cells) < 3:
                continue
            c_toks = _content_tokens(" ".join(cells))
            ov = len(c_toks & ans_toks) / max(1, len(c_toks))
            if ov > 0.2:
                print(f"  [{d.metadata['source'][:2]} p.{d.metadata['page']}] ov={ov:.2f} | {line[:180]}")



=== Q2 ===

=== Q5 ===
  [01 p.7] ov=0.67 | | 합격자 발표 및 등록 | 합격자 발표 및 등록 | 합격자 발표 및 등록 | 합격자 발표 및 등록 | 합격자 발표 및 등록 | 합격자 발표 및 등록 |
  [01 p.7] ov=0.75 | | 합격자 발표 |  | 합격자 발표 | 충원합격자 발표 | 발표방법 |  |
  [01 p.7] ov=0.25 | |  |  | 2026.08.13.(목) | 2026.12.23(수) ~ 29(화) | 본 대학 입학처 홈페이지 |  |
  [01 p.8] ov=0.50 | | 전형방법 | 전형방법 | 전형방법 |
  [01 p.8] ov=0.33 | | 전형방법 | 1. 전체 모집단위 | 1. 전체 모집단위 |
  [01 p.16] ov=0.38 | | 홈페이지 문의 : | 홈페이지 문의 : | 본 대학 입학처 홈페이지 > 재외국민/외국인 > 재외국민/외국인 Q&A |


In [186]:
t2 = testset_holdout[1]
docs2, _ = retrieve_v15(t2["q"], top_n=8)
ans2 = ho2[1]["pred"]
ans_t = _content_tokens(ans2)
for d in docs2[:4]:
    if not d.metadata["source"].startswith("04") or d.metadata["page"] != 9:
        continue
    for s in iter_sentences(d.metadata["raw"]):
        st = _content_tokens(s)
        if len(st) < 4:
            continue
        ov = len(st & ans_t) / len(st)
        if ov > 0.15:
            print(f"  ov={ov:.2f} | {s[:220]}")


  ov=1.00 | [04_2027학년도_정시_모집요강.pdf p.9]
  ov=0.30 | 원서접수 중 유의사항을 반드시 확인하고 안내문의 진행요령에 따라 작성하여야 합니다.
  ov=1.00 | 원서접수는 전형료 결제 후 접수가 완료됩니다.
  ov=0.91 |  접수가 완료된 후에는 학과 및 전형 변경, 지원 포기 등은 불가능하므로 「모집요강」을 정확히 숙지하고
  ov=0.56 | 전형 유형, 지원학과/전공, 지원자격 등을 정확히 입력한 후 확인하고 접수해야 합니다.
  ov=0.20 | 되므로 변경되는 즉시 본 대학 입학처로 변경 신청을 해야 합니다.
  ov=0.50 | 수험표 분실 시에는 원서접수 사이트에 로그인하여 재출력이 가능합니다.


In [187]:
DATE_DOT = re.compile(r"\d{4}\.\d{2}\.\d{2}")


def missing_neighbor_details(answer, docs, max_items=2):
    """답변이 거의 그대로 반영한 문장의 다음 문장 / 같은 표 행의 다른 셀을 확인."""
    ans_toks = _content_tokens(answer)
    body = _strip_cite(answer)

    # 1) 문장: 답변과 0.8 이상 겹치는 문장의 다음 문장
    for d in docs[:4]:
        sents = list(iter_sentences(d.metadata.get("raw", "")))
        for i, s in enumerate(sents[:-1]):
            st = _content_tokens(s)
            if len(st) < 4:
                continue
            if len(st & ans_toks) / len(st) >= 0.8:
                missing = [t for t in _noun_tokens(sents[i + 1]) if t not in answer]
                if len(missing) >= 1:
                    return missing[:max_items]

    # 2) 표: 답변의 날짜/금액이 있는 행의 다른 셀
    facts = DATE_DOT.findall(body) + [n for n in NUM_RE.findall(body) if len(n.replace(",", "")) >= 5]
    if facts:
        for d in docs[:4]:
            if d.metadata["kind"] != "table":
                continue
            for line in d.metadata.get("raw", "").splitlines():
                if line.count("|") < 2 or not any(f in line for f in facts):
                    continue
                cells = [c.strip() for c in line.strip("|").split("|")]
                missing = [c for c in cells
                           if len(c) >= 3 and c not in answer and not is_value(c) and c not in GENERIC_DETAILS]
                if missing:
                    return missing[:max_items]
    return []


for idx in (1, 4):
    t = testset_holdout[idx]
    docs, _ = retrieve_v15(t["q"], top_n=8)
    print(f"[Q{idx+1}] missing:", missing_neighbor_details(ho2[idx]["pred"], docs))


[Q2] missing: ['안내문', '요령']
[Q5] missing: ['등록', '지정']


In [188]:
def norm_dates(text):
    out = set(re.findall(r"\d{4}\.\d{2}\.\d{2}", text))
    for y, m, d in re.findall(r"(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일", text):
        out.add(f"{y}.{int(m):02d}.{int(d):02d}")
    return out


def missing_neighbor_details(answer, docs, max_items=2):
    ans_toks = _content_tokens(answer)
    body = _strip_cite(answer)

    # 1) 답변이 거의 그대로 쓴 문장 → 다음 2문장의 세부정보 확인
    best = (0.0, [], 0)
    for d in docs[:4]:
        sents = [s for s in iter_sentences(d.metadata.get("raw", "")) if not s.startswith("[")]
        for i, s in enumerate(sents):
            st = _content_tokens(s)
            if len(st) < 4:
                continue
            ov = len(st & ans_toks) / len(st)
            if ov > best[0]:
                best = (ov, sents, i)
    if best[0] >= 0.8:
        for nxt in best[1][best[2] + 1: best[2] + 3]:
            missing = [t for t in _noun_tokens(nxt) if t not in answer]
            if missing:
                return missing[:max_items]

    # 2) 답변의 날짜가 있는 표 행 → 같은 행의 다른 셀 확인
    facts = norm_dates(body) | {n for n in NUM_RE.findall(body) if len(n.replace(",", "")) >= 5}
    if facts:
        for d in docs[:4]:
            if d.metadata["kind"] != "table":
                continue
            for line in d.metadata.get("raw", "").splitlines():
                if line.count("|") < 2 or not any(f in line for f in facts):
                    continue
                cells = [c.strip() for c in line.strip("|").split("|")]
                missing = [c for c in cells
                           if len(c) >= 3 and c not in answer and not is_value(c) and c not in GENERIC_DETAILS]
                if missing:
                    return missing[:max_items]
    return []


for idx in (1, 4):
    t = testset_holdout[idx]
    docs, _ = retrieve_v15(t["q"], top_n=8)
    print(f"[Q{idx+1}] missing:", missing_neighbor_details(ho2[idx]["pred"], docs))


[Q2] missing: ['숙지']
[Q5] missing: ['2026.08.13.(목)', '2026.12.23(수) ~ 29(화)']


In [189]:
DETAIL_STOP = {"숙지", "확인", "작성", "입력", "안내", "경우", "내용", "정확", "변경", "가능", "경우에는", "모집요강"}
DATE_CELL = re.compile(r"^[\d.,~()\s월화수목금토일]+$")


def missing_neighbor_details(answer, docs, max_items=2):
    ans_toks = _content_tokens(answer)
    body = _strip_cite(answer)

    best = (0.0, [], 0)
    for d in docs[:4]:
        sents = [s for s in iter_sentences(d.metadata.get("raw", "")) if not s.startswith("[")]
        for i, s in enumerate(sents):
            st = _content_tokens(s)
            if len(st) < 4:
                continue
            ov = len(st & ans_toks) / len(st)
            if ov > best[0]:
                best = (ov, sents, i)
    if best[0] >= 0.8:
        for nxt in best[1][best[2] + 1: best[2] + 3]:
            missing = [t for t in _noun_tokens(nxt)
                       if t not in answer and t not in DETAIL_STOP and not t.isdigit()]
            if missing:
                return missing[:max_items]

    facts = norm_dates(body) | {n for n in NUM_RE.findall(body) if len(n.replace(",", "")) >= 5}
    if facts:
        for d in docs[:4]:
            if d.metadata["kind"] != "table":
                continue
            for line in d.metadata.get("raw", "").splitlines():
                if line.count("|") < 2 or not any(f in line for f in facts):
                    continue
                cells = [c.strip() for c in line.strip("|").split("|")]
                missing = [c for c in cells
                           if len(c) >= 3 and c not in answer and not is_value(c)
                           and not DATE_CELL.match(c) and c not in GENERIC_DETAILS]
                if missing:
                    return missing[:max_items]
    return []


for idx in (1, 4):
    t = testset_holdout[idx]
    docs, _ = retrieve_v15(t["q"], top_n=8)
    print(f"[Q{idx+1}] missing:", missing_neighbor_details(ho2[idx]["pred"], docs))


[Q2] missing: ['전공', '유형']
[Q5] missing: ['본 대학 입학처 홈페이지']


In [190]:
NEG_RE = re.compile(r"(불가|반환하지|제외|없음|않|미포함|불허)")


def missing_neighbor_details(answer, docs, max_items=2):
    ans_toks = _content_tokens(answer)
    body = _strip_cite(answer)

    # 0) 답변과 핵심어를 공유하는 예외/부정 문장
    for d in docs[:4]:
        for s in iter_sentences(d.metadata.get("raw", "")):
            if not NEG_RE.search(s):
                continue
            st = _content_tokens(s)
            if len(st & ans_toks) < 1:
                continue
            missing = [t for t in _noun_tokens(s)
                       if t not in answer and t not in DETAIL_STOP and not t.isdigit()]
            if missing:
                return missing[:max_items]

    # 1) 답변이 거의 그대로 쓴 문장 → 다음 2문장
    best = (0.0, [], 0)
    for d in docs[:4]:
        sents = [s for s in iter_sentences(d.metadata.get("raw", "")) if not s.startswith("[")]
        for i, s in enumerate(sents):
            st = _content_tokens(s)
            if len(st) < 4:
                continue
            ov = len(st & ans_toks) / len(st)
            if ov > best[0]:
                best = (ov, sents, i)
    if best[0] >= 0.8:
        for nxt in best[1][best[2] + 1: best[2] + 3]:
            missing = [t for t in _noun_tokens(nxt)
                       if t not in answer and t not in DETAIL_STOP and not t.isdigit()]
            if missing:
                return missing[:max_items]

    # 2) 답변의 날짜가 있는 표 행
    facts = norm_dates(body) | {n for n in NUM_RE.findall(body) if len(n.replace(",", "")) >= 5}
    if facts:
        for d in docs[:4]:
            if d.metadata["kind"] != "table":
                continue
            for line in d.metadata.get("raw", "").splitlines():
                if line.count("|") < 2 or not any(f in line for f in facts):
                    continue
                cells = [c.strip() for c in line.strip("|").split("|")]
                missing = [c for c in cells
                           if len(c) >= 3 and c not in answer and not is_value(c)
                           and not DATE_CELL.match(c) and c not in GENERIC_DETAILS]
                if missing:
                    return missing[:max_items]
    return []


for idx in (1, 4):
    t = testset_holdout[idx]
    docs, _ = retrieve_v15(t["q"], top_n=8)
    print(f"[Q{idx+1}] missing:", missing_neighbor_details(ho2[idx]["pred"], docs))


[Q2] missing: ['사용', '자료']
[Q5] missing: ['연락', '두절']


In [191]:
def missing_neighbor_details(answer, docs, max_items=2):
    ans_toks = _content_tokens(answer)
    body = _strip_cite(answer)

    # 1) 답변의 날짜/금액이 있는 표 행 → 같은 행의 다른 셀
    facts = norm_dates(body) | {n for n in NUM_RE.findall(body) if len(n.replace(",", "")) >= 5}
    if facts:
        for d in docs[:4]:
            if d.metadata["kind"] != "table":
                continue
            for line in d.metadata.get("raw", "").splitlines():
                if line.count("|") < 2 or not any(f in line for f in facts):
                    continue
                cells = [c.strip() for c in line.strip("|").split("|")]
                missing = [c for c in cells
                           if len(c) >= 3 and c not in answer and not is_value(c)
                           and not DATE_CELL.match(c) and c not in GENERIC_DETAILS]
                if missing:
                    return missing[:max_items]

    # 2) 답변과 가장 관련 깊은 예외/부정 문장
    best_neg = (0, [])
    for d in docs[:4]:
        for s in iter_sentences(d.metadata.get("raw", "")):
            if not NEG_RE.search(s):
                continue
            shared = _content_tokens(s) & ans_toks
            score = sum(len(t) for t in shared)
            if score > best_neg[0]:
                best_neg = (score, [t for t in _noun_tokens(s)
                                    if t not in answer and t not in DETAIL_STOP and not t.isdigit()])
    if best_neg[0] >= 2 and best_neg[1]:
        return best_neg[1][:max_items]

    # 3) 답변이 거의 그대로 쓴 문장 → 다음 2문장
    best = (0.0, [], 0)
    for d in docs[:4]:
        sents = [s for s in iter_sentences(d.metadata.get("raw", "")) if not s.startswith("[")]
        for i, s in enumerate(sents):
            st = _content_tokens(s)
            if len(st) < 4:
                continue
            ov = len(st & ans_toks) / len(st)
            if ov > best[0]:
                best = (ov, sents, i)
    if best[0] >= 0.8:
        for nxt in best[1][best[2] + 1: best[2] + 3]:
            missing = [t for t in _noun_tokens(nxt)
                       if t not in answer and t not in DETAIL_STOP and not t.isdigit()]
            if missing:
                return missing[:max_items]
    return []


for idx in (1, 4):
    t = testset_holdout[idx]
    docs, _ = retrieve_v15(t["q"], top_n=8)
    print(f"[Q{idx+1}] missing:", missing_neighbor_details(ho2[idx]["pred"], docs))


[Q2] missing: ['전공', '유형']
[Q5] missing: ['본 대학 입학처 홈페이지']


In [192]:
def missing_neighbor_details(answer, docs, max_items=2):
    ans_toks = _content_tokens(answer)
    body = _strip_cite(answer)

    facts = norm_dates(body) | {n for n in NUM_RE.findall(body) if len(n.replace(",", "")) >= 5}
    if facts:
        for d in docs[:4]:
            if d.metadata["kind"] != "table":
                continue
            for line in d.metadata.get("raw", "").splitlines():
                if line.count("|") < 2 or not any(f in line for f in facts):
                    continue
                cells = [c.strip() for c in line.strip("|").split("|")]
                missing = [c for c in cells
                           if len(c) >= 3 and c not in answer and not is_value(c)
                           and not DATE_CELL.match(c) and c not in GENERIC_DETAILS]
                if missing:
                    return missing[:max_items]

    best_neg = (0, [])
    for d in docs[:4]:
        for s in iter_sentences(d.metadata.get("raw", "")):
            if not NEG_RE.search(s):
                continue
            shared = _content_tokens(s) & ans_toks
            score = sum(len(t) for t in shared)
            missing = [t for t in _noun_tokens(s)
                       if t not in answer and t not in DETAIL_STOP and not t.isdigit()]
            if missing and score > best_neg[0]:
                best_neg = (score, missing)
    if best_neg[0] >= 2:
        return best_neg[1][:max_items]

    best = (0.0, [], 0)
    for d in docs[:4]:
        sents = [s for s in iter_sentences(d.metadata.get("raw", "")) if not s.startswith("[")]
        for i, s in enumerate(sents):
            st = _content_tokens(s)
            if len(st) < 4:
                continue
            ov = len(st & ans_toks) / len(st)
            if ov > best[0]:
                best = (ov, sents, i)
    if best[0] >= 0.8:
        for nxt in best[1][best[2] + 1: best[2] + 3]:
            missing = [t for t in _noun_tokens(nxt)
                       if t not in answer and t not in DETAIL_STOP and not t.isdigit()]
            if missing:
                return missing[:max_items]
    return []


for idx in (1, 4):
    t = testset_holdout[idx]
    docs, _ = retrieve_v15(t["q"], top_n=8)
    print(f"[Q{idx+1}] missing:", missing_neighbor_details(ho2[idx]["pred"], docs))


[Q2] missing: ['사용', '자료']
[Q5] missing: ['본 대학 입학처 홈페이지']


In [193]:
def focus_docs(answer, docs):
    prefixes = {f.strip()[:2] for f, _ in CITE_RE.findall(answer)}
    focus = [d for d in docs if d.metadata["source"][:2] in prefixes]
    return (focus or docs)[:4]


def missing_neighbor_details(answer, docs, max_items=2):
    ans_toks = _content_tokens(answer)
    body = _strip_cite(answer)
    focus = focus_docs(answer, docs)

    facts = norm_dates(body) | {n for n in NUM_RE.findall(body) if len(n.replace(",", "")) >= 5}
    if facts:
        for d in focus:
            if d.metadata["kind"] != "table":
                continue
            for line in d.metadata.get("raw", "").splitlines():
                if line.count("|") < 2 or not any(f in line for f in facts):
                    continue
                cells = [c.strip() for c in line.strip("|").split("|")]
                missing = [c for c in cells
                           if len(c) >= 3 and c not in answer and not is_value(c)
                           and not DATE_CELL.match(c) and c not in GENERIC_DETAILS]
                if missing:
                    return missing[:max_items]

    best_neg = (0, [])
    for d in focus:
        for s in iter_sentences(d.metadata.get("raw", "")):
            if not NEG_RE.search(s):
                continue
            shared = _content_tokens(s) & ans_toks
            score = sum(len(t) for t in shared)
            missing = [t for t in _noun_tokens(s)
                       if t not in answer and t not in DETAIL_STOP and not t.isdigit()]
            if missing and score > best_neg[0]:
                best_neg = (score, missing)
    if best_neg[0] >= 2:
        return best_neg[1][:max_items]

    best = (0.0, [], 0)
    for d in focus:
        sents = [s for s in iter_sentences(d.metadata.get("raw", "")) if not s.startswith("[")]
        for i, s in enumerate(sents):
            st = _content_tokens(s)
            if len(st) < 4:
                continue
            ov = len(st & ans_toks) / len(st)
            if ov > best[0]:
                best = (ov, sents, i)
    if best[0] >= 0.8:
        for nxt in best[1][best[2] + 1: best[2] + 3]:
            missing = [t for t in _noun_tokens(nxt)
                       if t not in answer and t not in DETAIL_STOP and not t.isdigit()]
            if missing:
                return missing[:max_items]
    return []


for idx in (1, 4):
    t = testset_holdout[idx]
    docs, _ = retrieve_v15(t["q"], top_n=8)
    print(f"[Q{idx+1}] missing:", missing_neighbor_details(ho2[idx]["pred"], docs))


[Q2] missing: ['사용', '자료']
[Q5] missing: ['본 대학 입학처 홈페이지']


In [194]:
t2 = testset_holdout[1]
docs2, _ = retrieve_v15(t2["q"], top_n=8)
focus2 = focus_docs(ho2[1]["pred"], docs2)
print("focus:", [(d.metadata["source"][:2], d.metadata["page"], d.metadata["kind"]) for d in focus2])
for d in focus2:
    if d.metadata["source"].startswith("04") and d.metadata["page"] == 9:
        print(f"\n--- p.9 sentences (kind={d.metadata['kind']}) ---")
        for i, s in enumerate(iter_sentences(d.metadata["raw"])):
            flag = "NEG" if NEG_RE.search(s) else "   "
            print(f"  [{i}] {flag} {s[:150]}")


focus: [('04', 9, 'text'), ('04', 8, 'text'), ('04', 10, 'text'), ('04', 8, 'text')]

--- p.9 sentences (kind=text) ---
  [0]     [04_2027학년도_정시_모집요강.pdf p.9]
  [1]     원서접수 중 유의사항을 반드시 확인하고 안내문의 진행요령에 따라 작성하여야 합니다.
  [2]     원서접수는 전형료 결제 후 접수가 완료됩니다.
  [3] NEG  접수가 완료된 후에는 학과 및 전형 변경, 지원 포기 등은 불가능하므로 「모집요강」을 정확히 숙지하고
  [4]     전형 유형, 지원학과/전공, 지원자격 등을 정확히 입력한 후 확인하고 접수해야 합니다.
  [5]     성명, 주민등록번호는 반드시 주민등록상에 기재된 내용과 같아야 하며, 주소나 전화번호가 전형기간 중 사용
  [6]     되므로 변경되는 즉시 본 대학 입학처로 변경 신청을 해야 합니다.
  [7]     (개명 등으로 인해 학교생활기록부와 이름이 서로 다를 경우 주민등록초본 또는 기본증명서 제출)
  [8]     수험표 분실 시에는 원서접수 사이트에 로그인하여 재출력이 가능합니다.
  [9]     입학원서 입력사항이 사실과 달라 발생하는 모든 불이익은 지원자의 책임입니다.
  [10]     서류제출 해당자는 지정된 날짜까지 온라인으로 제출해야 하며, 각각의 제출서류를 저용량 PDF로 합본하여


In [195]:
for i, c in enumerate(chunks_v15):
    if "반환하지 않음" in c.metadata["raw"]:
        print(f"idx={i} {c.metadata['source'][:2]} p.{c.metadata['page']} {c.metadata['kind']} len={len(c.metadata['raw'])}")
        idx = c.metadata["raw"].find("반환하지")
        print("  ...", c.metadata["raw"][max(0, idx-120):idx+80].replace("\n", " "))


idx=6 01 p.5 table len=1356
  ... 빠짐없이 정확하게 입력 및 확인 |  |  |  | |  |  | ▼ | ▼ | ▼ |  | |  |  | 5. 전형료 결제 전형료 결제 후에는 입학원서 기재사항을 수정하거나 원서접수를 취소할 수 없으며, 전형료는 반환하지 않음 |  |  |  | |  |  | ▼ | ▼ | ▼ |  | |  |  | 6. 수험표 확인 접수가 완료된 것을 원서와 수험표를 
idx=8 01 p.6 table len=713
  ...  후에는 학과 및 전형 변경, 지원 포기 등은 불가능합니다. 따라서 「모집요강」을 정확히 숙지하여야 하며 전형 유형, 지원학과/전공, 지원자격 등을 정확히 확인한 후 접수하여야 합니다. (제출서류 및 전형료는 일절 반환하지 않음) 성명, 주민등록번호는 반드시 주민등록상에 기재된 내용과 같아야 하며, 주소나 전화번호가 전형기간 중 사용되므로 변경되는 즉시 본
idx=166 03 p.35 text len=825
  ... 학과 / 전공 등을 선택하여 입력 ② 모든 사항을 빠짐없이 정확하게 입력 및 확인(학생부 온라인 제공 동의)  전형료 결제 전형료 결제 후에는 입학원서 기재사항을 수정하거나 원서접수를 취소할 수 없으며, 전형료는 반환하지 않음  수험표 확인 접수가 완료된 것을 원서와 수험표를 통해 직접 확인  서류제출(해당자만) ① 온라인 원서접수 사이트를 통해 제출
idx=168 03 p.36 text len=599
  ... 접수가 완료된 후에는 학과 및 전형 변경, 지원 포기 등은 불가능하므로 「모집요강」을 정확히 숙지하고 전형 유형, 지원학과/전공, 지원자격 등을 정확히 입력한 후 확인하고 접수해야 합니다. (제출서류 및 전형료는 반환하지 않음) 4. 성명, 주민등록번호는 반드시 주민등록상에 기재된 내용과 같아야 하며, 주소나 전화번호가 전형기간 중 사용 되므로 변경되는
idx=390 04 p.8 text len=804
  ... 학과 / 전공 등을 선택하여 입력 ② 모든 사

In [196]:
def retrieve_v15(query, top_n=6, expand_top=2, rrf_k=60):
    route, kw, kw_route = route_queries(query)
    lists = [bm25_v15.invoke(query), dense_v15.invoke(query),
             bm25_v15.invoke(kw), dense_v15.invoke(kw_route)]
    weights = [1.0, 1.0, 1.15, 1.15]

    rare = rare_terms(kw, 1)
    if rare:
        tok = rare[0]
        if sum(tok in c.metadata["raw"] for c in chunks_v15) < 80:
            lists.append(bm25_v15.invoke(tok))
            weights.append(1.0)

    scores, docs = rrf_fuse(lists, weights, rrf_k)
    prior = page_prior(kw_route)
    tdoc = target_doc(query)
    for key, d in docs.items():
        r = prior.get((d.metadata["source"], d.metadata["page"]))
        if r:
            scores[key] += 0.45 / (40 + r)
        if tdoc and d.metadata["source"].startswith(tdoc):
            scores[key] *= 1.12
        if route == "table" and d.metadata["kind"] == "table":
            scores[key] *= 1.15
        if route in ("procedure", "eligibility", "page") and d.metadata["kind"] == "text":
            scores[key] *= 1.12

    top_score = max(scores.values()) if scores else 0.0
    if route == "table" and tdoc:
        table_dept_boost(scores, docs, query, kw, tdoc, top_score)
    if route == "page" and tdoc and tdoc in TOC_PAGES:
        pool = [c for c in chunks_v15
                if c.metadata["source"].startswith(tdoc) and c.metadata["page"] in TOC_PAGES[tdoc]]
        inject(scores, docs, kw, pool, top_score, factor=1.12, limit=2)
    if route == "schedule":
        pool = [c for c in chunks_v15 if c.metadata["source"].startswith("05")]
        inject(scores, docs, kw, pool, top_score, factor=1.15, limit=3)

    top = [docs[k] for k in sorted(scores, key=scores.get, reverse=True)[:top_n]]
    out, seen = [], set()
    for n, d in enumerate(top):
        if d.page_content not in seen:
            out.append(d)
            seen.add(d.page_content)
        if n >= expand_top:
            continue
        i = CHUNK_INDEX_V15[d.page_content]
        for j in range(max(0, i - 1), min(len(chunks_v15), i + 2)):
            c = chunks_v15[j]
            if c.metadata["source"] == d.metadata["source"] and c.page_content not in seen:
                out.append(c)
                seen.add(c.page_content)

    # 페이지 단위 형제 청크 확장 (같은 페이지의 다른 청크 중 질문과 가장 관련 있는 1개)
    for d in top[:2]:
        sibs = [c for c in chunks_v15
                if c.metadata["source"] == d.metadata["source"]
                and c.metadata["page"] == d.metadata["page"]
                and c.page_content not in seen]
        if sibs:
            best = max(sibs, key=lambda c: overlap_score(c, kw))
            if overlap_score(best, kw) > 0:
                out.append(best)
                seen.add(best.page_content)
    return out, route


t2 = testset_holdout[1]
docs2, _ = retrieve_v15(t2["q"], top_n=8)
print("Q2 docs:", [(d.metadata["source"][:2], d.metadata["page"], len(d.metadata["raw"])) for d in docs2])
r2 = ask_v20(t2["q"], return_meta=True)
print(f"\n{r2['sec']:.1f}s retry={r2['retried']} conf={r2['conf']}")
print(r2["answer"][:420])


Q2 docs: [('04', 9, 594), ('04', 8, 394), ('04', 10, 825), ('03', 36, 599), ('03', 35, 390), ('03', 37, 767), ('04', 8, 804), ('03', 35, 825), ('01', 5, 1356), ('01', 6, 713), ('04', 8, 159), ('03', 35, 160)]

4.3s retry=True conf=0.875
정시 원서접수가 완료된 후에는 학과 및 전형 변경, 지원 포기가 불가능합니다. 접수는 전형료 결제 후 완료되므로, 접수 전에 전형 유형과 지원학과·전공, 지원자격 등을 정확히 확인해야 합니다. (출처: 04_2027학년도_정시_모집요강.pdf p.9)


In [197]:
def ask_v21(question, top_n=8, t_budget=10.0, return_meta=False):
    t0 = time.perf_counter()
    route = route_query(question)
    tdoc = target_doc(question)

    if route == "page" and tdoc:
        hit = toc_lookup(question, tdoc)
        if hit:
            topic = hit["title"]
            answer = (f"{topic}{_particle(topic)} {doc_title(hit['source'])} "
                      f"{hit['page']}페이지에 안내되어 있습니다. "
                      f"(출처: {hit['source']} p.{hit['doc_page']})")
            toc_doc = next(c for c in chunks_v15
                           if c.metadata["source"] == hit["source"] and c.metadata["page"] == hit["doc_page"])
            g = ground_answer(answer, [toc_doc], question)
            sec = time.perf_counter() - t0
            if return_meta:
                return {"answer": answer, "route": "page_lookup", "sec": sec, "conf": 0.99,
                        "cov": 1.0, "ground": g, "retried": False,
                        "docs": [(hit["source"][:2], hit["doc_page"])]}
            return answer, 1, sec

    docs, route = retrieve_v15(question, top_n=top_n)
    kw = search_keywords(question)
    ctx = "\n\n".join(f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.metadata['raw']}"
                      for i, d in enumerate(docs, 1))
    answer = llm.invoke([("system", DRAFT_PROMPT_V5),
                         ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}")]).content
    g = ground_answer(answer, docs, question)
    conf, cov = make_confidence(route, kw, docs, g)
    retried = False

    missing_items = missing_list_items(question, answer, docs)
    missing_conds = missing_conditions(answer, docs, max_sent=1)
    missing_neigh = missing_neighbor_details(answer, docs)
    weak = (g["score"] < 0.78 or g["not_found"] or bool(missing_items) or bool(missing_conds)
            or bool(missing_neigh) or re.search(
                r"(규정이 없|안내되어 있지 않|나와 있지 않|확인할 수 없|제공된 자료에는.{0,20}없)", answer))
    if weak and time.perf_counter() - t0 < t_budget:
        retried = True
        hints = []
        if missing_items:
            hints.append("[Exact items to include]\n" + ", ".join(missing_items))
        if missing_conds:
            hints.append("[Conditions that must be reflected]\n" + "\n".join(missing_conds))
        if missing_neigh:
            hints.append("[Related details from the same evidence row/sentence]\n" + ", ".join(missing_neigh))
        answer = llm.invoke([("system", REPAIR_PROMPT_V2),
                             ("human", f"[Evidence]\n{ctx}\n\n[Question]\n{question}\n\n[Draft]\n{answer}\n\n"
                                       + "\n\n".join(hints))]).content
        g = ground_answer(answer, docs, question)
        conf, cov = make_confidence(route, kw, docs, g)

    sec = time.perf_counter() - t0
    if return_meta:
        return {"answer": answer, "route": route, "sec": sec, "conf": conf, "cov": cov,
                "ground": g, "retried": retried,
                "docs": [(d.metadata["source"][:2], d.metadata["page"]) for d in docs]}
    return answer, 1 + int(retried), sec


for idx in (1, 4):
    t = testset_holdout[idx]
    r = ask_v21(t["q"], return_meta=True)
    print(f"\n[Q{idx+1}] {r['sec']:.1f}s retry={r['retried']} conf={r['conf']}")
    print(r["answer"][:420])



[Q2] 4.0s retry=True conf=0.875
아니요. 정시는 접수가 완료된 후 학과 및 전형 변경이나 지원 포기가 불가능하므로, 지원학과·전공과 지원자격 등을 정확히 확인한 뒤 접수해야 합니다. 원서접수는 전형료 결제 후 완료되며, 접수 완료 여부는 원서와 수험표를 통해 직접 확인해야 합니다. (출처: 04_2027학년도_정시_모집요강.pdf p.9)

[Q5] 3.8s retry=True conf=1.0
재외국민 전형 합격자는 2026.08.13.(목)에 발표하며, 충원합격자는 2026.12.23(수)부터 29(화)까지 발표합니다. 발표는 본 대학 입학처 홈페이지에서 확인할 수 있고, 충원합격자 통지기간은 2026.12.23(수)부터 29(화) 18:00까지입니다. (출처: 01_2027학년도_재외국민_모집요강.hwp p.7)


In [198]:
t2 = testset_holdout[1]
docs2, _ = retrieve_v15(t2["q"], top_n=8)
a2 = llm.invoke([("system", DRAFT_PROMPT_V5),
                 ("human", f"[Evidence]\n" + "\n\n".join(
                     f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.metadata['raw']}"
                     for i, d in enumerate(docs2, 1)) + f"\n\n[Question]\n{t2['q']}")]).content
print("DRAFT:", a2[:260])
print("\nfocus:", [(d.metadata["source"][:2], d.metadata["page"], len(d.metadata["raw"])) for d in focus_docs(a2, docs2)])
print("hint:", missing_neighbor_details(a2, docs2))

# 환불 문장이 실제로 어느 청크에 있는지
print("\n[환불 문장 후보]")
for d in docs2:
    for s in iter_sentences(d.metadata["raw"]):
        if "반환" in s:
            print(f"  {d.metadata['source'][:2]} p.{d.metadata['page']} | {s[:120]}")


DRAFT: 정시 원서접수가 완료된 뒤에는 학과·전형 변경이나 지원 포기가 불가능합니다. 원서접수는 전형료 결제 후 완료되며, 접수가 완료됐는지는 원서와 수험표를 통해 직접 확인해야 합니다. (출처: 04_2027학년도_정시_모집요강.pdf p.9, p.8)

focus: [('04', 9, 594), ('04', 8, 394), ('04', 10, 825), ('04', 8, 804)]
hint: ['사용', '자료']

[환불 문장 후보]
  01 p.5 | 전형료 결제 전형료 결제 후에는 입학원서 기재사항을 수정하거나 원서접수를 취소할 수 없으며, 전형료는 반환하지 않음
  01 p.6 | (제출서류 및 전형료는 일절 반환하지 않음) 성명, 주민등록번호는 반드시 주민등록상에 기재된 내용과 같아야 하며, 주소나 전화번호가 전형기간 중 사용되므로 변경되는 즉시 본 대학 입학처로 변경 신청을 해야 합니다.


In [199]:
for d in docs2:
    if d.metadata["source"].startswith("04") and d.metadata["page"] == 8:
        print(f"\n[len={len(d.metadata['raw'])}] has반환={'반환' in d.metadata['raw']}")
        print("  ", d.metadata["raw"][:230].replace("\n", " "))

DRAFT_PROMPT_V6 = DRAFT_PROMPT_V5.replace(
    "- Answer in Korean",
    "- If the evidence states a restriction, exception, or non-refundable/not-allowed condition about the action asked, "
    "state it too, even if the question does not ask for it.\n- Answer in Korean"
)
print("\n[V6 적용 여부]", "restriction" in DRAFT_PROMPT_V6)

ctx2 = "\n\n".join(f"[근거 {i}] {d.metadata['source']} p.{d.metadata['page']}\n{d.metadata['raw']}"
                   for i, d in enumerate(docs2, 1))
a2b = llm.invoke([("system", DRAFT_PROMPT_V6),
                  ("human", f"[Evidence]\n{ctx2}\n\n[Question]\n{t2['q']}")]).content
print("\nV6 DRAFT:", a2b[:320])



[len=394] has반환=False
   [04_2027학년도_정시_모집요강.pdf p.8] 으로 간주합니다. 3. 장애인복지법 제32조에 의하여 장애인등록을 필하고, 각종 장애 또는 지체로 인하여 입학전형 진행과정에서 지원이 필요한 경우 사전 요청바랍니다. 4. 장애학생의 지원 및 선발에 대한 차별은 없으며, 입학 시 본교의 장애학생지원에 관한 규정을 적용합니다. ※ 본 대학교는 원서접수 대행기관을 통해 원서접수를 위탁 처리하고, 수집한 개인정

[len=804] has반환=True
   [04_2027학년도_정시_모집요강.pdf p.8] 34 원서접수 기간 원서접수 방법 모집시기 접수기간 인터넷 접수 정시 2027.01.04.(월) ~ 07.(목) 본 대학 입학처 홈페이지 인터넷 원서접수 시 사진 업로드를 위하여 지원자 본인의 증명사진 파일(jpg, gif 파일형식)을 준비하시기 바랍니다. [최근 3개월 내 사진으로 인물 뒤 배경이 있는 사진 또는 스냅사진은 사용이 불가함] 원서접수 사이트 

[len=159] has반환=False
   [04_2027학년도_정시_모집요강.pdf p.8] | 원서접수 사이트 접속 | |---| |  | | 회원가입 및 로그인 | |  | | 유의사항 확인 | |  | | 원서작성 | |  | | 전형료 결제 | |  | | 수험표 확인 | |  | | 서류제출(해당자만) |

[V6 적용 여부] False

V6 DRAFT: 정시 원서 접수가 완료된 후에는 학과 및 전형 변경이 불가능합니다. 접수는 전형료 결제 후 완료되므로, 결제 후 원서 기재사항을 수정하거나 접수를 취소할 수 없습니다. (출처: 04_2027학년도_정시_모집요강.pdf p.9)


In [200]:
ho2 = eval_meta(testset_holdout, 0, 5, "holdout-v21", fn=ask_v21)


 1. O  6.4s conf=0.93 g=1.00 [procedure] 지역의사가 의무복무를 안 지키면 어떻게 되나요? 단계별로 알려주세요.
 2. X  4.1s conf=1.00 g=1.00 [eligibility] 정시 원서 접수하고 나서 학과를 바꿀 수 있나요? 접수는 언제 완료되나요?
 3. O  6.3s conf=1.00 g=1.00 [procedure] 지역의사 하다가 그만두면 돈을 내야 한다는데, 금액별로 납부 기한이 어떻게 
 4. O  5.8s conf=1.00 g=1.00 [general] 지역의사가 될 사람이 수련할 수 있는 전문과목은 모두 몇 개인가요? 수련병원
 5. O  4.4s conf=1.00 g=1.00 [general] 재외국민 전형 합격자 발표는 언제이고, 충원합격자 발표는 언제까지 하나요?
[holdout-v21 1-5] 4/5 | 평균 5.40s | 최대 6.35s | 재시도 5


In [201]:
ho2 += eval_meta(testset_holdout, 5, 10, "holdout-v21", fn=ask_v21)


 6. O  6.3s conf=1.00 g=1.00 [table] 정시에서 학교폭력 6호 처분을 받으면 몇 점 깎이나요?
 7. O  2.3s conf=1.00 g=1.00 [general] 2027학년도 수시에서 새로 생긴 학과나 전형이 있나요?
 8. O  4.4s conf=1.00 g=1.00 [general] 가천대 길병원은 연구중심병원으로 언제 선정됐고, 이후 평가는 어땠나요?
 9. O  1.9s conf=0.96 g=1.00 [general] 가천대 의과대학 의학교육 인증평가는 언제부터 언제까지 유효한가요?
10. X  5.1s conf=1.00 g=1.00 [procedure] 지역의사가 의무복무지를 벗어나서 수련하면 그 기간은 어떻게 되나요?
[holdout-v21 6-10] 4/5 | 평균 3.99s | 최대 6.26s | 재시도 3


In [202]:
import json

r10 = ho2[9]
print("[Q10]", r10["q"][:70])
print("  PRED:", r10["pred"][:400])
print("  WHY :", r10["why"][:250])
print("  GOLD:", testset_holdout[9]["gold"][:200])

def summarize(rows):
    ok = sum(r["ok"] for r in rows)
    secs = [r["sec"] for r in rows]
    return {"n": len(rows), "acc": f"{ok}/{len(rows)}", "mean_sec": round(sum(secs)/len(secs), 2),
            "max_sec": round(max(secs), 2), "retries": sum(r.get("retried", False) for r in rows)}

print("\nheld-out v21:", summarize(ho2))
print("held-out v20:", summarize(ho20))
print("colloquial v20:", summarize(h20))
print("original v20:", summarize(o20))


[Q10] 지역의사가 의무복무지를 벗어나서 수련하면 그 기간은 어떻게 되나요?
  PRED: 의무복무지역이 아닌 곳에서 수련하면 수련기간 전체가 의무복무 기간에 산입되지 않으며, 전문의 자격 취득 후에도 본인의 의무복무지역에서 10년을 의무복무해야 합니다. 다만, 의무복무지역 내 9개 과목 수련이 불가능한 경우 수련기간 동안에 한해 별도 지정을 신청할 수 있고, 지정되면 해당 지역이 의무복무지역으로 간주되어 그 지역 내 수련기간은 의무복무 기간에 산입될 수 있습니다. 수련기간 종료 또는 수련과정 중단 시 별도 지정의 효력은 소멸하며, 종전 의무복무지역으로 복귀해야 합니다. (출처: 02_2027학년도_지역의사선발전형_안내.pdf p.15)
  WHY : 정답은 의무복무지역 외 수련 기간은 의무복무 기간에 산입되지 않는다고 했지만, 예측은 별도 지정 시 해당 지역의 수련 기간이 산입될 수 있다고 덧붙여 정답의 핵심 내용을 제한하는 예외를 제시합니다. 또한 ‘부득이한 경우’라는 조건도 빠져 있습니다.
  GOLD: 부득이한 경우 의무복무지역 외 수련이 가능하지만, 그 수련기간은 의무복무 기간에 불산입됩니다.

held-out v21: {'n': 10, 'acc': '8/10', 'mean_sec': 4.69, 'max_sec': 6.35, 'retries': 8}
held-out v20: {'n': 10, 'acc': '5/10', 'mean_sec': 3.01, 'max_sec': 6.33, 'retries': 3}
colloquial v20: {'n': 20, 'acc': '20/20', 'mean_sec': 3.12, 'max_sec': 6.41, 'retries': 8}
original v20: {'n': 20, 'acc': '18/20', 'mean_sec': 2.66, 'max_sec': 5.86, 'retries': 5}


In [203]:
import inspect

NAMES_A = ["doc_title", "korean_tokenizer", "search_keywords", "route_queries", "target_doc",
           "page_prior", "rrf_fuse", "rare_terms", "inject", "table_dept_boost", "overlap_score"]
for n in NAMES_A:
    obj = globals().get(n)
    if obj is None:
        print(f"### MISSING {n}")
        continue
    try:
        print(f"\n### {n}\n" + inspect.getsource(obj))
    except Exception as e:
        print(f"### {n} (no source: {e})")



### doc_title
def doc_title(source):
    return DOC_TITLES.get(source[:2], source)


### korean_tokenizer
def korean_tokenizer(text):
    return [t.form for t in kiwi.tokenize(text) if t.tag in ALLOWED]


### search_keywords
def search_keywords(q, k=10):
    """질문 → 검색용 키워드(일반명사 제거 + 동의어 치환)."""
    out = []
    for t in kiwi.tokenize(q):
        if t.tag not in {"NNG", "NNP", "SL", "SN"} or len(t.form) < 2:
            continue
        w = t.form
        if w in GENERIC or any(key in w for key in SYNONYMS):
            continue
        if w not in out:
            out.append(w)
    for key, val in SYNONYMS.items():
        if key in q:
            for v in val.split():
                if v not in out:
                    out.append(v)
    return " ".join(out[:k])


### route_queries
def route_queries(q):
    kw = search_keywords(q)
    return route_query(q), kw, (kw + ROUTE_SUFFIX[route_query(q)]).strip()


### target_doc
def target_doc(q):
    for prefix, pat in DOC_KEYWORDS:
      

In [204]:
NAMES_B = ["toc_lookup", "retrieve_v15", "build_contextual_chunks", "parse_table2", "row_summary2",
           "dedup_cells", "is_value", "dept_terms"]
for n in NAMES_B:
    obj = globals().get(n)
    if obj is None:
        print(f"### MISSING {n}")
        continue
    try:
        print(f"\n### {n}\n" + inspect.getsource(obj))
    except Exception as e:
        print(f"### {n} (no source: {e})")



### toc_lookup
def toc_lookup(question, tdoc):
    if not tdoc or tdoc not in TOC_PAGES:
        return None
    kw = search_keywords(question)
    skip = {"페이지", "목차", "요강", "모집", "전형", "안내", "정시모집", "수시모집", "대학"}
    toks = [t for t in kw.split() if len(t) >= 2 and t not in skip]
    best = (0, None, None, None)
    for c in chunks_v15:
        if not (c.metadata["source"].startswith(tdoc) and c.metadata["page"] in TOC_PAGES[tdoc]):
            continue
        for line in c.metadata["raw"].splitlines():
            m = re.match(r"^\s*(\d{1,3})\s+(.+)$", line.strip())
            if not m:
                continue
            num, title = m.group(1), m.group(2).strip()
            score = sum(len(t) for t in toks if t in title)
            if score > best[0]:
                best = (score, num, title, c)
    if best[0] < 2:
        return None
    return {"page": best[1], "title": best[2], "doc_page": best[3].metadata["page"],
            "source": best[3].metadata["source"]}


### 

In [205]:
NAMES_C = ["_strip_cite", "_particle", "ground_answer", "make_confidence", "norm_dates",
           "iter_sentences", "_content_tokens", "_noun_tokens", "focus_docs"]
for n in NAMES_C:
    obj = globals().get(n)
    if obj is None:
        print(f"### MISSING {n}")
        continue
    try:
        print(f"\n### {n}\n" + inspect.getsource(obj))
    except Exception as e:
        print(f"### {n} (no source: {e})")
print("\n### PATTERNS")
for pn in ("CITE_RE", "NUM_RE", "NUMERIC", "NEG_RE", "DATE_CELL"):
    v = globals().get(pn)
    print(f"  {pn} = {v.pattern!r}" if v is not None else f"  {pn} MISSING")



### _strip_cite
def _strip_cite(s):
    return CITE_RE.sub("", s)


### _particle
def _particle(word):
    last = word.strip()[-1]
    if not ("가" <= last <= "힣"):
        return "은"
    return "은" if (ord(last) - 0xAC00) % 28 else "는"


### ground_answer
def ground_answer(answer, docs, question=""):
    """답변의 숫자·날짜·개체·인용을 근거와 대조해 grounding 점수를 계산."""
    evidence = _norm("\n".join(d.metadata.get("raw", d.page_content) for d in docs))
    body = _strip_cite(answer)

    nums = [n for n in NUM_RE.findall(body) if len(n.replace(",", "")) >= 1]
    dates = DATE_RE.findall(body)
    ents = [e for e in ENT_RE.findall(body) if len(e) >= 3]

    num_ok = sum(_norm(n) in evidence for n in nums) / len(nums) if nums else 1.0
    date_ok = sum(_norm(d) in evidence for d in dates) / len(dates) if dates else 1.0
    ent_ok = sum(_norm(e) in evidence for e in ents) / len(ents) if ents else 1.0

    cites = CITE_RE.findall(answer)
    valid = 0
    for fname, page in cites:
        key = fname.stri

In [206]:
NAMES_D = ["missing_list_items", "missing_conditions", "missing_neighbor_details", "ask_v21"]
for n in NAMES_D:
    obj = globals().get(n)
    if obj is None:
        print(f"### MISSING {n}")
        continue
    try:
        print(f"\n### {n}\n" + inspect.getsource(obj))
    except Exception as e:
        print(f"### {n} (no source: {e})")



### missing_list_items
def missing_list_items(question, answer, docs, max_items=4):
    """질문이 목록을 요구할 때, 근거의 쉼표 항목이 답변에 그대로 있는지 확인."""
    if not LIST_INTENT.search(question):
        return []
    kws = [t for t in search_keywords(question).split() if len(t) >= 2][:4]
    missing = []
    for d in docs[:3]:
        for sent in re.split(r"[.\n]", d.metadata.get("raw", "")):
            if ("," not in sent and "·" not in sent) or len(sent) < 15:
                continue
            if kws and not any(t in sent for t in kws):
                continue
            items = [x.strip(" ①②③④⑤⑥⑦⑧⑨⑩") for x in re.split(r"[,·]", sent)]
            items = [x for x in items if 2 <= len(x) <= 14]
            if len(items) < 3:
                continue
            for it in items:
                if it not in answer and it not in missing:
                    missing.append(it)
            if missing:
                return missing[:max_items]
    return []


### missing_conditions
def missing_cond

In [207]:
print("### DRAFT_PROMPT_V5\n" + DRAFT_PROMPT_V5)
print("\n\n### REPAIR_PROMPT_V2\n" + REPAIR_PROMPT_V2)
print("\n\n### CONSTANTS")
for n in ("TOC_PAGES", "DOC_TITLES", "DOC_KEYWORDS", "ROUTE_PATTERNS", "ROUTE_SUFFIX", "SYNONYMS",
          "GENERIC", "GENERIC_COLS", "GENERIC_DETAILS", "DETAIL_STOP", "ALLOWED", "NOT_FOUND", "SYMBOLS"):
    print(f"\n{n} = {globals().get(n)!r}")


### DRAFT_PROMPT_V5
You are an admissions counselor at Gachon University. Answer using ONLY the evidence documents.

Procedure:
1. Scan EVERY evidence passage for sentences containing the key terms of the question, including passages near the end.
2. Collect all facts that answer any part of the question.
3. Write the answer from those facts only.

Rules:
- Include EVERY key fact the question asks for (numbers, dates, targets, conditions, exceptions), and also closely related details stated in the same evidence sentence (e.g., venue, refund policy, issuing authority).
- If the evidence lists several conditions or exceptions, list ALL of them, including additional requirements stated in the same passage.
- Copy the exact wording of list items (e.g., "국내외 연수", "주거 안정") instead of shortening them.
- For tables, align the row label and column headers by position. Never swap columns.
- For numbered columns (1호, 2호, ...), count every cell from the left, including dashes; match each value to 

In [208]:
for n in ("expand_header", "_norm", "lexical_coverage", "route_query", "evaluate_meta", "eval_meta"):
    obj = globals().get(n)
    print(f"\n### {n}")
    try:
        print(inspect.getsource(obj))
    except Exception as e:
        print("  (no source:", e, ")")
for pn in ("COND_MARK", "LIST_INTENT", "DATE_RE", "ENT_RE"):
    v = globals().get(pn)
    print(f"{pn} = {v.pattern!r}" if v is not None else f"{pn} MISSING")



### expand_header
def expand_header(header_rows, width):
    rows = []
    for r in header_rows:
        r = list(r) + [""] * (width - len(r))
        last_nonempty = max((j for j, v in enumerate(r) if v), default=-1)
        for j in range(last_nonempty):
            if not r[j] and j > 0 and r[j - 1]:
                r[j] = r[j - 1]
        rows.append(r)
    cols = []
    for j in range(width):
        parts = []
        for r in rows:
            v = r[j]
            if v and (not parts or parts[-1] != v):
                parts.append(v)
        cols.append(" ".join(parts))
    return cols


### _norm
def _norm(s):
    return re.sub(r"\s+", "", s)


### lexical_coverage
def lexical_coverage(kw, docs):
    toks = [t for t in kw.split() if len(t) > 1]
    if not toks:
        return 0.0
    ev = _norm(" ".join(d.metadata.get("raw", "") for d in docs[:5]))
    return sum(_norm(t) in ev for t in toks) / len(toks)


### route_query
def route_query(q):
    if re.search(r"(면접.*(일정|일정표|날짜